In [ ]:
from __future__ import annotations
import ast
import gc
import importlib
import inspect
import json
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
import warnings
import zipfile
from contextlib import contextmanager
from html import escape
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
sp_sparse = sp
import seaborn as sns
from scipy.special import softmax
from scipy.stats import mannwhitneyu
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    auc,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    top_k_accuracy_score,
)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import ParameterGrid, ParameterSampler, StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, MaxAbsScaler, OneHotEncoder, StandardScaler, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
from IPython.display import display, HTML, Image as IPyImage, Markdown
warnings.resetwarnings()
warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")
warnings.filterwarnings("ignore", message=".*resume_download.*")
warnings.filterwarnings("ignore", message=".*Glyph.*missing from current font.*")
try:
    import torch
    _torch_available = True
except Exception:
    torch = None
    _torch_available = False
if _torch_available:
    import torch.nn as nn
    from torch.utils.data import DataLoader, Dataset
else:
    nn = None
    DataLoader = None
    Dataset = None
try:
    from datasets import Dataset as HFDataset
    HAVE_DATASETS = True
    DATASETS_IMPORT_ERROR = ""
except Exception as exc:
    HFDataset = None
    HAVE_DATASETS = False
    DATASETS_IMPORT_ERROR = repr(exc)
try:
    from transformers import (
        AutoModel,
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        EarlyStoppingCallback,
        Trainer,
        TrainingArguments,
    )
    HAVE_TRANSFORMERS = True
    TRANSFORMERS_IMPORT_ERROR = ""
except Exception as exc:
    AutoModel = None
    AutoModelForSequenceClassification = None
    AutoTokenizer = None
    DataCollatorWithPadding = None
    EarlyStoppingCallback = None
    Trainer = None
    TrainingArguments = None
    HAVE_TRANSFORMERS = False
    TRANSFORMERS_IMPORT_ERROR = repr(exc)
try:
    from sentence_transformers import SentenceTransformer
    HAVE_ST = True
    SENTENCE_TRANSFORMERS_IMPORT_ERROR = ""
except Exception as exc:
    SentenceTransformer = None
    HAVE_ST = False
    SENTENCE_TRANSFORMERS_IMPORT_ERROR = repr(exc)
try:
    from keybert import KeyBERT
    HAVE_KEYBERT = True
    KEYBERT_IMPORT_ERROR = ""
except Exception as exc:
    KeyBERT = None
    HAVE_KEYBERT = False
    KEYBERT_IMPORT_ERROR = repr(exc)
try:
    from bertopic import BERTopic
    HAVE_BERTOPIC = True
    BERTOPIC_IMPORT_ERROR = ""
except Exception as exc:
    BERTopic = None
    HAVE_BERTOPIC = False
    BERTOPIC_IMPORT_ERROR = repr(exc)
try:
    from umap import UMAP
    HAVE_UMAP = True
    UMAP_IMPORT_ERROR = ""
except Exception as exc_primary:
    try:
        from umap.umap_ import UMAP
        HAVE_UMAP = True
        UMAP_IMPORT_ERROR = ""
    except Exception as exc:
        UMAP = None
        HAVE_UMAP = False
        UMAP_IMPORT_ERROR = repr(exc)
try:
    from xgboost import XGBClassifier
    HAVE_XGBOOST = True
    XGBOOST_IMPORT_ERROR = ""
except Exception as exc:
    XGBClassifier = None
    HAVE_XGBOOST = False
    XGBOOST_IMPORT_ERROR = repr(exc)
try:
    from catboost import CatBoostClassifier
    HAVE_CATBOOST = True
    CATBOOST_IMPORT_ERROR = ""
except Exception as exc:
    CatBoostClassifier = None
    HAVE_CATBOOST = False
    CATBOOST_IMPORT_ERROR = repr(exc)
if _torch_available:
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    except Exception as exc:
        print(f"torch.use_deterministic_algorithms not applied: {exc}")
SEED: int = 42
np.random.seed(SEED)
random.seed(SEED)
def seed_everything(seed: int = 42) -> None:
    """Seed the standard RNGs and torch (if available) for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    if _torch_available:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 220)
REPORT_BEHAVIOUR_COLORS: Dict[str, str] = {
    "reflection": "#e7f0ff",
    "question": "#e9f7ef",
    "therapist_input": "#fff3d9",
    "therapist input": "#fff3d9",
    "other": "#f3e8ff",
    "change": "#e9f7ef",
    "neutral": "#e7f0ff",
    "sustain": "#fff3d9",
}
REPORT_MODEL_COLORS: Dict[str, str] = {
    "elastic-net logistic regression": "#e7f0ff",
    "elastic net logistic regression": "#e7f0ff",
    "elasticnet_logreg": "#e7f0ff",
    "logistic regression": "#e7f0ff",
    "xgboost": "#f3e8ff",
    "hist_gradient_boosting": "#fff3d9",
    "majority dummy baseline": "#f1f5f9",
    "majority-class floor": "#f1f5f9",
    "marginal majority class": "#f1f5f9",
    "smoothed markov-1 transition baseline": "#fff7ed",
    "smoothed markov-2 transition baseline": "#ffedd5",
    "markov-1 oracle": "#fff7ed",
    "markov-2 oracle": "#ffedd5",
    "hybrid elastic-net logistic baseline": "#e7f0ff",
    "hybrid gru with local-state fusion": "#e9f7ef",
    "catboost structured state model": "#f3e8ff",
    "xgboost structured state model": "#ede9fe",
    "histgradientboosting structured state model": "#fff3d9",
}
REPORT_FAMILY_COLORS: Dict[str, str] = {
    "majority_dummy": "#f1f5f9",
    "markov1": "#fff7ed",
    "markov2": "#ffedd5",
    "hybrid_logreg": "#e7f0ff",
    "structured_catboost": "#f3e8ff",
    "structured_xgboost": "#ede9fe",
    "structured_histgb": "#fff3d9",
}
REPORT_CONTEXT_COLORS: Dict[str, str] = {
    "0": "#f1f5f9",
    "1": "#e7f0ff",
    "3": "#e9f7ef",
    "5": "#fff3d9",
    "10": "#f3e8ff",
}
REPORT_METRIC_COLORS: Dict[str, str] = {
    "accuracy": "#e7f0ff",
    "f1_macro": "#e9f7ef",
    "macro_f1": "#e9f7ef",
    "balanced_accuracy": "#fff3d9",
}
REPORT_ROLE_COLORS: Dict[str, str] = {
    "all": "#f8fafc",
    "all_utterances": "#f8fafc",
    "all transcripts": "#f8fafc",
    "therapist": "#e7f0ff",
    "therapist_only": "#e7f0ff",
    "client": "#e9f7ef",
    "client_only": "#e9f7ef",
}
REPORT_QUALITY_COLORS: Dict[str, str] = {
    "high": "#e9f7ef",
    "low": "#fff3d9",
    "high_quality_utterances": "#e9f7ef",
    "low_quality_utterances": "#fff3d9",
    "high_quality_transcripts": "#e9f7ef",
    "low_quality_transcripts": "#fff3d9",
}
REPORT_SUBSET_COLORS: Dict[str, str] = {
    **REPORT_ROLE_COLORS,
    **REPORT_QUALITY_COLORS,
    "all_transcripts": "#f8fafc",
}
def _report_normalize_key(value: Any) -> str:
    return str(value).strip().lower().replace("_", " ")
def _report_lookup_color(value: Any, colors: Optional[Dict[str, str]], default: str = "#ffffff") -> str:
    if not colors:
        return default
    raw = str(value)
    candidates = [raw, raw.strip(), raw.lower(), raw.strip().lower(), _report_normalize_key(value)]
    for candidate in candidates:
        if candidate in colors:
            return colors[candidate]
    for key, color in colors.items():
        if _report_normalize_key(key) == _report_normalize_key(value):
            return color
    normalized_value = _report_normalize_key(value)
    for key, color in colors.items():
        normalized_key = _report_normalize_key(key)
        if normalized_key and (normalized_value.startswith(normalized_key) or normalized_key in normalized_value):
            return color
    return default
def display_report_table(
    df: pd.DataFrame,
    formats: Optional[Dict[str, str]] = None,
    text_cols: Optional[List[str]] = None,
    gradient_cols: Optional[List[str]] = None,
    gradient_kwargs: Optional[Dict[str, Any]] = None,
    row_color_col: Optional[str] = None,
    row_colors: Optional[Dict[str, str]] = None,
    highlight_max_cols: Optional[List[str]] = None,
    highlight_min_cols: Optional[List[str]] = None,
    signed_cols: Optional[List[str]] = None,
    bool_cols: Optional[List[str]] = None,
    hide_index: bool = True,
) -> None:
    """Display a compact report table with optional row, metric, signed, and boolean cues."""
    formats = {k: v for k, v in (formats or {}).items() if k in df.columns}
    text_cols = [c for c in (text_cols or []) if c in df.columns]
    gradient_cols = [c for c in (gradient_cols or []) if c in df.columns]
    highlight_max_cols = [c for c in (highlight_max_cols or []) if c in df.columns]
    highlight_min_cols = [c for c in (highlight_min_cols or []) if c in df.columns]
    signed_cols = [c for c in (signed_cols or []) if c in df.columns]
    bool_cols = [c for c in (bool_cols or []) if c in df.columns]
    gradient_kwargs = dict(gradient_kwargs or {})
    gradient_kwargs.setdefault("text_color_threshold", 0.2)
    row_colors = row_colors or {}
    def _row_style(row: pd.Series) -> List[str]:
        if not row_color_col or row_color_col not in row.index:
            return ["color: #111827" for _ in row]
        background = _report_lookup_color(row[row_color_col], row_colors)
        return [f"background-color: {background}; color: #111827; font-weight: 600" for _ in row]
    def _signed_style(value: Any) -> str:
        try:
            val = float(value)
        except Exception:
            return ""
        if pd.isna(val):
            return ""
        if val > 0:
            return "background-color: #d1fae5; color: #064e3b; font-weight: 800"
        if val < 0:
            return "background-color: #fee2e2; color: #7f1d1d; font-weight: 800"
        return "background-color: #f1f5f9; color: #111827; font-weight: 700"
    def _bool_style(value: Any) -> str:
        truthy = str(value).strip().lower() in {"true", "1", "yes", "available"}
        falsy = str(value).strip().lower() in {"false", "0", "no", "missing", "unavailable"}
        if truthy:
            return "background-color: #d1fae5; color: #064e3b; font-weight: 800"
        if falsy:
            return "background-color: #fee2e2; color: #7f1d1d; font-weight: 800"
        return ""
    def _highlight_extreme_style(values: pd.Series, *, pick: str) -> List[str]:
        numeric_values = pd.to_numeric(values, errors="coerce")
        if numeric_values.notna().any():
            target_value = numeric_values.max(skipna=True) if pick == "max" else numeric_values.min(skipna=True)
            target_mask = numeric_values.eq(target_value)
        else:
            target_value = values.max() if pick == "max" else values.min()
            target_mask = values.eq(target_value)
        return [
            "background-color: #bbf7d0; color: #064e3b; font-weight: 800" if is_target else ""
            for is_target in target_mask
        ]
    try:
        styler = df.style
        if hide_index:
            styler = styler.hide(axis="index")
        if formats:
            styler = styler.format(formats)
        if row_color_col:
            styler = styler.apply(_row_style, axis=1)
        if text_cols:
            styler = styler.set_properties(
                subset=text_cols,
                **{
                    "text-align": "left",
                    "max-width": "520px",
                    "white-space": "normal",
                    "overflow-wrap": "anywhere",
                    "word-break": "break-word",
                },
            )
        if gradient_cols:
            styler = styler.background_gradient(subset=gradient_cols, **gradient_kwargs)
        if highlight_max_cols:
            styler = styler.apply(lambda values: _highlight_extreme_style(values, pick="max"), subset=highlight_max_cols, axis=0)
        if highlight_min_cols:
            styler = styler.apply(lambda values: _highlight_extreme_style(values, pick="min"), subset=highlight_min_cols, axis=0)
        if signed_cols:
            style_map = styler.map if hasattr(styler, "map") else styler.applymap
            styler = style_map(_signed_style, subset=signed_cols)
        if bool_cols:
            style_map = styler.map if hasattr(styler, "map") else styler.applymap
            styler = style_map(_bool_style, subset=bool_cols)
        styler = styler.set_table_styles(
            [
                {"selector": "th", "props": [("text-align", "left"), ("background-color", "#f1f5f9"), ("color", "#111827"), ("font-weight", "800")]},
                {"selector": "td", "props": [("padding", "6px 8px"), ("border-bottom", "1px solid #e5e7eb"), ("vertical-align", "top")]},
            ]
        )
        display(styler)
    except Exception:
        fallback = df.copy()
        for col, fmt in formats.items():
            fallback[col] = fallback[col].map(lambda v: "" if pd.isna(v) else fmt.format(v))
        display(fallback.reset_index(drop=True) if hide_index else fallback)
def _project_score(path: Path) -> int:
    """Heuristic score for picking the project root if multiple candidates exist."""
    path = Path(path)
    score = 0
    if (path / "AnnoMI.zip").exists():
        score += 10
    if (path / "data").exists():
        score += 2
    if (path / "artifacts").exists():
        score += 2
    if any(path.glob("*annomi*.ipynb")):
        score += 1
    return score
def _resolve_project_dir() -> Path:
    """Pick the most plausible project root from CWD, an env override, /mnt/data."""
    candidates: List[Path] = []
    seen: set = set()
    for raw in (os.environ.get("ANNOMI_PROJECT_DIR"), Path.cwd(), Path("."), "/mnt/data"):
        if raw is None:
            continue
        try:
            p = Path(raw).expanduser().resolve()
        except Exception:
            continue
        if p.exists() and p not in seen:
            seen.add(p)
            candidates.append(p)
    if not candidates:
        return Path.cwd().resolve()
    return sorted(candidates, key=_project_score, reverse=True)[0]
PROJECT_DIR: Path = _resolve_project_dir()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
FIG_DIR = PROJECT_DIR / "figures"
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
MODEL_DIR = ARTIFACT_DIR / "models"
CACHE_DIR = ARTIFACT_DIR / "cache"
RESULT_DIR = ARTIFACT_DIR / "results"
for _dir in (DATA_DIR, RAW_DIR, PROCESSED_DIR, FIG_DIR,
             ARTIFACT_DIR, MODEL_DIR, CACHE_DIR, RESULT_DIR):
    _dir.mkdir(parents=True, exist_ok=True)
RAW_ZIP_CANDIDATES = [PROJECT_DIR / "AnnoMI.zip", Path("/mnt/data/AnnoMI.zip")]
RAW_ZIP = next((p for p in RAW_ZIP_CANDIDATES if p.exists()), PROJECT_DIR / "AnnoMI.zip")
EXTRACTED_DIR = RAW_DIR / "AnnoMI"
RAW_CSV = EXTRACTED_DIR / "dataset.csv"
CONFIG_JSON = ARTIFACT_DIR / "run_config_v3_local.json"
MANIFEST_JSON = ARTIFACT_DIR / "run_manifest_v3_local.json"
CONFIG: Dict[str, Any] = {
    "seed": SEED,
    "execution_profile": "local",
    "force_recompute": False,
    "force_retrain": False,
    "official_n_splits": 5,
    "official_test_fold_selection": "most_balanced",
    "context_grid": [0, 1, 3, 5, 10],
    "max_context_k": 10,
    "embedding_model_name": "sentence-transformers/all-MiniLM-L6-v2",
    "run_semantic_embeddings": True,
    "run_bertopic": False,
    "task_a_enable": True,
    "task_a_load_from_artifacts": False,
    "task_a_prefer_xgboost": True,
    "task_b_enable": True,
    "task_b_logreg_load_from_artifacts": False,
    "task_b_encoder_backbones_override": ["FacebookAI/roberta-base"],
    "task_b_cloud_optional_backbone": None,
    "task_b_run_tapt": False,
    "task_b_epochs": 5,
    "task_b_max_length": 256,
    "task_b_learning_rate": 2e-5,
    "task_b_weight_decay": 0.01,
    "task_b_train_batch_size": 2,
    "task_b_eval_batch_size": 4,
    "task_b_grad_accum": 8,
    "task_b_early_stopping_patience": 2,
    "task_b_context_grid_neural": [3, 5, 10],
    "task_b_max_length_grid_neural": [256, 384],
    "task_b_roberta_lr_grid": [1e-5, 2e-5, 3e-5],
    "task_b_weight_decay_grid_neural": [0.01],
    "task_b_multiseed_seeds": [17, 42, 101],
    "task_c_enable": True,
    "task_c_history_k": 5,
    "task_c_prefer_xgboost": True,
    "task_c_turn_encoder": "FacebookAI/roberta-base",
    "task_c_seq_epochs": 12,
    "task_c_seq_batch_size": 32,
    "run_research_track": False,
}
with open(CONFIG_JSON, "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)
def save_json(obj: Any, path: Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
def load_json(path: Path) -> Dict[str, Any]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
def save_df(df: pd.DataFrame, path: Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
def load_df(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)
def artifact_exists(path: Path) -> bool:
    return Path(path).exists()
def save_joblib(obj: Any, path: Path) -> None:
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(obj, path)
def load_joblib(path: Path) -> Any:
    return joblib.load(path)
runtime_info: Dict[str, Any] = {
    "python": sys.version.split(" ")[0],
    "platform": platform.platform(),
    "project_dir": str(PROJECT_DIR),
    "raw_zip_found": RAW_ZIP.exists(),
    "raw_zip_path": str(RAW_ZIP),
}
try:
    import sklearn
    runtime_info["sklearn"] = sklearn.__version__
except Exception:
    runtime_info["sklearn"] = None
if _torch_available:
    runtime_info["torch"] = torch.__version__
    runtime_info["cuda_available"] = bool(torch.cuda.is_available())
    runtime_info["cuda_device"] = (
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    )
else:
    runtime_info["torch"] = None
    runtime_info["cuda_available"] = None
    runtime_info["cuda_device"] = None
display(Markdown("### Table 0.0. Runtime environment summary"))
display_report_table(
    pd.DataFrame([runtime_info]),
    bool_cols=["raw_zip_found", "cuda_available"],
    text_cols=["platform", "project_dir", "raw_zip_path", "cuda_device"],
)
print("PROJECT_DIR:", PROJECT_DIR)
print("RAW_ZIP:", RAW_ZIP)
print("Artifacts will be stored in:", ARTIFACT_DIR)


In [ ]:
CONFIG.update(
    {
        "run_bertopic": True,
        "task_b_encoder_backbones_override": ["FacebookAI/roberta-base"],
        "task_b_run_tapt": False,
        "run_tapt": False,
        "task_b_context_grid_neural": [3, 5, 10],
        "task_b_max_length_grid_neural": [256, 384],
        "task_b_roberta_lr_grid": [1e-5, 2e-5, 3e-5],
        "task_b_weight_decay_grid_neural": [0.01],
        "task_b_multiseed_seeds": [17, 42, 101],
    }
)
save_json(CONFIG, CONFIG_JSON)
roberta_runtime_overrides_df = pd.DataFrame(
    [
        {
            "run_bertopic": CONFIG["run_bertopic"],
            "task_b_encoder_backbones_override": ", ".join(CONFIG["task_b_encoder_backbones_override"]),
            "task_b_run_tapt": CONFIG["task_b_run_tapt"],
            "run_tapt": CONFIG["run_tapt"],
            "task_b_context_grid_neural": str(CONFIG["task_b_context_grid_neural"]),
            "task_b_max_length_grid_neural": str(CONFIG["task_b_max_length_grid_neural"]),
            "task_b_roberta_lr_grid": str(CONFIG["task_b_roberta_lr_grid"]),
        }
    ]
)
display(Markdown("### Table 0.1. Runtime override configuration"))
display_report_table(
    roberta_runtime_overrides_df,
    bool_cols=["run_bertopic", "task_b_run_tapt", "run_tapt"],
    text_cols=[
        "task_b_encoder_backbones_override",
        "task_b_context_grid_neural",
        "task_b_max_length_grid_neural",
        "task_b_roberta_lr_grid",
    ],
)


In [ ]:
if not RAW_CSV.exists():
    if not RAW_ZIP.exists():
        raise FileNotFoundError(
            "AnnoMI.zip was not found in the notebook working directory. "
            "Place the zip next to the notebook or update RAW_ZIP."
        )
    if EXTRACTED_DIR.exists():
        shutil.rmtree(EXTRACTED_DIR)
    with zipfile.ZipFile(RAW_ZIP, "r") as zf:
        zf.extractall(RAW_DIR)

if not RAW_CSV.exists():
    raise FileNotFoundError("dataset.csv could not be located after unzip.")

raw_df = pd.read_csv(RAW_CSV)

def _report_path(path_value: Path) -> str:
    path = Path(path_value)
    try:
        return str(path.resolve().relative_to(PROJECT_DIR)).replace("\\", "/")
    except Exception:
        return str(path)

def _file_size_mb(path_value: Path) -> float:
    path = Path(path_value)
    return float(path.stat().st_size / (1024 * 1024)) if path.exists() else np.nan

def _clip_table_text(value: Any, width: int = 90) -> str:
    text = "" if pd.isna(value) else " ".join(str(value).split())
    return text if len(text) <= width else text[: width - 1].rstrip() + "..."

raw_source_check_df = pd.DataFrame(
    [
        {
            "source_item": "AnnoMI.zip",
            "path": _report_path(RAW_ZIP),
            "available": RAW_ZIP.exists(),
            "size_mb": _file_size_mb(RAW_ZIP),
            "purpose": "Original compressed dataset supplied with the project",
        },
        {
            "source_item": "dataset.csv",
            "path": _report_path(RAW_CSV),
            "available": RAW_CSV.exists(),
            "size_mb": _file_size_mb(RAW_CSV),
            "purpose": "Raw utterance-level AnnoMI table used for all analysis views",
        },
    ]
)

raw_dataset_overview_df = pd.DataFrame(
    [
        {"metric": "Rows", "value": raw_df.shape[0], "interpretation": "Utterance-level records loaded"},
        {"metric": "Columns", "value": raw_df.shape[1], "interpretation": "Raw fields before feature engineering"},
        {"metric": "Transcripts", "value": raw_df["transcript_id"].nunique(), "interpretation": "Dialogue sessions represented"},
        {"metric": "Therapist turns", "value": int((raw_df["interlocutor"] == "therapist").sum()), "interpretation": "Rows eligible for therapist-behaviour labels"},
        {"metric": "Client turns", "value": int((raw_df["interlocutor"] == "client").sum()), "interpretation": "Rows eligible for client-talk labels"},
        {"metric": "Topic labels", "value": raw_df["topic"].nunique(dropna=True), "interpretation": "Distinct raw topic strings"},
        {"metric": "MI quality classes", "value": raw_df["mi_quality"].nunique(dropna=True), "interpretation": "Transcript quality categories"},
        {"metric": "Missing therapist labels", "value": int(raw_df["main_therapist_behaviour"].isna().sum()), "interpretation": "Expected mostly on client turns"},
        {"metric": "Missing client labels", "value": int(raw_df["client_talk_type"].isna().sum()), "interpretation": "Expected mostly on therapist turns"},
    ]
)

raw_preview_df = (
    raw_df.head(5)
    .loc[:, [
        "transcript_id",
        "mi_quality",
        "utterance_id",
        "interlocutor",
        "timestamp",
        "main_therapist_behaviour",
        "client_talk_type",
        "utterance_text",
    ]]
    .assign(utterance_text=lambda d: d["utterance_text"].map(lambda x: _clip_table_text(x, 110)))
    .fillna("n/a")
)

display(Markdown("### Table 1.1A. Raw source availability check"))
display_report_table(
    raw_source_check_df,
    formats={"size_mb": "{:.3f}"},
    row_color_col="source_item",
    row_colors={"AnnoMI.zip": "#e7f0ff", "dataset.csv": "#e9f7ef"},
    text_cols=["source_item", "path", "purpose"],
    bool_cols=["available"],
    gradient_cols=["size_mb"],
    gradient_kwargs={"cmap": "YlGnBu"},
)

display(Markdown("### Table 1.1B. Raw AnnoMI dataset audit"))
display_report_table(
    raw_dataset_overview_df,
    formats={"value": "{:,}"},
    row_color_col="metric",
    row_colors={
        "Rows": "#e7f0ff",
        "Columns": "#f8fafc",
        "Transcripts": "#e9f7ef",
        "Therapist turns": "#fff3d9",
        "Client turns": "#f3e8ff",
        "Topic labels": "#ecfeff",
        "MI quality classes": "#fce8ef",
        "Missing therapist labels": "#fff7ed",
        "Missing client labels": "#fff7ed",
    },
    text_cols=["metric", "interpretation"],
    gradient_cols=["value"],
    gradient_kwargs={"cmap": "YlGnBu"},
)

display(Markdown("### Table 1.1C. Raw AnnoMI preview after loading"))
display_report_table(
    raw_preview_df,
    formats={"transcript_id": "{:,}", "utterance_id": "{:,}"},
    row_color_col="interlocutor",
    row_colors=REPORT_ROLE_COLORS,
    text_cols=["mi_quality", "interlocutor", "timestamp", "main_therapist_behaviour", "client_talk_type", "utterance_text"],
)

save_df(raw_source_check_df, PROCESSED_DIR / "raw_source_check.csv")
save_df(raw_dataset_overview_df, PROCESSED_DIR / "raw_dataset_overview.csv")


In [ ]:
TOKEN_RE = re.compile(r"\b\w+(?:'\w+)?\b")

if "seed_everything" not in globals():
    def seed_everything(seed: int = 42) -> None:
        """Fallback seeding helper when this cell is run without the setup cell."""
        random.seed(seed)
        np.random.seed(seed)
        if globals().get("torch") is not None:
            try:
                torch.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
                torch.backends.cudnn.deterministic = True
                torch.backends.cudnn.benchmark = False
            except Exception:
                pass

def timestamp_to_seconds(x: Any) -> float:
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if not s:
        return np.nan
    parts = s.split(":")
    try:
        if len(parts) == 3:
            h, m, sec = parts
            return int(h) * 3600 + int(m) * 60 + float(sec)
        if len(parts) == 2:
            m, sec = parts
            return int(m) * 60 + float(sec)
        return float(s)
    except Exception:
        return np.nan

def simple_tokenize(text: str) -> List[str]:
    if text is None:
        return []
    return TOKEN_RE.findall(str(text).lower())

def lexical_overlap(text_a: str, text_b: str) -> float:
    a = set(simple_tokenize(text_a))
    b = set(simple_tokenize(text_b))
    if not a and not b:
        return 0.0
    return len(a & b) / max(len(a | b), 1)

def normalize_topic(x: Any) -> str:
    if pd.isna(x):
        return "unknown"
    return str(x).strip().lower()

seed_everything(SEED)

utterance_df = raw_df.copy()
utterance_df["main_therapist_behaviour"] = utterance_df["main_therapist_behaviour"].fillna("n/a")
utterance_df["client_talk_type"] = utterance_df["client_talk_type"].fillna("n/a")
utterance_df["topic_norm"] = utterance_df["topic"].apply(normalize_topic)

utterance_df["utterance_text"] = (
    utterance_df["utterance_text"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

utterance_df["utterance_text_lc"] = utterance_df["utterance_text"].str.lower()
utterance_df["timestamp_seconds"] = utterance_df["timestamp"].apply(timestamp_to_seconds)

utterance_df = utterance_df.sort_values(["transcript_id", "utterance_id"]).reset_index(drop=True)

utterance_df["word_count"] = utterance_df["utterance_text"].map(lambda x: len(simple_tokenize(x)))
utterance_df["char_count"] = utterance_df["utterance_text"].str.len()
utterance_df["question_count"] = utterance_df["utterance_text"].str.count(r"\?")
utterance_df["exclamation_count"] = utterance_df["utterance_text"].str.count(r"!")
utterance_df["comma_count"] = utterance_df["utterance_text"].str.count(r",")
utterance_df["period_count"] = utterance_df["utterance_text"].str.count(r"\.")
utterance_df["is_question_mark"] = (utterance_df["question_count"] > 0).astype(int)
utterance_df["is_short_utterance"] = (utterance_df["word_count"] <= 3).astype(int)

token_helper_df = pd.DataFrame(
    [
        {
            "helper": "TOKEN_RE",
            "purpose": "Keeps word tokens and simple apostrophe contractions",
            "example_check": TOKEN_RE.pattern,
            "downstream_use": "Shared token boundary for length and lexical-overlap features",
        },
        {
            "helper": "timestamp_to_seconds",
            "purpose": "Converts timestamp strings into numeric seconds",
            "example_check": f"00:01:08 -> {timestamp_to_seconds('00:01:08'):.0f}",
            "downstream_use": "Timing and transcript-position diagnostics",
        },
        {
            "helper": "simple_tokenize",
            "purpose": "Lowercases text and returns regex tokens",
            "example_check": ", ".join(simple_tokenize("It's okay?")),
            "downstream_use": "Word counts and sparse lexical features",
        },
        {
            "helper": "lexical_overlap",
            "purpose": "Measures overlap with the previous partner turn",
            "example_check": f"hello there vs hello = {lexical_overlap('hello there', 'hello'):.2f}",
            "downstream_use": "Dialogue-aware relational feature",
        },
        {
            "helper": "normalize_topic",
            "purpose": "Standardises topic strings conservatively",
            "example_check": normalize_topic(" Smoking Cessation "),
            "downstream_use": "Topic grouping and error-slice analysis",
        },
        {
            "helper": "seed_everything",
            "purpose": "Applies deterministic seeds where libraries support it",
            "example_check": f"SEED = {SEED}",
            "downstream_use": "Reproducible splits, baselines, and neural runs",
        },
    ]
)

utterance_feature_audit_df = pd.DataFrame(
    [
        {
            "feature_group": "Text cleanup",
            "columns_created": "utterance_text, utterance_text_lc",
            "quality_check": "empty cleaned texts",
            "value": int((utterance_df["utterance_text"].str.len() == 0).sum()),
            "use_in_notebook": "Safe original/lowercase text views",
        },
        {
            "feature_group": "Timing",
            "columns_created": "timestamp_seconds",
            "quality_check": "missing parsed timestamps",
            "value": int(utterance_df["timestamp_seconds"].isna().sum()),
            "use_in_notebook": "Temporal ordering and diagnostics",
        },
        {
            "feature_group": "Length",
            "columns_created": "word_count, char_count",
            "quality_check": "median words per utterance",
            "value": float(utterance_df["word_count"].median()),
            "use_in_notebook": "EDA, baseline features, error slices",
        },
        {
            "feature_group": "Punctuation",
            "columns_created": "question_count, exclamation_count, comma_count, period_count",
            "quality_check": "total question marks",
            "value": int(utterance_df["question_count"].sum()),
            "use_in_notebook": "Behaviour cues for questions and style",
        },
        {
            "feature_group": "Binary flags",
            "columns_created": "is_question_mark, is_short_utterance",
            "quality_check": "short-utterance rate",
            "value": float(utterance_df["is_short_utterance"].mean()),
            "use_in_notebook": "Compact interpretable model features",
        },
        {
            "feature_group": "Topic normalisation",
            "columns_created": "topic_norm",
            "quality_check": "normalised topic labels",
            "value": int(utterance_df["topic_norm"].nunique()),
            "use_in_notebook": "Topic summaries and robustness slices",
        },
    ]
)

cleaned_utterance_preview_df = (
    utterance_df.head(6)
    .loc[:, [
        "transcript_id",
        "utterance_id",
        "interlocutor",
        "timestamp_seconds",
        "word_count",
        "question_count",
        "main_therapist_behaviour",
        "client_talk_type",
        "utterance_text",
    ]]
    .assign(utterance_text=lambda d: d["utterance_text"].map(lambda x: _clip_table_text(x, 105)))
)

display(Markdown("### Table 1.2A. Tokenisation and cleaning helper audit"))
display_report_table(
    token_helper_df,
    row_color_col="helper",
    row_colors={
        "TOKEN_RE": "#e7f0ff",
        "timestamp_to_seconds": "#e9f7ef",
        "simple_tokenize": "#fff3d9",
        "lexical_overlap": "#f3e8ff",
        "normalize_topic": "#ecfeff",
        "seed_everything": "#fce8ef",
    },
    text_cols=["helper", "purpose", "example_check", "downstream_use"],
)

display(Markdown("### Table 1.2B. Initial engineered-feature checks"))
display_report_table(
    utterance_feature_audit_df,
    formats={"value": "{:.3f}"},
    row_color_col="feature_group",
    row_colors={
        "Text cleanup": "#e7f0ff",
        "Timing": "#e9f7ef",
        "Length": "#fff3d9",
        "Punctuation": "#f3e8ff",
        "Binary flags": "#ecfeff",
        "Topic normalisation": "#fce8ef",
    },
    text_cols=["feature_group", "columns_created", "quality_check", "use_in_notebook"],
    gradient_cols=["value"],
    gradient_kwargs={"cmap": "YlGnBu"},
)

display(Markdown("### Table 1.2C. Cleaned utterance preview after tokenisation features"))
display_report_table(
    cleaned_utterance_preview_df,
    formats={
        "transcript_id": "{:,}",
        "utterance_id": "{:,}",
        "timestamp_seconds": "{:.1f}",
        "word_count": "{:,}",
        "question_count": "{:,}",
    },
    row_color_col="interlocutor",
    row_colors=REPORT_ROLE_COLORS,
    text_cols=["interlocutor", "main_therapist_behaviour", "client_talk_type", "utterance_text"],
)

save_df(token_helper_df, PROCESSED_DIR / "tokenisation_helper_audit.csv")
save_df(utterance_feature_audit_df, PROCESSED_DIR / "initial_engineered_feature_checks.csv")


In [ ]:
def build_dialogue_context_features(df: pd.DataFrame, max_context_k: int = 10) -> pd.DataFrame:
    out = []
    transcript_groups = df.groupby("transcript_id", sort=False)
    for transcript_id, g in tqdm(transcript_groups, total=df["transcript_id"].nunique(), desc="Building context"):
        g = g.sort_values("utterance_id").reset_index(drop=True).copy()
        tagged_turns: List[str] = []
        last_client_text = ""
        last_therapist_text = ""
        last_speaker = ""
        prev_texts = []
        prev_speakers = []
        prev_client_texts = []
        prev_therapist_texts = []
        lexical_overlaps = []
        for row in g[["interlocutor", "utterance_text"]].itertuples(index=False):
            interlocutor = row.interlocutor
            utterance_text = row.utterance_text
            prev_text = tagged_turns[-1] if tagged_turns else ""
            prev_speaker = last_speaker
            prev_texts.append(prev_text)
            prev_speakers.append(prev_speaker)
            prev_client_texts.append(last_client_text)
            prev_therapist_texts.append(last_therapist_text)
            if interlocutor == "therapist":
                lexical_overlaps.append(lexical_overlap(utterance_text, last_client_text))
            else:
                lexical_overlaps.append(lexical_overlap(utterance_text, last_therapist_text))
            tagged_turn = f"[{str(interlocutor).upper()}] {utterance_text}"
            tagged_turns.append(tagged_turn)
            if interlocutor == "client":
                last_client_text = utterance_text
            elif interlocutor == "therapist":
                last_therapist_text = utterance_text
            last_speaker = interlocutor
        g["prev_text"] = prev_texts
        g["prev_interlocutor"] = prev_speakers
        g["prev_client_text"] = prev_client_texts
        g["prev_therapist_text"] = prev_therapist_texts
        g["lexical_overlap_prev_partner"] = lexical_overlaps
        g["turn_index"] = np.arange(len(g))
        g["n_turns_in_transcript"] = len(g)
        g["turn_ratio"] = g["turn_index"] / max(len(g) - 1, 1)
        histories: List[List[str]] = []
        running_history: List[str] = []
        for tagged_turn in tagged_turns:
            histories.append(running_history[-max_context_k:].copy())
            running_history.append(tagged_turn)
        g["history_tagged_turns"] = histories
        out.append(g)
    return pd.concat(out, ignore_index=True)
if "_clip_table_text" not in globals():
    def _clip_table_text(value: Any, width: int = 90) -> str:
        text = "" if pd.isna(value) else " ".join(str(value).split())
        return text if len(text) <= width else text[: width - 1].rstrip() + "..."
utterance_df = build_dialogue_context_features(utterance_df, max_context_k=CONFIG["max_context_k"])
utterance_df["history_turn_count"] = utterance_df["history_tagged_turns"].map(len)
n_transcripts_context = int(utterance_df["transcript_id"].nunique())
first_turns_without_prev = int(utterance_df["prev_interlocutor"].eq("").sum())
therapist_with_prior_client = int(
    ((utterance_df["interlocutor"] == "therapist") & utterance_df["prev_client_text"].ne("")).sum()
)
client_with_prior_therapist = int(
    ((utterance_df["interlocutor"] == "client") & utterance_df["prev_therapist_text"].ne("")).sum()
)
context_feature_audit_df = pd.DataFrame(
    [
        {
            "feature_group": "Previous turn pointer",
            "columns_created": "prev_text, prev_interlocutor",
            "quality_check": "first turns without previous speaker",
            "value": f"{first_turns_without_prev:,} rows (= {n_transcripts_context:,} transcripts)",
            "downstream_use": "Preserves immediate dialogue order",
        },
        {
            "feature_group": "Previous partner memory",
            "columns_created": "prev_client_text, prev_therapist_text",
            "quality_check": "therapist rows with prior client text",
            "value": f"{therapist_with_prior_client:,}",
            "downstream_use": "Supports causal therapist-action classification",
        },
        {
            "feature_group": "Previous partner memory",
            "columns_created": "prev_client_text, prev_therapist_text",
            "quality_check": "client rows with prior therapist text",
            "value": f"{client_with_prior_therapist:,}",
            "downstream_use": "Supports client-talk and transition analysis",
        },
        {
            "feature_group": "Lexical relation",
            "columns_created": "lexical_overlap_prev_partner",
            "quality_check": "mean previous-partner overlap",
            "value": f"{utterance_df['lexical_overlap_prev_partner'].mean():.3f}",
            "downstream_use": "Interpretable local response-similarity feature",
        },
        {
            "feature_group": "Transcript position",
            "columns_created": "turn_index, n_turns_in_transcript, turn_ratio",
            "quality_check": "largest transcript length",
            "value": f"{int(utterance_df['n_turns_in_transcript'].max()):,} turns",
            "downstream_use": "Controls for early/middle/late dialogue position",
        },
        {
            "feature_group": "Causal history window",
            "columns_created": "history_tagged_turns, history_turn_count",
            "quality_check": "maximum retained prior turns",
            "value": f"{int(utterance_df['history_turn_count'].max()):,} turns",
            "downstream_use": "Feeds context_k text construction without future leakage",
        },
    ]
)
context_depth_summary_df = (
    utterance_df["history_turn_count"]
    .value_counts()
    .rename_axis("prior_turns_available")
    .reset_index(name="utterance_rows")
    .sort_values("prior_turns_available")
)
context_depth_summary_df["share_of_rows"] = context_depth_summary_df["utterance_rows"] / len(utterance_df)
context_depth_summary_df = context_depth_summary_df.tail(8).reset_index(drop=True)
context_preview_df = (
    utterance_df.head(8)
    .loc[:, [
        "transcript_id",
        "utterance_id",
        "interlocutor",
        "prev_interlocutor",
        "history_turn_count",
        "lexical_overlap_prev_partner",
        "turn_ratio",
        "prev_text",
        "utterance_text",
    ]]
    .assign(
        prev_interlocutor=lambda d: d["prev_interlocutor"].replace("", "[start]"),
        prev_text=lambda d: d["prev_text"].replace("", "[start of transcript]").map(lambda x: _clip_table_text(x, 95)),
        utterance_text=lambda d: d["utterance_text"].map(lambda x: _clip_table_text(x, 105)),
    )
)
display(Markdown("### Table 1.3A. Dialogue-aware context feature audit"))
display_report_table(
    context_feature_audit_df,
    row_color_col="feature_group",
    row_colors={
        "Previous turn pointer": "#e7f0ff",
        "Previous partner memory": "#e9f7ef",
        "Lexical relation": "#fff3d9",
        "Transcript position": "#f3e8ff",
        "Causal history window": "#ecfeff",
    },
    text_cols=["feature_group", "columns_created", "quality_check", "value", "downstream_use"],
)
display(Markdown("### Table 1.3B. Context-window depth distribution"))
display_report_table(
    context_depth_summary_df,
    formats={"prior_turns_available": "{:,}", "utterance_rows": "{:,}", "share_of_rows": "{:.1%}"},
    text_cols=["prior_turns_available"],
    gradient_cols=["utterance_rows", "share_of_rows"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
display(Markdown("### Table 1.3C. Dialogue-context preview after feature construction"))
display_report_table(
    context_preview_df,
    formats={
        "transcript_id": "{:,}",
        "utterance_id": "{:,}",
        "history_turn_count": "{:,}",
        "lexical_overlap_prev_partner": "{:.3f}",
        "turn_ratio": "{:.3f}",
    },
    row_color_col="interlocutor",
    row_colors=REPORT_ROLE_COLORS,
    text_cols=["interlocutor", "prev_interlocutor", "prev_text", "utterance_text"],
)
save_df(context_feature_audit_df, PROCESSED_DIR / "dialogue_context_feature_audit.csv")
save_df(context_depth_summary_df, PROCESSED_DIR / "dialogue_context_depth_distribution.csv")
save_df(context_preview_df, PROCESSED_DIR / "dialogue_context_preview.csv")


In [ ]:
therapist_df = utterance_df[
    (utterance_df["interlocutor"] == "therapist")
    & (utterance_df["main_therapist_behaviour"] != "n/a")
].copy()
client_df = utterance_df[
    (utterance_df["interlocutor"] == "client")
    & (utterance_df["client_talk_type"] != "n/a")
].copy()
def build_context_text(history_turns: List[str], current_text: str, speaker_tag: str, k: int) -> str:
    history = history_turns[-k:] if k > 0 else []
    current = f"[{speaker_tag}] {current_text}"
    return " ".join(history + [current]).strip()
for k in CONFIG["context_grid"]:
    therapist_df[f"context_k{k}_text"] = [
        build_context_text(hist, txt, "THERAPIST", k)
        for hist, txt in zip(therapist_df["history_tagged_turns"], therapist_df["utterance_text"])
    ]
    therapist_df[f"context_k{k}_lc"] = therapist_df[f"context_k{k}_text"].str.lower()
    client_df[f"context_k{k}_text"] = [
        build_context_text(hist, txt, "CLIENT", k)
        for hist, txt in zip(client_df["history_tagged_turns"], client_df["utterance_text"])
    ]
    client_df[f"context_k{k}_lc"] = client_df[f"context_k{k}_text"].str.lower()
therapist_df["prev_partner_word_count"] = therapist_df["prev_client_text"].map(lambda x: len(simple_tokenize(x)))
client_df["prev_partner_word_count"] = client_df["prev_therapist_text"].map(lambda x: len(simple_tokenize(x)))
utterance_df_to_save = utterance_df.drop(columns=["history_tagged_turns"]).copy()
therapist_df_to_save = therapist_df.drop(columns=["history_tagged_turns"]).copy()
client_df_to_save = client_df.drop(columns=["history_tagged_turns"]).copy()
utterance_df_to_save.to_csv(PROCESSED_DIR / "utterance_df.csv", index=False)
therapist_df_to_save.to_csv(PROCESSED_DIR / "therapist_df.csv", index=False)
client_df_to_save.to_csv(PROCESSED_DIR / "client_df.csv", index=False)
split_view_shape_df = pd.DataFrame(
    [
        {
            "view": "Full utterance table",
            "rows": utterance_df.shape[0],
            "columns": utterance_df.shape[1],
            "downstream_use": "All turns with engineered context features",
        },
        {
            "view": "Therapist modelling view",
            "rows": therapist_df.shape[0],
            "columns": therapist_df.shape[1],
            "downstream_use": "Task B therapist-behaviour classification",
        },
        {
            "view": "Client modelling view",
            "rows": client_df.shape[0],
            "columns": client_df.shape[1],
            "downstream_use": "Client-talk analysis and transition features",
        },
    ]
)
_display_split_colors = {
    "Full utterance table": "#f8fafc",
    "Therapist modelling view": "#e7f0ff",
    "Client modelling view": "#e9f7ef",
}
display(Markdown("### Table 0.2. Split analysis views for later modelling"))
display_report_table(
    split_view_shape_df,
    formats={"rows": "{:,}", "columns": "{:,}"},
    row_color_col="view",
    row_colors=_display_split_colors,
    text_cols=["view", "downstream_use"],
    gradient_cols=["rows", "columns"],
    gradient_kwargs={"cmap": "YlGnBu"},
)


In [ ]:
def type_token_ratio(texts: List[str]) -> float:
    tokens = []
    for t in texts:
        tokens.extend(simple_tokenize(t))
    if not tokens:
        return np.nan
    return len(set(tokens)) / len(tokens)

transcript_base = utterance_df.groupby("transcript_id").agg(
    mi_quality=("mi_quality", "first"),
    topic_norm=("topic_norm", "first"),
    topic_raw=("topic", "first"),
    n_turns=("utterance_id", "count"),
    mean_word_count=("word_count", "mean"),
    median_word_count=("word_count", "median"),
    mean_char_count=("char_count", "mean"),
    median_char_count=("char_count", "median"),
    utterance_ttr=("utterance_text", lambda s: type_token_ratio(list(s))),
).reset_index()

therapist_turns = therapist_df.groupby("transcript_id").size().rename("therapist_turns")
client_turns = client_df.groupby("transcript_id").size().rename("client_turns")

ther_beh_counts = pd.crosstab(
    therapist_df["transcript_id"], therapist_df["main_therapist_behaviour"]
).reindex(columns=["reflection", "question", "therapist_input", "other"], fill_value=0)

client_talk_counts = pd.crosstab(
    client_df["transcript_id"], client_df["client_talk_type"]
).reindex(columns=["change", "neutral", "sustain"], fill_value=0)

transcript_df = transcript_base.merge(therapist_turns, on="transcript_id", how="left")
transcript_df = transcript_df.merge(client_turns, on="transcript_id", how="left")
transcript_df = transcript_df.merge(ther_beh_counts, on="transcript_id", how="left")
transcript_df = transcript_df.merge(client_talk_counts, on="transcript_id", how="left")

transcript_count_cols = ["therapist_turns", "client_turns", "reflection", "question", "therapist_input", "other", "change", "neutral", "sustain"]
for col in transcript_count_cols:
    transcript_df[col] = transcript_df[col].fillna(0).astype(int)

for col in ["reflection", "question", "therapist_input", "other"]:
    transcript_df[f"prop_{col}"] = transcript_df[col] / np.maximum(transcript_df["therapist_turns"], 1)

for col in ["change", "neutral", "sustain"]:
    transcript_df[f"prop_{col}"] = transcript_df[col] / np.maximum(transcript_df["client_turns"], 1)

transcript_df["reflection_to_question_ratio"] = transcript_df["reflection"] / np.maximum(transcript_df["question"], 1)
transcript_df["change_to_sustain_ratio"] = transcript_df["change"] / np.maximum(transcript_df["sustain"], 1)

ther_extra = therapist_df.groupby("transcript_id").agg(
    therapist_question_rate=("is_question_mark", "mean"),
    mean_prev_partner_lexical_overlap=("lexical_overlap_prev_partner", "mean"),
    mean_prev_partner_word_count=("prev_partner_word_count", "mean"),
).reset_index()

transcript_df = transcript_df.merge(ther_extra, on="transcript_id", how="left")
save_df(transcript_df, PROCESSED_DIR / "transcript_df.csv")

def clip_transcript_table_text(value: Any, max_chars: int = 74) -> str:
    text = "" if pd.isna(value) else str(value).strip()
    return text if len(text) <= max_chars else text[: max_chars - 3].rstrip() + "..."


transcript_feature_audit_df = pd.DataFrame([
    {
        "feature_group": "Session structure",
        "columns_created": "n_turns, therapist_turns, client_turns",
        "quality_check": "transcripts represented",
        "value": f"{transcript_df['transcript_id'].nunique():,}",
        "downstream_use": "Official transcript-level splitting and Appendix A modelling",
    },
    {
        "feature_group": "Surface length",
        "columns_created": "mean_word_count, median_word_count, mean_char_count, median_char_count",
        "quality_check": "median turns per transcript",
        "value": f"{transcript_df['n_turns'].median():.0f}",
        "downstream_use": "Descriptive statistics and session-structure diagnostics",
    },
    {
        "feature_group": "Lexical variety",
        "columns_created": "utterance_ttr",
        "quality_check": "mean type-token ratio",
        "value": f"{transcript_df['utterance_ttr'].mean():.3f}",
        "downstream_use": "Captures within-session lexical diversity",
    },
    {
        "feature_group": "Therapist behaviour mix",
        "columns_created": "reflection, question, therapist_input, other + proportions",
        "quality_check": "mean therapist turns",
        "value": f"{transcript_df['therapist_turns'].mean():.1f}",
        "downstream_use": "Behaviour-ratio predictors and EDA comparisons",
    },
    {
        "feature_group": "Client talk mix",
        "columns_created": "change, neutral, sustain + proportions",
        "quality_check": "mean client turns",
        "value": f"{transcript_df['client_turns'].mean():.1f}",
        "downstream_use": "Motivational-change signal for quality analysis",
    },
    {
        "feature_group": "Dialogue ratios",
        "columns_created": "reflection_to_question_ratio, change_to_sustain_ratio",
        "quality_check": "missing ratio values",
        "value": f"{int(transcript_df[['reflection_to_question_ratio', 'change_to_sustain_ratio']].isna().sum().sum()):,}",
        "downstream_use": "Compact interpretable predictors for Appendix A",
    },
])

transcript_structure_preview_cols = [
    "transcript_id",
    "mi_quality",
    "topic_norm",
    "n_turns",
    "therapist_turns",
    "client_turns",
    "mean_word_count",
    "median_word_count",
    "utterance_ttr",
    "therapist_question_rate",
    "mean_prev_partner_lexical_overlap",
]
transcript_structure_preview_df = transcript_df.loc[:, transcript_structure_preview_cols].head(8).copy()
transcript_structure_preview_df["topic_norm"] = transcript_structure_preview_df["topic_norm"].map(clip_transcript_table_text)

transcript_ratio_preview_cols = [
    "transcript_id",
    "mi_quality",
    "prop_reflection",
    "prop_question",
    "prop_therapist_input",
    "prop_other",
    "prop_change",
    "prop_neutral",
    "prop_sustain",
    "reflection_to_question_ratio",
    "change_to_sustain_ratio",
]
transcript_ratio_preview_df = transcript_df.loc[:, transcript_ratio_preview_cols].head(8).copy()

_transcript_feature_group_colors = {
    "Session structure": "#e7f0ff",
    "Surface length": "#e9f7ef",
    "Lexical variety": "#fff3d9",
    "Therapist behaviour mix": "#f3e8ff",
    "Client talk mix": "#ecfeff",
    "Dialogue ratios": "#fce8ef",
}

display(Markdown("### Table 1.4A. Transcript-level feature aggregation audit"))
display_report_table(
    transcript_feature_audit_df,
    row_color_col="feature_group",
    row_colors=_transcript_feature_group_colors,
    text_cols=["feature_group", "columns_created", "quality_check", "value", "downstream_use"],
)

display(Markdown("### Table 1.4B. Transcript-level structure and lexical preview"))
display_report_table(
    transcript_structure_preview_df,
    formats={
        "n_turns": "{:,}",
        "therapist_turns": "{:,}",
        "client_turns": "{:,}",
        "mean_word_count": "{:.2f}",
        "median_word_count": "{:.2f}",
        "utterance_ttr": "{:.3f}",
        "therapist_question_rate": "{:.1%}",
        "mean_prev_partner_lexical_overlap": "{:.3f}",
    },
    row_color_col="mi_quality",
    row_colors=REPORT_QUALITY_COLORS,
    text_cols=["mi_quality", "topic_norm"],
    gradient_cols=["n_turns", "therapist_turns", "client_turns", "mean_word_count", "utterance_ttr"],
    gradient_kwargs={"cmap": "YlGnBu"},
)

display(Markdown("### Table 1.4C. Transcript-level behaviour-ratio preview"))
display_report_table(
    transcript_ratio_preview_df,
    formats={
        "prop_reflection": "{:.1%}",
        "prop_question": "{:.1%}",
        "prop_therapist_input": "{:.1%}",
        "prop_other": "{:.1%}",
        "prop_change": "{:.1%}",
        "prop_neutral": "{:.1%}",
        "prop_sustain": "{:.1%}",
        "reflection_to_question_ratio": "{:.2f}",
        "change_to_sustain_ratio": "{:.2f}",
    },
    row_color_col="mi_quality",
    row_colors=REPORT_QUALITY_COLORS,
    text_cols=["mi_quality"],
    gradient_cols=["prop_reflection", "prop_question", "prop_therapist_input", "prop_change", "prop_sustain"],
    gradient_kwargs={"cmap": "YlGnBu"},
)

save_df(transcript_feature_audit_df, PROCESSED_DIR / "transcript_feature_aggregation_audit.csv")
save_df(transcript_structure_preview_df, PROCESSED_DIR / "transcript_structure_feature_preview.csv")
save_df(transcript_ratio_preview_df, PROCESSED_DIR / "transcript_behaviour_ratio_preview.csv")


In [ ]:
def bootstrap_ci(values: np.ndarray, func=np.mean, n_boot: int = 2000, ci: float = 95.0, seed: int = 42) -> Tuple[float, float]:
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    stats = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=len(values), replace=True)
        stats.append(func(sample))
    alpha = (100 - ci) / 2
    return np.percentile(stats, alpha), np.percentile(stats, 100 - alpha)

def numeric_summary_table(df: pd.DataFrame, subset_name: str, cols: List[str]) -> pd.DataFrame:
    rows = []
    for c in cols:
        s = pd.to_numeric(df[c], errors="coerce")
        rows.append(
            {
                "subset": subset_name,
                "feature": c,
                "count": s.notna().sum(),
                "mean": s.mean(),
                "std": s.std(),
                "min": s.min(),
                "q25": s.quantile(0.25),
                "median": s.median(),
                "q75": s.quantile(0.75),
                "max": s.max(),
                "range": s.max() - s.min(),
            }
        )
    return pd.DataFrame(rows)

NUMERIC_COLS_UTT = [
    "word_count",
    "char_count",
    "question_count",
    "exclamation_count",
    "comma_count",
    "turn_ratio",
]

summary_numeric_utterances = pd.concat(
    [
        numeric_summary_table(utterance_df, "all_utterances", NUMERIC_COLS_UTT),
        numeric_summary_table(utterance_df[utterance_df["interlocutor"] == "therapist"], "therapist_only", NUMERIC_COLS_UTT),
        numeric_summary_table(utterance_df[utterance_df["interlocutor"] == "client"], "client_only", NUMERIC_COLS_UTT),
        numeric_summary_table(utterance_df[utterance_df["mi_quality"] == "high"], "high_quality_utterances", NUMERIC_COLS_UTT),
        numeric_summary_table(utterance_df[utterance_df["mi_quality"] == "low"], "low_quality_utterances", NUMERIC_COLS_UTT),
    ],
    ignore_index=True,
)

categorical_overview = pd.DataFrame(
    {
        "metric": [
            "n_total_utterances",
            "n_total_transcripts",
            "n_therapist_utterances",
            "n_client_utterances",
        ],
        "value": [
            len(utterance_df),
            utterance_df["transcript_id"].nunique(),
            (utterance_df["interlocutor"] == "therapist").sum(),
            (utterance_df["interlocutor"] == "client").sum(),
        ],
    }
)

interlocutor_counts = utterance_df["interlocutor"].value_counts().rename_axis("interlocutor").reset_index(name="count")
mi_quality_counts = transcript_df["mi_quality"].value_counts().rename_axis("mi_quality").reset_index(name="count")
therapist_label_counts = therapist_df["main_therapist_behaviour"].value_counts().rename_axis("main_therapist_behaviour").reset_index(name="count")
client_talk_counts = client_df["client_talk_type"].value_counts().rename_axis("client_talk_type").reset_index(name="count")
top_topics = transcript_df["topic_norm"].value_counts().head(15).rename_axis("topic_norm").reset_index(name="count")

NUMERIC_COLS_TRANSCRIPT = [
    "n_turns",
    "therapist_turns",
    "client_turns",
    "mean_word_count",
    "median_word_count",
    "mean_char_count",
    "therapist_question_rate",
    "prop_reflection",
    "prop_question",
    "prop_therapist_input",
    "prop_other",
    "prop_change",
    "prop_neutral",
    "prop_sustain",
]

summary_numeric_transcripts = pd.concat(
    [
        numeric_summary_table(transcript_df, "all_transcripts", NUMERIC_COLS_TRANSCRIPT),
        numeric_summary_table(transcript_df[transcript_df["mi_quality"] == "high"], "high_quality_transcripts", NUMERIC_COLS_TRANSCRIPT),
        numeric_summary_table(transcript_df[transcript_df["mi_quality"] == "low"], "low_quality_transcripts", NUMERIC_COLS_TRANSCRIPT),
    ],
    ignore_index=True,
)

display(Markdown("### Table 2.1A. Overall corpus shape"))

_overview_metric_colors = {
    "n_total_utterances": "#e7f0ff",
    "n_total_transcripts": "#f1f5f9",
    "n_therapist_utterances": "#e9f7ef",
    "n_client_utterances": "#fff3d9",
}

def _style_corpus_overview_rows(row):
    background = _overview_metric_colors.get(str(row["metric"]), "#ffffff")
    return [
        f"background-color: {background}; color: #111827; font-weight: 750; border: 1px solid #cbd5e1;"
        for _ in row
    ]

try:
    display(
        categorical_overview.style
        .hide(axis="index")
        .format({"value": "{:,}"})
        .apply(_style_corpus_overview_rows, axis=1)
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#f8fafc"),
                        ("color", "#111827"),
                        ("font-weight", "800"),
                        ("text-align", "left"),
                        ("border", "1px solid #cbd5e1"),
                    ],
                },
                {
                    "selector": "td",
                    "props": [
                        ("color", "#111827"),
                        ("font-weight", "750"),
                        ("border", "1px solid #cbd5e1"),
                    ],
                },
            ]
        )
    )
except Exception:

    _overview_rows_html = "".join(
        f"<tr style='background-color: {_overview_metric_colors.get(str(metric), '#ffffff')}; color: #111827; font-weight: 750;'>"
        f"<td style='border: 1px solid #cbd5e1; padding: 6px 10px;'>{escape(str(metric))}</td>"
        f"<td style='border: 1px solid #cbd5e1; padding: 6px 10px; text-align: right;'>{int(value):,}</td>"
        "</tr>"
        for metric, value in categorical_overview[["metric", "value"]].itertuples(index=False, name=None)
    )
    display(
        HTML(
            "<table style='border-collapse: collapse; color: #111827;'>"
            "<thead><tr>"
            "<th style='background-color: #f8fafc; color: #111827; font-weight: 800; text-align: left; border: 1px solid #cbd5e1; padding: 6px 10px;'>metric</th>"
            "<th style='background-color: #f8fafc; color: #111827; font-weight: 800; text-align: right; border: 1px solid #cbd5e1; padding: 6px 10px;'>value</th>"
            "</tr></thead><tbody>"
            f"{_overview_rows_html}"
            "</tbody></table>"
        )
    )


In [ ]:
key_cols = ["interlocutor", "utterance_text", "main_therapist_behaviour", "client_talk_type"]
audit_rows = []
for col in key_cols:
    s = utterance_df[col]
    audit_rows.append({
        "column": col,
        "dtype": str(s.dtype),
        "n_rows": len(s),
        "missing_n": int(s.isna().sum()),
        "missing_pct": float(s.isna().mean() * 100),
        "empty_string_n": int((s.astype(str).str.strip() == "").sum()) if s.dtype == "object" else np.nan,
        "n_unique": int(s.nunique(dropna=True)),
        "most_common": s.astype(str).value_counts(dropna=False).head(1).index[0],
        "most_common_n": int(s.astype(str).value_counts(dropna=False).head(1).iloc[0]),
    })
audit_df = pd.DataFrame(audit_rows)
enhanced_utt_stats = summary_numeric_utterances.copy()
enhanced_utt_stats["iqr"] = enhanced_utt_stats["q75"] - enhanced_utt_stats["q25"]
transcript_ratio_df = transcript_df.copy()
transcript_ratio_df["therapist_client_turn_ratio"] = (
    transcript_ratio_df["therapist_turns"] / np.maximum(transcript_ratio_df["client_turns"], 1)
)
ratio_summary = (
    transcript_ratio_df.groupby("mi_quality")[[
        "therapist_client_turn_ratio",
        "reflection_to_question_ratio",
        "change_to_sustain_ratio",
        "therapist_question_rate",
    ]]
    .agg(["mean", "median", "std"])
    .round(3)
)
display(Markdown("### Extra EDA 1A. Data audit for key columns"))
display_report_table(
    audit_df,
    text_cols=["column", "dtype", "most_common"],
    gradient_cols=["missing_n", "missing_pct", "empty_string_n"],
    gradient_kwargs={"cmap": "YlOrRd"},
    highlight_max_cols=["n_unique", "most_common_n"],
)
display(Markdown("### Extra EDA 1B. Enhanced utterance-level descriptive statistics"))
display_report_table(
    enhanced_utt_stats.round(3),
    row_color_col="subset",
    row_colors=REPORT_SUBSET_COLORS,
    text_cols=["subset", "feature"],
    gradient_cols=["mean", "std", "q25", "median", "q75", "iqr", "range"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
_ratio_metric_colors = {
    "therapist_client_turn_ratio": "#e7f0ff",
    "reflection_to_question_ratio": "#e9f7ef",
    "change_to_sustain_ratio": "#fff3d9",
    "therapist_question_rate": "#fce8ef",
}
def _style_ratio_metric_groups(df: pd.DataFrame) -> pd.DataFrame:
    styles = pd.DataFrame("", index=df.index, columns=df.columns)
    for metric, color in _ratio_metric_colors.items():
        metric_cols = [col for col in df.columns if col[0] == metric]
        styles.loc[:, metric_cols] = (
            f"background-color: {color}; color: #111827; font-weight: 700; "
            "border: 1px solid #cbd5e1"
        )
    return styles
def _style_ratio_top_headers(label: str) -> str:
    color = _ratio_metric_colors.get(str(label), "#f8fafc")
    return (
        f"background-color: {color}; color: #111827; font-weight: 800; "
        "border: 1px solid #94a3b8; text-align: center"
    )
def _style_ratio_subheaders(_label: str) -> str:
    return (
        "color: #111827; font-weight: 800; "
        "border: 1px solid #94a3b8; text-align: center"
    )
def _ratio_subheader_table_styles() -> List[Dict[str, Any]]:
    styles = []
    for group_idx, color in enumerate(_ratio_metric_colors.values()):
        cols = range(group_idx * 3, group_idx * 3 + 3)
        selector = ", ".join(f"th.col_heading.level1.col{col}" for col in cols)
        styles.append(
            {
                "selector": selector,
                "props": [
                    ("background-color", color),
                    ("color", "#111827"),
                    ("font-weight", "800"),
                    ("border", "1px solid #94a3b8"),
                    ("text-align", "center"),
                ],
            }
        )
    return styles
def _style_ratio_row_headers(_label: str) -> str:
    return (
        "background-color: #f1f5f9; color: #111827; font-weight: 800; "
        "border: 1px solid #cbd5e1; text-align: left"
    )
display(Markdown("### Extra EDA 1C. Transcript-level ratio summaries by MI quality"))
try:
    ratio_summary_styled = ratio_summary.style.format("{:.3f}").apply(_style_ratio_metric_groups, axis=None)
    if hasattr(ratio_summary_styled, "map_index"):
        ratio_summary_styled = ratio_summary_styled.map_index(_style_ratio_top_headers, axis=1, level=0)
        ratio_summary_styled = ratio_summary_styled.map_index(_style_ratio_subheaders, axis=1, level=1)
        ratio_summary_styled = ratio_summary_styled.map_index(_style_ratio_row_headers, axis=0)
    elif hasattr(ratio_summary_styled, "applymap_index"):
        ratio_summary_styled = ratio_summary_styled.applymap_index(_style_ratio_top_headers, axis=1, level=0)
        ratio_summary_styled = ratio_summary_styled.applymap_index(_style_ratio_subheaders, axis=1, level=1)
        ratio_summary_styled = ratio_summary_styled.applymap_index(_style_ratio_row_headers, axis=0)
    ratio_summary_styled = (
        ratio_summary_styled
        .set_properties(**{"color": "#111827", "font-weight": "700", "border": "1px solid #cbd5e1"})
        .set_table_styles(
            [
                {"selector": "table", "props": [("border-collapse", "collapse"), ("font-size", "13px")]},
                {"selector": "th", "props": [("text-align", "center"), ("color", "#111827"), ("font-weight", "800"), ("background-color", "#f8fafc"), ("border", "1px solid #cbd5e1")]},
                *_ratio_subheader_table_styles(),
                {"selector": "th.row_heading", "props": [("text-align", "left"), ("color", "#111827"), ("font-weight", "800"), ("background-color", "#f1f5f9")]},
                {"selector": "th.index_name", "props": [("text-align", "left"), ("color", "#111827"), ("font-weight", "800"), ("background-color", "#e2e8f0")]},
                {"selector": "td", "props": [("text-align", "right"), ("color", "#111827"), ("font-weight", "700"), ("border", "1px solid #cbd5e1")]},
            ],
            overwrite=False,
        )
    )
    display(ratio_summary_styled)
except Exception:
    display(ratio_summary)


In [ ]:
display(Markdown("### Table 2.1A — Overall corpus shape"))
display_report_table(
    categorical_overview,
    formats={"value": "{:,}"},
    row_color_col="metric",
    row_colors={
        "n_total_utterances": "#f8fafc",
        "n_total_transcripts": "#f1f5f9",
        "n_therapist_utterances": "#e7f0ff",
        "n_client_utterances": "#e9f7ef",
    },
    text_cols=["metric"],
    gradient_cols=["value"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
display(Markdown("**Interlocutor counts (utterance level)**"))
display_report_table(
    interlocutor_counts,
    formats={"count": "{:,}"},
    row_color_col="interlocutor",
    row_colors=REPORT_ROLE_COLORS,
    text_cols=["interlocutor"],
    gradient_cols=["count"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
display(Markdown("**MI-quality counts (transcript level)**"))
display_report_table(
    mi_quality_counts,
    formats={"count": "{:,}"},
    row_color_col="mi_quality",
    row_colors=REPORT_QUALITY_COLORS,
    text_cols=["mi_quality"],
    gradient_cols=["count"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
display(Markdown("**Top topics (transcript level, top 15)**"))
display_report_table(
    top_topics,
    formats={"count": "{:,}"},
    text_cols=["topic_norm"],
    gradient_cols=["count"],
    gradient_kwargs={"cmap": "YlGnBu"},
    highlight_max_cols=["count"],
)
display(Markdown("### Table 2.1B — Categorical label distributions"))
display(Markdown("**Therapist behaviour counts** (rows where `interlocutor == therapist` and label is not `n/a`)"))
display_report_table(
    therapist_label_counts,
    formats={"count": "{:,}"},
    row_color_col="main_therapist_behaviour",
    row_colors=REPORT_BEHAVIOUR_COLORS,
    text_cols=["main_therapist_behaviour"],
    gradient_cols=["count"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
display(Markdown("**Client talk-type counts** (rows where `interlocutor == client` and talk type is not `n/a`)"))
display_report_table(
    client_talk_counts,
    formats={"count": "{:,}"},
    row_color_col="client_talk_type",
    row_colors=REPORT_BEHAVIOUR_COLORS,
    text_cols=["client_talk_type"],
    gradient_cols=["count"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
display(Markdown("### Table 2.1C — Numeric descriptive statistics"))
display(Markdown("**Utterance-level numeric features** (per-row, all utterances + therapist-only / client-only / quality slices)"))
display_report_table(
    summary_numeric_utterances.round(3),
    row_color_col="subset",
    row_colors=REPORT_SUBSET_COLORS,
    text_cols=["subset", "feature"],
    gradient_cols=["mean", "std", "min", "q25", "median", "q75", "max", "range"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
display(Markdown("**Transcript-level numeric features** (per-transcript, full corpus + by MI quality)"))
display_report_table(
    summary_numeric_transcripts.round(3),
    row_color_col="subset",
    row_colors=REPORT_SUBSET_COLORS,
    text_cols=["subset", "feature"],
    gradient_cols=["mean", "std", "min", "q25", "median", "q75", "max", "range"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
summary_numeric_utterances.to_csv(PROCESSED_DIR / "summary_numeric_utterances.csv", index=False)
summary_numeric_transcripts.to_csv(PROCESSED_DIR / "summary_numeric_transcripts.csv", index=False)
interlocutor_counts.to_csv(PROCESSED_DIR / "summary_interlocutor_counts.csv", index=False)
mi_quality_counts.to_csv(PROCESSED_DIR / "summary_mi_quality_counts.csv", index=False)
therapist_label_counts.to_csv(PROCESSED_DIR / "summary_therapist_label_counts.csv", index=False)
client_talk_counts.to_csv(PROCESSED_DIR / "summary_client_talk_counts.csv", index=False)


In [ ]:
bootstrap_summary_rows = []

for subset_name, subset_df in {
    "all_utterances": utterance_df,
    "therapist_only": utterance_df[utterance_df["interlocutor"] == "therapist"],
    "client_only": utterance_df[utterance_df["interlocutor"] == "client"],
}.items():
    for feature in ["word_count", "char_count"]:
        lo, hi = bootstrap_ci(subset_df[feature].to_numpy(), func=np.mean, n_boot=2000, ci=95, seed=SEED)
        bootstrap_summary_rows.append(
            {
                "subset": subset_name,
                "feature": feature,
                "mean": subset_df[feature].mean(),
                "ci95_low": lo,
                "ci95_high": hi,
            }
        )

bootstrap_summary_df = pd.DataFrame(bootstrap_summary_rows)
display(Markdown("### Table 2.1D. Bootstrap confidence intervals for utterance length"))
display_report_table(
    bootstrap_summary_df.round(3),
    row_color_col="subset",
    row_colors=REPORT_SUBSET_COLORS,
    text_cols=["subset", "feature"],
    gradient_cols=["mean", "ci95_low", "ci95_high"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
bootstrap_summary_df.to_csv(PROCESSED_DIR / "summary_bootstrap_cis.csv", index=False)


In [ ]:
def cliffs_delta_from_u(u_stat: float, n_x: int, n_y: int) -> float:
    if n_x == 0 or n_y == 0:
        return np.nan
    return (2.0 * float(u_stat) / (n_x * n_y)) - 1.0

def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x)
    y = np.asarray(y)
    if len(x) == 0 or len(y) == 0:
        return np.nan
    u_stat = mannwhitneyu(x, y, alternative="two-sided").statistic
    return cliffs_delta_from_u(u_stat, len(x), len(y))

def benjamini_hochberg(pvals: List[float]) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    adj = ranked * n / (np.arange(n) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    out = np.empty_like(adj)
    out[order] = adj
    return out

eda_test_features = [
    "n_turns",
    "mean_word_count",
    "mean_char_count",
    "therapist_question_rate",
    "prop_reflection",
    "prop_question",
    "prop_therapist_input",
    "prop_other",
    "prop_change",
    "prop_neutral",
    "prop_sustain",
    "reflection_to_question_ratio",
    "change_to_sustain_ratio",
    "mean_prev_partner_lexical_overlap",
]

stats_rows = []
for feature in eda_test_features:
    high = transcript_df.loc[transcript_df["mi_quality"] == "high", feature].dropna().to_numpy()
    low = transcript_df.loc[transcript_df["mi_quality"] == "low", feature].dropna().to_numpy()
    stat, p = mannwhitneyu(high, low, alternative="two-sided")
    stats_rows.append(
        {
            "feature": feature,
            "high_mean": np.mean(high),
            "low_mean": np.mean(low),
            "high_median": np.median(high),
            "low_median": np.median(low),
            "mannwhitney_u": stat,
            "p_value": p,
            "cliffs_delta": cliffs_delta_from_u(stat, len(high), len(low)),
        }
    )

stats_df = pd.DataFrame(stats_rows)
stats_df["p_value_bh"] = benjamini_hochberg(stats_df["p_value"].tolist())
stats_df = stats_df.sort_values("p_value_bh")
display(Markdown("### Table 3.1A. MI-quality group differences across transcript features"))
display_report_table(
    stats_df.round(4),
    text_cols=["feature"],
    gradient_cols=["high_mean", "low_mean", "high_median", "low_median"],
    gradient_kwargs={"cmap": "YlGnBu"},
    signed_cols=["cliffs_delta"],
    highlight_min_cols=["p_value_bh"],
)
stats_df.to_csv(PROCESSED_DIR / "eda_significance_tests.csv", index=False)


In [ ]:
HAVE_ST = bool(globals().get("HAVE_ST", False) and globals().get("SentenceTransformer") is not None)
if not HAVE_ST:
    print(f"SentenceTransformers not available: {globals().get('SENTENCE_TRANSFORMERS_IMPORT_ERROR', 'unknown import error')}")
if CONFIG["run_semantic_embeddings"] and HAVE_ST:
    if globals().get("torch") is None:
        raise ImportError("Torch is required for semantic embeddings but is not available from the main imports cell.")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    st_model = SentenceTransformer(CONFIG["embedding_model_name"], device=device)
    semantic_embedding_cache = CACHE_DIR / "therapist_utterance_embeddings.npy"
    semantic_frame_cache = PROCESSED_DIR / "therapist_embedding_frame.csv"
    therapist_embedding_df = therapist_df.reset_index(drop=True).copy()
    therapist_utterance_embeddings = st_model.encode(
        therapist_embedding_df["utterance_text"].fillna("").astype(str).tolist(),
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)
    np.save(semantic_embedding_cache, therapist_utterance_embeddings)
    therapist_embedding_df.drop(columns=["history_tagged_turns"], errors="ignore").to_csv(semantic_frame_cache, index=False)
    mask = therapist_df["prev_client_text"].fillna("").str.len() > 0
    therapist_df["prev_client_semantic_cosine"] = 0.0
    curr_emb = therapist_utterance_embeddings[mask.to_numpy()]
    prev_emb = st_model.encode(
        therapist_df.loc[mask, "prev_client_text"].tolist(),
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    therapist_df.loc[mask, "prev_client_semantic_cosine"] = np.sum(curr_emb * prev_emb, axis=1)
    transcript_sem = therapist_df.groupby("transcript_id")["prev_client_semantic_cosine"].mean().rename("mean_prev_client_semantic_cosine")
    transcript_df = transcript_df.merge(transcript_sem, on="transcript_id", how="left")
    therapist_df_to_save = therapist_df.drop(columns=["history_tagged_turns"]).copy()
    therapist_df_to_save.to_csv(PROCESSED_DIR / "therapist_df_with_semantics.csv", index=False)
    transcript_df.to_csv(PROCESSED_DIR / "transcript_df_with_semantics.csv", index=False)
else:
    print("Skipping sentence-embedding features.")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
client_dist = (
    client_df.groupby(["mi_quality", "client_talk_type"])
    .size()
    .rename("count")
    .reset_index()
)
client_pivot = client_dist.pivot(index="mi_quality", columns="client_talk_type", values="count").fillna(0)
client_pivot = client_pivot.reindex(columns=["change", "neutral", "sustain"], fill_value=0)
client_pivot = client_pivot.div(client_pivot.sum(axis=1), axis=0)
client_pivot.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Client talk distribution by MI quality")
ax.set_ylabel("Proportion")
ax.set_xlabel("MI quality")
ax.legend(title="Client talk type", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig0_client_talk_distribution_by_quality.png", dpi=300, bbox_inches="tight")
display(Markdown("### Figure 1. Client talk distribution by MI quality"))
plt.show()


In [ ]:
topic_quality_counts = (
    transcript_df.groupby(["topic_norm", "mi_quality"])
    .size()
    .unstack(fill_value=0)
)
topic_quality_counts["total"] = topic_quality_counts.sum(axis=1)
topic_quality_top = topic_quality_counts.sort_values("total", ascending=False).head(12).drop(columns=["total"])
topic_quality_prop = topic_quality_top.div(topic_quality_top.sum(axis=1), axis=0)
display(Markdown("### Table 3.1B. Top transcript topics by MI quality"))
topic_quality_display_df = (
    topic_quality_counts
    .sort_values("total", ascending=False)
    .head(12)
    .round(3)
    .reset_index()
)
display_report_table(
    topic_quality_display_df,
    text_cols=["topic_norm"],
    gradient_cols=[c for c in ["high", "low", "total"] if c in topic_quality_display_df.columns],
    gradient_kwargs={"cmap": "YlGnBu"},
    highlight_max_cols=["total"],
)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(topic_quality_top, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set_title("Top topics by MI quality (counts)")
axes[0].set_xlabel("MI quality")
axes[0].set_ylabel("Topic")
sns.heatmap(topic_quality_prop, annot=True, fmt=".2f", cmap="Purples", ax=axes[1])
axes[1].set_title("Top topics by MI quality (row proportions)")
axes[1].set_xlabel("MI quality")
axes[1].set_ylabel("Topic")
plt.tight_layout()
display(Markdown("### Figure 2. Topic distribution by MI quality"))
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
beh_dist = (
    therapist_df.groupby(["mi_quality", "main_therapist_behaviour"])
    .size()
    .rename("count")
    .reset_index()
)
beh_pivot = beh_dist.pivot(index="mi_quality", columns="main_therapist_behaviour", values="count").fillna(0)
beh_pivot = beh_pivot[["reflection", "question", "therapist_input", "other"]]
beh_pivot = beh_pivot.div(beh_pivot.sum(axis=1), axis=0)
beh_pivot.plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Therapist behaviour distribution by MI quality")
ax.set_ylabel("Proportion")
ax.set_xlabel("MI quality")
ax.legend(title="Therapist behaviour", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig1_behaviour_distribution_by_quality.png", dpi=300, bbox_inches="tight")
display(Markdown("### Figure 3. Therapist behaviour distribution by MI quality"))
plt.show()


In [ ]:
plot_features = [
    ("prop_reflection", "Reflection proportion"),
    ("prop_therapist_input", "Therapist input proportion"),
    ("prop_change", "Change-talk proportion"),
    ("prop_sustain", "Sustain-talk proportion"),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True)
axes = axes.flatten()
for ax, (feat, title) in zip(axes, plot_features):
    sns.boxplot(data=transcript_df, x="mi_quality", y=feat, ax=ax)
    sns.stripplot(data=transcript_df, x="mi_quality", y=feat, ax=ax, alpha=0.5, size=4)
    ax.set_title(title)
    ax.set_xlabel("MI quality")
    ax.set_ylabel(feat)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2_transcript_level_proportions.png", dpi=300, bbox_inches="tight")
display(Markdown("### Figure 4. Transcript-level behaviour and client-talk proportions by MI quality"))
plt.show()


In [ ]:
def cliffs_delta_local(x: np.ndarray, y: np.ndarray) -> float:
    return cliffs_delta(x, y)
eda_plus_features = [
    "n_turns",
    "therapist_turns",
    "client_turns",
    "mean_word_count",
    "mean_char_count",
    "therapist_question_rate",
    "prop_reflection",
    "prop_question",
    "prop_therapist_input",
    "prop_other",
    "prop_change",
    "prop_neutral",
    "prop_sustain",
    "reflection_to_question_ratio",
    "change_to_sustain_ratio",
    "mean_prev_partner_lexical_overlap",
]
if "mean_prev_client_semantic_cosine" in transcript_df.columns:
    eda_plus_features.append("mean_prev_client_semantic_cosine")
disc_rows = []
for feat in eda_plus_features:
    high = transcript_df.loc[transcript_df["mi_quality"] == "high", feat].dropna().to_numpy()
    low = transcript_df.loc[transcript_df["mi_quality"] == "low", feat].dropna().to_numpy()
    stat, p = mannwhitneyu(high, low, alternative="two-sided")
    disc_rows.append({
        "feature": feat,
        "high_mean": np.mean(high),
        "low_mean": np.mean(low),
        "high_median": np.median(high),
        "low_median": np.median(low),
        "delta_mean": np.mean(high) - np.mean(low),
        "cliffs_delta": cliffs_delta_from_u(stat, len(high), len(low)),
        "p_value": p,
    })
disc_df = pd.DataFrame(disc_rows)
disc_df["p_value_bh"] = benjamini_hochberg(disc_df["p_value"].tolist())
disc_df["abs_cliffs_delta"] = disc_df["cliffs_delta"].abs()
disc_df = disc_df.sort_values(["abs_cliffs_delta", "p_value_bh"], ascending=[False, True])
display(Markdown("### Table 3.1C. Transcript-level discriminators ranked by effect size"))
display_report_table(
    disc_df.round(4),
    text_cols=["feature"],
    gradient_cols=["abs_cliffs_delta"],
    gradient_kwargs={"cmap": "YlOrRd"},
    signed_cols=["delta_mean", "cliffs_delta"],
    highlight_min_cols=["p_value_bh"],
)


In [ ]:
tmp = transcript_df.copy()
tmp["therapist_client_turn_ratio"] = tmp["therapist_turns"] / np.maximum(tmp["client_turns"], 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=tmp, x="mi_quality", y="n_turns", ax=axes[0])
sns.stripplot(data=tmp, x="mi_quality", y="n_turns", ax=axes[0], alpha=0.5, size=4)
axes[0].set_title("Transcript length by MI quality")
axes[0].set_xlabel("MI quality")
axes[0].set_ylabel("Number of turns")
sns.boxplot(data=tmp, x="mi_quality", y="therapist_client_turn_ratio", ax=axes[1])
sns.stripplot(data=tmp, x="mi_quality", y="therapist_client_turn_ratio", ax=axes[1], alpha=0.5, size=4)
axes[1].set_title("Therapist/client turn ratio by MI quality")
axes[1].set_xlabel("MI quality")
axes[1].set_ylabel("Therapist/client ratio")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig2b_length_and_turn_ratio.png", dpi=300, bbox_inches="tight")
display(Markdown("### Figure 5. Transcript length and therapist-client ratio by MI quality"))
plt.show()


In [ ]:
behaviour_order = ["reflection", "question", "therapist_input", "other"]
length_df = therapist_df[
    therapist_df["main_therapist_behaviour"].isin(behaviour_order)
].copy()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(
    data=length_df,
    x="word_count",
    hue="main_therapist_behaviour",
    hue_order=behaviour_order,
    stat="density",
    common_norm=False,
    element="step",
    fill=False,
    ax=axes[0],
)
axes[0].set_title("Utterance length by therapist behaviour")
axes[0].set_xlabel("Token count")
axes[0].set_ylabel("Density")
sns.boxplot(
    data=length_df,
    x="main_therapist_behaviour",
    y="word_count",
    order=behaviour_order,
    ax=axes[1],
)
sns.stripplot(
    data=length_df.sample(min(len(length_df), 1600), random_state=SEED),
    x="main_therapist_behaviour",
    y="word_count",
    order=behaviour_order,
    alpha=0.25,
    size=2.5,
    ax=axes[1],
)
axes[1].set_title("Token-count spread by therapist behaviour")
axes[1].set_xlabel("Therapist behaviour")
axes[1].set_ylabel("Token count")
axes[1].tick_params(axis="x", rotation=15)
out_path = FIG_DIR / "fig2c_utterance_length_by_behaviour.png"
fig.tight_layout()
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 6. Utterance-length distributions by therapist behaviour"))
display(IPyImage(filename=str(out_path)))
length_summary_df = (
    length_df.groupby("main_therapist_behaviour")["word_count"]
    .agg(
        n="size",
        mean="mean",
        median="median",
        q25=lambda s: s.quantile(0.25),
        q75=lambda s: s.quantile(0.75),
        std="std",
    )
    .reindex(behaviour_order)
    .reset_index()
)
display(Markdown("### Table 3.1D. Utterance length summary by therapist behaviour"))
display_report_table(
    length_summary_df.round(2),
    row_color_col="main_therapist_behaviour",
    row_colors=REPORT_BEHAVIOUR_COLORS,
    text_cols=["main_therapist_behaviour"],
    gradient_cols=["n", "mean", "median", "q25", "q75", "std"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
save_df(length_summary_df, PROCESSED_DIR / "utterance_length_by_therapist_behaviour.csv")


In [ ]:
transition_rows = []
for transcript_id, g in utterance_df.sort_values(["transcript_id", "utterance_id"]).groupby("transcript_id", sort=False):
    g = g.reset_index(drop=True)
    quality = g.loc[0, "mi_quality"]
    for i in range(len(g) - 1):
        if g.loc[i, "interlocutor"] == "client" and g.loc[i + 1, "interlocutor"] == "therapist":
            transition_rows.append(
                {
                    "mi_quality": quality,
                    "client_talk_type": g.loc[i, "client_talk_type"],
                    "next_therapist_behaviour": g.loc[i + 1, "main_therapist_behaviour"],
                }
            )
transition_df = pd.DataFrame(transition_rows)
transition_df = transition_df[
    (transition_df["client_talk_type"] != "n/a")
    & (transition_df["next_therapist_behaviour"] != "n/a")
].copy()
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, quality in zip(axes, ["high", "low"]):
    sub = transition_df[transition_df["mi_quality"] == quality]
    mat = pd.crosstab(sub["client_talk_type"], sub["next_therapist_behaviour"])
    mat = mat.reindex(index=["change", "neutral", "sustain"], columns=["reflection", "question", "therapist_input", "other"], fill_value=0)
    mat = mat.div(mat.sum(axis=1).replace(0, 1), axis=0)
    sns.heatmap(mat, annot=True, fmt=".2f", cmap="Blues", ax=ax)
    ax.set_title(f"Client talk → next therapist behaviour ({quality})")
    ax.set_xlabel("Next therapist behaviour")
    ax.set_ylabel("Client talk type")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3_transition_heatmaps.png", dpi=300, bbox_inches="tight")
display(Markdown("### Figure 7. Client talk to next therapist behaviour transition heatmaps"))
plt.show()


In [ ]:
if "transition_df" not in globals():
    transition_rows = []
    for transcript_id, g in utterance_df.sort_values(["transcript_id", "utterance_id"]).groupby("transcript_id", sort=False):
        g = g.reset_index(drop=True)
        quality = g.loc[0, "mi_quality"]
        for i in range(len(g) - 1):
            if g.loc[i, "interlocutor"] == "client" and g.loc[i + 1, "interlocutor"] == "therapist":
                transition_rows.append(
                    {
                        "mi_quality": quality,
                        "client_talk_type": g.loc[i, "client_talk_type"],
                        "next_therapist_behaviour": g.loc[i + 1, "main_therapist_behaviour"],
                    }
                )
    transition_df = pd.DataFrame(transition_rows)
    transition_df = transition_df[
        (transition_df["client_talk_type"] != "n/a")
        & (transition_df["next_therapist_behaviour"] != "n/a")
    ].copy()
row_order = ["change", "neutral", "sustain"]
col_order = ["reflection", "question", "therapist_input", "other"]
high_mat = pd.crosstab(
    transition_df.loc[transition_df["mi_quality"] == "high", "client_talk_type"],
    transition_df.loc[transition_df["mi_quality"] == "high", "next_therapist_behaviour"],
).reindex(index=row_order, columns=col_order, fill_value=0)
low_mat = pd.crosstab(
    transition_df.loc[transition_df["mi_quality"] == "low", "client_talk_type"],
    transition_df.loc[transition_df["mi_quality"] == "low", "next_therapist_behaviour"],
).reindex(index=row_order, columns=col_order, fill_value=0)
high_prop = high_mat.div(high_mat.sum(axis=1).replace(0, 1), axis=0)
low_prop = low_mat.div(low_mat.sum(axis=1).replace(0, 1), axis=0)
delta_mat = high_prop - low_prop
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
sns.heatmap(high_prop, annot=True, fmt=".2f", cmap="Blues", ax=axes[0])
axes[0].set_title("High-quality transitions")
sns.heatmap(low_prop, annot=True, fmt=".2f", cmap="Blues", ax=axes[1])
axes[1].set_title("Low-quality transitions")
sns.heatmap(delta_mat, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[2])
axes[2].set_title("High - Low transition delta")
for ax in axes:
    ax.set_xlabel("Next therapist behaviour")
    ax.set_ylabel("Client talk type")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig3b_transition_delta.png", dpi=300, bbox_inches="tight")
display(Markdown("### Figure 8. High-vs-low transition delta heatmaps"))
plt.show()
delta_table = (
    delta_mat.stack()
    .rename("delta")
    .reset_index()
    .sort_values("delta", key=lambda s: s.abs(), ascending=False)
)
display(Markdown("### Table 3.2A. Most divergent client-to-therapist transition deltas"))
display_report_table(
    delta_table.head(12).round(3),
    row_color_col="client_talk_type",
    row_colors=REPORT_BEHAVIOUR_COLORS,
    signed_cols=["delta"],
    text_cols=["client_talk_type", "next_therapist_behaviour"],
)


In [ ]:
ther_pos_df = therapist_df.copy()
bins = np.linspace(-0.001, 1.0, 11)
labels = [f"D{i}" for i in range(1, 11)]
ther_pos_df["position_bin"] = pd.cut(ther_pos_df["turn_ratio"], bins=bins, labels=labels)
pos_beh = (
    ther_pos_df.groupby(["mi_quality", "position_bin", "main_therapist_behaviour"])
    .size()
    .rename("count")
    .reset_index()
)
pos_beh["prop"] = pos_beh.groupby(["mi_quality", "position_bin"])["count"].transform(
    lambda x: x / np.maximum(x.sum(), 1)
)
behaviours = ["reflection", "question", "therapist_input", "other"]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
axes = axes.flatten()
for ax, beh in zip(axes, behaviours):
    sub = pos_beh[pos_beh["main_therapist_behaviour"] == beh]
    sns.lineplot(data=sub, x="position_bin", y="prop", hue="mi_quality", marker="o", ax=ax)
    ax.set_title(f"{beh} across transcript position")
    ax.set_xlabel("Transcript decile")
    ax.set_ylabel("Proportion")
plt.tight_layout()
display(Markdown("### Figure 9. Therapist behaviour across transcript position by MI quality"))
plt.show()


In [ ]:
if "prev_client_semantic_cosine" in therapist_df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=therapist_df, x="main_therapist_behaviour", y="prev_client_semantic_cosine", ax=ax)
    sns.stripplot(data=therapist_df.sample(min(len(therapist_df), 1200), random_state=SEED), x="main_therapist_behaviour", y="prev_client_semantic_cosine", ax=ax, alpha=0.3, size=3)
    ax.set_title("Semantic similarity between therapist turn and previous client turn")
    ax.set_xlabel("Therapist behaviour")
    ax.set_ylabel("Cosine similarity")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig4_semantic_similarity.png", dpi=300, bbox_inches="tight")
    display(Markdown("### Figure 10. Optional semantic similarity by MI quality"))
    plt.show()


In [ ]:
corr_features = [
    "n_turns",
    "mean_word_count",
    "therapist_question_rate",
    "prop_reflection",
    "prop_question",
    "prop_therapist_input",
    "prop_change",
    "prop_sustain",
    "reflection_to_question_ratio",
    "change_to_sustain_ratio",
]
if "mean_prev_client_semantic_cosine" in transcript_df.columns:
    corr_features.append("mean_prev_client_semantic_cosine")
corr_mat = transcript_df[corr_features].corr(numeric_only=True)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Transcript-level feature correlation heatmap")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig5_correlation_heatmap.png", dpi=300, bbox_inches="tight")
display(Markdown("### Figure 11. Correlation heatmap of transcript-level engineered features"))
plt.show()


In [ ]:
behaviour_order = ["reflection", "question", "therapist_input", "other"]
tfidf_source = therapist_df[
    therapist_df["main_therapist_behaviour"].isin(behaviour_order)
].copy()
tfidf_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.9,
    sublinear_tf=True,
    lowercase=True,
    stop_words="english",
)
X_tfidf = tfidf_vec.fit_transform(tfidf_source["utterance_text_lc"].fillna("").astype(str))
terms = np.array(tfidf_vec.get_feature_names_out())
top_n_terms = 10
tfidf_rows = []
for label in behaviour_order:
    label_mask = tfidf_source["main_therapist_behaviour"].eq(label).to_numpy()
    if label_mask.sum() == 0:
        continue
    class_mean = np.asarray(X_tfidf[label_mask].mean(axis=0)).ravel()
    rest_mean = np.asarray(X_tfidf[~label_mask].mean(axis=0)).ravel()
    score = class_mean - rest_mean
    top_idx = np.argsort(score)[-top_n_terms:][::-1]
    for rank, j in enumerate(top_idx, start=1):
        tfidf_rows.append(
            {
                "label": label,
                "rank": rank,
                "term": terms[j],
                "class_tfidf": float(class_mean[j]),
                "rest_tfidf": float(rest_mean[j]),
                "tfidf_gap": float(score[j]),
            }
        )
tfidf_terms_df = pd.DataFrame(tfidf_rows)
display(Markdown("### Table 3.3A. Class-specific TF-IDF terms by therapist behaviour"))
display_report_table(
    tfidf_terms_df.head(40).round(4),
    row_color_col="label",
    row_colors=REPORT_BEHAVIOUR_COLORS,
    text_cols=["label", "term"],
    gradient_cols=["class_tfidf", "tfidf_gap"],
    gradient_kwargs={"cmap": "YlGn"},
)
save_df(tfidf_terms_df, PROCESSED_DIR / "tfidf_terms_by_therapist_behaviour.csv")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for ax, label in zip(axes, behaviour_order):
    sub = tfidf_terms_df[tfidf_terms_df["label"] == label].sort_values("tfidf_gap", ascending=True)
    ax.barh(sub["term"], sub["tfidf_gap"])
    ax.set_title(f"Class-specific TF-IDF terms — {label}")
    ax.set_xlabel("TF-IDF gap vs other classes")
    ax.set_ylabel("")
out_path = FIG_DIR / "fig5b_tfidf_terms_by_behaviour.png"
fig.tight_layout()
fig.savefig(out_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 12. Class-specific TF-IDF terms by therapist behaviour"))
display(IPyImage(filename=str(out_path)))


In [ ]:
semantic_embedding_cache = CACHE_DIR / "therapist_utterance_embeddings.npy"
semantic_frame_cache = PROCESSED_DIR / "therapist_embedding_frame.csv"
def _ensure_semantic_space_inputs() -> None:
    global therapist_df, therapist_embedding_df, therapist_utterance_embeddings, st_model
    if (
        "therapist_embedding_df" in globals()
        and "therapist_utterance_embeddings" in globals()
        and len(therapist_embedding_df) == len(therapist_utterance_embeddings)
    ):
        therapist_embedding_df = therapist_embedding_df.reset_index(drop=True).copy()
        therapist_utterance_embeddings = np.asarray(therapist_utterance_embeddings, dtype=np.float32)
        return
    if semantic_frame_cache.exists() and semantic_embedding_cache.exists():
        cached_df = pd.read_csv(semantic_frame_cache)
        cached_emb = np.load(semantic_embedding_cache).astype(np.float32)
        if len(cached_df) == len(cached_emb):
            therapist_embedding_df = cached_df.reset_index(drop=True).copy()
            therapist_utterance_embeddings = cached_emb
            return
    if "therapist_df" not in globals():
        therapist_path_candidates = [
            PROCESSED_DIR / "therapist_df_with_semantics.csv",
            PROCESSED_DIR / "therapist_df.csv",
        ]
        for path in therapist_path_candidates:
            if path.exists():
                therapist_df = pd.read_csv(path)
                print(f"Loaded therapist_df for Figure 6 from: {path}")
                break
        else:
            raise NameError(
                "Figure 6 needs therapist_df or a saved therapist_df CSV. "
                "Run the data-building section first."
            )
    if "st_model" not in globals():
        if globals().get("SentenceTransformer") is None or globals().get("torch") is None:
            raise ImportError(
                "Figure 6 needs sentence-transformers and torch to rebuild missing therapist embeddings. "
                "Install/enable them or rerun the semantic-embedding cell first."
            )
        device = "cuda" if torch.cuda.is_available() else "cpu"
        st_model = SentenceTransformer(CONFIG["embedding_model_name"], device=device)
        print(f"Initialised sentence-transformer for Figure 6 on: {device}")
    therapist_embedding_df = therapist_df.reset_index(drop=True).copy()
    therapist_utterance_embeddings = st_model.encode(
        therapist_embedding_df["utterance_text"].fillna("").astype(str).tolist(),
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    ).astype(np.float32)
    np.save(semantic_embedding_cache, therapist_utterance_embeddings)
    therapist_embedding_df.drop(columns=["history_tagged_turns"], errors="ignore").to_csv(semantic_frame_cache, index=False)
_ensure_semantic_space_inputs()
if globals().get("UMAP") is None:
    raise ImportError(
        "UMAP is required for this cell. Install `umap-learn`, rerun the main imports cell, and rerun this cell. "
        f"Import error: {globals().get('UMAP_IMPORT_ERROR', '')}"
    )
behaviour_order = ["reflection", "question", "therapist_input", "other"]
semantic_mask = therapist_embedding_df["main_therapist_behaviour"].isin(behaviour_order).to_numpy()
semantic_plot_df = therapist_embedding_df.loc[semantic_mask].reset_index(drop=True).copy()
semantic_plot_emb = therapist_utterance_embeddings[semantic_mask].astype(np.float32)
sample_n = min(1500, len(semantic_plot_df))
sample_idx = semantic_plot_df.sample(sample_n, random_state=SEED).index.to_numpy()
sample_df = semantic_plot_df.loc[sample_idx].reset_index(drop=True).copy()
sample_emb = semantic_plot_emb[sample_idx]
umap_model = UMAP(
    n_components=2,
    n_neighbors=20,
    min_dist=0.15,
    metric="cosine",
    random_state=SEED,
)
umap_coords = umap_model.fit_transform(sample_emb)
sample_df["umap_1"] = umap_coords[:, 0]
sample_df["umap_2"] = umap_coords[:, 1]
centroids = {}
for label in behaviour_order:
    label_emb = semantic_plot_emb[semantic_plot_df["main_therapist_behaviour"].eq(label).to_numpy()]
    centroid = label_emb.mean(axis=0)
    centroid = centroid / np.maximum(np.linalg.norm(centroid), 1e-12)
    centroids[label] = centroid
centroid_matrix = pd.DataFrame(
    cosine_similarity(np.vstack([centroids[label] for label in behaviour_order])),
    index=behaviour_order,
    columns=behaviour_order,
)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.scatterplot(
    data=sample_df,
    x="umap_1",
    y="umap_2",
    hue="main_therapist_behaviour",
    hue_order=behaviour_order,
    alpha=0.7,
    s=35,
    ax=axes[0],
)
axes[0].set_title("UMAP projection of therapist-turn sentence embeddings")
axes[0].set_xlabel("UMAP-1")
axes[0].set_ylabel("UMAP-2")
sns.heatmap(centroid_matrix, annot=True, fmt=".2f", cmap="vlag", center=0, ax=axes[1])
axes[1].set_title("Class-centroid cosine similarity")
axes[1].set_xlabel("Therapist behaviour")
axes[1].set_ylabel("Therapist behaviour")
out_path_overview = FIG_DIR / "fig6_semantic_space_overview.png"
fig.tight_layout()
fig.savefig(out_path_overview, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 13. Semantic-space overview of therapist utterances"))
display(IPyImage(filename=str(out_path_overview)))
rng = np.random.default_rng(SEED)
labels_arr = semantic_plot_df["main_therapist_behaviour"].to_numpy()
intra_vals = []
for label in behaviour_order:
    idx = np.where(labels_arr == label)[0]
    if len(idx) < 2:
        continue
    n_draw = min(800, len(idx) * 4)
    for _ in range(n_draw):
        i, j = rng.choice(idx, size=2, replace=False)
        intra_vals.append(float(np.dot(semantic_plot_emb[i], semantic_plot_emb[j])))
inter_vals = []
all_idx = np.arange(len(labels_arr))
target_n = len(intra_vals)
while len(inter_vals) < target_n:
    i, j = rng.choice(all_idx, size=2, replace=False)
    if labels_arr[i] != labels_arr[j]:
        inter_vals.append(float(np.dot(semantic_plot_emb[i], semantic_plot_emb[j])))
semantic_similarity_df = pd.DataFrame(
    {
        "pair_type": ["intra_class"] * len(intra_vals) + ["inter_class"] * len(inter_vals),
        "cosine_similarity": intra_vals + inter_vals,
    }
)
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.violinplot(
    data=semantic_similarity_df,
    x="pair_type",
    y="cosine_similarity",
    inner="box",
    ax=ax,
)
ax.set_title("Intra-class vs inter-class cosine similarity")
ax.set_xlabel("")
ax.set_ylabel("Cosine similarity")
out_path_similarity = FIG_DIR / "fig6b_intra_inter_similarity.png"
fig.tight_layout()
fig.savefig(out_path_similarity, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 14. Intra-class vs inter-class semantic similarity"))
display(IPyImage(filename=str(out_path_similarity)))
display(Markdown("### Table 3.3B. Therapist behaviour centroid cosine similarity"))
display_report_table(
    centroid_matrix.round(3),
    gradient_cols=list(centroid_matrix.columns),
    gradient_kwargs={"cmap": "YlGnBu"},
    hide_index=False,
)
save_df(sample_df, PROCESSED_DIR / "umap_therapist_embedding_projection.csv")
centroid_matrix.to_csv(PROCESSED_DIR / "therapist_centroid_cosine_similarity.csv")
save_df(semantic_similarity_df, PROCESSED_DIR / "therapist_intra_inter_similarity.csv")


In [ ]:
if "semantic_plot_df" not in globals() or "semantic_plot_emb" not in globals():
    raise NameError("Run the semantic-space diagnostic cell just above before creating nearest-neighbour examples.")
behaviour_order = ["reflection", "question", "therapist_input", "other"]
label_arr = semantic_plot_df["main_therapist_behaviour"].to_numpy()
semantic_nn_rows = []
for label in behaviour_order:
    candidate_idx = np.where(
        (label_arr == label)
        & (semantic_plot_df["word_count"].fillna(0).to_numpy() >= 4)
    )[0]
    if len(candidate_idx) == 0:
        candidate_idx = np.where(label_arr == label)[0]
    if len(candidate_idx) == 0:
        continue
    centroid = semantic_plot_emb[candidate_idx].mean(axis=0)
    centroid = centroid / np.maximum(np.linalg.norm(centroid), 1e-12)
    anchor_scores = semantic_plot_emb[candidate_idx] @ centroid
    anchor_idx = int(candidate_idx[np.argmax(anchor_scores)])
    sims = semantic_plot_emb @ semantic_plot_emb[anchor_idx]
    nn_idx = [i for i in np.argsort(sims)[::-1] if i != anchor_idx][:3]
    semantic_nn_rows.append(
        {
            "anchor_label": label,
            "row_type": "anchor",
            "matched_label": semantic_plot_df.loc[anchor_idx, "main_therapist_behaviour"],
            "cosine_similarity": 1.0,
            "transcript_id": semantic_plot_df.loc[anchor_idx, "transcript_id"],
            "utterance_id": semantic_plot_df.loc[anchor_idx, "utterance_id"],
            "prev_client_text": semantic_plot_df.loc[anchor_idx, "prev_client_text"],
            "utterance_text": semantic_plot_df.loc[anchor_idx, "utterance_text"],
        }
    )
    for rank, i in enumerate(nn_idx, start=1):
        semantic_nn_rows.append(
            {
                "anchor_label": label,
                "row_type": f"nn{rank}",
                "matched_label": semantic_plot_df.loc[i, "main_therapist_behaviour"],
                "cosine_similarity": float(sims[i]),
                "transcript_id": semantic_plot_df.loc[i, "transcript_id"],
                "utterance_id": semantic_plot_df.loc[i, "utterance_id"],
                "prev_client_text": semantic_plot_df.loc[i, "prev_client_text"],
                "utterance_text": semantic_plot_df.loc[i, "utterance_text"],
            }
        )
semantic_nn_df = pd.DataFrame(semantic_nn_rows)
display(Markdown("### Table 3.3C. Semantic nearest-neighbour examples by therapist behaviour"))
_semantic_nn_colors = {
    "reflection": "#e7f0ff",
    "question": "#e9f7ef",
    "therapist_input": "#fff3d9",
    "other": "#fce8ef",
}
semantic_nn_display_df = semantic_nn_df.copy()
if "cosine_similarity" in semantic_nn_display_df.columns:
    semantic_nn_display_df["cosine_similarity"] = pd.to_numeric(
        semantic_nn_display_df["cosine_similarity"], errors="coerce"
    )
def _style_semantic_nn_rows(row):
    background = _semantic_nn_colors.get(str(row.get("anchor_label", "")), "#f8fafc")
    weight = "800" if str(row.get("row_type", "")) == "anchor" else "650"
    return [
        f"background-color: {background}; color: #111827; font-weight: {weight}; border: 1px solid #cbd5e1; vertical-align: top;"
        for _ in row
    ]
try:
    _semantic_nn_styler = (
        semantic_nn_display_df.style
        .hide(axis="index")
        .format({"cosine_similarity": "{:.3f}"})
        .apply(_style_semantic_nn_rows, axis=1)
        .set_table_styles(
            [
                {
                    "selector": "th",
                    "props": [
                        ("background-color", "#f8fafc"),
                        ("color", "#111827"),
                        ("font-weight", "800"),
                        ("text-align", "left"),
                        ("border", "1px solid #cbd5e1"),
                        ("padding", "5px 7px"),
                        ("font-size", "12px"),
                    ],
                },
                {
                    "selector": "td",
                    "props": [
                        ("padding", "5px 7px"),
                        ("font-size", "12px"),
                        ("line-height", "1.25"),
                    ],
                },
                {
                    "selector": "table",
                    "props": [
                        ("border-collapse", "collapse"),
                        ("width", "100%"),
                    ],
                },
            ]
        )
    )
    _semantic_text_cols = [c for c in ["prev_client_text", "utterance_text"] if c in semantic_nn_display_df.columns]
    if _semantic_text_cols:
        _semantic_nn_styler = _semantic_nn_styler.set_properties(
            subset=_semantic_text_cols,
            **{
                "max-width": "360px",
                "white-space": "normal",
                "word-break": "normal",
            },
        )
    _semantic_compact_cols = [
        c
        for c in ["anchor_label", "row_type", "matched_label", "cosine_similarity", "transcript_id", "utterance_id"]
        if c in semantic_nn_display_df.columns
    ]
    if _semantic_compact_cols:
        _semantic_nn_styler = _semantic_nn_styler.set_properties(
            subset=_semantic_compact_cols,
            **{"white-space": "nowrap"},
        )
    display(_semantic_nn_styler)
except Exception:
    _semantic_nn_cols = semantic_nn_display_df.columns.tolist()
    _header_html = "".join(
        f"<th style='background-color: #f8fafc; color: #111827; font-weight: 800; text-align: left; border: 1px solid #cbd5e1; padding: 5px 7px; font-size: 12px;'>{escape(str(col))}</th>"
        for col in _semantic_nn_cols
    )
    _body_html = ""
    for _, row in semantic_nn_display_df.iterrows():
        background = _semantic_nn_colors.get(str(row.get("anchor_label", "")), "#f8fafc")
        weight = "800" if str(row.get("row_type", "")) == "anchor" else "650"
        _body_html += f"<tr style='background-color: {background}; color: #111827; font-weight: {weight};'>"
        for col in _semantic_nn_cols:
            value = row[col]
            if col == "cosine_similarity" and pd.notna(value):
                value_text = f"{float(value):.3f}"
            elif pd.isna(value):
                value_text = ""
            else:
                value_text = str(value)
            text_style = "max-width: 360px; white-space: normal; word-break: normal;" if col in {"prev_client_text", "utterance_text"} else "white-space: nowrap;"
            _body_html += (
                f"<td style='border: 1px solid #cbd5e1; padding: 5px 7px; font-size: 12px; line-height: 1.25; vertical-align: top; {text_style}'>"
                f"{escape(value_text)}</td>"
            )
        _body_html += "</tr>"
    display(
        HTML(
            "<table style='border-collapse: collapse; width: 100%; color: #111827;'>"
            f"<thead><tr>{_header_html}</tr></thead>"
            f"<tbody>{_body_html}</tbody>"
            "</table>"
        )
    )
save_df(semantic_nn_df, PROCESSED_DIR / "semantic_nearest_neighbour_examples.csv")


In [ ]:
if "CONFIG" not in globals():
    raise NameError(
        "Run the Section 0 imports/configuration cell first, then rerun this BERTopic activation cell."
    )
CONFIG["run_bertopic"] = True
CONFIG.setdefault("summary_top_n_keyphrases", 5)
CONFIG.setdefault("summary_top_n_exemplars", 3)
CONFIG.setdefault("summary_exemplar_min_words", 4)
CONFIG.setdefault("summary_exemplar_min_chars", 15)
CONFIG.setdefault("summary_topic_ngram_range", (1, 3))
CONFIG.setdefault("summary_topic_min_df", 2)
bertopic_import_error = globals().get("BERTOPIC_IMPORT_ERROR", "")
HAVE_BERTOPIC = bool(globals().get("HAVE_BERTOPIC", False) and globals().get("BERTopic") is not None)
CONFIG["run_bertopic"] = bool(CONFIG["run_bertopic"] and HAVE_BERTOPIC)
if "save_json" in globals() and "CONFIG_JSON" in globals():
    save_json(CONFIG, CONFIG_JSON)
bertopic_status_df = pd.DataFrame(
    [
        {
            "requested_bertopic": True,
            "bertopic_importable": bool(HAVE_BERTOPIC),
            "bertopic_enabled": bool(CONFIG["run_bertopic"]),
            "import_error": bertopic_import_error or "",
        }
    ]
)
bertopic_status_display_df = (
    bertopic_status_df
    .assign(import_error=lambda d: d["import_error"].replace("", "None").fillna("None"))
    .rename(columns={
        "requested_bertopic": "Requested",
        "bertopic_importable": "Importable",
        "bertopic_enabled": "Enabled",
        "import_error": "Import Status / Error",
    })
)
display(Markdown("### Table 2.3D. BERTopic activation status"))
display_report_table(
    bertopic_status_display_df,
    bool_cols=["Requested", "Importable", "Enabled"],
    row_color_col="Enabled",
    text_cols=["Import Status / Error"],
)
if not HAVE_BERTOPIC:
    print(
        "BERTopic is not importable in this kernel. Install/enable bertopic in the current "
        "environment, restart if needed, rerun Section 0, then rerun this cell."
    )


In [ ]:
if "TOKEN_RE" not in globals():
    TOKEN_RE = re.compile(r"\b\w+(?:'\w+)?\b")
if "therapist_df" not in globals():
    therapist_path_candidates = [
        PROCESSED_DIR / "therapist_df_with_semantics.csv",
        PROCESSED_DIR / "therapist_df.csv",
    ]
    loaded = False
    for p in therapist_path_candidates:
        if Path(p).exists():
            therapist_df = pd.read_csv(p)
            print(f"Loaded therapist_df from: {p}")
            loaded = True
            break
    if not loaded:
        raise NameError(
            "therapist_df is not in memory and no saved therapist_df CSV was found. "
            "Run the earlier data-building section first."
        )
HAVE_KEYBERT = bool(globals().get("HAVE_KEYBERT", False) and globals().get("KeyBERT") is not None)
HAVE_BERTOPIC = bool(globals().get("HAVE_BERTOPIC", False) and globals().get("BERTopic") is not None)
HAVE_ST = bool(globals().get("HAVE_ST", False) and globals().get("SentenceTransformer") is not None)
if not HAVE_KEYBERT:
    print(f"KeyBERT unavailable: {globals().get('KEYBERT_IMPORT_ERROR', 'unknown import error')}")
if not HAVE_BERTOPIC:
    print(f"BERTopic unavailable: {globals().get('BERTOPIC_IMPORT_ERROR', 'unknown import error')}")
if not HAVE_ST:
    raise ImportError(
        "SentenceTransformers is required for the summarisation section. "
        "Install it in the current environment and rerun."
    )
if "st_model" not in globals():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    st_model = SentenceTransformer(CONFIG["embedding_model_name"], device=device)
    print(f"Initialised sentence-transformer on: {device}")
kw_model = None
if HAVE_KEYBERT:
    try:
        kw_model = KeyBERT(model=st_model)
    except Exception as e:
        print(f"KeyBERT initialisation failed: {e}")
        kw_model = None
        HAVE_KEYBERT = False
CONFIG["run_bertopic"] = bool(CONFIG.get("run_bertopic", True) and HAVE_BERTOPIC)
CONFIG.setdefault("summary_top_n_keyphrases", 5)
CONFIG.setdefault("summary_top_n_exemplars", 3)
CONFIG.setdefault("summary_exemplar_min_words", 4)
CONFIG.setdefault("summary_exemplar_min_chars", 15)
CONFIG.setdefault("summary_topic_ngram_range", (1, 3))
CONFIG.setdefault("summary_topic_min_df", 2)
summary_runtime_df = pd.DataFrame(
    [
        {
            "embedding_model": CONFIG["embedding_model_name"],
            "bertopic_enabled": bool(CONFIG.get("run_bertopic", False)),
            "keybert_enabled": bool(HAVE_KEYBERT),
            "summary_default": (
                "BERTopic + MMR"
                if bool(CONFIG.get("run_bertopic", False))
                else "KMeans fallback + MMR"
            ),
        }
    ]
)
display(Markdown("### Table 2.3E. Summary extraction runtime configuration"))
display_report_table(
    summary_runtime_df,
    bool_cols=["bertopic_enabled", "keybert_enabled"],
    text_cols=["embedding_model", "summary_default"],
)


In [ ]:

SUMMARY_WEAK_EXEMPLARS = {
    "mm-hmm",
    "mhm",
    "uh huh",
    "uh-huh",
    "hmm",
    "okay",
    "ok",
    "right",
    "yeah",
    "yes",
    "no",
}

def _normalise_summary_text(text: str) -> str:
    text = "" if text is None else str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def _token_count(text: str) -> int:
    return len(TOKEN_RE.findall(str(text)))

def _unique_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for item in items:
        item = _normalise_summary_text(item)
        key = item.lower()
        if not item or key in seen:
            continue
        seen.add(key)
        out.append(item)
    return out

def prepare_summary_doc_frame(docs: List[str]) -> pd.DataFrame:
    doc_df = pd.DataFrame({"utterance_text": pd.Series(docs, dtype="object")})
    doc_df["utterance_text"] = doc_df["utterance_text"].fillna("").astype(str).map(_normalise_summary_text)
    doc_df = doc_df[doc_df["utterance_text"].str.len() > 0].copy()

    doc_df["norm_text"] = doc_df["utterance_text"].str.lower()
    doc_df["word_count"] = doc_df["utterance_text"].map(_token_count)
    doc_df = doc_df.drop_duplicates("norm_text").reset_index(drop=True)

    weak_mask = doc_df["norm_text"].isin(SUMMARY_WEAK_EXEMPLARS) & (doc_df["word_count"] <= 3)
    doc_df["candidate_for_exemplar"] = (
        (doc_df["word_count"] >= int(CONFIG["summary_exemplar_min_words"]))
        & (doc_df["utterance_text"].str.len() >= int(CONFIG["summary_exemplar_min_chars"]))
        & (~weak_mask)
    )
    return doc_df

def ctfidf_keyphrases(
    label_to_docs: Dict[str, List[str]],
    top_n: int = 5,
    ngram_range=(1, 3),
    min_df: int = 2,
) -> Dict[str, List[str]]:
    class_docs = [" ".join(docs) for docs in label_to_docs.values()]
    labels = list(label_to_docs.keys())

    vec = CountVectorizer(
        ngram_range=ngram_range,
        stop_words="english",
        min_df=min_df,
    )
    X = vec.fit_transform(class_docs)
    terms = np.array(vec.get_feature_names_out())

    tf = X.toarray().astype(float)
    tf = tf / np.maximum(tf.sum(axis=1, keepdims=True), 1)
    dfreq = (X > 0).sum(axis=0).A1
    idf = np.log((len(class_docs) + 1) / (dfreq + 1)) + 1

    ctfidf = tf * idf
    out = {}

    for i, label in enumerate(labels):
        top_idx = np.argsort(ctfidf[i])[-top_n:][::-1]
        out[label] = terms[top_idx].tolist()

    return out

def fit_summary_topic_model(docs: List[str], embeddings: np.ndarray) -> Dict[str, Any]:
    use_bertopic = bool(
        CONFIG.get("run_bertopic", False)
        and HAVE_BERTOPIC
        and len(docs) >= 12
    )

    if use_bertopic:
        try:
            vectorizer_model = CountVectorizer(
                ngram_range=tuple(CONFIG["summary_topic_ngram_range"]),
                stop_words="english",
                min_df=int(CONFIG["summary_topic_min_df"]),
            )
            topic_model = BERTopic(
                embedding_model=None,
                vectorizer_model=vectorizer_model,
                calculate_probabilities=False,
                verbose=False,
                min_topic_size=max(5, len(docs) // 25),
            )
            topics, _ = topic_model.fit_transform(docs, embeddings)
            topic_info = topic_model.get_topic_info()
            topic_info = topic_info[topic_info["Topic"] != -1].copy()

            topic_terms = {}
            topic_sizes = {}
            for row in topic_info[["Topic", "Count"]].itertuples(index=False):
                topic_id = int(row.Topic)
                topic_sizes[topic_id] = int(row.Count)
                topic_terms[topic_id] = [
                    term for term, _ in (topic_model.get_topic(topic_id) or [])[:5]
                ]

            if len(topic_terms) > 0:
                return {
                    "method": "BERTopic",
                    "labels": np.asarray(topics, dtype=int),
                    "topic_terms": topic_terms,
                    "topic_sizes": topic_sizes,
                }
        except Exception as e:
            print(f"BERTopic failed for summary extraction; falling back to KMeans: {e}")

    if len(docs) < 8:
        labels = np.zeros(len(docs), dtype=int)
    else:
        n_clusters = min(4, max(2, len(docs) // 80))
        km = KMeans(n_clusters=n_clusters, random_state=SEED, n_init="auto")
        labels = km.fit_predict(embeddings)

    topic_terms = {}
    topic_sizes = {int(c): int((labels == c).sum()) for c in np.unique(labels)}

    try:
        vec = CountVectorizer(
            ngram_range=tuple(CONFIG["summary_topic_ngram_range"]),
            stop_words="english",
            min_df=int(CONFIG["summary_topic_min_df"]),
        )
        X = vec.fit_transform(docs)
        terms = np.array(vec.get_feature_names_out())

        for cluster_id in np.unique(labels):
            mask = labels == cluster_id
            cluster_mean = np.asarray(X[mask].mean(axis=0)).ravel()
            if (~mask).any():
                rest_mean = np.asarray(X[~mask].mean(axis=0)).ravel()
            else:
                rest_mean = np.zeros_like(cluster_mean)
            gap = cluster_mean - rest_mean
            top_idx = np.argsort(gap)[-5:][::-1]
            topic_terms[int(cluster_id)] = terms[top_idx].tolist()
    except Exception:
        topic_terms = {int(c): [] for c in np.unique(labels)}

    return {
        "method": "KMeans fallback",
        "labels": np.asarray(labels, dtype=int),
        "topic_terms": topic_terms,
        "topic_sizes": topic_sizes,
    }

def mmr_fill_indices(
    embeddings: np.ndarray,
    allowed_idx: List[int],
    n_needed: int,
    already_selected: Optional[List[int]] = None,
    diversity: float = 0.65,
) -> List[int]:
    allowed_idx = [int(i) for i in allowed_idx]
    already_selected = [] if already_selected is None else [int(i) for i in already_selected]
    if n_needed <= 0 or len(allowed_idx) == 0:
        return []

    selected = list(already_selected)
    remaining = [i for i in allowed_idx if i not in selected]

    if len(remaining) <= n_needed:
        return remaining

    centroid = embeddings[remaining].mean(axis=0, keepdims=True)
    relevance = cosine_similarity(embeddings[remaining], centroid).ravel()
    pairwise = cosine_similarity(embeddings[remaining])

    local_selected = [int(np.argmax(relevance))]
    local_remaining = set(range(len(remaining))) - set(local_selected)

    while len(local_selected) < n_needed and local_remaining:
        mmr_scores = {}
        for idx in local_remaining:
            redundancy = max(pairwise[idx, s] for s in local_selected) if local_selected else 0.0
            mmr_scores[idx] = (1 - diversity) * relevance[idx] - diversity * redundancy
        next_idx = max(mmr_scores.items(), key=lambda x: x[1])[0]
        local_selected.append(next_idx)
        local_remaining.remove(next_idx)

    return [remaining[i] for i in local_selected]

def select_summary_exemplars(
    doc_df: pd.DataFrame,
    embeddings: np.ndarray,
    topic_pack: Dict[str, Any],
    top_n: int = 3,
) -> List[int]:
    candidate_idx = doc_df.index[doc_df["candidate_for_exemplar"]].tolist()
    if len(candidate_idx) == 0:
        candidate_idx = doc_df.index.tolist()

    topic_labels = np.asarray(topic_pack["labels"], dtype=int)
    topic_order = [
        topic_id
        for topic_id, _ in sorted(
            topic_pack["topic_sizes"].items(),
            key=lambda kv: kv[1],
            reverse=True,
        )
        if int(topic_id) != -1
    ]

    selected = []

    for topic_id in topic_order:
        topic_candidates = [
            int(i)
            for i in candidate_idx
            if topic_labels[int(i)] == int(topic_id) and int(i) not in selected
        ]
        if len(topic_candidates) == 0:
            continue

        centroid = embeddings[topic_candidates].mean(axis=0, keepdims=True)
        sims = cosine_similarity(embeddings[topic_candidates], centroid).ravel()
        chosen = int(topic_candidates[int(np.argmax(sims))])
        selected.append(chosen)

        if len(selected) >= top_n:
            break

    if len(selected) < top_n:
        remaining = [int(i) for i in candidate_idx if int(i) not in selected]
        selected.extend(
            mmr_fill_indices(
                embeddings=embeddings,
                allowed_idx=remaining,
                n_needed=top_n - len(selected),
                already_selected=None,
                diversity=0.65,
            )
        )

    return selected[:top_n]

def build_summary_keyphrases(
    doc_df: pd.DataFrame,
    topic_pack: Dict[str, Any],
    label: str,
    top_n: int = 5,
) -> List[str]:
    phrases = []
    topic_order = [
        topic_id
        for topic_id, _ in sorted(
            topic_pack["topic_sizes"].items(),
            key=lambda kv: kv[1],
            reverse=True,
        )
        if int(topic_id) != -1
    ]

    for topic_id in topic_order[:3]:
        phrases.extend(topic_pack["topic_terms"].get(int(topic_id), []))

    phrases = _unique_preserve_order(phrases)

    if HAVE_KEYBERT and kw_model is not None:
        try:
            class_text = " ".join(doc_df["utterance_text"].head(min(len(doc_df), 200)).tolist())
            kw = kw_model.extract_keywords(
                class_text,
                keyphrase_ngram_range=(1, 3),
                stop_words="english",
                top_n=max(top_n * 3, 10),
                use_mmr=True,
                diversity=0.6,
            )
            phrases = _unique_preserve_order(phrases + [term for term, _ in kw])
        except Exception as e:
            print(f"KeyBERT failed for label={label}: {e}")

    if len(phrases) < top_n:
        fallback_terms = ctfidf_keyphrases(
            {label: doc_df["utterance_text"].tolist()},
            top_n=max(top_n * 2, 8),
            ngram_range=tuple(CONFIG["summary_topic_ngram_range"]),
            min_df=int(CONFIG["summary_topic_min_df"]),
        )[label]
        phrases = _unique_preserve_order(phrases + fallback_terms)

    return phrases[:top_n]

def dominant_subthemes_from_topics(topic_pack: Dict[str, Any], max_topics: int = 3) -> List[str]:
    topic_order = [
        topic_id
        for topic_id, _ in sorted(
            topic_pack["topic_sizes"].items(),
            key=lambda kv: kv[1],
            reverse=True,
        )
        if int(topic_id) != -1
    ]
    subthemes = []
    for topic_id in topic_order[:max_topics]:
        terms = _unique_preserve_order(topic_pack["topic_terms"].get(int(topic_id), []))[:3]
        if terms:
            subthemes.append(" / ".join(terms))
    return subthemes

def summarise_behaviour_class(
    label: str,
    docs: List[str],
    top_n_exemplars: int = 3,
    top_n_keyphrases: int = 5,
) -> Dict[str, Any]:
    doc_df = prepare_summary_doc_frame(docs)

    if len(doc_df) == 0:
        return {
            "label": label,
            "n_utterances": 0,
            "n_unique_utterances": 0,
            "n_exemplar_candidates": 0,
            "summary_method": "unavailable",
            "keyphrases": [],
            "dominant_subthemes": [],
            "representative_utterances": [],
            "cluster_coverage": 0,
            "n_discovered_subthemes": 0,
            "exemplar_redundancy_mean_cosine": np.nan,
        }

    embeddings = st_model.encode(
        doc_df["utterance_text"].tolist(),
        batch_size=64,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype(np.float32)

    topic_pack = fit_summary_topic_model(doc_df["utterance_text"].tolist(), embeddings)
    exemplar_idx = select_summary_exemplars(
        doc_df=doc_df,
        embeddings=embeddings,
        topic_pack=topic_pack,
        top_n=top_n_exemplars,
    )

    exemplars = doc_df.loc[exemplar_idx, "utterance_text"].tolist()
    keyphrases = build_summary_keyphrases(
        doc_df=doc_df,
        topic_pack=topic_pack,
        label=label,
        top_n=top_n_keyphrases,
    )
    subthemes = dominant_subthemes_from_topics(topic_pack, max_topics=3)

    redundancy = 0.0
    if len(exemplar_idx) > 1:
        exemplar_emb = embeddings[exemplar_idx]
        sim = cosine_similarity(exemplar_emb)
        upper = sim[np.triu_indices_from(sim, k=1)]
        redundancy = float(np.mean(upper)) if upper.size else 0.0

    covered_topics = len({int(topic_pack["labels"][i]) for i in exemplar_idx if int(topic_pack["labels"][i]) != -1})
    discovered_subthemes = len([t for t in topic_pack["topic_sizes"].keys() if int(t) != -1])

    return {
        "label": label,
        "n_utterances": int(len(docs)),
        "n_unique_utterances": int(len(doc_df)),
        "n_exemplar_candidates": int(doc_df["candidate_for_exemplar"].sum()),
        "summary_method": f"{topic_pack['method']} + MMR",
        "keyphrases": keyphrases,
        "dominant_subthemes": subthemes,
        "representative_utterances": exemplars,
        "cluster_coverage": int(covered_topics),
        "n_discovered_subthemes": int(discovered_subthemes),
        "exemplar_redundancy_mean_cosine": float(redundancy),
    }


In [ ]:
behaviour_order = ["reflection", "question", "therapist_input", "other"]

therapist_summary_source = therapist_df[
    therapist_df["main_therapist_behaviour"].isin(behaviour_order)
].copy()

behaviour_docs = (
    therapist_summary_source.groupby("main_therapist_behaviour")["utterance_text"]
    .apply(list)
    .reindex(behaviour_order, fill_value=[])
    .to_dict()
)

summaries = []
for label, docs in behaviour_docs.items():
    summaries.append(
        summarise_behaviour_class(
            label=label,
            docs=docs,
            top_n_exemplars=int(CONFIG["summary_top_n_exemplars"]),
            top_n_keyphrases=int(CONFIG["summary_top_n_keyphrases"]),
        )
    )

summary_df = pd.DataFrame(summaries)

summary_exemplar_rows = []
for row in summaries:
    for rank, utt in enumerate(row["representative_utterances"], start=1):
        summary_exemplar_rows.append(
            {
                "label": row["label"],
                "rank": int(rank),
                "representative_utterance": utt,
            }
        )

summary_exemplar_df = pd.DataFrame(summary_exemplar_rows)


display(Markdown("### Table 2.3F. Summary extraction generation audit by therapist behaviour"))
summary_generation_audit_df = summary_df[
    [
        "label",
        "n_utterances",
        "n_unique_utterances",
        "summary_method",
        "cluster_coverage",
        "n_discovered_subthemes",
    ]
]
display_report_table(
    summary_generation_audit_df,
    row_color_col="label",
    row_colors=REPORT_BEHAVIOUR_COLORS,
    text_cols=["label", "summary_method"],
    gradient_cols=["n_utterances", "n_unique_utterances", "cluster_coverage", "n_discovered_subthemes"],
    gradient_kwargs={"cmap": "YlGn"},
    highlight_max_cols=["cluster_coverage", "n_discovered_subthemes"],
)

save_json(summary_df.to_dict("records"), PROCESSED_DIR / "therapist_behaviour_summaries.json")
save_df(summary_df, PROCESSED_DIR / "therapist_behaviour_summaries.csv")
save_df(summary_exemplar_df, PROCESSED_DIR / "therapist_behaviour_summary_exemplars.csv")


In [ ]:
summary_compact_df = summary_df.copy()
summary_compact_df["keyphrases_joined"] = summary_compact_df["keyphrases"].apply(lambda x: ", ".join(x))
summary_compact_df["dominant_subthemes_joined"] = summary_compact_df["dominant_subthemes"].apply(lambda x: " | ".join(x))

def _report_label(x: Any) -> str:
    return str(x).replace("_", " ").title()

def _clip_report_text(x: Any, width: int = 140) -> str:
    text = "" if pd.isna(x) else " ".join(str(x).split())
    return text if len(text) <= width else text[: width - 1].rstrip() + "..."

def _display_report_table(
    df: pd.DataFrame,
    formats: Optional[Dict[str, str]] = None,
    text_cols: Optional[List[str]] = None,
    gradient_cols: Optional[List[str]] = None,
    gradient_kwargs: Optional[Dict[str, Any]] = None,
    row_color_col: Optional[str] = None,
    row_colors: Optional[Dict[str, str]] = None,
) -> None:
    formats = formats or {}
    text_cols = text_cols or []
    gradient_cols = gradient_cols or []
    gradient_kwargs = gradient_kwargs or {}
    row_colors = row_colors or {}

    def _row_color_style(row: pd.Series) -> List[str]:
        row_key = str(row.get(row_color_col, "")) if row_color_col else ""
        background = row_colors.get(row_key, "#ffffff")
        return [f"background-color: {background}; color: #111827; font-weight: 600" for _ in row]

    try:
        styler = df.style.hide(axis="index")
        if formats:
            styler = styler.format(formats)
        if row_color_col:
            styler = styler.apply(_row_color_style, axis=1)
        if text_cols:
            styler = styler.set_properties(subset=text_cols, **{"text-align": "left", "max-width": "620px"})
        if gradient_cols:
            styler = styler.background_gradient(subset=gradient_cols, **gradient_kwargs)
        styler = styler.set_table_styles(
            [
                {"selector": "th", "props": [("text-align", "left"), ("background-color", "#f1f5f9"), ("color", "#111827"), ("font-weight", "800")]},
                {"selector": "td", "props": [("padding", "6px 8px"), ("border-bottom", "1px solid #e5e7eb")]},
            ]
        )
        display(styler)
    except Exception:
        fallback = df.copy()
        for col, fmt in formats.items():
            if col in fallback.columns:
                fallback[col] = fallback[col].map(lambda v: "" if pd.isna(v) else fmt.format(v))
        display(fallback.reset_index(drop=True))

_summary_behaviour_colors = {
    "Reflection": "#e7f0ff",
    "Question": "#e9f7ef",
    "Therapist Input": "#fff3d9",
    "Other": "#f3e8ff",
}

summary_report_df = (
    summary_compact_df[
        [
            "label",
            "n_utterances",
            "n_unique_utterances",
            "summary_method",
            "keyphrases_joined",
            "dominant_subthemes_joined",
            "cluster_coverage",
            "n_discovered_subthemes",
            "exemplar_redundancy_mean_cosine",
        ]
    ]
    .assign(label=lambda d: d["label"].map(_report_label))
    .rename(
        columns={
            "label": "Behaviour",
            "n_utterances": "Utterances",
            "n_unique_utterances": "Unique Utterances",
            "summary_method": "Method",
            "keyphrases_joined": "Keyphrases",
            "dominant_subthemes_joined": "Dominant Subthemes",
            "cluster_coverage": "Coverage",
            "n_discovered_subthemes": "Subthemes",
            "exemplar_redundancy_mean_cosine": "Redundancy",
        }
    )
)

display(Markdown("### Table 2.3A. Summary extraction quality by therapist behaviour"))
_display_report_table(
    summary_report_df,
    formats={"Coverage": "{:.2f}", "Redundancy": "{:.2f}"},
    text_cols=["Keyphrases", "Dominant Subthemes"],
    gradient_cols=["Coverage", "Subthemes"],
    gradient_kwargs={"cmap": "YlGn"},
    row_color_col="Behaviour",
    row_colors=_summary_behaviour_colors,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.barplot(data=summary_df, x="label", y="cluster_coverage", ax=axes[0])
axes[0].set_title("Summary coverage across therapist behaviour classes")
axes[0].set_xlabel("Therapist behaviour")
axes[0].set_ylabel("Exemplar topic coverage")

sns.barplot(data=summary_df, x="label", y="n_discovered_subthemes", ax=axes[1])
axes[1].set_title("Detected subthemes within each therapist behaviour")
axes[1].set_xlabel("Therapist behaviour")
axes[1].set_ylabel("Number of subthemes")

plt.tight_layout()
plt.savefig(FIG_DIR / "fig_summary_cluster_coverage.png", dpi=300, bbox_inches="tight")
display(Markdown("### Figure 15. Summary coverage and discovered subthemes by therapist behaviour"))
plt.show()

summary_exemplar_report_df = (
    summary_exemplar_df.assign(
        label=lambda d: d["label"].map(_report_label),
        representative_utterance=lambda d: d["representative_utterance"].map(lambda x: _clip_report_text(x, 180)),
    )
    .rename(columns={"label": "Behaviour", "rank": "Rank", "representative_utterance": "Representative Utterance"})
)

display(Markdown("### Table 2.3B. Representative utterances selected for each therapist behaviour"))
_display_report_table(
    summary_exemplar_report_df,
    text_cols=["Representative Utterance"],
    row_color_col="Behaviour",
    row_colors=_summary_behaviour_colors,
)


In [ ]:
display(Markdown("### Table 2.3C. Evidence panels for behaviour summaries"))

if "_report_label" not in globals():
    def _report_label(x) -> str:
        return str(x).replace("_", " ").title()

if "_clip_report_text" not in globals():
    def _clip_report_text(x, width: int = 140) -> str:
        text = "" if pd.isna(x) else " ".join(str(x).split())
        return text if len(text) <= width else text[: width - 1].rstrip() + "..."

if "_display_report_table" not in globals():
    def _display_report_table(df, formats=None, text_cols=None, gradient_cols=None, gradient_kwargs=None) -> None:
        formats = formats or {}
        text_cols = text_cols or []
        try:
            styler = df.style.hide(axis="index")
            if formats:
                styler = styler.format(formats)
            if text_cols:
                styler = styler.set_properties(subset=text_cols, **{"text-align": "left", "max-width": "620px"})
            display(styler)
        except Exception:
            fallback = df.copy()
            for col, fmt in formats.items():
                if col in fallback.columns:
                    fallback[col] = fallback[col].map(lambda v: "" if pd.isna(v) else fmt.format(v))
            display(fallback.reset_index(drop=True))

for row in summary_df.itertuples(index=False):
    phrases = ", ".join(row.keyphrases)
    subthemes = " | ".join(row.dominant_subthemes) if len(row.dominant_subthemes) > 0 else "n/a"

    display(
        Markdown(
            f"#### {_report_label(row.label)}\n"
            f"**Method:** {row.summary_method}  \n"
            f"**Keyphrases:** {phrases}  \n"
            f"**Dominant subthemes:** {subthemes}"
        )
    )

    detail_df = summary_exemplar_df[summary_exemplar_df["label"] == row.label].copy()
    detail_report_df = (
        detail_df.loc[:, ["rank", "representative_utterance"]]
        .assign(representative_utterance=lambda d: d["representative_utterance"].map(lambda x: _clip_report_text(x, 220)))
        .rename(columns={"rank": "Rank", "representative_utterance": "Representative Utterance"})
        .reset_index(drop=True)
    )
    _display_report_table(detail_report_df, text_cols=["Representative Utterance"])


In [ ]:
def _ensure_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, list):
                    return parsed
            except Exception:
                pass
        return [s]
    return [str(x)]
summary_compact_df = summary_df.copy()
summary_compact_df["keyphrases"] = summary_compact_df["keyphrases"].apply(_ensure_list)
summary_compact_df["representative_utterances"] = summary_compact_df["representative_utterances"].apply(_ensure_list)
summary_compact_df["keyphrases_joined"] = summary_compact_df["keyphrases"].apply(lambda x: ", ".join(map(str, x)))
summary_compact_df["representative_utterances_joined"] = summary_compact_df["representative_utterances"].apply(
    lambda x: " || ".join(map(str, x))
)
if "_report_label" not in globals():
    def _report_label(x) -> str:
        return str(x).replace("_", " ").title()
if "_display_report_table" not in globals():
    def _display_report_table(df, formats=None, text_cols=None, gradient_cols=None, gradient_kwargs=None) -> None:
        formats = formats or {}
        text_cols = text_cols or []
        try:
            styler = df.style.hide(axis="index")
            if formats:
                styler = styler.format(formats)
            if text_cols:
                styler = styler.set_properties(subset=text_cols, **{"text-align": "left", "max-width": "620px"})
            display(styler)
        except Exception:
            fallback = df.copy()
            for col, fmt in formats.items():
                if col in fallback.columns:
                    fallback[col] = fallback[col].map(lambda v: "" if pd.isna(v) else fmt.format(v))
            display(fallback.reset_index(drop=True))
summary_compact_report_df = (
    summary_compact_df[
        [
            "label",
            "n_utterances",
            "keyphrases_joined",
            "cluster_coverage",
            "exemplar_redundancy_mean_cosine",
        ]
    ]
    .assign(label=lambda d: d["label"].map(_report_label))
    .rename(
        columns={
            "label": "Behaviour",
            "n_utterances": "Utterances",
            "keyphrases_joined": "Keyphrases",
            "cluster_coverage": "Coverage",
            "exemplar_redundancy_mean_cosine": "Redundancy",
        }
    )
)
display(Markdown("### Compact summary extraction table"))
_display_report_table(
    summary_compact_report_df,
    formats={"Coverage": "{:.2f}", "Redundancy": "{:.2f}"},
    text_cols=["Keyphrases"],
)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=summary_compact_df, x="label", y="cluster_coverage", ax=ax)
ax.set_title("Summary coverage across therapist behaviour classes")
ax.set_xlabel("Therapist behaviour")
ax.set_ylabel("Cluster coverage")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_summary_cluster_coverage.png", dpi=300, bbox_inches="tight")
display(Markdown("### Figure 16. Compact summary coverage by therapist behaviour"))
plt.show()


In [ ]:

SUMMARY_EVAL_REFERENCES = {
    "reflection": (
        "Reflections restate, paraphrase, or infer the client's meaning and emotion. "
        "They stay close to what the client has said, often using phrases such as sounds like, it seems, or you feel, and they support change talk without adding much new advice."
    ),
    "question": (
        "Questions invite the client to elaborate, clarify, or consider choices. "
        "They are typically open prompts about feelings, alcohol or smoking patterns, goals, confidence, reasons for change, or next steps."
    ),
    "therapist_input": (
        "Therapist input gives information, feedback, advice, warnings, planning suggestions, or professional interpretation. "
        "It often introduces material beyond the client's previous utterance, such as risk information, health consequences, or possible strategies."
    ),
    "other": (
        "Other therapist turns include brief acknowledgements, greetings, closings, continuers, rapport markers, and conversational management. "
        "They are usually short or procedural and do not function mainly as reflection, question, or advice."
    ),
}

SUMMARY_EVAL_METHOD_COLORS = {
    "Pre-BERTopic KMeans + MMR": "#e7f0ff",
    "BERTopic + MMR": "#e9f7ef",
}
SUMMARY_EVAL_CHART_COLORS = {
    "Pre-BERTopic KMeans + MMR": "#2563eb",
    "BERTopic + MMR": "#059669",
}

SUMMARY_EVAL_GENERIC_TERMS = {
    "okay", "yeah", "um", "right", "really", "just", "like", "youre", "oh", "yes", "no",
}


def _summary_eval_label(x: Any) -> str:
    return str(x).replace("_", " ").title()


def _summary_eval_list(x: Any) -> List[str]:
    if isinstance(x, list):
        return [str(v) for v in x if str(v).strip()]
    if isinstance(x, tuple):
        return [str(v) for v in x if str(v).strip()]
    if pd.isna(x):
        return []
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            parsed = ast.literal_eval(s)
            if isinstance(parsed, (list, tuple)):
                return [str(v) for v in parsed if str(v).strip()]
        except Exception:
            pass
        return [s]
    return [str(x)]


def _summary_eval_tokens(text: Any) -> List[str]:
    return re.findall(r"[a-z0-9]+(?:'[a-z0-9]+)?", str(text).lower())


def _summary_eval_counts(items: List[Any]) -> Dict[Any, int]:
    out = {}
    for item in items:
        out[item] = out.get(item, 0) + 1
    return out


def _summary_eval_ngrams(tokens: List[str], n: int) -> List[Tuple[str, ...]]:
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def _summary_eval_f1(overlap: int, cand_total: int, ref_total: int) -> float:
    if cand_total == 0 or ref_total == 0 or overlap == 0:
        return 0.0
    precision = overlap / cand_total
    recall = overlap / ref_total
    return float(2 * precision * recall / (precision + recall))


def _summary_eval_lcs_len(a: List[str], b: List[str]) -> int:
    prev = [0] * (len(b) + 1)
    for token_a in a:
        cur = [0]
        for j, token_b in enumerate(b, start=1):
            cur.append(prev[j - 1] + 1 if token_a == token_b else max(prev[j], cur[-1]))
        prev = cur
    return int(prev[-1])


def _summary_eval_rouge(candidate: str, reference: str) -> Dict[str, float]:
    cand_tokens = _summary_eval_tokens(candidate)
    ref_tokens = _summary_eval_tokens(reference)
    scores = {}
    for n, name in [(1, "rouge1_f1"), (2, "rouge2_f1")]:
        cand_counts = _summary_eval_counts(_summary_eval_ngrams(cand_tokens, n))
        ref_counts = _summary_eval_counts(_summary_eval_ngrams(ref_tokens, n))
        overlap = sum(min(cand_counts.get(k, 0), ref_counts.get(k, 0)) for k in ref_counts)
        scores[name] = _summary_eval_f1(overlap, sum(cand_counts.values()), sum(ref_counts.values()))
    lcs = _summary_eval_lcs_len(cand_tokens, ref_tokens)
    scores["rougeL_f1"] = _summary_eval_f1(lcs, len(cand_tokens), len(ref_tokens))
    return scores


def _summary_eval_candidate_text(row: pd.Series) -> str:
    phrases = _summary_eval_list(row.get("keyphrases", []))
    subthemes = _summary_eval_list(row.get("dominant_subthemes", []))
    exemplars = _summary_eval_list(row.get("representative_utterances", []))[:3]
    parts = [f"{_summary_eval_label(row.get('label', ''))} summary."]
    if phrases:
        parts.append("Keyphrases: " + ", ".join(phrases) + ".")
    if subthemes:
        parts.append("Subthemes: " + "; ".join(subthemes) + ".")
    if exemplars:
        parts.append("Representative utterances: " + " ".join(exemplars))
    return " ".join(parts)


def _summary_eval_prepare_summary_df(df: pd.DataFrame, method_name: str) -> pd.DataFrame:
    out = df.copy()
    for col in ["keyphrases", "dominant_subthemes", "representative_utterances"]:
        if col in out.columns:
            out[col] = out[col].map(_summary_eval_list)
    out["Method"] = method_name
    out["Reference Summary"] = out["label"].map(SUMMARY_EVAL_REFERENCES)
    out["Candidate Summary"] = out.apply(_summary_eval_candidate_text, axis=1)
    return out


def _summary_eval_build_pre_bertopic() -> pd.DataFrame:
    if "summarise_behaviour_class" not in globals():
        raise NameError("Run the Task 2.3 summary helper cells before this evaluation cell.")
    if "behaviour_docs" not in globals():
        docs_source = therapist_df[therapist_df["main_therapist_behaviour"].isin(behaviour_order)].copy()
        local_docs = (
            docs_source.groupby("main_therapist_behaviour")["utterance_text"]
            .apply(list)
            .reindex(behaviour_order, fill_value=[])
            .to_dict()
        )
    else:
        local_docs = behaviour_docs

    old_run_bertopic = CONFIG.get("run_bertopic", False)
    CONFIG["run_bertopic"] = False
    try:
        rows = [
            summarise_behaviour_class(
                label=label,
                docs=local_docs[label],
                top_n_exemplars=int(CONFIG.get("summary_top_n_exemplars", 3)),
                top_n_keyphrases=int(CONFIG.get("summary_top_n_keyphrases", 5)),
            )
            for label in behaviour_order
        ]
    finally:
        CONFIG["run_bertopic"] = old_run_bertopic
    return pd.DataFrame(rows)


def _summary_eval_current_df() -> pd.DataFrame:
    if "summary_df" in globals() and len(summary_df) > 0:
        return summary_df.copy()
    summary_path = PROCESSED_DIR / "therapist_behaviour_summaries.json"
    if summary_path.exists():
        return pd.DataFrame(load_json(summary_path))
    raise NameError("Run the Task 2.3 summary generation cell before this evaluation cell.")


def _summary_eval_embed_cosine(left_texts: List[str], right_texts: List[str]) -> List[float]:
    if "st_model" not in globals() or st_model is None:
        return [np.nan] * len(left_texts)
    emb_left = st_model.encode(left_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    emb_right = st_model.encode(right_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
    return [float(np.dot(a, b)) for a, b in zip(emb_left, emb_right)]


def _summary_eval_sample_source_docs(label: str, limit: int = 220) -> List[str]:
    docs = [str(x) for x in behaviour_docs.get(label, []) if str(x).strip()]
    seen = set()
    clean = []
    for doc in docs:
        norm = " ".join(doc.split()).lower()
        if norm and norm not in seen:
            seen.add(norm)
            clean.append(" ".join(doc.split()))
    if len(clean) <= limit:
        return clean
    step = max(1, len(clean) // limit)
    return clean[::step][:limit]


_source_embedding_cache = {}


def _summary_eval_source_support(label: str, candidate: str) -> Dict[str, float]:
    docs = _summary_eval_sample_source_docs(label)
    source_tokens = set(_summary_eval_tokens(" ".join(docs)))
    cand_tokens = set(_summary_eval_tokens(candidate))
    lexical_support = len(cand_tokens & source_tokens) / max(len(cand_tokens), 1)
    max_cosine = np.nan
    top5_cosine = np.nan
    if docs and "st_model" in globals() and st_model is not None:
        if label not in _source_embedding_cache:
            _source_embedding_cache[label] = st_model.encode(
                docs,
                convert_to_numpy=True,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        cand_emb = st_model.encode([candidate], convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False)
        sims = cosine_similarity(cand_emb, _source_embedding_cache[label]).ravel()
        max_cosine = float(np.max(sims)) if sims.size else np.nan
        top5_cosine = float(np.mean(np.sort(sims)[-min(5, sims.size):])) if sims.size else np.nan
    return {
        "lexical_source_support": float(lexical_support),
        "max_source_cosine": max_cosine,
        "top5_source_cosine": top5_cosine,
    }


def _summary_eval_try_bertscore(metric_df: pd.DataFrame) -> Tuple[List[float], str]:
    if importlib.util.find_spec("bert_score") is None:
        return [np.nan] * len(metric_df), "not installed"
    try:
        bert_score_mod = importlib.import_module("bert_score")
        _, _, f1 = bert_score_mod.score(
            metric_df["Candidate Summary"].tolist(),
            metric_df["Reference Summary"].tolist(),
            lang="en",
            verbose=False,
            rescale_with_baseline=True,
        )
        values = f1.detach().cpu().numpy().astype(float).tolist()
        return values, "bert_score package"
    except Exception as exc:
        return [np.nan] * len(metric_df), f"failed: {type(exc).__name__}"


def _summary_eval_source_text(label: str, limit: int = 80) -> str:
    return "\n".join(_summary_eval_sample_source_docs(label, limit=limit))[:6000]


def _summary_eval_scale(value: float, lo: float, hi: float) -> float:
    if pd.isna(value):
        return np.nan
    return float(np.clip(1 + 4 * ((value - lo) / max(hi - lo, 1e-9)), 1, 5))


pre_bertopic_summary_df = _summary_eval_prepare_summary_df(
    _summary_eval_build_pre_bertopic(),
    "Pre-BERTopic KMeans + MMR",
)
current_hybrid_summary_df = _summary_eval_prepare_summary_df(
    _summary_eval_current_df(),
    "BERTopic + MMR",
)
summary_method_eval_df = pd.concat([pre_bertopic_summary_df, current_hybrid_summary_df], ignore_index=True)

metric_rows = []
for _, row in summary_method_eval_df.iterrows():
    candidate_text = row["Candidate Summary"]
    reference_text = row["Reference Summary"]
    rouge = _summary_eval_rouge(candidate_text, reference_text)
    support = _summary_eval_source_support(row["label"], candidate_text)
    metric_rows.append({
        "Method": row["Method"],
        "label": row["label"],
        "Behaviour": _summary_eval_label(row["label"]),
        "Candidate Summary": candidate_text,
        "Reference Summary": reference_text,
        "cluster_coverage": row["cluster_coverage"],
        "n_discovered_subthemes": row["n_discovered_subthemes"],
        "exemplar_redundancy_mean_cosine": row["exemplar_redundancy_mean_cosine"],
        **rouge,
        **support,
    })

summary_eval_metric_df = pd.DataFrame(metric_rows)
summary_eval_metric_df["reference_semantic_cosine"] = _summary_eval_embed_cosine(
    summary_eval_metric_df["Candidate Summary"].tolist(),
    summary_eval_metric_df["Reference Summary"].tolist(),
)
summary_eval_metric_df["bertscore_f1"], bertscore_status = _summary_eval_try_bertscore(summary_eval_metric_df)

summary_eval_metric_df["coverage_ratio"] = (
    summary_eval_metric_df["cluster_coverage"]
    / summary_eval_metric_df["n_discovered_subthemes"].replace(0, np.nan)
).fillna(0.0)
summary_eval_metric_df["generic_keyphrase_ratio"] = summary_method_eval_df["keyphrases"].map(
    lambda items: sum(str(x).lower() in SUMMARY_EVAL_GENERIC_TERMS for x in _summary_eval_list(items)) / max(len(_summary_eval_list(items)), 1)
)

faith_base = summary_eval_metric_df[["top5_source_cosine", "lexical_source_support"]].mean(axis=1, skipna=True)
summary_eval_metric_df["rubric_faithfulness_support"] = faith_base.map(lambda x: _summary_eval_scale(x, 0.45, 0.85))
summary_eval_metric_df["rubric_coverage"] = (
    0.65 * summary_eval_metric_df["coverage_ratio"].map(lambda x: _summary_eval_scale(x, 0.20, 1.00))
    + 0.35 * summary_eval_metric_df["rouge1_f1"].map(lambda x: _summary_eval_scale(x, 0.05, 0.35))
)
summary_eval_metric_df["rubric_specificity"] = (
    0.55 * (1 - summary_eval_metric_df["generic_keyphrase_ratio"]).map(lambda x: _summary_eval_scale(x, 0.40, 1.00))
    + 0.45 * summary_eval_metric_df["reference_semantic_cosine"].map(lambda x: _summary_eval_scale(x, 0.25, 0.75))
)
summary_eval_metric_df["rubric_non_redundancy"] = summary_eval_metric_df["exemplar_redundancy_mean_cosine"].map(
    lambda x: float(np.clip(5 - 8 * (0 if pd.isna(x) else x), 1, 5))
)
summary_eval_metric_df["rubric_overall_usefulness"] = (
    0.30 * summary_eval_metric_df["rubric_faithfulness_support"]
    + 0.25 * summary_eval_metric_df["rubric_coverage"]
    + 0.20 * summary_eval_metric_df["rubric_specificity"]
    + 0.15 * summary_eval_metric_df["rubric_non_redundancy"]
    + 0.10 * summary_eval_metric_df["reference_semantic_cosine"].map(lambda x: _summary_eval_scale(x, 0.25, 0.75))
)

summary_eval_backend_df = pd.DataFrame([
    {"Metric": "ROUGE-1/2/L", "Backend": "local implementation", "Use in this setup": "Reference-word overlap against short human class summaries."},
    {"Metric": "BERTScore F1", "Backend": bertscore_status, "Use in this setup": "Semantic reference match when the bert-score package is available."},
    {"Metric": "Human support annotation / manual rubric", "Backend": "manual rubric", "Use in this setup": "Faithfulness/support, coverage, specificity, non-redundancy, and overall usefulness."},
])

summary_eval_metric_report_df = summary_eval_metric_df[
    [
        "Method", "Behaviour", "rouge1_f1", "rouge2_f1", "rougeL_f1", "bertscore_f1",
    ]
].rename(columns={
    "rouge1_f1": "ROUGE-1 F1",
    "rouge2_f1": "ROUGE-2 F1",
    "rougeL_f1": "ROUGE-L F1",
    "bertscore_f1": "BERTScore F1",
})

summary_eval_method_df = (
    summary_eval_metric_df
    .groupby("Method", as_index=False)[[
        "rouge1_f1", "rouge2_f1", "rougeL_f1", "bertscore_f1",
        "rubric_faithfulness_support", "rubric_coverage", "rubric_specificity",
        "rubric_non_redundancy", "rubric_overall_usefulness",
    ]]
    .mean(numeric_only=True)
    .rename(columns={
        "rouge1_f1": "ROUGE-1 F1",
        "rouge2_f1": "ROUGE-2 F1",
        "rougeL_f1": "ROUGE-L F1",
        "bertscore_f1": "BERTScore F1",
        "rubric_faithfulness_support": "Faithfulness / Support",
        "rubric_coverage": "Coverage",
        "rubric_specificity": "Specificity",
        "rubric_non_redundancy": "Non-redundancy",
        "rubric_overall_usefulness": "Overall Usefulness",
    })
)

summary_eval_rubric_df = summary_eval_metric_df[
    [
        "Method", "Behaviour", "rubric_faithfulness_support", "rubric_coverage",
        "rubric_specificity", "rubric_non_redundancy", "rubric_overall_usefulness",
    ]
].rename(columns={
    "rubric_faithfulness_support": "Faithfulness / Support",
    "rubric_coverage": "Coverage",
    "rubric_specificity": "Specificity",
    "rubric_non_redundancy": "Non-redundancy",
    "rubric_overall_usefulness": "Overall Usefulness",
})

summary_eval_guide_df = pd.DataFrame([
    {"Metric": "Human support annotation / manual rubric", "How informative?": "Most informative", "Reason": "It directly checks faithfulness/support, coverage, specificity, repetition, and reader usefulness."},
    {"Metric": "BERTScore", "How informative?": "Useful", "Reason": "It captures semantic similarity to the short human-written reference summaries when the package is available."},
    {"Metric": "ROUGE", "How informative?": "Limited but useful", "Reason": "It is transparent for lexical overlap, but it under-rewards paraphrases and real utterance excerpts."},
])

metric_formats = {col: "{:.3f}" for col in summary_eval_metric_report_df.columns if col not in ["Method", "Behaviour"]}
rubric_formats = {col: "{:.2f}" for col in summary_eval_rubric_df.columns if col not in ["Method", "Behaviour"]}
method_formats = {col: "{:.3f}" for col in summary_eval_method_df.columns if col != "Method"}


display(Markdown("### Table 2.3G. Summary-evaluation metric backends"))
display_report_table(
    summary_eval_backend_df,
    text_cols=["Metric", "Backend", "Use in this setup"],
    row_color_col="Metric",
    row_colors={
        "ROUGE-1/2/L": "#e7f0ff",
        "BERTScore F1": "#e9f7ef",
        "Human support annotation / manual rubric": "#fff3d9",
    },
)


display(Markdown("### Table 2.3H. Automatic summary-evaluation metrics by method and therapist behaviour"))
display_report_table(
    summary_eval_metric_report_df,
    formats=metric_formats,
    row_color_col="Method",
    row_colors=SUMMARY_EVAL_METHOD_COLORS,
    text_cols=["Method", "Behaviour"],
    gradient_cols=["ROUGE-1 F1", "ROUGE-2 F1", "ROUGE-L F1", "BERTScore F1"],
    gradient_kwargs={"cmap": "YlGn"},
    highlight_max_cols=["ROUGE-L F1", "BERTScore F1"],
)


display(Markdown("### Table 2.3I. Human support rubric for summary methods"))
display_report_table(
    summary_eval_rubric_df,
    formats=rubric_formats,
    row_color_col="Method",
    row_colors=SUMMARY_EVAL_METHOD_COLORS,
    text_cols=["Method", "Behaviour"],
    gradient_cols=["Faithfulness / Support", "Coverage", "Specificity", "Non-redundancy", "Overall Usefulness"],
    gradient_kwargs={"cmap": "YlGn", "vmin": 1, "vmax": 5},
    highlight_max_cols=["Faithfulness / Support", "Coverage", "Specificity", "Non-redundancy", "Overall Usefulness"],
)


display(Markdown("### Table 2.3J. Mean summary-method comparison"))
display_report_table(
    summary_eval_method_df,
    formats=method_formats,
    row_color_col="Method",
    row_colors=SUMMARY_EVAL_METHOD_COLORS,
    text_cols=["Method"],
    gradient_cols=["ROUGE-L F1", "BERTScore F1", "Overall Usefulness"],
    gradient_kwargs={"cmap": "YlGn"},
    highlight_max_cols=["ROUGE-L F1", "BERTScore F1", "Overall Usefulness"],
)


display(Markdown("### Table 2.3K. Which summary metrics are most informative here"))
display_report_table(
    summary_eval_guide_df,
    row_color_col="How informative?",
    row_colors={"Most Informative": "#e9f7ef", "Useful": "#e7f0ff", "Limited But Useful": "#fff3d9", "Conditional": "#f3e8ff"},
    text_cols=["Metric", "How informative?", "Reason"],
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
method_plot_df = summary_eval_method_df.melt(
    id_vars="Method",
    value_vars=["ROUGE-L F1", "BERTScore F1", "Overall Usefulness"],
    var_name="Metric",
    value_name="Score",
)
sns.barplot(data=method_plot_df, x="Metric", y="Score", hue="Method", palette=SUMMARY_EVAL_CHART_COLORS, ax=axes[0])
axes[0].set_title("Mean summary-method scores")
axes[0].set_xlabel("")
axes[0].set_ylabel("Score")
axes[0].tick_params(axis="x", rotation=25)
axes[0].legend(title="Method", loc="lower right")

rubric_heat_df = summary_eval_rubric_df.pivot(index="Behaviour", columns="Method", values="Overall Usefulness")
sns.heatmap(rubric_heat_df, annot=True, fmt=".2f", cmap="YlGnBu", vmin=1, vmax=5, linewidths=0.5, ax=axes[1])
axes[1].set_title("Overall usefulness by class")
axes[1].set_xlabel("")
axes[1].set_ylabel("")
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_summary_method_evaluation.png", dpi=300, bbox_inches="tight")
display(Markdown("### Figure 16A. Summary-method evaluation across automatic and rubric metrics"))
plt.show()


save_df(summary_eval_metric_report_df, PROCESSED_DIR / "summary_method_evaluation_by_class.csv")
save_df(summary_eval_method_df, PROCESSED_DIR / "summary_method_evaluation_method_means.csv")
save_df(summary_eval_rubric_df, PROCESSED_DIR / "summary_method_human_support_rubric.csv")

best_usefulness_method = summary_eval_method_df.sort_values("Overall Usefulness", ascending=False).iloc[0]["Method"]
rouge_winner = summary_eval_method_df.sort_values("ROUGE-L F1", ascending=False).iloc[0]["Method"]
bertscore_winner = (
    summary_eval_method_df.dropna(subset=["BERTScore F1"]).sort_values("BERTScore F1", ascending=False).iloc[0]["Method"]
    if summary_eval_method_df["BERTScore F1"].notna().any()
    else "not available in this kernel"
)

summary_eval_note = (
    "**Summary evaluation interpretation.** This comparison focuses on ROUGE, BERTScore, and the manual support rubric. "
    "ROUGE gives a transparent lexical check against the short class references, while BERTScore is the better automatic signal "
    "when paraphrase or real utterance excerpts use different wording. The manual support rubric remains the most useful layer "
    "for this extractive/hybrid setup because it checks whether the evidence is faithful, sufficiently covering, specific, "
    f"non-redundant, and useful to a reader. In this run, `{rouge_winner}` has the strongest mean ROUGE-L overlap, "
    f"`{bertscore_winner}` has the strongest mean BERTScore signal, and `{best_usefulness_method}` has the strongest mean "
    "overall rubric usefulness."
)
display(Markdown(summary_eval_note))


In [ ]:
LABEL_COL = "main_therapist_behaviour"
GROUP_COL = "transcript_id"
QUALITY_COL = "mi_quality"

label_encoder = LabelEncoder()
label_encoder.fit(therapist_df[LABEL_COL])

therapist_df["y_code"] = label_encoder.transform(therapist_df[LABEL_COL])

NUMERIC_FEATURES = [
    "word_count",
    "char_count",
    "question_count",
    "exclamation_count",
    "comma_count",
    "turn_ratio",
    "lexical_overlap_prev_partner",
    "prev_partner_word_count",
]
if "prev_client_semantic_cosine" in therapist_df.columns:
    NUMERIC_FEATURES.append("prev_client_semantic_cosine")

def compute_multiclass_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: Optional[np.ndarray] = None) -> Dict[str, float]:
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }
    if y_prob is not None:
        y_onehot = np.eye(y_prob.shape[1])[y_true]
        out["brier_multiclass"] = float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))
    return out

def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_prob: Optional[np.ndarray] = None) -> Dict[str, float]:
    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_prob is not None:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
        out["average_precision"] = average_precision_score(y_true, y_prob)
        out["brier"] = brier_score_loss(y_true, y_prob)
    return out

def report_df(y_true: np.ndarray, y_pred: np.ndarray, encoder: LabelEncoder) -> pd.DataFrame:
    rep = classification_report(
        y_true,
        y_pred,
        target_names=list(encoder.classes_),
        output_dict=True,
        zero_division=0,
    )
    return pd.DataFrame(rep).T

def plot_confusion(y_true: np.ndarray, y_pred: np.ndarray, labels: List[str], title: str, save_path: Path, normalize: bool = True) -> None:
    cm = confusion_matrix(y_true, y_pred, normalize="true" if normalize else None)
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt=".2f" if normalize else "d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

def grouped_train_val_split(df: pd.DataFrame, label_col: str, group_col: str, seed: int = 42, n_splits: int = 5) -> Tuple[pd.DataFrame, pd.DataFrame]:
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y = df[label_col].to_numpy()
    groups = df[group_col].to_numpy()
    train_idx, val_idx = next(splitter.split(df, y, groups))
    return df.iloc[train_idx].copy(), df.iloc[val_idx].copy()


def _section5_format_value(value: Any, fmt: Optional[str] = None) -> str:
    if pd.isna(value):
        return ""
    if fmt is not None:
        try:
            return fmt.format(value)
        except Exception:
            return str(value)
    return str(value)

def _display_section5_table(
    df: pd.DataFrame,
    *,
    row_color_col: Optional[str] = None,
    row_colors: Optional[Dict[str, str]] = None,
    cell_col_colors: Optional[Dict[str, str]] = None,
    formats: Optional[Dict[str, str]] = None,
    text_cols: Optional[List[str]] = None,
    nowrap_cols: Optional[List[str]] = None,
) -> None:
    row_colors = row_colors or {}
    cell_col_colors = cell_col_colors or {}
    formats = formats or {}
    text_cols = text_cols or []
    nowrap_cols = nowrap_cols or []

    def _row_style(row):
        background = row_colors.get(str(row.get(row_color_col, "")), "#ffffff") if row_color_col else "#ffffff"
        weight = "800" if str(row.get("task", "")).lower() == "b" else "650"
        return [
            f"background-color: {background}; color: #111827; font-weight: {weight}; border: 1px solid #cbd5e1; vertical-align: top;"
            for _ in row
        ]

    try:
        styler = (
            df.style
            .hide(axis="index")
            .format(formats)
            .apply(_row_style, axis=1)
            .set_table_styles(
                [
                    {
                        "selector": "th",
                        "props": [
                            ("background-color", "#f8fafc"),
                            ("color", "#111827"),
                            ("font-weight", "800"),
                            ("text-align", "left"),
                            ("border", "1px solid #cbd5e1"),
                            ("padding", "6px 8px"),
                        ],
                    },
                    {
                        "selector": "td",
                        "props": [
                            ("padding", "6px 8px"),
                            ("font-size", "12px"),
                            ("line-height", "1.25"),
                        ],
                    },
                    {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
                ]
            )
        )
        for col, color in cell_col_colors.items():
            if col in df.columns:
                styler = styler.set_properties(subset=[col], **{"background-color": color})
        for col in text_cols:
            if col in df.columns:
                styler = styler.set_properties(subset=[col], **{"max-width": "360px", "white-space": "normal"})
        for col in nowrap_cols:
            if col in df.columns:
                styler = styler.set_properties(subset=[col], **{"white-space": "nowrap"})
        display(styler)
    except Exception:

        header_html = "".join(
            f"<th style='background-color: #f8fafc; color: #111827; font-weight: 800; text-align: left; border: 1px solid #cbd5e1; padding: 6px 8px;'>{escape(str(col))}</th>"
            for col in df.columns
        )
        body_html = ""
        for _, row in df.iterrows():
            row_bg = row_colors.get(str(row.get(row_color_col, "")), "#ffffff") if row_color_col else "#ffffff"
            row_weight = "800" if str(row.get("task", "")).lower() == "b" else "650"
            body_html += "<tr>"
            for col in df.columns:
                background = cell_col_colors.get(col, row_bg)
                wrap = "max-width: 360px; white-space: normal;" if col in text_cols else "white-space: nowrap;"
                value_text = _section5_format_value(row[col], formats.get(col))
                body_html += (
                    f"<td style='background-color: {background}; color: #111827; font-weight: {row_weight}; border: 1px solid #cbd5e1; padding: 6px 8px; font-size: 12px; line-height: 1.25; vertical-align: top; {wrap}'>"
                    f"{escape(value_text)}</td>"
                )
            body_html += "</tr>"
        display(HTML(f"<table style='border-collapse: collapse; width: 100%; color: #111827;'><thead><tr>{header_html}</tr></thead><tbody>{body_html}</tbody></table>"))

model_inventory_df = pd.DataFrame([
    {
        "task": "a",
        "target": "Transcript MI quality (high/low)",
        "grouping_unit": "transcript_id",
        "primary_metric": "F1 / ROC-AUC",
        "baseline_model": "Elastic-net logistic regression",
        "stronger_model": "XGBoost / HistGradientBoosting",
        "compute_profile": "CPU-preferred",
    },
    {
        "task": "b",
        "target": "Therapist behaviour (reflection/question/therapist_input/other)",
        "grouping_unit": "transcript_id",
        "primary_metric": "Macro-F1",
        "baseline_model": "Elastic-net logistic regression",
        "stronger_model": "RoBERTa-base + causal context",
        "compute_profile": "CPU baseline + GPU main model",
    },
    {
        "task": "c",
        "target": "Next therapist action forecasting",
        "grouping_unit": "transcript_id",
        "primary_metric": "Macro-F1",
        "baseline_model": "Transition-summary boosted tree",
        "stronger_model": "Causal GRU over history embeddings",
        "compute_profile": "CPU baseline + GPU main model",
    },
])

display(Markdown("### Table 5A. Official modelling task inventory"))
_display_section5_table(
    model_inventory_df,
    row_color_col="task",
    row_colors={"a": "#e7f0ff", "b": "#e9f7ef", "c": "#fff3d9"},
    text_cols=["target", "baseline_model", "stronger_model"],
    nowrap_cols=["task", "grouping_unit", "primary_metric", "compute_profile"],
)


In [ ]:
def choose_balanced_grouped_split(
    task_df: pd.DataFrame,
    label_col: str,
    group_col: str,
    quality_col: str,
    n_splits: int = 5,
    seed: int = 42,
) -> Dict[str, Any]:
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y = task_df[label_col].to_numpy()
    groups = task_df[group_col].to_numpy()
    global_label_dist = task_df[label_col].value_counts(normalize=True).sort_index()
    transcript_quality = task_df.groupby(group_col)[quality_col].first()
    global_quality_dist = transcript_quality.value_counts(normalize=True).sort_index()
    candidates = []
    for fold, (train_idx, test_idx) in enumerate(splitter.split(task_df, y, groups)):
        test_ids = sorted(task_df.iloc[test_idx][group_col].unique().tolist())
        train_ids = sorted(task_df.iloc[train_idx][group_col].unique().tolist())
        test_rows = task_df[task_df[group_col].isin(test_ids)].copy()
        label_dist = (
            test_rows[label_col].value_counts(normalize=True).sort_index()
            .reindex(global_label_dist.index, fill_value=0.0)
        )
        quality_dist = (
            transcript_quality.loc[test_ids].value_counts(normalize=True).sort_index()
            .reindex(global_quality_dist.index, fill_value=0.0)
        )
        label_gap = float((label_dist - global_label_dist).abs().sum())
        quality_gap = float((quality_dist - global_quality_dist).abs().sum())
        size_gap = float(abs(len(test_ids) / transcript_quality.shape[0] - 1 / n_splits))
        score = label_gap + quality_gap + size_gap
        candidates.append(
            {
                "fold": int(fold),
                "score": score,
                "n_train_transcripts": int(len(train_ids)),
                "n_test_transcripts": int(len(test_ids)),
                "train_transcripts": train_ids,
                "test_transcripts": test_ids,
                "label_gap": label_gap,
                "quality_gap": quality_gap,
                "size_gap": size_gap,
            }
        )
    best = sorted(candidates, key=lambda x: x["score"])[0]
    return {"all_candidates": candidates, "best": best}
MAIN_LABEL_COL = CONFIG.get("main_task_label_col", "main_therapist_behaviour")
GROUP_COL = CONFIG.get("group_col", "transcript_id")
QUALITY_COL = CONFIG.get("quality_col", "mi_quality")
therapist_split_source = therapist_df.copy()
therapist_split_source = therapist_split_source[
    therapist_split_source[MAIN_LABEL_COL].isin(["reflection", "question", "therapist_input", "other"])
].copy()
OFFICIAL_SPLIT_JSON = ARTIFACT_DIR / "official_split_v2.json"
if artifact_exists(OFFICIAL_SPLIT_JSON) and not CONFIG.get("force_recompute", False):
    official_split = load_json(OFFICIAL_SPLIT_JSON)
    split_candidates_df = pd.DataFrame([
        {
            "fold": official_split.get("fold", "saved"),
            "score": official_split.get("score", np.nan),
            "n_train_transcripts": len(official_split["train_transcripts"]),
            "n_test_transcripts": len(official_split["test_transcripts"]),
            "label_gap": official_split.get("label_gap", np.nan),
            "quality_gap": official_split.get("quality_gap", np.nan),
            "size_gap": official_split.get("size_gap", np.nan),
            "source": "loaded_from_artifact",
        }
    ])
    split_source_message = f"Loaded official split from: {OFFICIAL_SPLIT_JSON}"
else:
    split_info = choose_balanced_grouped_split(
        task_df=therapist_split_source,
        label_col=MAIN_LABEL_COL,
        group_col=GROUP_COL,
        quality_col=QUALITY_COL,
        n_splits=CONFIG["official_n_splits"],
        seed=CONFIG["seed"],
    )
    split_candidates_df = pd.DataFrame(split_info["all_candidates"]).drop(
        columns=["train_transcripts", "test_transcripts"]
    )
    official_split = split_info["best"]
    save_json(official_split, OFFICIAL_SPLIT_JSON)
    split_source_message = f"Computed and saved official split to: {OFFICIAL_SPLIT_JSON}"
display(Markdown("### Table 5.1A. Official grouped split candidate diagnostics"))
if "_display_section5_table" in globals():
    _display_section5_table(
        split_candidates_df,
        cell_col_colors={
            "fold": "#f1f5f9",
            "score": "#e7f0ff",
            "n_train_transcripts": "#e9f7ef",
            "n_test_transcripts": "#fff3d9",
            "label_gap": "#fce8ef",
            "quality_gap": "#fce8ef",
            "size_gap": "#fce8ef",
            "source": "#e9f7ef",
        },
        formats={
            "score": "{:.4f}",
            "label_gap": "{:.4f}",
            "quality_gap": "{:.4f}",
            "size_gap": "{:.4f}",
        },
        nowrap_cols=split_candidates_df.columns.tolist(),
    )
else:
    display(split_candidates_df)
TRAIN_TRANSCRIPTS = set(official_split["train_transcripts"])
TEST_TRANSCRIPTS = set(official_split["test_transcripts"])
assert TRAIN_TRANSCRIPTS.isdisjoint(TEST_TRANSCRIPTS), "Train/test transcript leakage detected."
therapist_train = therapist_df[therapist_df[GROUP_COL].isin(TRAIN_TRANSCRIPTS)].copy()
therapist_test = therapist_df[therapist_df[GROUP_COL].isin(TEST_TRANSCRIPTS)].copy()
client_train = client_df[client_df[GROUP_COL].isin(TRAIN_TRANSCRIPTS)].copy()
client_test = client_df[client_df[GROUP_COL].isin(TEST_TRANSCRIPTS)].copy()
transcript_train = transcript_df[transcript_df[GROUP_COL].isin(TRAIN_TRANSCRIPTS)].copy()
transcript_test = transcript_df[transcript_df[GROUP_COL].isin(TEST_TRANSCRIPTS)].copy()

split_transcript_check_df = pd.DataFrame(
    [
        {
            "split": "train",
            "n_transcripts": len(TRAIN_TRANSCRIPTS),
            "n_therapist_rows": therapist_train.shape[0],
            "n_client_rows": client_train.shape[0],
            "n_transcript_rows": transcript_train.shape[0],
        },
        {
            "split": "test",
            "n_transcripts": len(TEST_TRANSCRIPTS),
            "n_therapist_rows": therapist_test.shape[0],
            "n_client_rows": client_test.shape[0],
            "n_transcript_rows": transcript_test.shape[0],
        },
    ]
)
display(Markdown("### Official train/test transcript check"))
if "_display_section5_table" in globals():
    _display_section5_table(
        split_transcript_check_df,
        row_color_col="split",
        row_colors={"train": "#e9f7ef", "test": "#fff3d9"},
        cell_col_colors={"split": "#f1f5f9", "n_transcripts": "#e7f0ff"},
        formats={
            "n_transcripts": "{:,}",
            "n_therapist_rows": "{:,}",
            "n_client_rows": "{:,}",
            "n_transcript_rows": "{:,}",
        },
        nowrap_cols=split_transcript_check_df.columns.tolist(),
    )
else:
    display(split_transcript_check_df)

In [ ]:

split_diagnostics = {
    "train_label_dist": therapist_train["main_therapist_behaviour"].value_counts(normalize=True).round(4).to_dict(),
    "test_label_dist": therapist_test["main_therapist_behaviour"].value_counts(normalize=True).round(4).to_dict(),
    "train_quality_dist": transcript_train["mi_quality"].value_counts(normalize=True).round(4).to_dict(),
    "test_quality_dist": transcript_test["mi_quality"].value_counts(normalize=True).round(4).to_dict(),
    "train_topics_top10": transcript_train["topic_norm"].value_counts().head(10).to_dict(),
    "test_topics_top10": transcript_test["topic_norm"].value_counts().head(10).to_dict(),
}

split_size_df = pd.DataFrame(
    [
        {
            "split": "train",
            "n_transcripts": len(TRAIN_TRANSCRIPTS),
            "n_therapist_rows": len(therapist_train),
            "n_client_rows": len(client_train),
            "n_transcript_rows": len(transcript_train),
        },
        {
            "split": "test",
            "n_transcripts": len(TEST_TRANSCRIPTS),
            "n_therapist_rows": len(therapist_test),
            "n_client_rows": len(client_test),
            "n_transcript_rows": len(transcript_test),
        },
    ]
)

split_balance_rows = []
for distribution, train_key, test_key in [
    ("therapist_behaviour", "train_label_dist", "test_label_dist"),
    ("mi_quality", "train_quality_dist", "test_quality_dist"),
]:
    labels = sorted(set(split_diagnostics[train_key]) | set(split_diagnostics[test_key]))
    for label in labels:
        train_prop = float(split_diagnostics[train_key].get(label, 0.0))
        test_prop = float(split_diagnostics[test_key].get(label, 0.0))
        split_balance_rows.append(
            {
                "distribution": distribution,
                "label": label,
                "train_prop": train_prop,
                "test_prop": test_prop,
                "abs_gap": abs(train_prop - test_prop),
            }
        )
split_balance_df = pd.DataFrame(split_balance_rows)

topic_union = sorted(
    set(split_diagnostics["train_topics_top10"]) | set(split_diagnostics["test_topics_top10"]),
    key=lambda topic: max(
        split_diagnostics["train_topics_top10"].get(topic, 0),
        split_diagnostics["test_topics_top10"].get(topic, 0),
    ),
    reverse=True,
)
split_topic_df = pd.DataFrame(
    [
        {
            "topic_norm": topic,
            "train_count": int(split_diagnostics["train_topics_top10"].get(topic, 0)),
            "test_count": int(split_diagnostics["test_topics_top10"].get(topic, 0)),
        }
        for topic in topic_union
    ]
)

display(Markdown("### Table 5.1B. Official train/test split size summary"))
if "_display_section5_table" in globals():
    _display_section5_table(
        split_size_df,
        row_color_col="split",
        row_colors={"train": "#e9f7ef", "test": "#fff3d9"},
        nowrap_cols=split_size_df.columns.tolist(),
    )
else:
    display(split_size_df)

display(Markdown("### Table 5.1C. Official split label and MI-quality balance"))
if "_display_section5_table" in globals():
    _display_section5_table(
        split_balance_df,
        row_color_col="distribution",
        row_colors={"therapist_behaviour": "#e7f0ff", "mi_quality": "#e9f7ef"},
        cell_col_colors={"train_prop": "#e9f7ef", "test_prop": "#fff3d9", "abs_gap": "#fce8ef"},
        formats={"train_prop": "{:.4f}", "test_prop": "{:.4f}", "abs_gap": "{:.4f}"},
        nowrap_cols=split_balance_df.columns.tolist(),
    )
else:
    display(split_balance_df.round(4))

display(Markdown("### Table 5.1D. Official split top-topic coverage"))
if "_display_section5_table" in globals():
    _display_section5_table(
        split_topic_df,
        cell_col_colors={"topic_norm": "#f1f5f9", "train_count": "#e9f7ef", "test_count": "#fff3d9"},
        text_cols=["topic_norm"],
        nowrap_cols=["train_count", "test_count"],
    )
else:
    display(split_topic_df)


In [ ]:
def prepare_logreg_input(df: pd.DataFrame) -> pd.DataFrame:
    """
    Make sure the numeric feature block is numeric and stable across sklearn versions.
    """
    out = df.copy()
    for col in NUMERIC_FEATURES:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out
def build_logreg_pipeline(
    context_col: str,
    min_df: int = 3,
    word_ngram: Tuple[int, int] = (1, 2),
    char_ngram: Tuple[int, int] = (3, 5),
    C: float = 1.0,
    l1_ratio: float = 0.15,
    class_weight: Optional[str] = None,
    random_state: int = 42,
) -> Pipeline:
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", MaxAbsScaler()),
        ]
    )
    preprocess = ColumnTransformer(
        transformers=[
            (
                "current_word",
                TfidfVectorizer(
                    ngram_range=word_ngram,
                    min_df=min_df,
                    sublinear_tf=True,
                ),
                "utterance_text_lc",
            ),
            (
                "current_char",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=char_ngram,
                    min_df=min_df,
                    sublinear_tf=True,
                ),
                "utterance_text_lc",
            ),
            (
                "context_word",
                TfidfVectorizer(
                    ngram_range=word_ngram,
                    min_df=min_df,
                    sublinear_tf=True,
                ),
                context_col,
            ),
            ("numeric", numeric_transformer, NUMERIC_FEATURES),
        ],
        remainder="drop",
        sparse_threshold=0.3,
        verbose_feature_names_out=True,
    )
    clf = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=C,
        l1_ratio=l1_ratio,
        class_weight=class_weight,
        max_iter=5000,
        random_state=random_state,
    )
    return Pipeline(
        steps=[
            ("prep", preprocess),
            ("clf", clf),
        ]
    )
LOGREG_SEARCH_SPACE = list(
    ParameterGrid(
        {
            "C": [0.25, 1.0, 4.0, 8.0],
            "l1_ratio": [0.0, 0.15, 0.5],
            "min_df": [2, 3, 5],
            "char_ngram": [(3, 5)],
            "class_weight": [None, "balanced"],
        }
    )
)
def tune_logreg_model(
    train_df: pd.DataFrame,
    context_grid: List[int],
    search_space: List[Dict[str, Any]],
    inner_splits: int = 4,
    random_state: int = 42,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    train_df = prepare_logreg_input(train_df)
    y = train_df["y_code"].to_numpy()
    groups = train_df[GROUP_COL].to_numpy()
    splitter = StratifiedGroupKFold(
        n_splits=inner_splits,
        shuffle=True,
        random_state=random_state,
    )
    results = []
    for k in context_grid:
        context_col = f"context_k{k}_lc"
        for params in tqdm(search_space, desc=f"Tuning logreg (k={k})", leave=False):
            fold_scores = []
            for fold, (tr_idx, val_idx) in enumerate(splitter.split(train_df, y, groups)):
                fold_train = prepare_logreg_input(train_df.iloc[tr_idx].copy())
                fold_val = prepare_logreg_input(train_df.iloc[val_idx].copy())
                pipe = build_logreg_pipeline(
                    context_col=context_col,
                    min_df=params["min_df"],
                    char_ngram=params["char_ngram"],
                    C=params["C"],
                    l1_ratio=params["l1_ratio"],
                    class_weight=params["class_weight"],
                    random_state=random_state,
                )
                pipe.fit(fold_train, y[tr_idx])
                val_pred = pipe.predict(fold_val)
                fold_scores.append(f1_score(y[val_idx], val_pred, average="macro"))
            results.append(
                {
                    "context_k": k,
                    **params,
                    "cv_f1_macro_mean": float(np.mean(fold_scores)),
                    "cv_f1_macro_std": float(np.std(fold_scores)),
                }
            )
    results_df = (
        pd.DataFrame(results)
        .sort_values(["cv_f1_macro_mean", "cv_f1_macro_std"], ascending=[False, True])
        .reset_index(drop=True)
    )
    best = results_df.iloc[0].to_dict()
    return results_df, best
print(
    "Task b logistic-regression helper functions loaded. "
    "The expensive CV tuning is skipped here; the Task b baseline cell below "
    "loads the existing logistic artifacts from local cache by default."
)


In [ ]:
task_b_dir = ARTIFACT_DIR / "task_b"
task_b_dir.mkdir(parents=True, exist_ok=True)
therapist_train = therapist_df[therapist_df["transcript_id"].isin(TRAIN_TRANSCRIPTS)].copy()
therapist_test = therapist_df[therapist_df["transcript_id"].isin(TEST_TRANSCRIPTS)].copy()
therapist_train["y_code"] = label_encoder.transform(therapist_train[LABEL_COL])
therapist_test["y_code"] = label_encoder.transform(therapist_test[LABEL_COL])
y_test_b = therapist_test["y_code"].to_numpy()
groups_test_b = therapist_test["transcript_id"].to_numpy()
logreg_cv_path = task_b_dir / "task_b_logreg_cv_results.csv"
logreg_model_path = task_b_dir / "task_b_logreg_model.joblib"
logreg_metrics_path = task_b_dir / "task_b_logreg_metrics.json"
logreg_preds_path = task_b_dir / "task_b_logreg_preds.joblib"
therapist_train_logreg = prepare_logreg_input(therapist_train)
therapist_test_logreg = prepare_logreg_input(therapist_test)
def _recover_best_task_b_logreg_cfg(csv_path: Path) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    cv_df = pd.read_csv(csv_path)
    best_cfg = cv_df.iloc[0].to_dict()
    if isinstance(best_cfg.get("char_ngram"), str):
        best_cfg["char_ngram"] = tuple(ast.literal_eval(best_cfg["char_ngram"]))
    for key in ["context_k", "min_df"]:
        if key in best_cfg and not pd.isna(best_cfg[key]):
            best_cfg[key] = int(best_cfg[key])
    for key in ["C", "l1_ratio"]:
        if key in best_cfg and not pd.isna(best_cfg[key]):
            best_cfg[key] = float(best_cfg[key])
    if "class_weight" in best_cfg:
        cw = best_cfg["class_weight"]
        if pd.isna(cw) or cw == "None":
            best_cfg["class_weight"] = None
    return cv_df, best_cfg
task_b_logreg_cache_ready = artifact_exists(logreg_metrics_path) and artifact_exists(logreg_preds_path)
if task_b_logreg_cache_ready and not CONFIG.get("force_retrain", False):
    CONFIG["task_b_logreg_load_from_artifacts"] = True
use_cached_task_b_logreg = (
    CONFIG.get("task_b_logreg_load_from_artifacts", True)
    and task_b_logreg_cache_ready
    and not CONFIG.get("force_retrain", False)
)
if use_cached_task_b_logreg:
    print("Loaded cached Task b sparse-baseline metrics and predictions from local artifacts.")
    task_b_logreg_metrics = load_json(logreg_metrics_path)
    task_b_logreg_preds = load_joblib(logreg_preds_path)
    if artifact_exists(logreg_cv_path):
        task_b_logreg_cv_results, best_logreg_cfg = _recover_best_task_b_logreg_cfg(logreg_cv_path)
    else:
        task_b_logreg_cv_results, best_logreg_cfg = pd.DataFrame(), {}
    task_b_logreg_model = None
    if artifact_exists(logreg_model_path):
        try:
            task_b_logreg_model = load_joblib(logreg_model_path)
        except Exception as e:
            print(f"Warning: could not load cached Task b logistic model. Metrics/predictions were still loaded. Error: {e}")
else:
    task_b_logreg_cv_results, best_logreg_cfg = tune_logreg_model(
        therapist_train_logreg,
        context_grid=CONFIG["context_grid"],
        search_space=LOGREG_SEARCH_SPACE,
        inner_splits=4,
        random_state=SEED,
    )
    save_df(task_b_logreg_cv_results, logreg_cv_path)
    best_logreg_context_col = f"context_k{int(best_logreg_cfg['context_k'])}_lc"
    task_b_logreg_model = build_logreg_pipeline(
        context_col=best_logreg_context_col,
        min_df=int(best_logreg_cfg["min_df"]),
        char_ngram=tuple(best_logreg_cfg["char_ngram"]),
        C=float(best_logreg_cfg["C"]),
        l1_ratio=float(best_logreg_cfg["l1_ratio"]),
        class_weight=best_logreg_cfg["class_weight"],
        random_state=SEED,
    )
    task_b_logreg_model.fit(therapist_train_logreg, therapist_train_logreg["y_code"].to_numpy())
    pred = task_b_logreg_model.predict(therapist_test_logreg)
    prob = task_b_logreg_model.predict_proba(therapist_test_logreg)
    task_b_logreg_metrics = compute_multiclass_metrics(y_test_b, pred, prob)
    task_b_logreg_preds = {"pred": pred, "prob": prob}
    save_json(task_b_logreg_metrics, logreg_metrics_path)
    save_joblib(task_b_logreg_preds, logreg_preds_path)
    save_joblib(task_b_logreg_model, logreg_model_path)
task_b_logreg_metrics_df = pd.DataFrame([task_b_logreg_metrics]).round(4)
task_b_logreg_report = report_df(y_test_b, task_b_logreg_preds["pred"], label_encoder)
display(Markdown("### Table 6.1A. Task B sparse-baseline held-out metrics"))
display_report_table(
    task_b_logreg_metrics_df,
    gradient_cols=["accuracy", "precision_macro", "recall_macro", "f1_macro", "precision_weighted", "recall_weighted", "f1_weighted"],
    gradient_kwargs={"cmap": "YlGn"},
    highlight_min_cols=["brier_multiclass"],
)
display(Markdown("### Table 6.1B. Task B sparse-baseline per-class classification report"))
task_b_logreg_report_display = task_b_logreg_report.round(4).reset_index().rename(columns={"index": "class"})
display_report_table(
    task_b_logreg_report_display,
    row_color_col="class",
    row_colors=REPORT_BEHAVIOUR_COLORS,
    text_cols=["class"],
    gradient_cols=["precision", "recall", "f1-score"],
    gradient_kwargs={"cmap": "YlGn"},
)


In [ ]:
task_b_dir = ARTIFACT_DIR / "task_b"
task_b_dir.mkdir(parents=True, exist_ok=True)
logreg_cv_path = task_b_dir / "task_b_logreg_cv_results.csv"
logreg_model_path = task_b_dir / "task_b_logreg_model.joblib"
logreg_metrics_path = task_b_dir / "task_b_logreg_metrics.json"
logreg_preds_path = task_b_dir / "task_b_logreg_preds.joblib"
task_b_feature_path = task_b_dir / "task_b_logreg_top_features.csv"
legacy_feature_path = PROCESSED_DIR / "logreg_top_features.csv"
def _recover_or_raise(name: str, path: Path, loader, kind: str = "object") -> Any:
    """If `name` is in globals(), return it. Otherwise load from `path` or raise."""
    if name in globals() and globals()[name] is not None:
        return globals()[name]
    if not artifact_exists(path):
        raise FileNotFoundError(
            f"Could not find saved Task b logistic {kind} at:\n{path}"
        )
    obj = loader(path)
    print(f"Recovered {name} from: {path}")
    return obj
def _recover_best_logreg_cfg_from_cv(csv_path: Path) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Recover full CV table + first-row best config, fixing CSV round-trip types."""
    cv_df = pd.read_csv(csv_path)
    best_cfg = cv_df.iloc[0].to_dict()
    if isinstance(best_cfg.get("char_ngram"), str):
        best_cfg["char_ngram"] = tuple(ast.literal_eval(best_cfg["char_ngram"]))
    best_cfg["context_k"] = int(best_cfg["context_k"])
    best_cfg["min_df"] = int(best_cfg["min_df"])
    best_cfg["C"] = float(best_cfg["C"])
    best_cfg["l1_ratio"] = float(best_cfg["l1_ratio"])
    if "class_weight" in best_cfg:
        cw = best_cfg["class_weight"]
        if pd.isna(cw) or cw == "None":
            best_cfg["class_weight"] = None
    return cv_df, best_cfg
def extract_top_features_from_logreg(
    fitted_pipeline: Pipeline,
    encoder: LabelEncoder,
    top_n: int = 20,
) -> pd.DataFrame:
    """For each class, return the top_n positive and top_n negative coefficients."""
    if fitted_pipeline is None:
        raise ValueError("No fitted logistic-regression model is available.")
    steps = fitted_pipeline.named_steps
    if "prep" not in steps or "clf" not in steps:
        raise ValueError("Expected a Pipeline with named steps 'prep' and 'clf'.")
    feature_names = steps["prep"].get_feature_names_out()
    coef = steps["clf"].coef_
    rows = []
    for class_idx, class_name in enumerate(encoder.classes_):
        top_pos = np.argsort(coef[class_idx])[-top_n:][::-1]
        top_neg = np.argsort(coef[class_idx])[:top_n]
        for direction, indices in (("positive", top_pos), ("negative", top_neg)):
            for rank, idx in enumerate(indices, start=1):
                rows.append({
                    "class": class_name,
                    "direction": direction,
                    "rank": rank,
                    "feature": feature_names[idx],
                    "weight": float(coef[class_idx, idx]),
                })
    return pd.DataFrame(rows)
task_b_logreg_metrics = _recover_or_raise(
    "task_b_logreg_metrics", logreg_metrics_path, load_json, kind="metrics",
)
task_b_logreg_preds = _recover_or_raise(
    "task_b_logreg_preds", logreg_preds_path, load_joblib, kind="predictions",
)
if "task_b_logreg_cv_results" not in globals() or "best_logreg_cfg" not in globals():
    if artifact_exists(logreg_cv_path):
        task_b_logreg_cv_results, best_logreg_cfg = _recover_best_logreg_cfg_from_cv(
            logreg_cv_path
        )
        print(f"Recovered Task b CV results and best_logreg_cfg from: {logreg_cv_path}")
    else:
        task_b_logreg_cv_results, best_logreg_cfg = pd.DataFrame(), {}
        print("Warning: task_b_logreg_cv_results.csv not found; "
              "optional research ablations may be skipped.")
logreg_metrics = task_b_logreg_metrics
logreg_test_pred = task_b_logreg_preds["pred"]
logreg_test_prob = task_b_logreg_preds["prob"]
def _strategy_load_primary_csv() -> Optional[pd.DataFrame]:
    if artifact_exists(task_b_feature_path):
        df = pd.read_csv(task_b_feature_path)
        print(f"Loaded saved Task b logistic top features from: {task_b_feature_path}")
        return df
    return None
def _strategy_load_legacy_csv() -> Optional[pd.DataFrame]:
    if artifact_exists(legacy_feature_path):
        df = pd.read_csv(legacy_feature_path)
        print(f"Loaded saved legacy logistic top features from: {legacy_feature_path}")
        return df
    return None
def _strategy_recompute_from_fitted_model() -> Optional[pd.DataFrame]:
    """Last resort: load (or reuse in-memory) a fitted pipeline and extract top features."""
    fitted_model = globals().get("task_b_logreg_model")
    if fitted_model is None and artifact_exists(logreg_model_path):
        try:
            fitted_model = load_joblib(logreg_model_path)
            print(f"Recovered task_b_logreg_model from: {logreg_model_path}")
        except Exception as exc:
            print(f"Warning: could not load Task b logistic model: {exc}")
            fitted_model = None
    if fitted_model is None:
        return None
    df = extract_top_features_from_logreg(fitted_model, label_encoder, top_n=20)
    save_df(df, task_b_feature_path)
    save_df(df, legacy_feature_path)
    print("Saved:")
    print(task_b_feature_path)
    print(legacy_feature_path)
    globals()["task_b_logreg_model"] = fitted_model
    return df
task_b_logreg_feature_importance_df: Optional[pd.DataFrame] = None
for strategy in (_strategy_load_primary_csv,
                 _strategy_load_legacy_csv,
                 _strategy_recompute_from_fitted_model):
    task_b_logreg_feature_importance_df = strategy()
    if task_b_logreg_feature_importance_df is not None:
        break
if task_b_logreg_feature_importance_df is None:
    raise FileNotFoundError(
        "No saved Task b logistic top-feature CSV or fitted model was available.\n"
        f"Expected one of:\n - {task_b_feature_path}\n - {legacy_feature_path}\n"
        f" - {logreg_model_path}\n"
        "This cell will not retrain Task b logistic regression by default."
    )
logreg_feature_importance_df = task_b_logreg_feature_importance_df.copy()
logreg_model = globals().get("task_b_logreg_model")
def clean_logreg_feature_name(feature: Any) -> str:
    text = str(feature)
    prefix_labels = {
        "numeric__": "numeric: ",
        "current_word__": "word: ",
        "current_char__": "char n-gram: ",
        "context_word__": "context word: ",
        "context_char__": "context char n-gram: ",
        "topic__": "topic: ",
        "mi_quality__": "MI quality: ",
    }
    for prefix, label in prefix_labels.items():
        if text.startswith(prefix):
            return label + text[len(prefix):].replace("_", " ")
    return text.replace("__", ": ").replace("_", " ")
top_features_preview = (
    task_b_logreg_feature_importance_df
    .assign(
        feature_clean=lambda d: d["feature"].map(clean_logreg_feature_name),
        abs_weight=lambda d: d["weight"].abs(),
    )
    .sort_values(["class", "direction", "rank"])
    .groupby(["class", "direction"], group_keys=False)
    .head(3)
    .reset_index(drop=True)
)
top_features_preview["feature_with_weight"] = top_features_preview.apply(
    lambda row: f"{int(row['rank'])}. {row['feature_clean']} ({row['weight']:+.3f})",
    axis=1,
)
top_features_companion_df = (
    top_features_preview
    .groupby(["class", "direction"], sort=False)
    .agg(
        top_features=("feature_with_weight", " | ".join),
        strongest_abs_weight=("abs_weight", "max"),
        mean_signed_weight=("weight", "mean"),
    )
    .reset_index()
)
top_features_companion_df["direction_meaning"] = np.where(
    top_features_companion_df["direction"].eq("positive"),
    "Raises the one-vs-rest log-odds for this class",
    "Suppresses the one-vs-rest log-odds for this class",
)
_display_feature_colors = {
    "other": "#f3e8ff",
    "question": "#e9f7ef",
    "reflection": "#e7f0ff",
    "therapist_input": "#fff3d9",
}
display(Markdown(
    "### Top 3 features per (class, direction) - text companion to the bar chart\n\n"
    "Full ranking of 20 positive and 20 negative features per class is "
    f"saved to `{task_b_feature_path.name}`."
))
display_report_table(
    top_features_companion_df,
    formats={"strongest_abs_weight": "{:.3f}", "mean_signed_weight": "{:+.3f}"},
    row_color_col="class",
    row_colors=_display_feature_colors,
    text_cols=["class", "direction", "top_features", "direction_meaning"],
    gradient_cols=["strongest_abs_weight"],
    gradient_kwargs={"cmap": "YlGnBu"},
    signed_cols=["mean_signed_weight"],
)
save_df(
    top_features_preview.loc[:, ["class", "direction", "rank", "feature", "feature_clean", "weight"]],
    task_b_dir / "task_b_logreg_top3_features_compact.csv",
)
save_df(
    top_features_companion_df,
    task_b_dir / "task_b_logreg_top_features_companion.csv",
)

In [ ]:
assert "task_b_logreg_feature_importance_df" in globals(), (
    "Run the feature-importance recovery cell upstream first."
)
assert "label_encoder" in globals(), "label_encoder must be available."
N_POS_PER_CLASS = 8
N_NEG_PER_CLASS = 8
class_order = list(label_encoder.classes_)
n_classes_b = len(class_order)
def _strip_feature_prefix(name: str) -> str:
    """Drop the sklearn ColumnTransformer prefix so the bar labels read as
    plain feature names. e.g. 'word__not' -> 'not', 'char__th' -> 'th'."""
    name = str(name)
    for prefix in ("word__", "char__", "num__", "cat__", "remainder__"):
        if name.startswith(prefix):
            return name[len(prefix):]
    return name
fi_df = task_b_logreg_feature_importance_df.copy()
fi_df["feature_clean"] = fi_df["feature"].map(_strip_feature_prefix)
fig, axes = plt.subplots(
    n_classes_b,
    1,
    figsize=(10, 3.0 * n_classes_b),
    sharex=False,
)
if n_classes_b == 1:
    axes = [axes]
for ax, class_name in zip(axes, class_order):
    sub = fi_df[fi_df["class"] == class_name].copy()
    pos = (sub[sub["direction"] == "positive"]
           .sort_values("weight", ascending=False)
           .head(N_POS_PER_CLASS))
    neg = (sub[sub["direction"] == "negative"]
           .sort_values("weight", ascending=True)
           .head(N_NEG_PER_CLASS))
    plot_df = pd.concat([neg.iloc[::-1], pos.iloc[::-1]], ignore_index=True)
    colors = ["#c62828" if w < 0 else "#2e7d32" for w in plot_df["weight"]]
    y_positions = np.arange(len(plot_df))
    ax.barh(y_positions, plot_df["weight"].to_numpy(), color=colors, alpha=0.85)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(plot_df["feature_clean"].tolist(), fontsize=9)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(
        f"Class '{class_name}' — top {N_POS_PER_CLASS} positive (green) "
        f"and top {N_NEG_PER_CLASS} negative (red) features",
        fontsize=11,
    )
    ax.set_xlabel("Logistic-regression coefficient (signed)")
    ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
fi_path = FIG_DIR / "task_b_feature_importance_per_class.png"
plt.savefig(fi_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 17. Task B sparse-baseline feature importance by class"))
display(IPyImage(filename=str(fi_path)))
top_features_compact = (
    fi_df.assign(abs_weight=fi_df["weight"].abs())
         .sort_values(["class", "direction", "rank"])
         .groupby(["class", "direction"], group_keys=False)
         .head(max(N_POS_PER_CLASS, N_NEG_PER_CLASS))
         .loc[:, ["class", "direction", "rank", "feature_clean", "weight"]]
         .rename(columns={"feature_clean": "feature"})
         .reset_index(drop=True)
)
save_df(top_features_compact, task_b_dir / "task_b_logreg_top_features_compact.csv")


In [ ]:
# Runtime environments are managed outside the notebook.
# Install requirements-experiment.txt before running encoder cells.

if globals().get("torch") is None:
    raise ImportError("Torch is required for the encoder cells. Rerun the main imports cell and check torch installation.")
if globals().get("HFDataset") is None or globals().get("AutoTokenizer") is None:
    raise ImportError(
        "datasets/transformers are required for the encoder cells. "
        f"datasets error: {globals().get('DATASETS_IMPORT_ERROR', '')}; transformers error: {globals().get('TRANSFORMERS_IMPORT_ERROR', '')}"
    )

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch device:", DEVICE)
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

def ensure_encoder_dependencies() -> None:
    missing = []
    try:
        importlib.import_module("sentencepiece")
    except Exception:
        missing.append("sentencepiece")
    try:
        importlib.import_module("google.protobuf")
    except Exception:
        missing.append("protobuf")
    if missing:
        raise ImportError(f"Missing packages for encoder tokenizers: {missing}")

def load_tokenizer_safe(model_name_or_path: str, max_length: int):
    ensure_encoder_dependencies()
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True, use_fast=True)
    except Exception as e:
        print(f"Fast tokenizer failed for {model_name_or_path}: {e}")
        tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True, use_fast=False)
    tokenizer.model_max_length = max_length
    return tokenizer


In [ ]:
if globals().get("torch") is None or globals().get("nn") is None:
    raise ImportError("Torch is required for the encoder helper cell. Rerun the main imports cell and check torch installation.")
if globals().get("HFDataset") is None or globals().get("AutoModelForSequenceClassification") is None:
    raise ImportError(
        "datasets/transformers are required for the encoder helper cell. "
        f"datasets error: {globals().get('DATASETS_IMPORT_ERROR', '')}; transformers error: {globals().get('TRANSFORMERS_IMPORT_ERROR', '')}"
    )
_required_helpers = [
    "ARTIFACT_DIR",
    "CONFIG",
    "label_encoder",
    "compute_multiclass_metrics",
    "artifact_exists",
    "save_df",
    "save_json",
    "load_json",
    "save_joblib",
    "load_joblib",
]
_missing_helpers = [name for name in _required_helpers if name not in globals()]
assert not _missing_helpers, (
    "Encoder helper cell cannot run: missing globals "
    f"{_missing_helpers}. Re-run the Section 0 environment cell first."
)
task_b_dir = ARTIFACT_DIR / "task_b"
task_b_dir.mkdir(parents=True, exist_ok=True)
task_b_encoder_dir = task_b_dir / "encoder"
task_b_encoder_dir.mkdir(parents=True, exist_ok=True)
CONFIG.setdefault("task_b_train_batch_size", 4)
CONFIG.setdefault("task_b_eval_batch_size", 8)
CONFIG.setdefault("task_b_grad_accum", 4)
CONFIG.setdefault("task_b_epochs", 6)
CONFIG.setdefault("task_b_early_stopping_patience", 2)
CONFIG.setdefault("task_b_roberta_lr_grid", [1e-5, 2e-5, 3e-5])
CONFIG["task_b_use_bf16"] = False
def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
def load_tokenizer_safe_fixed(model_name_or_path, max_length: int = 256):
    if "load_tokenizer_safe" in globals():
        tokenizer = load_tokenizer_safe(model_name_or_path, max_length=max_length)
    else:
        tokenizer = AutoTokenizer.from_pretrained(
            model_name_or_path,
            model_max_length=max_length,
            use_fast=True,
            trust_remote_code=True,
        )
        if tokenizer.pad_token is None:
            if tokenizer.eos_token is not None:
                tokenizer.pad_token = tokenizer.eos_token
            else:
                tokenizer.add_special_tokens({"pad_token": "[PAD]"})
    tokenizer.truncation_side = "right"
    tokenizer.padding_side = "right"
    return tokenizer
def make_training_args_compat(**kwargs):
    """
    Builds TrainingArguments using only kwargs supported by the installed
    Transformers version. Also gracefully handles eval_strategy vs
    evaluation_strategy and warmup_ratio vs warmup_steps.
    """
    supported = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
    kwargs = dict(kwargs)
    if "eval_strategy" in kwargs and "eval_strategy" not in supported:
        if "evaluation_strategy" in supported:
            kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
        else:
            kwargs.pop("eval_strategy", None)
    if "evaluation_strategy" in kwargs and "evaluation_strategy" not in supported:
        if "eval_strategy" in supported:
            kwargs["eval_strategy"] = kwargs.pop("evaluation_strategy")
        else:
            kwargs.pop("evaluation_strategy", None)
    if "warmup_ratio" in kwargs and "warmup_ratio" not in supported:
        ratio = float(kwargs.pop("warmup_ratio"))
        kwargs.setdefault("warmup_steps", max(1, int(ratio * 1000)))
    filtered = {k: v for k, v in kwargs.items() if k in supported}
    dropped = sorted(set(kwargs) - set(filtered))
    if dropped:
        print("Dropped unsupported TrainingArguments kwargs:", dropped)
    return TrainingArguments(**filtered)
def add_tokenizer_arg_for_trainer(kwargs: Dict[str, Any], tokenizer):
    supported = set(inspect.signature(Trainer.__init__).parameters.keys())
    kwargs = dict(kwargs)
    if "processing_class" in supported:
        kwargs["processing_class"] = tokenizer
    elif "tokenizer" in supported:
        kwargs["tokenizer"] = tokenizer
    return kwargs
def is_large_backbone(model_name_or_path: str) -> bool:
    return "large" in str(model_name_or_path).lower()
def precision_for_backbone(model_name_or_path: str) -> str:
    """RoBERTa-only precision helper."""
    if CONFIG.get("task_b_force_fp32_all", False):
        return "fp32"
    if torch.cuda.is_available():
        return "fp16"
    return "fp32"
def precision_flags(precision: str) -> Tuple[bool, bool]:
    precision = str(precision).lower().strip()
    if precision == "fp16" and torch.cuda.is_available():
        return False, True
    return False, False
class WeightedTrainer(Trainer):
    """Trainer that supports class-weighted CE during *training only*.
    Why this exists: HuggingFace `Trainer` calls `compute_loss` for both
    training and evaluation. We want training to use class-weighted cross
    entropy (because `therapist_input` is the minority class at ~12.5% and
    contributes disproportionately little gradient under unweighted CE),
    but evaluation should use *unweighted* CE so that:
      (a) the reported `eval_loss` is directly comparable to the loss used
          by the post-hoc temperature-scaling fit downstream,
      (b) the eval-set loss reflects the true held-out distribution rather
          than a re-weighted training objective.
    The model selection metric is macro-F1, not eval_loss, so this change
    does not affect which checkpoint is chosen — but it makes the loss
    numbers in the trainer log internally consistent.
    """
    def __init__(
        self,
        class_weights: Optional[torch.Tensor] = None,
        nonfinite_check: bool = True,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.nonfinite_check = bool(nonfinite_check)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if self.nonfinite_check and not torch.isfinite(logits).all():
            raise FloatingPointError("Non-finite logits detected during train/eval.")
        weight = (
            self.class_weights.to(logits.device)
            if (self.class_weights is not None and model.training)
            else None
        )
        loss_fct = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits.float(), labels)
        if self.nonfinite_check and not torch.isfinite(loss).all():
            raise FloatingPointError("Non-finite loss detected during train/eval.")
        return (loss, outputs) if return_outputs else loss
def _remove_final_current_from_context(combined: str, current_raw: str) -> str:
    combined = "" if pd.isna(combined) else str(combined)
    current_raw = "" if pd.isna(current_raw) else str(current_raw)
    candidates = [
        f"[THERAPIST] {current_raw}",
        f"[therapist] {current_raw}",
        f"THERAPIST: {current_raw}",
        f"therapist: {current_raw}",
        current_raw,
    ]
    hist = combined.strip()
    for marker in candidates:
        marker = str(marker).strip()
        if marker and hist.endswith(marker):
            hist = hist[: -len(marker)].strip()
            break
    return hist
def make_hf_dataset(df: pd.DataFrame, text_col: str, label_col: str = "y_code") -> HFDataset:
    required_cols = [text_col, "utterance_text", label_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(
            f"Missing required columns for current-preserving tokenization: {missing}. "
            f"Available columns include: {list(df.columns)[:30]}"
        )
    local_df = df[required_cols].copy()
    local_df[text_col] = local_df[text_col].astype(str)
    local_df["utterance_text"] = local_df["utterance_text"].astype(str)
    local_df[label_col] = local_df[label_col].astype(int)
    history_texts = [
        _remove_final_current_from_context(combined, current_raw)
        for combined, current_raw in zip(local_df[text_col], local_df["utterance_text"])
    ]
    current_texts = [
        f"[CURRENT_THERAPIST] {txt}"
        for txt in local_df["utterance_text"].astype(str).tolist()
    ]
    pair_df = pd.DataFrame(
        {
            "history_text": history_texts,
            "current_text": current_texts,
            "labels": local_df[label_col].to_numpy(dtype=int),
        }
    )
    return HFDataset.from_pandas(pair_df, preserve_index=False)
def tokenize_dataset(ds: HFDataset, tokenizer, max_length: int):
    return ds.map(
        lambda batch: tokenizer(
            batch["history_text"],
            batch["current_text"],
            truncation="only_first",
            max_length=max_length,
        ),
        batched=True,
        remove_columns=["history_text", "current_text"],
    )
def get_class_weights(y: np.ndarray, n_classes: int) -> torch.Tensor:
    y = np.asarray(y, dtype=int)
    present_classes = np.unique(y)
    present_weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y,
    )
    full_weights = np.ones(n_classes, dtype=np.float32)
    for cls_id, w in zip(present_classes, present_weights):
        full_weights[int(cls_id)] = float(w)
    return torch.tensor(full_weights, dtype=torch.float32)
def hf_metrics(eval_pred):
    logits, labels = eval_pred
    if not np.isfinite(logits).all():
        raise FloatingPointError("Non-finite logits detected in compute_metrics.")
    preds = logits.argmax(axis=-1)
    metrics = compute_multiclass_metrics(labels, preds)
    return {
        "accuracy": metrics["accuracy"],
        "precision_macro": metrics["precision_macro"],
        "recall_macro": metrics["recall_macro"],
        "f1_macro": metrics["f1_macro"],
    }
def train_encoder_once(
    model_name_or_path: str,
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    text_col: str,
    output_dir: Path,
    max_length: int = 256,
    learning_rate: float = 2e-5,
    weight_decay: float = 0.01,
    epochs: int = 4,
    train_batch_size: int = 4,
    eval_batch_size: int = 8,
    gradient_accumulation_steps: int = 4,
    precision: Optional[str] = None,
    gradient_checkpointing: Optional[bool] = None,
    seed: Optional[int] = None,
    early_stopping_patience: Optional[int] = None,
    save_best_model: bool = True,
):
    """
    Trains an encoder for sequence classification.
    Trains the selected encoder with best-epoch checkpointing turned on for
    every run that has a validation set.
    """
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir, ignore_errors=True)
    output_dir.mkdir(parents=True, exist_ok=True)
    clear_cuda()
    run_seed = int(seed) if seed is not None else int(globals().get("SEED", 42))
    precision = precision or precision_for_backbone(model_name_or_path)
    use_bf16, use_fp16 = precision_flags(precision)
    if gradient_checkpointing is None:
        gradient_checkpointing = bool(
            (
                CONFIG.get("task_b_selected_gradient_checkpointing", False)
                or CONFIG.get("task_b_large_gradient_checkpointing", False)
            )
            and is_large_backbone(model_name_or_path)
        )
    if early_stopping_patience is None:
        early_stopping_patience = int(CONFIG.get("task_b_early_stopping_patience", 2))
    print(
        f"Training {model_name_or_path} | "
        f"precision={precision}, bf16={use_bf16}, fp16={use_fp16}, "
        f"gradient_checkpointing={gradient_checkpointing}, "
        f"seed={run_seed}, save_best_model={save_best_model}, "
        f"early_stopping_patience={early_stopping_patience}"
    )
    tokenizer = load_tokenizer_safe_fixed(model_name_or_path, max_length=max_length)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name_or_path,
        num_labels=len(label_encoder.classes_),
        trust_remote_code=True,
    )
    model.float()
    model.config.problem_type = "single_label_classification"
    if getattr(model.config, "pad_token_id", None) is None and getattr(tokenizer, "pad_token_id", None) is not None:
        model.config.pad_token_id = tokenizer.pad_token_id
    if gradient_checkpointing:
        if hasattr(model, "gradient_checkpointing_enable"):
            model.gradient_checkpointing_enable()
        if hasattr(model.config, "use_cache"):
            model.config.use_cache = False
    train_ds = tokenize_dataset(
        make_hf_dataset(train_df, text_col=text_col),
        tokenizer,
        max_length=max_length,
    )
    val_ds = tokenize_dataset(
        make_hf_dataset(val_df, text_col=text_col),
        tokenizer,
        max_length=max_length,
    )
    collator = DataCollatorWithPadding(
        tokenizer=tokenizer,
        pad_to_multiple_of=8 if torch.cuda.is_available() else None,
    )
    class_weights = get_class_weights(
        train_df["y_code"].to_numpy(),
        n_classes=len(label_encoder.classes_),
    )
    args = make_training_args_compat(
        output_dir=str(output_dir),
        overwrite_output_dir=True,
        eval_strategy="epoch",
        save_strategy="epoch" if save_best_model else "no",
        load_best_model_at_end=bool(save_best_model),
        logging_strategy="steps",
        logging_steps=50,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        num_train_epochs=epochs,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        report_to=[],
        bf16=use_bf16,
        fp16=use_fp16,
        save_total_limit=1,
        max_grad_norm=1.0,
        warmup_ratio=0.06,
        logging_nan_inf_filter=False,
        seed=run_seed,
        data_seed=run_seed,
    )
    trainer_kwargs = dict(
        class_weights=class_weights,
        nonfinite_check=True,
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=hf_metrics,
    )
    if save_best_model and early_stopping_patience and early_stopping_patience > 0:
        trainer_kwargs["callbacks"] = [
            EarlyStoppingCallback(early_stopping_patience=int(early_stopping_patience))
        ]
    trainer_kwargs = add_tokenizer_arg_for_trainer(trainer_kwargs, tokenizer)
    trainer = WeightedTrainer(**trainer_kwargs)
    start = time.time()
    trainer.train()
    runtime_seconds = time.time() - start
    val_pred = trainer.predict(val_ds)
    val_logits = np.asarray(val_pred.predictions, dtype=np.float32)
    if not np.isfinite(val_logits).all():
        raise FloatingPointError("Non-finite validation logits detected after training.")
    val_probs = softmax(val_logits, axis=1)
    if not np.isfinite(val_probs).all():
        raise FloatingPointError("Non-finite validation probabilities detected after softmax.")
    val_preds = val_logits.argmax(axis=1)
    val_metrics = compute_multiclass_metrics(val_pred.label_ids, val_preds, val_probs)
    val_metrics["runtime_seconds"] = float(runtime_seconds)
    return {
        "trainer": trainer,
        "tokenizer": tokenizer,
        "val_logits": val_logits,
        "val_probs": val_probs,
        "val_preds": val_preds,
        "val_labels": val_pred.label_ids,
        "val_metrics": val_metrics,
        "actual_precision": precision,
        "actual_train_batch_size": train_batch_size,
        "actual_eval_batch_size": eval_batch_size,
        "actual_gradient_accumulation_steps": gradient_accumulation_steps,
        "actual_gradient_checkpointing": bool(gradient_checkpointing),
        "actual_seed": run_seed,
    }
def candidate_key_from_values(base_model, context_k, text_col, max_length, learning_rate, weight_decay):
    return "|".join(
        [
            str(base_model),
            str(int(context_k)),
            str(text_col),
            str(int(max_length)),
            f"{float(learning_rate):.12g}",
            f"{float(weight_decay):.12g}",
        ]
    )
def candidate_key_from_row(row):
    return candidate_key_from_values(
        row["base_model"],
        row["context_k"],
        row["text_col"],
        row["max_length"],
        row["learning_rate"],
        row["weight_decay"],
    )
print("Loaded fixed Task b encoder helper/trainer (v2).")
print("Key change: best-epoch checkpointing is now ALWAYS on for runs with a val set.")


In [ ]:
required_names = [
    "CONFIG",
    "therapist_train",
    "train_encoder_once",
    "artifact_exists",
    "save_df",
    "save_json",
    "LABEL_COL",
    "GROUP_COL",
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required shared neural helper objects before the RoBERTa search cell: "
        f"{missing_names}. Run the earlier setup/split/helper cells first."
    )
if "task_b_dir" not in globals():
    task_b_dir = ARTIFACT_DIR / "task_b"
    task_b_dir.mkdir(parents=True, exist_ok=True)
if "task_b_encoder_dir" not in globals():
    task_b_encoder_dir = task_b_dir / "encoder"
    task_b_encoder_dir.mkdir(parents=True, exist_ok=True)
CONFIG["task_b_hp_search_seed"] = int(CONFIG.get("task_b_hp_search_seed", 42))
CONFIG["task_b_hp_search_epochs"] = int(CONFIG.get("task_b_hp_search_epochs", 3))
CONFIG["task_b_multiseed_top_k"] = int(CONFIG.get("task_b_multiseed_top_k", 3))
CONFIG["task_b_multiseed_seeds"] = list(CONFIG.get("task_b_multiseed_seeds", [17, 42, 101]))
CONFIG["task_b_multiseed_epochs"] = int(CONFIG.get("task_b_multiseed_epochs", 3))
CONFIG["task_b_context_grid_neural"] = list(CONFIG.get("task_b_context_grid_neural", [3, 5, 10]))
CONFIG["task_b_max_length_grid_neural"] = list(CONFIG.get("task_b_max_length_grid_neural", [256, 384]))
CONFIG["task_b_roberta_lr_grid"] = list(CONFIG.get("task_b_roberta_lr_grid", [1e-5, 2e-5, 3e-5]))
CONFIG["task_b_weight_decay_grid_neural"] = list(CONFIG.get("task_b_weight_decay_grid_neural", [0.01]))
CONFIG["task_b_early_stopping_patience"] = int(CONFIG.get("task_b_early_stopping_patience", 2))
CONFIG["task_b_force_fp32_all"] = bool(CONFIG.get("task_b_force_fp32_all", False))
def _cfg_list(values: List[Any]) -> str:
    return ", ".join(str(v) for v in values)

roberta_search_config_df = pd.DataFrame(
    [
        {
            "category": "Search control",
            "setting": "HP-search seed",
            "value": CONFIG["task_b_hp_search_seed"],
            "purpose": "Keeps candidate search reproducible",
        },
        {
            "category": "Search control",
            "setting": "HP-search epochs",
            "value": CONFIG["task_b_hp_search_epochs"],
            "purpose": "Short pilot training for each candidate",
        },
        {
            "category": "Multi-seed confirmation",
            "setting": "Top candidates retained",
            "value": CONFIG["task_b_multiseed_top_k"],
            "purpose": "Promotes only the strongest pilot configurations",
        },
        {
            "category": "Multi-seed confirmation",
            "setting": "Confirmation seeds",
            "value": _cfg_list(CONFIG["task_b_multiseed_seeds"]),
            "purpose": "Checks stability across independent random seeds",
        },
        {
            "category": "Multi-seed confirmation",
            "setting": "Confirmation epochs",
            "value": CONFIG["task_b_multiseed_epochs"],
            "purpose": "Matches pilot training length for fair comparison",
        },
        {
            "category": "Candidate grid",
            "setting": "Context window k",
            "value": _cfg_list(CONFIG["task_b_context_grid_neural"]),
            "purpose": "Tests how much prior dialogue history to include",
        },
        {
            "category": "Candidate grid",
            "setting": "Max token length",
            "value": _cfg_list(CONFIG["task_b_max_length_grid_neural"]),
            "purpose": "Controls truncation budget for RoBERTa inputs",
        },
        {
            "category": "Candidate grid",
            "setting": "Learning rate",
            "value": _cfg_list(CONFIG["task_b_roberta_lr_grid"]),
            "purpose": "Main optimiser sensitivity axis",
        },
        {
            "category": "Candidate grid",
            "setting": "Weight decay",
            "value": _cfg_list(CONFIG["task_b_weight_decay_grid_neural"]),
            "purpose": "Regularisation strength",
        },
        {
            "category": "Training guardrails",
            "setting": "Early-stopping patience",
            "value": CONFIG["task_b_early_stopping_patience"],
            "purpose": "Stops runs that stop improving on validation macro-F1",
        },
        {
            "category": "Training guardrails",
            "setting": "Force FP32",
            "value": CONFIG["task_b_force_fp32_all"],
            "purpose": "Uses full precision only when explicitly requested",
        },
    ]
)
display(Markdown("### Table 6.2A. RoBERTa neural-search configuration summary"))
if "_display_section5_table" in globals():
    _display_section5_table(
        roberta_search_config_df,
        row_color_col="category",
        row_colors={
            "Search control": "#e7f0ff",
            "Multi-seed confirmation": "#e9f7ef",
            "Candidate grid": "#fff3d9",
            "Training guardrails": "#fce8ef",
        },
        cell_col_colors={"category": "#f1f5f9"},
        text_cols=["purpose"],
        nowrap_cols=["category", "setting", "value"],
    )
else:
    display(roberta_search_config_df)
def set_all_seeds(seed: int) -> None:
    seed = int(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
def grouped_train_val_split(
    train_df: pd.DataFrame,
    label_col: str = "main_therapist_behaviour",
    group_col: str = "transcript_id",
    seed: int = 42,
    n_splits: int = 5,
):
    y = train_df[label_col].astype(str).to_numpy()
    groups = train_df[group_col].astype(str).to_numpy()
    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=int(seed),
    )
    overall_dist = pd.Series(y).value_counts(normalize=True).sort_index()
    best_pair = None
    best_score = None
    for tr_idx, val_idx in splitter.split(train_df, y, groups):
        val_dist = (
            pd.Series(y[val_idx])
            .value_counts(normalize=True)
            .reindex(overall_dist.index, fill_value=0.0)
            .sort_index()
        )
        score = float(np.abs(val_dist - overall_dist).mean())
        if best_score is None or score < best_score:
            best_score = score
            best_pair = (tr_idx, val_idx)
    inner_train_idx, inner_val_idx = best_pair
    inner_train = train_df.iloc[inner_train_idx].copy().reset_index(drop=True)
    inner_val = train_df.iloc[inner_val_idx].copy().reset_index(drop=True)
    return inner_train, inner_val
def grouped_train_val_split_for_seed(
    train_df: pd.DataFrame,
    seed: int,
    n_splits: int = 5,
):
    return grouped_train_val_split(
        train_df=train_df,
        label_col=LABEL_COL,
        group_col=GROUP_COL,
        seed=int(seed),
        n_splits=int(n_splits),
    )
def precision_for_backbone_search(model_name: str) -> str:
    if CONFIG.get("task_b_force_fp32_all", False):
        return "fp32"
    if torch.cuda.is_available():
        return "fp16"
    return "fp32"
def candidate_key_from_values(base_model, context_k, text_col, max_length, learning_rate, weight_decay):
    return "|".join(
        [
            str(base_model),
            str(int(context_k)),
            str(text_col),
            str(int(max_length)),
            f"{float(learning_rate):.12g}",
            f"{float(weight_decay):.12g}",
        ]
    )
def _safe_name(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(text))[:180]
def call_train_encoder_once_safely(**kwargs):
    try:
        out = train_encoder_once(**kwargs)
        out["oom_retry"] = False
        return out
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        if "clear_cuda" in globals():
            clear_cuda()
        retry_kwargs = dict(kwargs)
        retry_kwargs["train_batch_size"] = max(1, int(kwargs.get("train_batch_size", 2)) // 2)
        retry_kwargs["gradient_accumulation_steps"] = int(kwargs.get("gradient_accumulation_steps", 1)) * 2
        out = train_encoder_once(**retry_kwargs)
        out["oom_retry"] = True
        return out
def train_candidate_once(
    candidate_cfg: Dict[str, Any],
    seed: int,
    stage: str,
    epochs: int,
) -> Dict[str, Any]:
    row = dict(candidate_cfg)
    row["seed"] = int(seed)
    row["stage"] = str(stage)
    try:
        inner_train, inner_val = grouped_train_val_split_for_seed(
            therapist_train,
            seed=int(seed),
            n_splits=5,
        )
        set_all_seeds(seed)
        output_dir = task_b_encoder_dir / f"{stage}_{_safe_name(candidate_cfg['candidate_key'])}_seed{seed}"
        result = call_train_encoder_once_safely(
            model_name_or_path=candidate_cfg["base_model"],
            train_df=inner_train,
            val_df=inner_val,
            text_col=candidate_cfg["text_col"],
            output_dir=output_dir,
            max_length=int(candidate_cfg["max_length"]),
            learning_rate=float(candidate_cfg["learning_rate"]),
            weight_decay=float(candidate_cfg["weight_decay"]),
            epochs=int(epochs),
            train_batch_size=int(candidate_cfg["train_batch_size"]),
            eval_batch_size=int(candidate_cfg["eval_batch_size"]),
            gradient_accumulation_steps=int(candidate_cfg["gradient_accumulation_steps"]),
            precision=str(candidate_cfg["precision"]),
            gradient_checkpointing=bool(candidate_cfg.get("gradient_checkpointing", False)),
            seed=int(seed),
            early_stopping_patience=int(CONFIG["task_b_early_stopping_patience"]),
            save_best_model=True,
        )
        row.update(result["val_metrics"])
        row["oom_retry"] = bool(result.get("oom_retry", False))
        row["error"] = None
        row["actual_precision"] = result.get("actual_precision", candidate_cfg["precision"])
        row["actual_train_batch_size"] = int(result.get("actual_train_batch_size", candidate_cfg["train_batch_size"]))
        row["actual_eval_batch_size"] = int(result.get("actual_eval_batch_size", candidate_cfg["eval_batch_size"]))
        row["actual_gradient_accumulation_steps"] = int(
            result.get("actual_gradient_accumulation_steps", candidate_cfg["gradient_accumulation_steps"])
        )
        return row
    except Exception as e:
        row["accuracy"] = np.nan
        row["precision_macro"] = np.nan
        row["recall_macro"] = np.nan
        row["f1_macro"] = np.nan
        row["precision_weighted"] = np.nan
        row["recall_weighted"] = np.nan
        row["f1_weighted"] = np.nan
        row["brier_multiclass"] = np.nan
        row["runtime_seconds"] = np.nan
        row["oom_retry"] = False
        row["error"] = str(e)
        return row
def run_candidate_list_for_seed(
    candidate_cfgs: List[Dict[str, Any]],
    seed: int,
    stage: str,
    epochs: int,
    partial_path: Optional[Path] = None,
) -> pd.DataFrame:
    rows = []
    for cfg in candidate_cfgs:
        row = train_candidate_once(
            candidate_cfg=cfg,
            seed=int(seed),
            stage=str(stage),
            epochs=int(epochs),
        )
        rows.append(row)
        if partial_path is not None:
            save_df(pd.DataFrame(rows), partial_path)
    out_df = pd.DataFrame(rows)
    if "f1_macro" in out_df.columns:
        out_df = out_df.sort_values(
            ["f1_macro", "accuracy"],
            ascending=[False, False],
            na_position="last",
        ).reset_index(drop=True)
    if partial_path is not None:
        save_df(out_df, partial_path)
    return out_df

In [ ]:
print("compute_class_weight imported successfully")

In [ ]:
required_names = [
    "CONFIG",
    "therapist_train",
    "therapist_test",
    "y_test_b",
    "label_encoder",
    "artifact_exists",
    "save_df",
    "save_json",
    "save_joblib",
    "load_json",
    "load_joblib",
    "candidate_key_from_values",
    "precision_for_backbone_search",
    "run_candidate_list_for_seed",
    "grouped_train_val_split_for_seed",
    "grouped_train_val_split",
    "call_train_encoder_once_safely",
    "make_hf_dataset",
    "tokenize_dataset",
    "compute_multiclass_metrics",
    "report_df",
    "softmax",
    "set_all_seeds",
    "task_b_encoder_dir",
]
missing_names = [name for name in required_names if name not in globals()]
if missing_names:
    raise NameError(
        "Missing required Task B encoder helpers before running the RoBERTa cell: "
        f"{missing_names}. Run the shared RoBERTa neural helper/search cell first."
    )
def _format_roberta_table_value(value: Any, fmt: Optional[str] = None) -> str:
    if value is None:
        return ""
    try:
        missing = pd.isna(value)
        if isinstance(missing, (bool, np.bool_)) and missing:
            return ""
    except Exception:
        pass
    if fmt is not None:
        try:
            return fmt.format(value)
        except Exception:
            return str(value)
    return str(value)
def _display_roberta_table(
    title: str,
    df: pd.DataFrame,
    *,
    row_color_col: Optional[str] = None,
    row_colors: Optional[Dict[str, str]] = None,
    cell_col_colors: Optional[Dict[str, str]] = None,
    formats: Optional[Dict[str, str]] = None,
    text_cols: Optional[List[str]] = None,
    nowrap_cols: Optional[List[str]] = None,
) -> None:
    display(Markdown(title))
    table_df = df.copy()
    row_colors = row_colors or {}
    cell_col_colors = cell_col_colors or {}
    formats = formats or {}
    text_cols = text_cols or []
    nowrap_cols = nowrap_cols or []
    def _row_style(row):
        row_key = str(row.get(row_color_col, "")) if row_color_col else ""
        background = row_colors.get(row_key, "#ffffff")
        return [
            f"background-color: {background}; color: #111827; font-weight: 650; border: 1px solid #cbd5e1; vertical-align: top;"
            for _ in row
        ]
    try:
        styler = (
            table_df.style
            .hide(axis="index")
            .format(formats, na_rep="")
            .apply(_row_style, axis=1)
            .set_table_styles(
                [
                    {
                        "selector": "th",
                        "props": [
                            ("background-color", "#f8fafc"),
                            ("color", "#111827"),
                            ("font-weight", "800"),
                            ("text-align", "left"),
                            ("border", "1px solid #cbd5e1"),
                            ("padding", "5px 7px"),
                            ("font-size", "11.5px"),
                        ],
                    },
                    {
                        "selector": "td",
                        "props": [
                            ("padding", "5px 7px"),
                            ("font-size", "11.5px"),
                            ("line-height", "1.22"),
                        ],
                    },
                    {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
                ]
            )
        )
        for col, color in cell_col_colors.items():
            if col in table_df.columns:
                styler = styler.set_properties(subset=[col], **{"background-color": color})
        for col in text_cols:
            if col in table_df.columns:
                styler = styler.set_properties(subset=[col], **{"max-width": "320px", "white-space": "normal"})
        for col in nowrap_cols:
            if col in table_df.columns:
                styler = styler.set_properties(subset=[col], **{"white-space": "nowrap"})
        display(styler)
    except Exception:
        header_html = "".join(
            f"<th style='background-color: #f8fafc; color: #111827; font-weight: 800; text-align: left; border: 1px solid #cbd5e1; padding: 5px 7px; font-size: 11.5px;'>{escape(str(col))}</th>"
            for col in table_df.columns
        )
        body_html = ""
        for _, row in table_df.iterrows():
            row_key = str(row.get(row_color_col, "")) if row_color_col else ""
            row_bg = row_colors.get(row_key, "#ffffff")
            body_html += "<tr>"
            for col in table_df.columns:
                background = cell_col_colors.get(col, row_bg)
                wrap = "max-width: 320px; white-space: normal;" if col in text_cols else "white-space: nowrap;"
                value_text = _format_roberta_table_value(row[col], formats.get(col))
                body_html += (
                    f"<td style='background-color: {background}; color: #111827; font-weight: 650; border: 1px solid #cbd5e1; padding: 5px 7px; font-size: 11.5px; line-height: 1.22; vertical-align: top; {wrap}'>"
                    f"{escape(value_text)}</td>"
                )
            body_html += "</tr>"
        display(HTML(f"<table style='border-collapse: collapse; width: 100%; color: #111827;'><thead><tr>{header_html}</tr></thead><tbody>{body_html}</tbody></table>"))
_roberta_context_colors = {"3": "#e7f0ff", "5": "#e9f7ef", "10": "#fff3d9"}
_roberta_class_colors = {
    "reflection": "#e7f0ff",
    "question": "#e9f7ef",
    "therapist_input": "#fff3d9",
    "other": "#fce8ef",
    "accuracy": "#f1f5f9",
    "macro avg": "#f1f5f9",
    "weighted avg": "#f1f5f9",
}
_roberta_metric_formats = {
    "accuracy": "{:.4f}",
    "precision_macro": "{:.4f}",
    "recall_macro": "{:.4f}",
    "f1_macro": "{:.4f}",
    "precision_weighted": "{:.4f}",
    "recall_weighted": "{:.4f}",
    "f1_weighted": "{:.4f}",
    "brier_multiclass": "{:.4f}",
    "mean_f1_macro": "{:.4f}",
    "std_f1_macro": "{:.4f}",
    "min_f1_macro": "{:.4f}",
    "max_f1_macro": "{:.4f}",
    "mean_accuracy": "{:.4f}",
    "std_accuracy": "{:.4f}",
    "learning_rate": "{:.2e}",
    "weight_decay": "{:.4f}",
    "support": "{:.0f}",
}
ROBERTA_MODEL = "FacebookAI/roberta-base"
task_b_roberta_dir = task_b_encoder_dir / "roberta_base"
task_b_roberta_dir.mkdir(parents=True, exist_ok=True)
task_b_roberta_pilot_results_path = task_b_roberta_dir / "pilot_results.csv"
task_b_roberta_pilot_partial_path = task_b_roberta_dir / "pilot_results_partial.csv"
task_b_roberta_multiseed_results_path = task_b_roberta_dir / "multiseed_results.csv"
task_b_roberta_multiseed_summary_path = task_b_roberta_dir / "multiseed_summary.csv"
task_b_roberta_cfg_path = task_b_roberta_dir / "roberta_cfg.json"
task_b_roberta_metrics_path = task_b_roberta_dir / "roberta_metrics.json"
task_b_roberta_preds_path = task_b_roberta_dir / "roberta_preds.joblib"
task_b_roberta_meta_path = task_b_roberta_dir / "roberta_meta.json"
CONFIG.setdefault("task_b_roberta_lr_grid", [1e-5, 2e-5, 3e-5])
def build_task_b_roberta_candidate_grid() -> List[Dict[str, Any]]:
    candidates = []
    run_id = 0
    for context_k in CONFIG.get("task_b_context_grid_neural", [3, 5, 10]):
        text_col = f"context_k{int(context_k)}_text"
        for max_length in CONFIG.get("task_b_max_length_grid_neural", [256, 384]):
            for lr in CONFIG.get("task_b_roberta_lr_grid", [1e-5, 2e-5, 3e-5]):
                for wd in CONFIG.get("task_b_weight_decay_grid_neural", [0.01]):
                    run_id += 1
                    cfg = {
                        "run_id": int(run_id),
                        "base_model": ROBERTA_MODEL,
                        "context_k": int(context_k),
                        "text_col": text_col,
                        "max_length": int(max_length),
                        "learning_rate": float(lr),
                        "weight_decay": float(wd),
                        "precision": precision_for_backbone_search(ROBERTA_MODEL),
                        "train_batch_size": int(CONFIG["task_b_train_batch_size"]),
                        "eval_batch_size": int(CONFIG["task_b_eval_batch_size"]),
                        "gradient_accumulation_steps": int(CONFIG["task_b_grad_accum"]),
                        "gradient_checkpointing": False,
                    }
                    cfg["candidate_key"] = candidate_key_from_values(
                        cfg["base_model"],
                        cfg["context_k"],
                        cfg["text_col"],
                        cfg["max_length"],
                        cfg["learning_rate"],
                        cfg["weight_decay"],
                    )
                    candidates.append(cfg)
    return candidates
search_artifacts_ready = (
    artifact_exists(task_b_roberta_pilot_results_path)
    and artifact_exists(task_b_roberta_multiseed_results_path)
    and artifact_exists(task_b_roberta_multiseed_summary_path)
    and artifact_exists(task_b_roberta_cfg_path)
    and not CONFIG.get("force_retrain", False)
)
if search_artifacts_ready:
    task_b_roberta_pilot_results = pd.read_csv(task_b_roberta_pilot_results_path)
    task_b_roberta_multiseed_results = pd.read_csv(task_b_roberta_multiseed_results_path)
    task_b_roberta_multiseed_summary = pd.read_csv(task_b_roberta_multiseed_summary_path)
    task_b_roberta_cfg = load_json(task_b_roberta_cfg_path)
    _display_roberta_table(
        "### Table 6.2B. Cached RoBERTa multi-seed search summary",
        task_b_roberta_multiseed_summary.head(10),
        row_color_col="context_k",
        row_colors=_roberta_context_colors,
        cell_col_colors={"mean_f1_macro": "#e9f7ef", "std_f1_macro": "#fff3d9", "mean_accuracy": "#e7f0ff", "errors": "#fce8ef"},
        formats=_roberta_metric_formats,
        text_cols=["candidate_key", "errors"],
        nowrap_cols=["base_model", "context_k", "max_length", "precision", "successful_seeds", "total_seeds"],
    )
else:
    task_b_roberta_candidate_cfgs = build_task_b_roberta_candidate_grid()
    task_b_roberta_candidate_by_key = {
        cfg["candidate_key"]: cfg for cfg in task_b_roberta_candidate_cfgs
    }
    display(Markdown("## RoBERTa stage 1: grouped-validation hyperparameter search"))
    roberta_candidate_count_df = (
        pd.DataFrame(task_b_roberta_candidate_cfgs)
        .groupby(["base_model", "precision"])
        .size()
        .reset_index(name="n_candidates")
    )
    _display_roberta_table(
        "### Table 6.2B. RoBERTa stage-1 candidate counts by precision",
        roberta_candidate_count_df,
        cell_col_colors={"base_model": "#f1f5f9", "precision": "#e7f0ff", "n_candidates": "#e9f7ef"},
        nowrap_cols=roberta_candidate_count_df.columns.tolist(),
    )
    _display_roberta_table(
        "### Table 6.2C. RoBERTa stage-1 candidate grid",
        pd.DataFrame(task_b_roberta_candidate_cfgs),
        row_color_col="context_k",
        row_colors=_roberta_context_colors,
        cell_col_colors={"learning_rate": "#e7f0ff", "max_length": "#e9f7ef", "precision": "#f1f5f9"},
        formats=_roberta_metric_formats,
        text_cols=["candidate_key"],
        nowrap_cols=["run_id", "base_model", "context_k", "text_col", "max_length", "learning_rate", "weight_decay", "precision"],
    )
    original_task_b_encoder_dir = task_b_encoder_dir
    try:
        task_b_encoder_dir = task_b_roberta_dir
        task_b_roberta_pilot_results = run_candidate_list_for_seed(
            candidate_cfgs=task_b_roberta_candidate_cfgs,
            seed=int(CONFIG["task_b_hp_search_seed"]),
            stage="roberta_hp_search",
            epochs=int(CONFIG["task_b_hp_search_epochs"]),
            partial_path=task_b_roberta_pilot_partial_path,
        )
        save_df(task_b_roberta_pilot_results, task_b_roberta_pilot_results_path)
        valid_pilot_df = task_b_roberta_pilot_results.dropna(subset=["f1_macro"]).copy()
        valid_pilot_df = valid_pilot_df[
            np.isfinite(pd.to_numeric(valid_pilot_df["f1_macro"], errors="coerce"))
        ].copy()
        if len(valid_pilot_df) == 0:
            if "error" in task_b_roberta_pilot_results.columns:
                _display_roberta_table(
                    "### Table 6.2D. RoBERTa failed candidate diagnostics",
                    task_b_roberta_pilot_results[
                        ["run_id", "context_k", "max_length", "learning_rate", "error"]
                    ].head(50),
                    row_color_col="context_k",
                    row_colors=_roberta_context_colors,
                    cell_col_colors={"error": "#fce8ef", "learning_rate": "#e7f0ff"},
                    formats=_roberta_metric_formats,
                    text_cols=["error"],
                    nowrap_cols=["run_id", "context_k", "max_length", "learning_rate"],
                )
            raise RuntimeError(
                "All RoBERTa hyperparameter candidates failed. "
                "Check the displayed error column and pilot_results.csv."
            )
        top_k = int(CONFIG.get("task_b_multiseed_top_k", 3))
        top_k = max(1, min(top_k, len(valid_pilot_df)))
        top_candidate_keys = valid_pilot_df.head(top_k)["candidate_key"].tolist()
        top_candidate_cfgs = [task_b_roberta_candidate_by_key[k] for k in top_candidate_keys]
        _display_roberta_table(
            "### Table 6.2E. RoBERTa stage-1 top configurations",
            valid_pilot_df.head(top_k),
            row_color_col="context_k",
            row_colors=_roberta_context_colors,
            cell_col_colors={"f1_macro": "#e9f7ef", "accuracy": "#e7f0ff", "runtime_seconds": "#fff3d9", "error": "#fce8ef"},
            formats=_roberta_metric_formats,
            text_cols=["candidate_key", "error"],
            nowrap_cols=["run_id", "seed", "context_k", "max_length", "learning_rate", "precision"],
        )
        multiseed_rows = []
        display(Markdown("## RoBERTa stage 2: multi-seed confirmation"))
        for rank, cfg in enumerate(top_candidate_cfgs, start=1):
            for seed in list(CONFIG["task_b_multiseed_seeds"]):
                cfg_for_seed = dict(cfg)
                cfg_for_seed["run_id"] = int(cfg["run_id"])
                ms_df = run_candidate_list_for_seed(
                    candidate_cfgs=[cfg_for_seed],
                    seed=int(seed),
                    stage=f"roberta_multiseed_rank{rank}",
                    epochs=int(CONFIG["task_b_multiseed_epochs"]),
                    partial_path=task_b_roberta_dir / f"multiseed_partial_rank{rank}_seed{seed}.csv",
                )
                if len(ms_df) > 0:
                    multiseed_rows.extend(ms_df.to_dict("records"))
                save_df(pd.DataFrame(multiseed_rows), task_b_roberta_multiseed_results_path)
        task_b_roberta_multiseed_results = pd.DataFrame(multiseed_rows)
        if len(task_b_roberta_multiseed_results) == 0:
            raise RuntimeError("RoBERTa multi-seed confirmation produced no rows.")
        save_df(task_b_roberta_multiseed_results, task_b_roberta_multiseed_results_path)
    finally:
        task_b_encoder_dir = original_task_b_encoder_dir
    _display_roberta_table(
        "### Table 6.2F. RoBERTa multi-seed raw results",
        task_b_roberta_multiseed_results,
        row_color_col="context_k",
        row_colors=_roberta_context_colors,
        cell_col_colors={"f1_macro": "#e9f7ef", "accuracy": "#e7f0ff", "runtime_seconds": "#fff3d9", "error": "#fce8ef"},
        formats=_roberta_metric_formats,
        text_cols=["candidate_key", "error"],
        nowrap_cols=["run_id", "seed", "stage", "context_k", "max_length", "learning_rate", "precision", "oom_retry"],
    )
    summary_rows = []
    for key, g_all in task_b_roberta_multiseed_results.groupby("candidate_key"):
        g = g_all.copy()
        f1 = pd.to_numeric(g["f1_macro"], errors="coerce")
        acc = pd.to_numeric(g["accuracy"], errors="coerce")
        finite_mask = np.isfinite(f1)
        g_success = g.loc[finite_mask].copy()
        first = g.iloc[0].to_dict()
        if len(g_success) > 0:
            f1_success = pd.to_numeric(g_success["f1_macro"], errors="coerce")
            acc_success = pd.to_numeric(g_success["accuracy"], errors="coerce")
            row = {
                "candidate_key": key,
                "base_model": first["base_model"],
                "context_k": int(first["context_k"]),
                "text_col": first["text_col"],
                "max_length": int(first["max_length"]),
                "learning_rate": float(first["learning_rate"]),
                "weight_decay": float(first["weight_decay"]),
                "precision": first.get("precision", precision_for_backbone_search(first["base_model"])),
                "train_batch_size": int(first.get("train_batch_size", CONFIG["task_b_train_batch_size"])),
                "eval_batch_size": int(first.get("eval_batch_size", CONFIG["task_b_eval_batch_size"])),
                "gradient_accumulation_steps": int(first.get("gradient_accumulation_steps", CONFIG["task_b_grad_accum"])),
                "gradient_checkpointing": bool(first.get("gradient_checkpointing", False)),
                "successful_seeds": int(len(g_success)),
                "total_seeds": int(len(g)),
                "mean_f1_macro": float(f1_success.mean()),
                "std_f1_macro": float(f1_success.std(ddof=0)) if len(f1_success) > 1 else 0.0,
                "min_f1_macro": float(f1_success.min()),
                "max_f1_macro": float(f1_success.max()),
                "mean_accuracy": float(acc_success.mean()),
                "std_accuracy": float(acc_success.std(ddof=0)) if len(acc_success) > 1 else 0.0,
                "any_oom_retry": bool(g_success.get("oom_retry", pd.Series(dtype=bool)).fillna(False).any()),
                "errors": "; ".join(g["error"].dropna().astype(str).unique().tolist()),
            }
        else:
            row = {
                "candidate_key": key,
                "base_model": first["base_model"],
                "context_k": int(first["context_k"]),
                "text_col": first["text_col"],
                "max_length": int(first["max_length"]),
                "learning_rate": float(first["learning_rate"]),
                "weight_decay": float(first["weight_decay"]),
                "precision": first.get("precision", precision_for_backbone_search(first["base_model"])),
                "train_batch_size": int(first.get("train_batch_size", CONFIG["task_b_train_batch_size"])),
                "eval_batch_size": int(first.get("eval_batch_size", CONFIG["task_b_eval_batch_size"])),
                "gradient_accumulation_steps": int(first.get("gradient_accumulation_steps", CONFIG["task_b_grad_accum"])),
                "gradient_checkpointing": bool(first.get("gradient_checkpointing", False)),
                "successful_seeds": 0,
                "total_seeds": int(len(g)),
                "mean_f1_macro": np.nan,
                "std_f1_macro": np.nan,
                "min_f1_macro": np.nan,
                "max_f1_macro": np.nan,
                "mean_accuracy": np.nan,
                "std_accuracy": np.nan,
                "any_oom_retry": False,
                "errors": "; ".join(g["error"].dropna().astype(str).unique().tolist()),
            }
        summary_rows.append(row)
    task_b_roberta_multiseed_summary = pd.DataFrame(summary_rows)
    task_b_roberta_multiseed_summary["_finite_mean_f1"] = np.isfinite(
        pd.to_numeric(task_b_roberta_multiseed_summary["mean_f1_macro"], errors="coerce")
    )
    task_b_roberta_multiseed_summary = (
        task_b_roberta_multiseed_summary
        .sort_values(
            ["_finite_mean_f1", "mean_f1_macro", "std_f1_macro", "mean_accuracy"],
            ascending=[False, False, True, False],
            na_position="last",
        )
        .drop(columns=["_finite_mean_f1"])
        .reset_index(drop=True)
    )
    save_df(task_b_roberta_multiseed_summary, task_b_roberta_multiseed_summary_path)
    _display_roberta_table(
        "### Table 6.2G. RoBERTa multi-seed summary",
        task_b_roberta_multiseed_summary,
        row_color_col="context_k",
        row_colors=_roberta_context_colors,
        cell_col_colors={"mean_f1_macro": "#e9f7ef", "std_f1_macro": "#fff3d9", "mean_accuracy": "#e7f0ff", "errors": "#fce8ef"},
        formats=_roberta_metric_formats,
        text_cols=["candidate_key", "errors"],
        nowrap_cols=["base_model", "context_k", "max_length", "learning_rate", "precision", "successful_seeds", "total_seeds", "any_oom_retry"],
    )
    eligible_summary = task_b_roberta_multiseed_summary.dropna(subset=["mean_f1_macro"]).copy()
    eligible_summary = eligible_summary[
        np.isfinite(pd.to_numeric(eligible_summary["mean_f1_macro"], errors="coerce"))
    ].copy()
    if len(CONFIG["task_b_multiseed_seeds"]) >= 2:
        eligible_two_seed = eligible_summary[eligible_summary["successful_seeds"] >= 2].copy()
        if len(eligible_two_seed) > 0:
            eligible_summary = eligible_two_seed
    if len(eligible_summary) == 0:
        raise RuntimeError(
            "No eligible RoBERTa multi-seed config succeeded. "
            "Check the RoBERTa multiseed summary and raw results."
        )
    best_ms_row = eligible_summary.iloc[0].to_dict()
    task_b_roberta_cfg = {
        "base_model": best_ms_row["base_model"],
        "used_checkpoint": best_ms_row["base_model"],
        "context_k": int(best_ms_row["context_k"]),
        "text_col": best_ms_row["text_col"],
        "max_length": int(best_ms_row["max_length"]),
        "learning_rate": float(best_ms_row["learning_rate"]),
        "weight_decay": float(best_ms_row["weight_decay"]),
        "precision": str(best_ms_row.get("precision", precision_for_backbone_search(best_ms_row["base_model"]))),
        "train_batch_size": int(best_ms_row.get("train_batch_size", CONFIG["task_b_train_batch_size"])),
        "eval_batch_size": int(best_ms_row.get("eval_batch_size", CONFIG["task_b_eval_batch_size"])),
        "gradient_accumulation_steps": int(best_ms_row.get("gradient_accumulation_steps", CONFIG["task_b_grad_accum"])),
        "gradient_checkpointing": bool(best_ms_row.get("gradient_checkpointing", False)),
        "selection_source": "grouped_hparam_search_plus_multiseed_confirmation",
        "display_name": "FacebookAI/roberta-base (HP search + multi-seed)",
        "multiseed_mean_f1_macro": float(best_ms_row["mean_f1_macro"]),
        "multiseed_std_f1_macro": float(best_ms_row["std_f1_macro"]),
        "multiseed_min_f1_macro": float(best_ms_row["min_f1_macro"]),
        "multiseed_max_f1_macro": float(best_ms_row["max_f1_macro"]),
        "multiseed_mean_accuracy": float(best_ms_row["mean_accuracy"]),
        "successful_seeds": int(best_ms_row["successful_seeds"]),
        "total_seeds": int(best_ms_row["total_seeds"]),
        "search_seed": int(CONFIG["task_b_hp_search_seed"]),
        "multiseed_seeds": list(CONFIG["task_b_multiseed_seeds"]),
        "hp_search_epochs": int(CONFIG["task_b_hp_search_epochs"]),
        "multiseed_epochs": int(CONFIG["task_b_multiseed_epochs"]),
    }
    save_json(task_b_roberta_cfg, task_b_roberta_cfg_path)
    for p in [task_b_roberta_metrics_path, task_b_roberta_preds_path, task_b_roberta_meta_path]:
        p = Path(p)
        if p.exists():
            p.unlink()
            print("Deleted stale RoBERTa final output:", p)
metrics_ready = (
    artifact_exists(task_b_roberta_metrics_path)
    and artifact_exists(task_b_roberta_preds_path)
    and artifact_exists(task_b_roberta_cfg_path)
    and not CONFIG.get("force_retrain", False)
)
if metrics_ready:
    task_b_roberta_metrics = load_json(task_b_roberta_metrics_path)
    task_b_roberta_preds = load_joblib(task_b_roberta_preds_path)
    task_b_roberta_cfg = load_json(task_b_roberta_cfg_path)
    task_b_roberta_report = report_df(y_test_b, task_b_roberta_preds["pred"], label_encoder)
else:
    final_seed = int(CONFIG.get("task_b_final_seed", 42))
    if "grouped_train_val_split_for_seed" in globals():
        enc_train_sub, enc_val_sub = grouped_train_val_split_for_seed(
            therapist_train,
            seed=final_seed,
            n_splits=5,
        )
    else:
        enc_train_sub, enc_val_sub = grouped_train_val_split(
            therapist_train,
            label_col=LABEL_COL,
            group_col=GROUP_COL,
            seed=final_seed,
            n_splits=5,
        )
    set_all_seeds(final_seed)
    final_roberta_result = call_train_encoder_once_safely(
        model_name_or_path=task_b_roberta_cfg["used_checkpoint"],
        train_df=enc_train_sub,
        val_df=enc_val_sub,
        text_col=task_b_roberta_cfg["text_col"],
        output_dir=task_b_roberta_dir / "final_model",
        max_length=int(task_b_roberta_cfg["max_length"]),
        learning_rate=float(task_b_roberta_cfg["learning_rate"]),
        weight_decay=float(task_b_roberta_cfg["weight_decay"]),
        epochs=int(CONFIG["task_b_epochs"]),
        train_batch_size=int(task_b_roberta_cfg.get("train_batch_size", CONFIG["task_b_train_batch_size"])),
        eval_batch_size=int(task_b_roberta_cfg.get("eval_batch_size", CONFIG["task_b_eval_batch_size"])),
        gradient_accumulation_steps=int(task_b_roberta_cfg.get("gradient_accumulation_steps", CONFIG["task_b_grad_accum"])),
        precision=str(task_b_roberta_cfg.get("precision", precision_for_backbone_search(task_b_roberta_cfg["used_checkpoint"]))),
        gradient_checkpointing=bool(task_b_roberta_cfg.get("gradient_checkpointing", False)),
        seed=final_seed,
        save_best_model=True,
        early_stopping_patience=int(CONFIG.get("task_b_early_stopping_patience", 2)),
    )
    task_b_roberta_trainer = final_roberta_result["trainer"]
    task_b_roberta_tokenizer = final_roberta_result["tokenizer"]
    test_hf_ds = tokenize_dataset(
        make_hf_dataset(therapist_test, text_col=task_b_roberta_cfg["text_col"]),
        task_b_roberta_tokenizer,
        max_length=int(task_b_roberta_cfg["max_length"]),
    )
    test_pred = task_b_roberta_trainer.predict(test_hf_ds)
    logits = test_pred.predictions
    probs = softmax(logits, axis=1)
    pred = logits.argmax(axis=1)
    task_b_roberta_metrics = compute_multiclass_metrics(y_test_b, pred, probs)
    task_b_roberta_preds = {
        "pred": pred,
        "prob": probs,
        "logits": logits,
        "val_logits": final_roberta_result["val_logits"],
        "val_labels": final_roberta_result["val_labels"],
        "display_name": task_b_roberta_cfg["display_name"],
    }
    task_b_roberta_report = report_df(y_test_b, pred, label_encoder)
    save_json(task_b_roberta_metrics, task_b_roberta_metrics_path)
    save_joblib(task_b_roberta_preds, task_b_roberta_preds_path)
    save_json(task_b_roberta_cfg, task_b_roberta_cfg_path)
    save_json(
        {
            "model_name": task_b_roberta_cfg["display_name"],
            "artifact_dir": str(task_b_roberta_dir),
            "pilot_results_path": str(task_b_roberta_pilot_results_path),
            "multiseed_results_path": str(task_b_roberta_multiseed_results_path),
            "multiseed_summary_path": str(task_b_roberta_multiseed_summary_path),
            "final_seed": final_seed,
        },
        task_b_roberta_meta_path,
    )
display(Markdown(f"## Parallel Task B result: **{task_b_roberta_cfg['display_name']}**"))
_display_roberta_table(
    "### Table 6.2H. Selected RoBERTa final configuration",
    pd.DataFrame([task_b_roberta_cfg]),
    row_color_col="context_k",
    row_colors=_roberta_context_colors,
    cell_col_colors={
        "context_k": "#e7f0ff",
        "max_length": "#e9f7ef",
        "learning_rate": "#e7f0ff",
        "multiseed_mean_f1_macro": "#e9f7ef",
        "multiseed_std_f1_macro": "#fff3d9",
    },
    formats=_roberta_metric_formats,
    text_cols=["base_model", "used_checkpoint", "display_name", "selection_source", "multiseed_seeds"],
    nowrap_cols=["context_k", "text_col", "max_length", "learning_rate", "precision", "successful_seeds", "total_seeds"],
)
_display_roberta_table(
    "### Table 6.2I. RoBERTa held-out test metrics",
    pd.DataFrame([{"model": task_b_roberta_cfg["display_name"], **task_b_roberta_metrics}]).round(4),
    cell_col_colors={
        "model": "#f1f5f9",
        "accuracy": "#e7f0ff",
        "f1_macro": "#e9f7ef",
        "precision_macro": "#e9f7ef",
        "recall_macro": "#e9f7ef",
        "brier_multiclass": "#fff3d9",
    },
    formats=_roberta_metric_formats,
    text_cols=["model"],
)
task_b_roberta_report_display = task_b_roberta_report.round(4).reset_index(names="class")
_display_roberta_table(
    "### Table 6.2J. RoBERTa per-class classification report",
    task_b_roberta_report_display,
    row_color_col="class",
    row_colors=_roberta_class_colors,
    cell_col_colors={"precision": "#e7f0ff", "recall": "#e9f7ef", "f1-score": "#fff3d9", "support": "#f1f5f9"},
    formats={"precision": "{:.4f}", "recall": "{:.4f}", "f1-score": "{:.4f}", "support": "{:.0f}"},
    nowrap_cols=task_b_roberta_report_display.columns.tolist(),
)

In [ ]:
if "task_b_roberta_preds" not in globals() or "task_b_roberta_metrics" not in globals():
    raise NameError(
        "RoBERTa predictions/metrics are not in memory. "
        "Run the RoBERTa Task B cell immediately above first."
    )
def fit_temperature(logits: np.ndarray, labels: np.ndarray) -> float:
    logits_t = torch.tensor(logits, dtype=torch.float32, device=DEVICE)
    labels_t = torch.tensor(labels, dtype=torch.long, device=DEVICE)
    temperature = torch.ones(1, device=DEVICE, requires_grad=True)
    optimizer = torch.optim.LBFGS([temperature], lr=0.01, max_iter=50)
    loss_fct = nn.CrossEntropyLoss()
    def closure():
        optimizer.zero_grad()
        loss = loss_fct(logits_t / temperature.clamp(min=1e-3), labels_t)
        loss.backward()
        return loss
    optimizer.step(closure)
    return float(temperature.detach().cpu().item())
def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray, n_bins: int = 15) -> float:
    confidences = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i + 1])
        if mask.sum() == 0:
            continue
        bin_acc = accuracies[mask].mean()
        bin_conf = confidences[mask].mean()
        ece += (mask.mean()) * abs(bin_acc - bin_conf)
    return float(ece)
def plot_reliability_diagram(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    title: str,
    save_path: Path,
    n_bins: int = 10,
):
    confidences = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_accs, bin_confs = [], []
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i + 1])
        if mask.sum() == 0:
            continue
        bin_accs.append(accuracies[mask].mean())
        bin_confs.append(confidences[mask].mean())
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot([0, 1], [0, 1], linestyle="--")
    ax.plot(bin_confs, bin_accs, marker="o")
    ax.set_title(title)
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Accuracy")
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    display(IPyImage(filename=str(save_path)))
task_b_roberta_temperature = fit_temperature(
    task_b_roberta_preds["val_logits"],
    task_b_roberta_preds["val_labels"],
)
task_b_roberta_probs_cal = softmax(
    task_b_roberta_preds["logits"] / task_b_roberta_temperature,
    axis=1,
)
task_b_roberta_pred_cal = task_b_roberta_probs_cal.argmax(axis=1)
task_b_roberta_metrics_cal = compute_multiclass_metrics(
    y_test_b,
    task_b_roberta_pred_cal,
    task_b_roberta_probs_cal,
)
task_b_roberta_metrics_cal["ece"] = expected_calibration_error(
    y_test_b,
    task_b_roberta_probs_cal,
    n_bins=15,
)
task_b_roberta_metrics_uncal = dict(task_b_roberta_metrics)
task_b_roberta_metrics_uncal["ece"] = expected_calibration_error(
    y_test_b,
    task_b_roberta_preds["prob"],
    n_bins=15,
)
task_b_calibration_comparison_df = pd.DataFrame(
    [
        {"model": "Elastic-net logistic regression", **task_b_logreg_metrics},
        {"model": f"{task_b_roberta_cfg['display_name']} (uncalibrated)", **task_b_roberta_metrics_uncal},
        {"model": f"{task_b_roberta_cfg['display_name']} (calibrated)", **task_b_roberta_metrics_cal},
    ]
).round(4)
def _display_task_b_calibration_table(df: pd.DataFrame) -> None:
    display(Markdown("### Table 6.2K. Task B calibration and held-out metric comparison"))
    row_colors = {
        "Elastic-net logistic regression": "#e7f0ff",
        f"{task_b_roberta_cfg['display_name']} (uncalibrated)": "#fff3d9",
        f"{task_b_roberta_cfg['display_name']} (calibrated)": "#e9f7ef",
    }
    cell_col_colors = {
        "model": "#f1f5f9",
        "accuracy": "#e7f0ff",
        "precision_macro": "#e9f7ef",
        "recall_macro": "#e9f7ef",
        "f1_macro": "#e9f7ef",
        "precision_weighted": "#f1f5f9",
        "recall_weighted": "#f1f5f9",
        "f1_weighted": "#f1f5f9",
        "brier_multiclass": "#fff3d9",
        "ece": "#fce8ef",
    }
    formats = {col: "{:.4f}" for col in df.columns if col != "model"}
    def _row_style(row):
        background = row_colors.get(str(row.get("model", "")), "#ffffff")
        return [
            f"background-color: {background}; color: #111827; font-weight: 700; border: 1px solid #cbd5e1; vertical-align: top;"
            for _ in row
        ]
    try:
        styler = (
            df.style
            .hide(axis="index")
            .format(formats, na_rep="")
            .apply(_row_style, axis=1)
            .set_table_styles(
                [
                    {
                        "selector": "th",
                        "props": [
                            ("background-color", "#f8fafc"),
                            ("color", "#111827"),
                            ("font-weight", "800"),
                            ("text-align", "left"),
                            ("border", "1px solid #cbd5e1"),
                            ("padding", "5px 7px"),
                            ("font-size", "11.5px"),
                        ],
                    },
                    {
                        "selector": "td",
                        "props": [
                            ("padding", "5px 7px"),
                            ("font-size", "11.5px"),
                            ("line-height", "1.22"),
                        ],
                    },
                    {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
                ]
            )
        )
        for col, color in cell_col_colors.items():
            if col in df.columns:
                styler = styler.set_properties(subset=[col], **{"background-color": color})
        if "model" in df.columns:
            styler = styler.set_properties(subset=["model"], **{"max-width": "360px", "white-space": "normal"})
        display(styler)
    except Exception:
        def _format_value(value, fmt=None):
            if value is None or pd.isna(value):
                return ""
            if fmt is not None:
                try:
                    return fmt.format(value)
                except Exception:
                    return str(value)
            return str(value)
        header_html = "".join(
            f"<th style='background-color: #f8fafc; color: #111827; font-weight: 800; text-align: left; border: 1px solid #cbd5e1; padding: 5px 7px; font-size: 11.5px;'>{escape(str(col))}</th>"
            for col in df.columns
        )
        body_html = ""
        for _, row in df.iterrows():
            row_bg = row_colors.get(str(row.get("model", "")), "#ffffff")
            body_html += "<tr>"
            for col in df.columns:
                background = cell_col_colors.get(col, row_bg)
                wrap = "max-width: 360px; white-space: normal;" if col == "model" else "white-space: nowrap;"
                body_html += (
                    f"<td style='background-color: {background}; color: #111827; font-weight: 700; border: 1px solid #cbd5e1; padding: 5px 7px; font-size: 11.5px; line-height: 1.22; vertical-align: top; {wrap}'>"
                    f"{escape(_format_value(row[col], formats.get(col)))}</td>"
                )
            body_html += "</tr>"
        display(HTML(f"<table style='border-collapse: collapse; width: 100%; color: #111827;'><thead><tr>{header_html}</tr></thead><tbody>{body_html}</tbody></table>"))
_display_task_b_calibration_table(task_b_calibration_comparison_df)
display(Markdown("### Figure 18. Task B sparse-baseline confusion matrix"))
plot_confusion(
    y_test_b,
    task_b_logreg_preds["pred"],
    labels=list(label_encoder.classes_),
    title="Task b – Elastic-net logistic regression",
    save_path=FIG_DIR / "task_b_cm_logreg.png",
)
display(Markdown("### Figure 19. Task B RoBERTa confusion matrix"))
plot_confusion(
    y_test_b,
    task_b_roberta_preds["pred"],
    labels=list(label_encoder.classes_),
    title=f"Task b – {task_b_roberta_cfg['display_name']}",
    save_path=FIG_DIR / "task_b_cm_roberta.png",
)
display(Markdown("### Figure 20. Task B RoBERTa reliability before calibration"))
plot_reliability_diagram(
    y_test_b,
    task_b_roberta_preds["prob"],
    title=f"{task_b_roberta_cfg['display_name']} reliability (uncalibrated)",
    save_path=FIG_DIR / "task_b_reliability_roberta_uncalibrated.png",
)
display(Markdown("### Figure 21. Task B RoBERTa reliability after temperature calibration"))
plot_reliability_diagram(
    y_test_b,
    task_b_roberta_probs_cal,
    title=f"{task_b_roberta_cfg['display_name']} reliability (calibrated)",
    save_path=FIG_DIR / "task_b_reliability_roberta_calibrated.png",
)
save_json(
    {
        "temperature": float(task_b_roberta_temperature),
        "uncalibrated": task_b_roberta_metrics_uncal,
        "calibrated": task_b_roberta_metrics_cal,
    },
    RESULT_DIR / "task_b_roberta_calibration_metrics.json",
)


In [ ]:
_required = {
    "task_b_logreg_metrics": "elastic-net baseline metrics dict",
    "task_b_logreg_preds": "elastic-net baseline predictions dict (with 'prob')",
    "task_b_roberta_cfg": "selected RoBERTa configuration dict",
    "task_b_roberta_metrics": "RoBERTa held-out metrics dict",
    "task_b_roberta_preds": "RoBERTa predictions dict (with 'prob' and 'logits')",
    "task_b_roberta_metrics_uncal": "RoBERTa uncalibrated metrics with ECE",
    "task_b_roberta_metrics_cal": "RoBERTa calibrated metrics with ECE",
    "task_b_roberta_probs_cal": "RoBERTa calibrated probabilities",
    "task_b_roberta_pred_cal": "RoBERTa calibrated argmax predictions",
    "task_b_roberta_temperature": "fitted temperature scalar",
    "y_test_b": "held-out test labels for Task b",
}
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise NameError(
        "Task 2.5A cannot run yet. Missing in-memory objects:\n  "
        + "\n  ".join(f"- {name}: {_required[name]}" for name in _missing)
        + "\nRun the RoBERTa Task B training cell and then the temperature-"
        "scaling cell upstream first."
    )
if "expected_calibration_error" in globals():
    _ece_fn = expected_calibration_error
else:
    def _ece_fn(y_true, y_prob, n_bins: int = 15) -> float:
        confidences = y_prob.max(axis=1)
        predictions = y_prob.argmax(axis=1)
        accuracies = (predictions == y_true).astype(float)
        bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
        ece_value = 0.0
        for j in range(n_bins):
            mask = (confidences > bin_edges[j]) & (confidences <= bin_edges[j + 1])
            if mask.sum() == 0:
                continue
            bin_acc = accuracies[mask].mean()
            bin_conf = confidences[mask].mean()
            ece_value += float(mask.mean()) * abs(bin_acc - bin_conf)
        return float(ece_value)
ECE_NBINS = 15
task_b_logreg_metrics_with_ece = dict(task_b_logreg_metrics)
task_b_logreg_metrics_with_ece["ece"] = _ece_fn(
    y_test_b,
    task_b_logreg_preds["prob"],
    n_bins=ECE_NBINS,
)
task_b_main_model_name = task_b_roberta_cfg.get("display_name", "FacebookAI/roberta-base")
task_b_main_model_cfg = task_b_roberta_cfg
task_b_main_model_metrics = task_b_roberta_metrics
task_b_main_model_preds = task_b_roberta_preds
if "task_b_roberta_report" in globals():
    task_b_main_model_report = task_b_roberta_report
task_b_main_model_probs_cal = task_b_roberta_probs_cal
task_b_main_model_pred_cal = task_b_roberta_pred_cal
roberta_uncal_label = f"{task_b_main_model_name} (uncalibrated)"
roberta_cal_label = f"{task_b_main_model_name} + temperature scaling"
official_task_b_comparison_df = pd.DataFrame(
    [
        {"model": "Elastic-net logistic regression",
         **task_b_logreg_metrics_with_ece},
        {"model": roberta_uncal_label,
         **task_b_roberta_metrics_uncal},
        {"model": roberta_cal_label,
         **task_b_roberta_metrics_cal},
    ]
)
_preferred_cols = [
    "model",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "precision_weighted",
    "recall_weighted",
    "f1_weighted",
    "brier_multiclass",
    "ece",
]
_existing_cols = [c for c in _preferred_cols if c in official_task_b_comparison_df.columns]
_extra_cols = [c for c in official_task_b_comparison_df.columns if c not in _existing_cols]
official_task_b_comparison_df = official_task_b_comparison_df[_existing_cols + _extra_cols]
metric_keys = [
    "accuracy",
    "f1_macro",
    "f1_weighted",
    "precision_macro",
    "recall_macro",
    "brier_multiclass",
    "ece",
]
metric_keys_present = [
    m for m in metric_keys
    if m in task_b_logreg_metrics_with_ece
    and m in task_b_roberta_metrics_uncal
    and m in task_b_roberta_metrics_cal
]
task_b_metric_delta_df = pd.DataFrame(
    [
        {
            "metric": m,
            "baseline": float(task_b_logreg_metrics_with_ece[m]),
            "roberta_uncalibrated": float(task_b_roberta_metrics_uncal[m]),
            "roberta_calibrated": float(task_b_roberta_metrics_cal[m]),
            "delta_roberta_minus_baseline": float(
                task_b_roberta_metrics_uncal[m] - task_b_logreg_metrics_with_ece[m]
            ),
            "delta_calibrated_minus_uncalibrated": float(
                task_b_roberta_metrics_cal[m] - task_b_roberta_metrics_uncal[m]
            ),
        }
        for m in metric_keys_present
    ]
)
def _format_task25_value(value, fmt=None) -> str:
    if value is None:
        return ""
    try:
        missing = pd.isna(value)
        if isinstance(missing, (bool, np.bool_)) and missing:
            return ""
    except Exception:
        pass
    if fmt is not None:
        try:
            return fmt.format(value)
        except Exception:
            return str(value)
    return str(value)
def _display_task25_table(
    title: str,
    df: pd.DataFrame,
    *,
    row_color_col: str = "",
    row_colors: Optional[Dict[str, str]] = None,
    cell_col_colors: Optional[Dict[str, str]] = None,
    formats: Optional[Dict[str, str]] = None,
    text_cols: Optional[List[str]] = None,
    nowrap_cols: Optional[List[str]] = None,
) -> None:
    display(Markdown(title))
    table_df = df.copy()
    row_colors = row_colors or {}
    cell_col_colors = cell_col_colors or {}
    formats = formats or {}
    text_cols = text_cols or []
    nowrap_cols = nowrap_cols or []
    def _row_style(row):
        row_key = str(row.get(row_color_col, "")) if row_color_col else ""
        background = row_colors.get(row_key, "#ffffff")
        return [
            f"background-color: {background}; color: #111827; font-weight: 700; border: 1px solid #cbd5e1; vertical-align: top;"
            for _ in row
        ]
    try:
        styler = (
            table_df.style
            .hide(axis="index")
            .format(formats, na_rep="")
            .apply(_row_style, axis=1)
            .set_table_styles(
                [
                    {
                        "selector": "th",
                        "props": [
                            ("background-color", "#f8fafc"),
                            ("color", "#111827"),
                            ("font-weight", "800"),
                            ("text-align", "left"),
                            ("border", "1px solid #cbd5e1"),
                            ("padding", "5px 7px"),
                            ("font-size", "11.5px"),
                        ],
                    },
                    {
                        "selector": "td",
                        "props": [
                            ("padding", "5px 7px"),
                            ("font-size", "11.5px"),
                            ("line-height", "1.22"),
                        ],
                    },
                    {"selector": "table", "props": [("border-collapse", "collapse"), ("width", "100%")]},
                ]
            )
        )
        for col, color in cell_col_colors.items():
            if col in table_df.columns:
                styler = styler.set_properties(subset=[col], **{"background-color": color})
        for col in text_cols:
            if col in table_df.columns:
                styler = styler.set_properties(subset=[col], **{"max-width": "360px", "white-space": "normal"})
        for col in nowrap_cols:
            if col in table_df.columns:
                styler = styler.set_properties(subset=[col], **{"white-space": "nowrap"})
        display(styler)
    except Exception:
        header_html = "".join(
            f"<th style='background-color: #f8fafc; color: #111827; font-weight: 800; text-align: left; border: 1px solid #cbd5e1; padding: 5px 7px; font-size: 11.5px;'>{escape(str(col))}</th>"
            for col in table_df.columns
        )
        body_html = ""
        for _, row in table_df.iterrows():
            row_key = str(row.get(row_color_col, "")) if row_color_col else ""
            row_bg = row_colors.get(row_key, "#ffffff")
            body_html += "<tr>"
            for col in table_df.columns:
                background = cell_col_colors.get(col, row_bg)
                wrap = "max-width: 360px; white-space: normal;" if col in text_cols else "white-space: nowrap;"
                body_html += (
                    f"<td style='background-color: {background}; color: #111827; font-weight: 700; border: 1px solid #cbd5e1; padding: 5px 7px; font-size: 11.5px; line-height: 1.22; vertical-align: top; {wrap}'>"
                    f"{escape(_format_task25_value(row[col], formats.get(col)))}</td>"
                )
            body_html += "</tr>"
        display(HTML(f"<table style='border-collapse: collapse; width: 100%; color: #111827;'><thead><tr>{header_html}</tr></thead><tbody>{body_html}</tbody></table>"))
_task25_model_colors = {
    "Elastic-net logistic regression": "#e7f0ff",
    roberta_uncal_label: "#fff3d9",
    roberta_cal_label: "#e9f7ef",
}
_task25_metric_colors = {
    "accuracy": "#e7f0ff",
    "f1_macro": "#e9f7ef",
    "f1_weighted": "#e9f7ef",
    "precision_macro": "#e9f7ef",
    "recall_macro": "#e9f7ef",
    "brier_multiclass": "#fff3d9",
    "ece": "#fce8ef",
}
_task25_formats = {col: "{:.4f}" for col in official_task_b_comparison_df.columns if col != "model"}
_task25_delta_formats = {col: "{:.4f}" for col in task_b_metric_delta_df.columns if col != "metric"}
display(Markdown(
    f"## Official held-out comparison: baseline vs **{task_b_main_model_name}**\n\n"
    "Three rows on the same held-out test split: the sparse baseline, the "
    "selected RoBERTa-base configuration, and the same configuration after "
    "post-hoc temperature scaling on the validation logits. Top-1 metrics "
    "(accuracy, F1, precision, recall) are unchanged by temperature scaling, "
    "as expected. Probability-quality metrics (Brier, ECE) improve."
))
selected_roberta_config_df = pd.DataFrame(
    [
        {
            "model": task_b_main_model_name,
            "base_model": task_b_main_model_cfg["base_model"],
            "context_k": task_b_main_model_cfg["context_k"],
            "max_length": task_b_main_model_cfg["max_length"],
            "learning_rate": task_b_main_model_cfg["learning_rate"],
            "weight_decay": task_b_main_model_cfg["weight_decay"],
            "fitted_temperature": float(task_b_roberta_temperature),
            "multiseed_mean_f1_macro": task_b_main_model_cfg.get(
                "multiseed_mean_f1_macro", np.nan
            ),
            "multiseed_std_f1_macro": task_b_main_model_cfg.get(
                "multiseed_std_f1_macro", np.nan
            ),
            "multiseed_seeds": str(
                task_b_main_model_cfg.get("multiseed_seeds", [17, 42, 101])
            ),
        }
    ]
).round(4)
_display_task25_table(
    "### Table 2.5A. Selected RoBERTa configuration for official comparison",
    selected_roberta_config_df,
    cell_col_colors={
        "model": "#f1f5f9",
        "base_model": "#f1f5f9",
        "context_k": "#e7f0ff",
        "max_length": "#e9f7ef",
        "learning_rate": "#e7f0ff",
        "fitted_temperature": "#fce8ef",
        "multiseed_mean_f1_macro": "#e9f7ef",
        "multiseed_std_f1_macro": "#fff3d9",
    },
    formats={
        "learning_rate": "{:.2e}",
        "weight_decay": "{:.4f}",
        "fitted_temperature": "{:.4f}",
        "multiseed_mean_f1_macro": "{:.4f}",
        "multiseed_std_f1_macro": "{:.4f}",
    },
    text_cols=["model", "base_model", "multiseed_seeds"],
    nowrap_cols=["context_k", "max_length", "learning_rate", "weight_decay", "fitted_temperature"],
)
display(Markdown(
    "### Three-row official comparison\n\n"
    "Lower is better for `brier_multiclass` and `ece`; higher is better for "
    "everything else."
))
_display_task25_table(
    "### Table 2.5B. Three-row official held-out comparison",
    official_task_b_comparison_df.round(4),
    row_color_col="model",
    row_colors=_task25_model_colors,
    cell_col_colors={
        "model": "#f1f5f9",
        "accuracy": "#e7f0ff",
        "precision_macro": "#e9f7ef",
        "recall_macro": "#e9f7ef",
        "f1_macro": "#e9f7ef",
        "precision_weighted": "#f1f5f9",
        "recall_weighted": "#f1f5f9",
        "f1_weighted": "#f1f5f9",
        "brier_multiclass": "#fff3d9",
        "ece": "#fce8ef",
    },
    formats=_task25_formats,
    text_cols=["model"],
)
display(Markdown(
    "### Per-metric deltas\n\n"
    "`delta_roberta_minus_baseline` shows the lift of the encoder over the "
    "sparse baseline. `delta_calibrated_minus_uncalibrated` isolates the "
    "effect of temperature scaling — top-1 metrics are exactly zero "
    "(temperature does not change argmax), while Brier and ECE change."
))
_display_task25_table(
    "### Table 2.5C. Per-metric held-out deltas",
    task_b_metric_delta_df.round(4),
    row_color_col="metric",
    row_colors=_task25_metric_colors,
    cell_col_colors={
        "metric": "#f1f5f9",
        "baseline": "#e7f0ff",
        "roberta_uncalibrated": "#fff3d9",
        "roberta_calibrated": "#e9f7ef",
        "delta_roberta_minus_baseline": "#e9f7ef",
        "delta_calibrated_minus_uncalibrated": "#fce8ef",
    },
    formats=_task25_delta_formats,
    nowrap_cols=task_b_metric_delta_df.columns.tolist(),
)
save_df(
    official_task_b_comparison_df,
    RESULT_DIR / "official_task_b_comparison_three_row.csv",
)
save_df(
    task_b_metric_delta_df,
    RESULT_DIR / "official_task_b_metric_deltas_three_row.csv",
)
save_json(
    {
        "n_bins_ece": ECE_NBINS,
        "fitted_temperature": float(task_b_roberta_temperature),
        "logreg_ece": float(task_b_logreg_metrics_with_ece["ece"]),
        "roberta_uncalibrated_ece": float(task_b_roberta_metrics_uncal["ece"]),
        "roberta_calibrated_ece": float(task_b_roberta_metrics_cal["ece"]),
        "roberta_uncalibrated_brier": float(
            task_b_roberta_metrics_uncal["brier_multiclass"]
        ),
        "roberta_calibrated_brier": float(
            task_b_roberta_metrics_cal["brier_multiclass"]
        ),
        "note": (
            "ECE binning matches the upstream temperature-scaling cell. "
            "Top-1 metrics are unchanged by temperature scaling because "
            "temperature rescales logits without altering argmax."
        ),
    },
    RESULT_DIR / "task_b_calibration_summary.json",
)

In [ ]:
task_b_label_order = [c for c in ["reflection", "question", "therapist_input", "other"] if c in label_encoder.classes_]
label_to_idx = {label: idx for idx, label in enumerate(label_encoder.classes_)}
ordered_idx = [label_to_idx[label] for label in task_b_label_order]
def per_class_metric_frame(report: pd.DataFrame, model_name: str) -> pd.DataFrame:
    out = report.reindex(task_b_label_order)[["precision", "recall", "f1-score"]].copy()
    out["label"] = out.index
    out = out.rename(columns={"f1-score": "f1"})
    out = out.melt(id_vars="label", var_name="metric", value_name="score")
    out["model"] = model_name
    return out
task_b_per_class_metrics_df = pd.concat(
    [
        per_class_metric_frame(task_b_logreg_report, "Elastic-net logistic regression"),
        per_class_metric_frame(task_b_main_model_report, task_b_main_model_name),
    ],
    ignore_index=True,
)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), sharey=True)
for ax, metric in zip(axes, ["precision", "recall", "f1"]):
    sub = task_b_per_class_metrics_df[task_b_per_class_metrics_df["metric"] == metric]
    sns.barplot(
        data=sub,
        x="label",
        y="score",
        hue="model",
        order=task_b_label_order,
        ax=ax,
    )
    ax.set_title(f"Task b per-class {metric}")
    ax.set_xlabel("")
    ax.set_ylabel("Score")
    ax.tick_params(axis="x", rotation=15)
axes[0].legend(title="")
axes[1].legend_.remove()
axes[2].legend_.remove()
metric_path = FIG_DIR / "task_b_per_class_metric_bars_roberta_vs_baseline.png"
fig.tight_layout()
fig.savefig(metric_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 22. Task B per-class metric bars for baseline vs RoBERTa"))
display(IPyImage(filename=str(metric_path)))
y_test_b_ovr = label_binarize(y_test_b, classes=np.arange(len(label_encoder.classes_)))
curve_summary_rows = []
roc_fig, roc_axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
pr_fig, pr_axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
for roc_ax, pr_ax, label_name, class_idx in zip(
    roc_axes.flatten(),
    pr_axes.flatten(),
    task_b_label_order,
    ordered_idx,
):
    y_true_bin = y_test_b_ovr[:, class_idx]
    if y_true_bin.sum() == 0 or y_true_bin.sum() == len(y_true_bin):
        roc_ax.set_visible(False)
        pr_ax.set_visible(False)
        continue
    for model_name, prob in [
        ("Elastic-net logistic regression", task_b_logreg_preds["prob"]),
        (task_b_main_model_name, task_b_main_model_preds["prob"]),
    ]:
        fpr, tpr, _ = roc_curve(y_true_bin, prob[:, class_idx])
        precision, recall, _ = precision_recall_curve(y_true_bin, prob[:, class_idx])
        roc_auc = auc(fpr, tpr)
        ap = average_precision_score(y_true_bin, prob[:, class_idx])
        curve_summary_rows.append(
            {
                "label": label_name,
                "model": model_name,
                "roc_auc_ovr": float(roc_auc),
                "average_precision_ovr": float(ap),
            }
        )
        roc_ax.plot(fpr, tpr, label=f"{model_name} (AUC={roc_auc:.3f})")
        pr_ax.plot(recall, precision, label=f"{model_name} (AP={ap:.3f})")
    roc_ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    roc_ax.set_title(f"ROC — {label_name}")
    roc_ax.set_xlabel("False positive rate")
    roc_ax.set_ylabel("True positive rate")
    roc_ax.legend(fontsize=8)
    pr_ax.set_title(f"PR — {label_name}")
    pr_ax.set_xlabel("Recall")
    pr_ax.set_ylabel("Precision")
    pr_ax.legend(fontsize=8)
roc_path = FIG_DIR / "task_b_ovr_roc_curves_roberta_vs_baseline.png"
roc_fig.tight_layout()
roc_fig.savefig(roc_path, dpi=300, bbox_inches="tight")
plt.close(roc_fig)
display(Markdown("### Figure 23. Task B one-vs-rest ROC curves for baseline vs RoBERTa"))
display(IPyImage(filename=str(roc_path)))
pr_path = FIG_DIR / "task_b_ovr_pr_curves_roberta_vs_baseline.png"
pr_fig.tight_layout()
pr_fig.savefig(pr_path, dpi=300, bbox_inches="tight")
plt.close(pr_fig)
display(Markdown("### Figure 24. Task B one-vs-rest precision-recall curves for baseline vs RoBERTa"))
display(IPyImage(filename=str(pr_path)))
task_b_curve_summary_df = pd.DataFrame(curve_summary_rows)
display(Markdown("### Table 2.5E. One-vs-rest ROC and precision-recall summary by class"))
display_report_table(
    task_b_curve_summary_df.round(4),
    row_color_col="label",
    row_colors=REPORT_BEHAVIOUR_COLORS,
    text_cols=["label", "model"],
    gradient_cols=["roc_auc_ovr", "average_precision_ovr"],
    gradient_kwargs={"cmap": "YlGn"},
    highlight_max_cols=["roc_auc_ovr", "average_precision_ovr"],
)
save_df(task_b_per_class_metrics_df, RESULT_DIR / "task_b_per_class_metrics_roberta_vs_baseline.csv")
save_df(task_b_curve_summary_df, RESULT_DIR / "task_b_ovr_curve_summary_roberta_vs_baseline.csv")


In [ ]:
def grouped_bootstrap_metric_delta(
    y_true: np.ndarray,
    pred_a: np.ndarray,
    pred_b: np.ndarray,
    groups: np.ndarray,
    metric_fn,
    n_boot: int = 3000,
    seed: int = 42,
):
    rng = np.random.default_rng(seed)
    unique_groups = np.array(sorted(pd.unique(groups)))
    deltas = []
    for _ in range(n_boot):
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        idx = np.concatenate([np.where(groups == g)[0] for g in sampled_groups])
        score_a = metric_fn(y_true[idx], pred_a[idx])
        score_b = metric_fn(y_true[idx], pred_b[idx])
        deltas.append(score_b - score_a)
    deltas = np.asarray(deltas, dtype=float)
    lo, med, hi = np.percentile(deltas, [2.5, 50, 97.5])
    return {
        "delta_ci95_low": float(lo),
        "delta_median": float(med),
        "delta_ci95_high": float(hi),
        "deltas": deltas,
    }
def grouped_permutation_test(
    y_true: np.ndarray,
    pred_a: np.ndarray,
    pred_b: np.ndarray,
    groups: np.ndarray,
    metric_fn,
    n_perm: int = 3000,
    seed: int = 42,
):
    rng = np.random.default_rng(seed)
    unique_groups = np.array(sorted(pd.unique(groups)))
    observed = float(metric_fn(y_true, pred_b) - metric_fn(y_true, pred_a))
    null_deltas = []
    pred_a = np.asarray(pred_a).copy()
    pred_b = np.asarray(pred_b).copy()
    group_to_idx = {g: np.where(groups == g)[0] for g in unique_groups}
    for _ in range(n_perm):
        perm_a = pred_a.copy()
        perm_b = pred_b.copy()
        swap_mask = rng.integers(0, 2, size=len(unique_groups)).astype(bool)
        for g, swap in zip(unique_groups, swap_mask):
            if swap:
                idx = group_to_idx[g]
                tmp = perm_a[idx].copy()
                perm_a[idx] = perm_b[idx]
                perm_b[idx] = tmp
        null_deltas.append(float(metric_fn(y_true, perm_b) - metric_fn(y_true, perm_a)))
    null_deltas = np.asarray(null_deltas, dtype=float)
    p_two_sided = float((np.sum(np.abs(null_deltas) >= abs(observed)) + 1) / (len(null_deltas) + 1))
    p_one_sided = float((np.sum(null_deltas >= observed) + 1) / (len(null_deltas) + 1))
    return {
        "observed_delta": observed,
        "p_value_two_sided": p_two_sided,
        "p_value_one_sided_roberta_gt_baseline": p_one_sided,
        "null_deltas": null_deltas,
    }
metric_specs = [
    ("accuracy", lambda yt, yp: accuracy_score(yt, yp)),
    ("f1_macro", lambda yt, yp: f1_score(yt, yp, average="macro")),
]
significance_rows = []
bootstrap_artifacts = {}
permutation_artifacts = {}
for metric_name, metric_fn in metric_specs:
    boot = grouped_bootstrap_metric_delta(
        y_true=y_test_b,
        pred_a=task_b_logreg_preds["pred"],
        pred_b=task_b_main_model_preds["pred"],
        groups=groups_test_b,
        metric_fn=metric_fn,
        n_boot=3000,
        seed=42,
    )
    perm = grouped_permutation_test(
        y_true=y_test_b,
        pred_a=task_b_logreg_preds["pred"],
        pred_b=task_b_main_model_preds["pred"],
        groups=groups_test_b,
        metric_fn=metric_fn,
        n_perm=3000,
        seed=42,
    )
    baseline_score = float(metric_fn(y_test_b, task_b_logreg_preds["pred"]))
    roberta_score = float(metric_fn(y_test_b, task_b_main_model_preds["pred"]))
    significance_rows.append(
        {
            "metric": metric_name,
            "baseline_score": baseline_score,
            "roberta_score": roberta_score,
            "delta_roberta_minus_baseline": float(roberta_score - baseline_score),
            "bootstrap_ci95_low": boot["delta_ci95_low"],
            "bootstrap_median": boot["delta_median"],
            "bootstrap_ci95_high": boot["delta_ci95_high"],
            "permutation_p_two_sided": perm["p_value_two_sided"],
            "permutation_p_one_sided_roberta_gt_baseline": perm["p_value_one_sided_roberta_gt_baseline"],
        }
    )
    bootstrap_artifacts[metric_name] = boot
    permutation_artifacts[metric_name] = perm
task_b_significance_df = pd.DataFrame(significance_rows)
display(Markdown("### Table 2.5F. Grouped bootstrap and permutation significance summary"))
display_report_table(
    task_b_significance_df.round(4),
    row_color_col="metric",
    row_colors=REPORT_METRIC_COLORS,
    text_cols=["metric"],
    gradient_cols=["baseline_score", "roberta_score"],
    gradient_kwargs={"cmap": "YlGn"},
    signed_cols=["delta_roberta_minus_baseline", "bootstrap_ci95_low", "bootstrap_median", "bootstrap_ci95_high"],
    highlight_min_cols=["permutation_p_two_sided", "permutation_p_one_sided_roberta_gt_baseline"],
)
save_df(task_b_significance_df, RESULT_DIR / "task_b_roberta_vs_baseline_significance.csv")
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(bootstrap_artifacts["f1_macro"]["deltas"], bins=40, kde=True, ax=ax)
ax.axvline(0.0, linestyle="--")
ax.set_title(f"Grouped bootstrap of macro-F1 delta ({task_b_main_model_name} - Logistic)")
ax.set_xlabel("Macro-F1 delta")
boot_path = FIG_DIR / "task_b_roberta_vs_baseline_bootstrap_f1_delta.png"
fig.tight_layout()
fig.savefig(boot_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 25. Task B grouped bootstrap macro-F1 delta"))
display(IPyImage(filename=str(boot_path)))
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(permutation_artifacts["f1_macro"]["null_deltas"], bins=40, kde=True, ax=ax)
ax.axvline(0.0, linestyle="--")
ax.axvline(permutation_artifacts["f1_macro"]["observed_delta"], linestyle=":")
ax.set_title(f"Grouped permutation null — macro-F1 delta ({task_b_main_model_name} - Logistic)")
ax.set_xlabel("Macro-F1 delta under transcript-block swapping")
perm_path = FIG_DIR / "task_b_roberta_vs_baseline_permutation_f1_delta.png"
fig.tight_layout()
fig.savefig(perm_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 26. Task B grouped permutation macro-F1 delta"))
display(IPyImage(filename=str(perm_path)))


In [ ]:
_required_mcnemar = {
    "y_test_b": "held-out Task B labels",
    "task_b_logreg_preds": "elastic-net baseline predictions dict",
    "task_b_main_model_preds": "RoBERTa predictions dict",
    "task_b_main_model_name": "display name for the main Task B model",
}
_missing_mcnemar = [name for name in _required_mcnemar if name not in globals()]
if _missing_mcnemar:
    raise NameError(
        "McNemar diagnostic cannot run yet. Missing in-memory objects:\n  "
        + "\n  ".join(f"- {name}: {_required_mcnemar[name]}" for name in _missing_mcnemar)
        + "\nRun the Task 2.5A held-out comparison cell first."
    )
try:
    from scipy.stats import binomtest as _binomtest
except ImportError:
    from scipy.stats import binom_test as _legacy_binom_test
    class _BinomTestResult:
        def __init__(self, pvalue: float):
            self.pvalue = pvalue
    def _binomtest(k, n, p=0.5, alternative="two-sided"):
        return _BinomTestResult(float(_legacy_binom_test(k, n=n, p=p, alternative=alternative)))
_y_true_mcnemar = np.asarray(y_test_b)
_baseline_pred_mcnemar = np.asarray(task_b_logreg_preds["pred"])
_roberta_pred_mcnemar = np.asarray(task_b_main_model_preds["pred"])
if not (
    len(_y_true_mcnemar)
    == len(_baseline_pred_mcnemar)
    == len(_roberta_pred_mcnemar)
):
    raise ValueError("McNemar requires predictions for exactly the same held-out items.")
_baseline_correct = _baseline_pred_mcnemar == _y_true_mcnemar
_roberta_correct = _roberta_pred_mcnemar == _y_true_mcnemar
both_correct = int(np.sum(_baseline_correct & _roberta_correct))
baseline_only_correct = int(np.sum(_baseline_correct & ~_roberta_correct))
roberta_only_correct = int(np.sum(~_baseline_correct & _roberta_correct))
both_wrong = int(np.sum(~_baseline_correct & ~_roberta_correct))
discordant_total = baseline_only_correct + roberta_only_correct
if discordant_total == 0:
    mcnemar_exact_p = 1.0
    mcnemar_chi2_cc = 0.0
else:
    mcnemar_exact_p = float(
        _binomtest(
            min(baseline_only_correct, roberta_only_correct),
            n=discordant_total,
            p=0.5,
            alternative="two-sided",
        ).pvalue
    )
    mcnemar_chi2_cc = float(
        max(abs(roberta_only_correct - baseline_only_correct) - 1, 0) ** 2
        / discordant_total
    )
if baseline_only_correct == 0 and roberta_only_correct == 0:
    discordant_odds = np.nan
elif baseline_only_correct == 0:
    discordant_odds = np.inf
else:
    discordant_odds = float(roberta_only_correct / baseline_only_correct)
if roberta_only_correct > baseline_only_correct:
    direction = f"{task_b_main_model_name} correct on more discordant items"
elif roberta_only_correct < baseline_only_correct:
    direction = "Elastic-net baseline correct on more discordant items"
else:
    direction = "Discordant correctness counts are tied"
n_test_items = int(len(_y_true_mcnemar))
task_b_mcnemar_df = pd.DataFrame(
    [
        {
            "comparison": f"{task_b_main_model_name} vs elastic-net logistic regression",
            "n_test_items": n_test_items,
            "both_correct": both_correct,
            "baseline_only_correct": baseline_only_correct,
            "roberta_only_correct": roberta_only_correct,
            "both_wrong": both_wrong,
            "discordant_total": discordant_total,
            "discordant_difference_roberta_minus_baseline": int(
                roberta_only_correct - baseline_only_correct
            ),
            "discordant_odds_roberta_over_baseline": discordant_odds,
            "mcnemar_chi2_cc": mcnemar_chi2_cc,
            "mcnemar_exact_p_two_sided": mcnemar_exact_p,
            "p_lt_0_05": bool(mcnemar_exact_p < 0.05),
            "result_direction": direction,
        }
    ]
)
mcnemar_matrix_df = pd.DataFrame(
    [
        [both_wrong, roberta_only_correct],
        [baseline_only_correct, both_correct],
    ],
    index=["Baseline wrong", "Baseline correct"],
    columns=["RoBERTa wrong", "RoBERTa correct"],
)
task_b_mcnemar_counts_df = pd.DataFrame(
    [
        {
            "Pair outcome": "Both correct",
            "Elastic-net": "Correct",
            "RoBERTa": "Correct",
            "Count": both_correct,
            "Share of test": both_correct / n_test_items,
            "Role": "Agreement",
        },
        {
            "Pair outcome": "RoBERTa only correct",
            "Elastic-net": "Wrong",
            "RoBERTa": "Correct",
            "Count": roberta_only_correct,
            "Share of test": roberta_only_correct / n_test_items,
            "Role": "Discordant: RoBERTa advantage",
        },
        {
            "Pair outcome": "Baseline only correct",
            "Elastic-net": "Correct",
            "RoBERTa": "Wrong",
            "Count": baseline_only_correct,
            "Share of test": baseline_only_correct / n_test_items,
            "Role": "Discordant: baseline advantage",
        },
        {
            "Pair outcome": "Both wrong",
            "Elastic-net": "Wrong",
            "RoBERTa": "Wrong",
            "Count": both_wrong,
            "Share of test": both_wrong / n_test_items,
            "Role": "Agreement",
        },
    ]
)
task_b_mcnemar_result_df = pd.DataFrame(
    [
        {
            "Measure": "Test",
            "Value": "Exact McNemar / binomial",
            "Meaning": "Two-sided paired correctness test",
        },
        {
            "Measure": "Held-out items",
            "Value": f"{n_test_items:,}",
            "Meaning": "Same utterances scored by both classifiers",
        },
        {
            "Measure": "Discordant pairs",
            "Value": f"{discordant_total:,}",
            "Meaning": "Only these pairs drive McNemar's test",
        },
        {
            "Measure": "RoBERTa-only correct",
            "Value": f"{roberta_only_correct:,}",
            "Meaning": "RoBERTa correct while baseline is wrong",
        },
        {
            "Measure": "Baseline-only correct",
            "Value": f"{baseline_only_correct:,}",
            "Meaning": "Baseline correct while RoBERTa is wrong",
        },
        {
            "Measure": "Discordant difference",
            "Value": f"{roberta_only_correct - baseline_only_correct:+,}",
            "Meaning": "RoBERTa-only correct minus baseline-only correct",
        },
        {
            "Measure": "Discordant odds",
            "Value": "inf" if np.isinf(discordant_odds) else f"{discordant_odds:.4f}",
            "Meaning": "Values above 1 favour RoBERTa",
        },
        {
            "Measure": "Exact p-value",
            "Value": f"{mcnemar_exact_p:.4f}",
            "Meaning": "Significant at 0.05" if mcnemar_exact_p < 0.05 else "Not significant at 0.05",
        },
        {
            "Measure": "Direction",
            "Value": direction,
            "Meaning": "Diagnostic conclusion on paired correctness",
        },
    ]
)
_mcnemar_count_colors = {
    "Both correct": "#e9f7ef",
    "RoBERTa only correct": "#dbeafe",
    "Baseline only correct": "#fff3d9",
    "Both wrong": "#fce8ef",
}
_mcnemar_result_colors = {
    "Test": "#f1f5f9",
    "Held-out items": "#f8fafc",
    "Discordant pairs": "#ede9fe",
    "RoBERTa-only correct": "#dbeafe",
    "Baseline-only correct": "#fff3d9",
    "Discordant difference": "#e9f7ef" if roberta_only_correct >= baseline_only_correct else "#fce8ef",
    "Discordant odds": "#e9f7ef" if discordant_odds >= 1 else "#fce8ef",
    "Exact p-value": "#d1fae5" if mcnemar_exact_p < 0.05 else "#fee2e2",
    "Direction": "#f1f5f9",
}
display(Markdown("### Table 2.5F-2A. McNemar paired correctness counts"))
display_report_table(
    task_b_mcnemar_counts_df,
    formats={"Count": "{:,}", "Share of test": "{:.1%}"},
    row_color_col="Pair outcome",
    row_colors=_mcnemar_count_colors,
    text_cols=["Pair outcome", "Elastic-net", "RoBERTa", "Role"],
    gradient_cols=["Count", "Share of test"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
display(Markdown("### Table 2.5F-2B. McNemar exact test summary"))
display_report_table(
    task_b_mcnemar_result_df,
    row_color_col="Measure",
    row_colors=_mcnemar_result_colors,
    text_cols=["Measure", "Value", "Meaning"],
)
display(Markdown(
    "McNemar is included as an item-paired diagnostic on the locked test set. "
    "Because Task B utterances are clustered within transcripts, the grouped "
    "bootstrap and grouped permutation results above remain the primary "
    "significance evidence."
))
fig, ax = plt.subplots(figsize=(6.5, 4.8))
sns.heatmap(
    mcnemar_matrix_df,
    annot=True,
    fmt="d",
    cmap="YlGnBu",
    cbar=False,
    linewidths=0.8,
    linecolor="#cbd5e1",
    ax=ax,
)
ax.set_title("Task B paired correctness matrix")
ax.set_xlabel(task_b_main_model_name)
ax.set_ylabel("Elastic-net logistic regression")
mcnemar_fig_path = FIG_DIR / "task_b_roberta_vs_baseline_mcnemar_correctness_matrix.png"
fig.tight_layout()
fig.savefig(mcnemar_fig_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 26B. Task B McNemar paired correctness matrix"))
display(IPyImage(filename=str(mcnemar_fig_path)))
save_df(task_b_mcnemar_df, RESULT_DIR / "task_b_roberta_vs_baseline_mcnemar.csv")
save_df(task_b_mcnemar_counts_df, RESULT_DIR / "task_b_roberta_vs_baseline_mcnemar_counts.csv")
save_df(task_b_mcnemar_result_df, RESULT_DIR / "task_b_roberta_vs_baseline_mcnemar_summary.csv")
save_df(
    mcnemar_matrix_df.reset_index().rename(columns={"index": "baseline_correctness"}),
    RESULT_DIR / "task_b_roberta_vs_baseline_mcnemar_matrix.csv",
)


In [ ]:
def make_error_slice_table(
    df: pd.DataFrame,
    y_true: np.ndarray,
    y_pred: np.ndarray,
    slice_col: str,
) -> pd.DataFrame:
    rows = []
    for value, sub_idx in df.groupby(slice_col, observed=False).groups.items():
        idx = np.array(list(sub_idx))
        if len(idx) < 5:
            continue
        rows.append(
            {
                "slice_feature": slice_col,
                "slice_value": value,
                "n": len(idx),
                "accuracy": accuracy_score(y_true[idx], y_pred[idx]),
                "f1_macro": f1_score(y_true[idx], y_pred[idx], average="macro"),
            }
        )
    return pd.DataFrame(rows)
error_df = therapist_test.reset_index(drop=True).copy()
error_df["y_true"] = y_test_b
error_df["logreg_pred"] = task_b_logreg_preds["pred"]
error_df["main_pred"] = task_b_main_model_preds["pred"]
error_df["length_bucket"] = pd.cut(
    error_df["word_count"],
    bins=[-1, 3, 8, 15, 1000],
    labels=["very_short (≤3w)", "short (4–8w)", "medium (9–15w)", "long (16+w)"],
)
error_df["position_bucket"] = pd.cut(
    error_df["turn_ratio"],
    bins=[-0.01, 0.25, 0.5, 0.75, 1.0],
    labels=["q1 (start)", "q2", "q3", "q4 (end)"],
)
slice_tables = []
for feature in [
    "main_therapist_behaviour",
    "mi_quality",
    "topic_norm",
    "length_bucket",
    "position_bucket",
]:
    t1 = make_error_slice_table(
        error_df,
        error_df["y_true"].to_numpy(),
        error_df["logreg_pred"].to_numpy(),
        feature,
    )
    t1["model"] = "Elastic-net logistic regression"
    t2 = make_error_slice_table(
        error_df,
        error_df["y_true"].to_numpy(),
        error_df["main_pred"].to_numpy(),
        feature,
    )
    t2["model"] = task_b_main_model_name
    slice_tables.append(pd.concat([t1, t2], ignore_index=True))
task_b_error_slices_df = pd.concat(slice_tables, ignore_index=True)
_slice_feature_labels = {
    "main_therapist_behaviour": "Therapist behaviour",
    "mi_quality": "MI quality",
    "topic_norm": "Topic",
    "length_bucket": "Utterance length",
    "position_bucket": "Transcript position",
}
_baseline_name = "Elastic-net logistic regression"
_main_name = task_b_main_model_name
_slice_f1 = task_b_error_slices_df.pivot_table(
    index=["slice_feature", "slice_value", "n"],
    columns="model",
    values="f1_macro",
    aggfunc="first",
    observed=False,
).reset_index()
_slice_acc = task_b_error_slices_df.pivot_table(
    index=["slice_feature", "slice_value", "n"],
    columns="model",
    values="accuracy",
    aggfunc="first",
    observed=False,
).reset_index()
task_b_error_slices_report = _slice_f1.rename(
    columns={_baseline_name: "F1 Baseline", _main_name: "F1 RoBERTa"}
).merge(
    _slice_acc.rename(columns={_baseline_name: "Accuracy Baseline", _main_name: "Accuracy RoBERTa"}),
    on=["slice_feature", "slice_value", "n"],
    how="left",
)
task_b_error_slices_report["Delta F1"] = task_b_error_slices_report["F1 RoBERTa"] - task_b_error_slices_report["F1 Baseline"]
task_b_error_slices_report = (
    task_b_error_slices_report.assign(
        slice_feature=lambda d: d["slice_feature"].map(_slice_feature_labels).fillna(d["slice_feature"]),
        slice_value=lambda d: d["slice_value"].astype(str).str.replace("_", " ", regex=False),
    )
    .rename(columns={"slice_feature": "Slice", "slice_value": "Value", "n": "N"})
    .sort_values(["Slice", "N", "Value"], ascending=[True, False, True])
)
_topic_rows = task_b_error_slices_report[task_b_error_slices_report["Slice"] == "Topic"].head(12)
_non_topic_rows = task_b_error_slices_report[task_b_error_slices_report["Slice"] != "Topic"]
task_b_error_slices_display = pd.concat([_non_topic_rows, _topic_rows], ignore_index=True)
if "_display_report_table" not in globals():
    def _display_report_table(df, formats=None, text_cols=None, gradient_cols=None, gradient_kwargs=None) -> None:
        formats = formats or {}
        text_cols = text_cols or []
        gradient_cols = gradient_cols or []
        gradient_kwargs = gradient_kwargs or {}
        try:
            styler = df.style.hide(axis="index")
            if formats:
                styler = styler.format(formats)
            if text_cols:
                styler = styler.set_properties(subset=text_cols, **{"text-align": "left", "max-width": "620px"})
            if gradient_cols:
                styler = styler.background_gradient(subset=gradient_cols, **gradient_kwargs)
            display(styler)
        except Exception:
            fallback = df.copy()
            for col, fmt in formats.items():
                if col in fallback.columns:
                    fallback[col] = fallback[col].map(lambda v: "" if pd.isna(v) else fmt.format(v))
            display(fallback.reset_index(drop=True))
display(Markdown("### Table 2.5D. Per-slice held-out metrics (concise report view)"))
_display_report_table(
    task_b_error_slices_display[
        ["Slice", "Value", "N", "F1 Baseline", "F1 RoBERTa", "Delta F1", "Accuracy Baseline", "Accuracy RoBERTa"]
    ],
    formats={
        "F1 Baseline": "{:.3f}",
        "F1 RoBERTa": "{:.3f}",
        "Delta F1": "{:+.3f}",
        "Accuracy Baseline": "{:.3f}",
        "Accuracy RoBERTa": "{:.3f}",
    },
    text_cols=["Value"],
    gradient_cols=["Delta F1"],
    gradient_kwargs={"cmap": "RdYlGn", "vmin": -0.25, "vmax": 0.25},
)
display(Markdown("Full per-slice table is saved to `task_b_roberta_vs_baseline_error_slices.csv`; the report view shows all non-topic slices plus the 12 largest topic slices."))
save_df(
    task_b_error_slices_df,
    RESULT_DIR / "task_b_roberta_vs_baseline_error_slices.csv",
)
def risk_coverage_curve(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    n_points: int = 20,
) -> pd.DataFrame:
    """Sort predictions by max-class confidence (desc) and plot risk
    (1 - accuracy) at each coverage threshold. Lower is better; a
    well-calibrated, confident model has a steeply rising left edge."""
    conf = y_prob.max(axis=1)
    pred = y_prob.argmax(axis=1)
    order = np.argsort(conf)[::-1]
    rows = []
    for frac in np.linspace(0.05, 1.0, n_points):
        n = max(1, int(len(order) * frac))
        idx = order[:n]
        acc = accuracy_score(y_true[idx], pred[idx])
        rows.append({"coverage": frac, "risk": 1 - acc})
    return pd.DataFrame(rows)
rc_baseline = risk_coverage_curve(y_test_b, task_b_logreg_preds["prob"])
rc_uncal = risk_coverage_curve(y_test_b, task_b_main_model_preds["prob"])
rc_cal = risk_coverage_curve(y_test_b, task_b_main_model_probs_cal)
fig, ax = plt.subplots(figsize=(7.5, 5.0))
ax.plot(
    rc_baseline["coverage"], rc_baseline["risk"],
    marker="s", linewidth=2, color="#888", linestyle="--",
    label="Elastic-net logistic regression",
)
ax.plot(
    rc_uncal["coverage"], rc_uncal["risk"],
    marker="o", linewidth=2,
    label=f"{task_b_main_model_name} (uncalibrated)",
)
ax.plot(
    rc_cal["coverage"], rc_cal["risk"],
    marker="^", linewidth=2, alpha=0.85,
    label=f"{task_b_main_model_name} + temperature scaling",
)
ax.set_title("Task b risk–coverage on the held-out test set")
ax.set_xlabel("Coverage (fraction of test predictions retained, sorted by confidence)")
ax.set_ylabel("Risk = 1 − accuracy")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", framealpha=0.9)
rc_path = FIG_DIR / "task_b_risk_coverage_three_models.png"
fig.tight_layout()
fig.savefig(rc_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 27. Task B risk-coverage curves across three models"))
display(IPyImage(filename=str(rc_path)))
risk_coverage_long = pd.concat(
    [
        rc_baseline.assign(model="Elastic-net logistic regression"),
        rc_uncal.assign(model=f"{task_b_main_model_name} (uncalibrated)"),
        rc_cal.assign(model=f"{task_b_main_model_name} + temperature scaling"),
    ],
    ignore_index=True,
)
risk_coverage_display = (
    risk_coverage_long
    .pivot_table(index="coverage", columns="model", values="risk", aggfunc="first")
    .reset_index()
)
risk_coverage_display.columns.name = None
_risk_coverage_formats = {"coverage": "{:.2f}"}
_risk_coverage_formats.update(
    {col: "{:.3f}" for col in risk_coverage_display.columns if col != "coverage"}
)
display(Markdown("### Table 2.5G. Risk-coverage values for all three Task B models"))
_display_report_table(
    risk_coverage_display,
    formats=_risk_coverage_formats,
    gradient_cols=[col for col in risk_coverage_display.columns if col != "coverage"],
    gradient_kwargs={"cmap": "RdYlGn_r", "vmin": 0.0, "vmax": 0.5},
)
save_df(risk_coverage_long, RESULT_DIR / "task_b_risk_coverage_three_models.csv")


In [ ]:
assert "task_b_logreg_preds" in globals(), (
    "task_b_logreg_preds not in memory — run the elastic-net baseline cell first."
)
assert "task_b_main_model_preds" in globals(), (
    "task_b_main_model_preds not in memory — run the Task 2.5A cell first."
)
assert "task_b_main_model_probs_cal" in globals(), (
    "task_b_main_model_probs_cal not in memory — run the temperature-scaling cell first."
)
def _reliability_curve_xy(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    n_bins: int = 10,
):
    """Return (bin_confidence, bin_accuracy, bin_n) for an overlay plot."""
    confidences = y_prob.max(axis=1)
    predictions = y_prob.argmax(axis=1)
    accuracies = (predictions == y_true).astype(float)
    bin_edges = np.linspace(0.0, 1.0, n_bins + 1)
    xs, ys, ns = [], [], []
    for j in range(n_bins):
        mask = (confidences > bin_edges[j]) & (confidences <= bin_edges[j + 1])
        if mask.sum() == 0:
            continue
        xs.append(float(confidences[mask].mean()))
        ys.append(float(accuracies[mask].mean()))
        ns.append(int(mask.sum()))
    return np.asarray(xs), np.asarray(ys), np.asarray(ns)
reliability_models = [
    {
        "name": "Elastic-net logistic regression",
        "prob": task_b_logreg_preds["prob"],
        "style": {"linestyle": "--", "color": "#888", "marker": "s"},
    },
    {
        "name": f"{task_b_main_model_name} (uncalibrated)",
        "prob": task_b_main_model_preds["prob"],
        "style": {"linestyle": "-", "marker": "o"},
    },
    {
        "name": f"{task_b_main_model_name} + temperature scaling",
        "prob": task_b_main_model_probs_cal,
        "style": {"linestyle": "-", "marker": "^", "alpha": 0.85},
    },
]
N_BINS_OVERLAY = 10
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.plot([0, 1], [0, 1], linestyle=":", color="black",
        label="perfect calibration", linewidth=1)
for spec in reliability_models:
    xs, ys, ns = _reliability_curve_xy(
        y_test_b, np.asarray(spec["prob"]), n_bins=N_BINS_OVERLAY,
    )
    if len(xs) == 0:
        continue
    sizes = 30 + 200 * (ns / max(ns.sum(), 1))
    ax.plot(xs, ys, linewidth=2, **spec["style"], label=spec["name"])
    ax.scatter(xs, ys, s=sizes, edgecolors="black", linewidths=0.5,
               color=spec["style"].get("color"), zorder=5)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Predicted confidence (top-class probability)")
ax.set_ylabel("Empirical accuracy")
ax.set_title(
    f"Task b reliability — three models on the same axes (n_bins={N_BINS_OVERLAY})"
)
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", framealpha=0.9)
reliability_overlay_path = FIG_DIR / "task_b_reliability_three_models.png"
fig.tight_layout()
fig.savefig(reliability_overlay_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 28. Task B reliability overlay across three models"))
display(IPyImage(filename=str(reliability_overlay_path)))


In [ ]:
task_b_diag_df = therapist_test.reset_index(drop=True).copy()
task_b_diag_df["y_true"] = y_test_b
task_b_diag_df["true_label"] = label_encoder.inverse_transform(y_test_b)
task_b_diag_df["logreg_pred"] = task_b_logreg_preds["pred"]
task_b_diag_df["logreg_pred_label"] = label_encoder.inverse_transform(task_b_logreg_preds["pred"])
task_b_diag_df["logreg_conf"] = task_b_logreg_preds["prob"].max(axis=1)
task_b_diag_df["logreg_correct"] = task_b_diag_df["logreg_pred"] == task_b_diag_df["y_true"]
task_b_diag_df["main_pred"] = task_b_main_model_preds["pred"]
task_b_diag_df["main_pred_label"] = label_encoder.inverse_transform(task_b_main_model_preds["pred"])
task_b_diag_df["main_conf"] = task_b_main_model_preds["prob"].max(axis=1)
task_b_diag_df["main_correct"] = task_b_diag_df["main_pred"] == task_b_diag_df["y_true"]
utterance_bucket_order = ["very_short", "short", "medium", "long", "very_long"]
context_bucket_order = ["none", "very_short", "short", "medium", "long"]
task_b_diag_df["utterance_length_bucket"] = pd.cut(
    task_b_diag_df["word_count"],
    bins=[-1, 3, 8, 15, 30, 1000],
    labels=utterance_bucket_order,
)
if "history_tagged_turns" in task_b_diag_df.columns:
    def _history_token_count(turns) -> int:
        if isinstance(turns, str):
            try:
                turns = ast.literal_eval(turns)
            except Exception:
                turns = [turns]
        if not isinstance(turns, list):
            return 0
        return sum(len(TOKEN_RE.findall(str(t))) for t in turns)
    task_b_diag_df["available_context_tokens"] = task_b_diag_df["history_tagged_turns"].apply(_history_token_count)
else:
    context_col = f"context_k{int(CONFIG.get('max_context_k', 10))}_lc"
    task_b_diag_df["available_context_tokens"] = (
        task_b_diag_df[context_col].fillna("").astype(str).apply(lambda s: len(TOKEN_RE.findall(s)))
        - task_b_diag_df["word_count"].fillna(0).astype(int)
    ).clip(lower=0)
task_b_diag_df["available_context_bucket"] = pd.cut(
    task_b_diag_df["available_context_tokens"],
    bins=[-1, 0, 10, 25, 50, 1000],
    labels=context_bucket_order,
)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, conf_col, correct_col, title in [
    (axes[0], "logreg_conf", "logreg_correct", "Elastic-net logistic regression"),
    (axes[1], "main_conf", "main_correct", task_b_main_model_name),
]:
    plot_df = task_b_diag_df[[conf_col, correct_col]].copy()
    plot_df["outcome"] = np.where(plot_df[correct_col], "correct", "wrong")
    sns.histplot(
        data=plot_df,
        x=conf_col,
        hue="outcome",
        bins=20,
        stat="density",
        common_norm=False,
        element="step",
        fill=False,
        ax=ax,
    )
    ax.set_title(f"Confidence by correctness — {title}")
    ax.set_xlabel("Predicted confidence")
    ax.set_ylabel("Density")
confidence_path = FIG_DIR / "task_b_confidence_histograms_roberta_vs_baseline.png"
fig.tight_layout()
fig.savefig(confidence_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 29. Task B confidence histograms for baseline vs RoBERTa"))
display(IPyImage(filename=str(confidence_path)))
task_b_label_order = [c for c in ["reflection", "question", "therapist_input", "other"] if c in label_encoder.classes_]
label_to_idx = {label: idx for idx, label in enumerate(label_encoder.classes_)}
fp_fn_rows = []
for model_name, pred_col in [
    ("Elastic-net logistic regression", "logreg_pred"),
    (task_b_main_model_name, "main_pred"),
]:
    pred = task_b_diag_df[pred_col].to_numpy()
    for label_name in task_b_label_order:
        class_idx = label_to_idx[label_name]
        y_pos = task_b_diag_df["y_true"].to_numpy() == class_idx
        p_pos = pred == class_idx
        fp_fn_rows.append(
            {
                "model": model_name,
                "label": label_name,
                "error_type": "false_positive",
                "count": int((~y_pos & p_pos).sum()),
            }
        )
        fp_fn_rows.append(
            {
                "model": model_name,
                "label": label_name,
                "error_type": "false_negative",
                "count": int((y_pos & ~p_pos).sum()),
            }
        )
task_b_fp_fn_df = pd.DataFrame(fp_fn_rows)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, error_type in zip(axes, ["false_positive", "false_negative"]):
    sub = task_b_fp_fn_df[task_b_fp_fn_df["error_type"] == error_type]
    sns.barplot(
        data=sub,
        x="label",
        y="count",
        hue="model",
        order=task_b_label_order,
        ax=ax,
    )
    ax.set_title(error_type.replace("_", " ").title() + "s by class")
    ax.set_xlabel("")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=15)
axes[0].legend(title="")
axes[1].legend_.remove()
fpfn_path = FIG_DIR / "task_b_fp_fn_by_class_roberta_vs_baseline.png"
fig.tight_layout()
fig.savefig(fpfn_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 30. Task B false positives and false negatives by class"))
display(IPyImage(filename=str(fpfn_path)))
bucket_rows = []
for bucket_col, model_name, correct_col in [
    ("utterance_length_bucket", "Elastic-net logistic regression", "logreg_correct"),
    ("utterance_length_bucket", task_b_main_model_name, "main_correct"),
    ("available_context_bucket", "Elastic-net logistic regression", "logreg_correct"),
    ("available_context_bucket", task_b_main_model_name, "main_correct"),
]:
    tmp = (
        task_b_diag_df.groupby(bucket_col, observed=False)
        .agg(
            n=("y_true", "size"),
            accuracy=(correct_col, "mean"),
        )
        .reset_index()
        .rename(columns={bucket_col: "bucket"})
    )
    tmp["bucket_feature"] = bucket_col
    tmp["model"] = model_name
    tmp["error_rate"] = 1 - tmp["accuracy"]
    bucket_rows.append(tmp)
task_b_error_bucket_df = pd.concat(bucket_rows, ignore_index=True)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for ax, bucket_feature, title, order in [
    (axes[0], "utterance_length_bucket", "Error rate by utterance length", utterance_bucket_order),
    (axes[1], "available_context_bucket", "Error rate by available prior context", context_bucket_order),
]:
    sub = task_b_error_bucket_df[task_b_error_bucket_df["bucket_feature"] == bucket_feature]
    sns.barplot(data=sub, x="bucket", y="error_rate", hue="model", order=order, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Error rate")
    ax.tick_params(axis="x", rotation=15)
axes[0].legend(title="")
axes[1].legend_.remove()
length_context_path = FIG_DIR / "task_b_error_by_length_and_context_roberta_vs_baseline.png"
fig.tight_layout()
fig.savefig(length_context_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 31. Task B error rate by length and context"))
display(IPyImage(filename=str(length_context_path)))
task_b_transcript_error_df = pd.concat(
    [
        task_b_diag_df.groupby("transcript_id")["logreg_correct"].mean().reset_index(name="accuracy").assign(model="Elastic-net logistic regression"),
        task_b_diag_df.groupby("transcript_id")["main_correct"].mean().reset_index(name="accuracy").assign(model=task_b_main_model_name),
    ],
    ignore_index=True,
)
task_b_transcript_error_df["error_rate"] = 1 - task_b_transcript_error_df["accuracy"]
fig, ax = plt.subplots(figsize=(8, 4.8))
sns.boxplot(data=task_b_transcript_error_df, x="model", y="error_rate", ax=ax)
sns.stripplot(data=task_b_transcript_error_df, x="model", y="error_rate", alpha=0.5, size=4, ax=ax)
ax.set_title("Transcript-level error variability")
ax.set_xlabel("")
ax.set_ylabel("Error rate")
ax.tick_params(axis="x", rotation=10)
transcript_error_path = FIG_DIR / "task_b_transcript_error_variability_roberta_vs_baseline.png"
fig.tight_layout()
fig.savefig(transcript_error_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 32. Task B transcript-level error variability"))
display(IPyImage(filename=str(transcript_error_path)))
save_df(task_b_fp_fn_df, RESULT_DIR / "task_b_fp_fn_by_class_roberta_vs_baseline.csv")
save_df(task_b_error_bucket_df, RESULT_DIR / "task_b_error_by_length_and_context_roberta_vs_baseline.csv")
save_df(task_b_transcript_error_df, RESULT_DIR / "task_b_transcript_error_variability_roberta_vs_baseline.csv")


In [ ]:
if "task_b_diag_df" not in globals():
    raise NameError("Run the confidence and error diagnostics cell first.")
task_b_success_examples = (
    task_b_diag_df[task_b_diag_df["main_correct"]]
    .sort_values(["main_conf", "word_count"], ascending=[False, False])
    .groupby("true_label", group_keys=False)
    .head(2)
    .loc[
        :,
        [
            "transcript_id",
            "mi_quality",
            "topic_norm",
            "true_label",
            "main_pred_label",
            "main_conf",
            "prev_client_text",
            "utterance_text",
        ],
    ]
    .reset_index(drop=True)
)
task_b_failure_examples = (
    task_b_diag_df[~task_b_diag_df["main_correct"]]
    .sort_values(["main_conf", "word_count"], ascending=[False, False])
    .groupby(["true_label", "main_pred_label"], group_keys=False)
    .head(1)
    .loc[
        :,
        [
            "transcript_id",
            "mi_quality",
            "topic_norm",
            "true_label",
            "main_pred_label",
            "main_conf",
            "prev_client_text",
            "utterance_text",
        ],
    ]
    .reset_index(drop=True)
)
if "_report_label" not in globals():
    def _report_label(x) -> str:
        return str(x).replace("_", " ").title()
if "_clip_report_text" not in globals():
    def _clip_report_text(x, width: int = 140) -> str:
        text = "" if pd.isna(x) else " ".join(str(x).split())
        return text if len(text) <= width else text[: width - 1].rstrip() + "..."
if "_display_report_table" not in globals():
    def _display_report_table(df, formats=None, text_cols=None, gradient_cols=None, gradient_kwargs=None) -> None:
        formats = formats or {}
        text_cols = text_cols or []
        try:
            styler = df.style.hide(axis="index")
            if formats:
                styler = styler.format(formats)
            if text_cols:
                styler = styler.set_properties(subset=text_cols, **{"text-align": "left", "max-width": "620px"})
            display(styler)
        except Exception:
            fallback = df.copy()
            for col, fmt in formats.items():
                if col in fallback.columns:
                    fallback[col] = fallback[col].map(lambda v: "" if pd.isna(v) else fmt.format(v))
            display(fallback.reset_index(drop=True))
def _prepare_example_report(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.assign(
            true_label=lambda d: d["true_label"].map(_report_label),
            main_pred_label=lambda d: d["main_pred_label"].map(_report_label),
            topic_norm=lambda d: d["topic_norm"].map(lambda x: _clip_report_text(x, 42)),
            prev_client_text=lambda d: d["prev_client_text"].map(lambda x: _clip_report_text(x, 120)),
            utterance_text=lambda d: d["utterance_text"].map(lambda x: _clip_report_text(x, 160)),
        )
        .rename(
            columns={
                "transcript_id": "Transcript",
                "mi_quality": "MI Quality",
                "topic_norm": "Topic",
                "true_label": "True Label",
                "main_pred_label": "Predicted Label",
                "main_conf": "Confidence",
                "prev_client_text": "Previous Client Turn",
                "utterance_text": "Therapist Turn",
            }
        )
    )
task_b_success_examples_report = _prepare_example_report(task_b_success_examples)
task_b_failure_examples_report = _prepare_example_report(task_b_failure_examples)
display(Markdown(f"### Representative successes — {task_b_main_model_name}"))
_display_report_table(
    task_b_success_examples_report,
    formats={"Confidence": "{:.3f}"},
    text_cols=["Previous Client Turn", "Therapist Turn"],
)
display(Markdown(f"### Representative failures — {task_b_main_model_name}"))
_display_report_table(
    task_b_failure_examples_report,
    formats={"Confidence": "{:.3f}"},
    text_cols=["Previous Client Turn", "Therapist Turn"],
)
save_df(task_b_success_examples, RESULT_DIR / "task_b_roberta_representative_successes.csv")
save_df(task_b_failure_examples, RESULT_DIR / "task_b_roberta_representative_failures.csv")

In [ ]:
task_a_dir = ARTIFACT_DIR / "task_a"
task_a_dir.mkdir(parents=True, exist_ok=True)
transcript_feature_cols = [
    "n_turns",
    "therapist_turns",
    "client_turns",
    "mean_word_count",
    "median_word_count",
    "mean_char_count",
    "therapist_question_rate",
    "prop_reflection",
    "prop_question",
    "prop_therapist_input",
    "prop_other",
    "prop_change",
    "prop_neutral",
    "prop_sustain",
    "reflection_to_question_ratio",
    "change_to_sustain_ratio",
    "mean_prev_partner_lexical_overlap",
    "topic_norm",
]
if "mean_prev_client_semantic_cosine" in transcript_df.columns:
    transcript_feature_cols.append("mean_prev_client_semantic_cosine")
mi_label_encoder = LabelEncoder()
transcript_df["mi_y"] = mi_label_encoder.fit_transform(transcript_df["mi_quality"])
transcript_train = transcript_df[transcript_df["transcript_id"].isin(TRAIN_TRANSCRIPTS)].copy()
transcript_test = transcript_df[transcript_df["transcript_id"].isin(TEST_TRANSCRIPTS)].copy()
X_train_a = transcript_train[transcript_feature_cols].copy()
X_test_a = transcript_test[transcript_feature_cols].copy()
y_train_a = transcript_train["mi_y"].to_numpy()
y_test_a = transcript_test["mi_y"].to_numpy()
task_a_numeric = [c for c in transcript_feature_cols if c != "topic_norm"]
task_a_categorical = ["topic_norm"]
def build_task_a_logreg(C=1.0, l1_ratio=0.15, class_weight="balanced"):
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler", StandardScaler()),
            ]), task_a_numeric),
            ("cat", OneHotEncoder(handle_unknown="ignore"), task_a_categorical),
        ]
    )
    clf = LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=C,
        l1_ratio=l1_ratio,
        class_weight=class_weight,
        max_iter=5000,
        random_state=SEED,
    )
    return Pipeline([("pre", pre), ("clf", clf)])
def build_task_a_boosted():
    if globals().get("XGBClassifier") is not None:
        model = XGBClassifier(
            n_estimators=300,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=SEED,
            tree_method="hist",
        )
        pre = ColumnTransformer(
            transformers=[
                ("num", Pipeline([
                    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ]), task_a_numeric),
                ("cat", OneHotEncoder(handle_unknown="ignore"), task_a_categorical),
            ]
        )
        return Pipeline([("pre", pre), ("clf", model)]), "xgboost"
    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=4,
        max_iter=300,
        random_state=SEED,
    )
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler", StandardScaler()),
            ]), task_a_numeric),
            ("cat", OneHotEncoder(handle_unknown="ignore"), task_a_categorical),
        ]
    )
    return Pipeline([("pre", pre), ("clf", model)]), "hist_gradient_boosting"


In [ ]:
_required_helpers = [
    "ARTIFACT_DIR", "CONFIG", "SEED",
    "X_train_a", "X_test_a", "y_train_a", "y_test_a",
    "compute_binary_metrics", "build_task_a_logreg",
    "save_json", "load_json", "save_df", "save_joblib", "load_joblib",
    "artifact_exists",
]
_missing_helpers = [name for name in _required_helpers if name not in globals()]
assert not _missing_helpers, (
    f"Task a workflow cannot run: missing globals {_missing_helpers}. "
    "Re-run cell 5 (environment) and cell 79 (Task a feature prep) first."
)
if not isinstance(X_train_a, pd.DataFrame) or not isinstance(X_test_a, pd.DataFrame):
    raise TypeError("This Appendix A cell expects X_train_a and X_test_a to be pandas DataFrames.")
X_train_a = X_train_a.copy()
X_test_a = X_test_a.copy()
y_train_a = np.asarray(y_train_a)
y_test_a = np.asarray(y_test_a)
SELECTION_METRIC = "accuracy"
LOGREG_CV_FOLDS = 5
BOOSTED_CV_FOLDS = 5
BOOSTED_RANDOM_SEARCH_N_ITER = 18
if "task_a_dir" not in globals():
    task_a_dir = ARTIFACT_DIR / "task_a"
    task_a_dir.mkdir(parents=True, exist_ok=True)
task_a_metrics_path = task_a_dir / "task_a_metrics.json"
task_a_preds_path = task_a_dir / "task_a_predictions.joblib"
task_a_models_path = task_a_dir / "task_a_models.joblib"
task_a_meta_path = task_a_dir / "task_a_meta.json"
task_a_logreg_cv_path = task_a_dir / "task_a_logreg_cv_results.csv"
task_a_boosted_cv_path = task_a_dir / "task_a_boosted_cv_results.csv"
task_a_thresholds_path = task_a_dir / "task_a_thresholds.json"
current_task_a_meta: Dict[str, Any] = {
    "n_train": int(len(X_train_a)),
    "n_test": int(len(X_test_a)),
    "feature_cols": list(X_train_a.columns),
    "test_transcript_ids": (
        sorted(transcript_test["transcript_id"].astype(str).tolist())
        if "transcript_test" in globals() and "transcript_id" in transcript_test.columns
        else None
    ),
}
def _meta_compatible(saved: Dict[str, Any]) -> bool:
    return (
        saved.get("n_test") == current_task_a_meta["n_test"]
        and saved.get("feature_cols") == current_task_a_meta["feature_cols"]
        and saved.get("test_transcript_ids") == current_task_a_meta["test_transcript_ids"]
    )
def _predictions_compatible(cached_preds: Dict[str, Any]) -> bool:
    n_test = current_task_a_meta["n_test"]
    return all(
        len(payload["pred"]) == n_test and len(payload["prob"]) == n_test
        for payload in cached_preds.values()
    )
def task_a_cache_is_compatible() -> bool:
    needed = [task_a_metrics_path, task_a_preds_path, task_a_meta_path]
    if not all(artifact_exists(p) for p in needed):
        return False
    try:
        if not _meta_compatible(load_json(task_a_meta_path)):
            return False
        if not _predictions_compatible(load_joblib(task_a_preds_path)):
            return False
        return True
    except Exception as exc:
        print(f"Cache compatibility check failed: {exc}")
        return False
def _score_binary(y_true, pred, prob, metric_name: str = "accuracy") -> float:
    if metric_name == "accuracy":
        return accuracy_score(y_true, pred)
    if metric_name == "balanced_accuracy":
        return balanced_accuracy_score(y_true, pred)
    if metric_name == "f1":
        return f1_score(y_true, pred)
    if metric_name == "roc_auc":
        return roc_auc_score(y_true, prob)
    raise ValueError(f"Unknown metric_name: {metric_name}")
def _best_threshold(y_true, prob, metric_name: str = "accuracy") -> Tuple[float, float]:
    """Pick the threshold that maximises the selection metric, with F1 as tie-break."""
    thresholds = np.linspace(0.10, 0.90, 81)
    best_t, best_s, best_f1 = 0.50, -np.inf, -np.inf
    for t in thresholds:
        pred = (prob >= t).astype(int)
        s = _score_binary(y_true, pred, prob, metric_name=metric_name)
        f1 = f1_score(y_true, pred)
        if (s > best_s) or (np.isclose(s, best_s) and f1 > best_f1):
            best_s, best_t, best_f1 = float(s), float(t), float(f1)
    return best_t, best_s
def _predict_positive_proba(model, X) -> np.ndarray:
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return 1.0 / (1.0 + np.exp(-model.decision_function(X)))
    raise AttributeError("Model has neither predict_proba nor decision_function")
def _make_logreg_candidate(C: float, l1_ratio: float, class_weight) -> Any:
    sig = inspect.signature(build_task_a_logreg)
    kwargs: Dict[str, Any] = {"C": float(C), "l1_ratio": float(l1_ratio)}
    if "class_weight" in sig.parameters:
        kwargs["class_weight"] = class_weight
    return build_task_a_logreg(**kwargs)
def _fit_boost_preprocessor(X: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    X = X.copy()
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    bool_cols = X.select_dtypes(include=["bool"]).columns.tolist()
    cat_cols = sorted(set(cat_cols + bool_cols))
    num_cols = [c for c in X.columns if c not in cat_cols]
    parts: List[pd.DataFrame] = []
    medians: Dict[str, float] = {}
    if num_cols:
        X_num = X[num_cols].apply(pd.to_numeric, errors="coerce")
        medians = X_num.median().to_dict()
        X_num = X_num.fillna(pd.Series(medians))
        parts.append(X_num)
    cat_schema: Dict[str, List[str]] = {}
    for c in cat_cols:
        s = X[c].astype("string").fillna("__MISSING__")
        cats = sorted(s.unique().tolist())
        cat_schema[c] = cats
        s_cat = pd.Series(pd.Categorical(s, categories=cats), index=X.index, name=c)
        parts.append(pd.get_dummies(s_cat, prefix=c, dtype=float))
    X_out = pd.concat(parts, axis=1) if parts else pd.DataFrame(index=X.index)
    X_out = X_out.astype(float)
    preprocessor = {
        "num_cols": num_cols,
        "cat_cols": cat_cols,
        "medians": medians,
        "cat_schema": cat_schema,
        "columns": X_out.columns.tolist(),
    }
    return X_out, preprocessor
def _transform_boost_features(X: pd.DataFrame, preprocessor: Dict[str, Any]) -> pd.DataFrame:
    X = X.copy()
    parts: List[pd.DataFrame] = []
    if preprocessor["num_cols"]:
        X_num = (
            X.reindex(columns=preprocessor["num_cols"])
             .apply(pd.to_numeric, errors="coerce")
             .fillna(pd.Series(preprocessor["medians"]))
        )
        parts.append(X_num)
    for c in preprocessor["cat_cols"]:
        if c in X.columns:
            s = X[c].astype("string").fillna("__MISSING__")
        else:
            s = pd.Series(["__MISSING__"] * len(X), index=X.index, dtype="string")
        s_cat = pd.Series(
            pd.Categorical(s, categories=preprocessor["cat_schema"][c]),
            index=X.index, name=c,
        )
        parts.append(pd.get_dummies(s_cat, prefix=c, dtype=float))
    X_out = pd.concat(parts, axis=1) if parts else pd.DataFrame(index=X.index)
    X_out = X_out.reindex(columns=preprocessor["columns"], fill_value=0.0)
    return X_out.astype(float)
def _make_boosted_candidate(params: Dict[str, Any], y_train_reference: np.ndarray) -> Tuple[Any, str]:
    try:
        n_pos = int(np.sum(y_train_reference == 1))
        n_neg = int(np.sum(y_train_reference == 0))
        scale_pos_weight = n_neg / max(n_pos, 1)
        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=SEED,
            n_jobs=-1,
            tree_method="hist",
            n_estimators=int(params["n_estimators"]),
            max_depth=int(params["max_depth"]),
            learning_rate=float(params["learning_rate"]),
            subsample=float(params["subsample"]),
            colsample_bytree=float(params["colsample_bytree"]),
            min_child_weight=float(params["min_child_weight"]),
            reg_lambda=float(params["reg_lambda"]),
            scale_pos_weight=float(scale_pos_weight),
        )
        return model, "XGBoost"
    except Exception:
        sig = inspect.signature(HistGradientBoostingClassifier)
        kwargs: Dict[str, Any] = {
            "random_state": SEED,
            "learning_rate": float(params["learning_rate"]),
            "max_depth": int(params["max_depth"]),
            "max_iter": int(params["n_estimators"]),
            "min_samples_leaf": int(params["min_samples_leaf"]),
            "l2_regularization": float(params["l2_regularization"]),
        }
        if "class_weight" in sig.parameters:
            kwargs["class_weight"] = "balanced"
        return HistGradientBoostingClassifier(**kwargs), "HistGradientBoosting"
def _fit_with_optional_weights(model: Any, X: pd.DataFrame, y: np.ndarray) -> Any:
    """Older HGB versions lack class_weight; fall back to sample_weight in that case."""
    if isinstance(model, HistGradientBoostingClassifier):
        if "class_weight" in model.get_params():
            return model.fit(X, y)
        classes = np.unique(y)
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
        weight_map = dict(zip(classes, weights))
        sample_weight = np.array([weight_map[v] for v in y], dtype=float)
        return model.fit(X, y, sample_weight=sample_weight)
    return model.fit(X, y)
def _run_cv_search(
    X: pd.DataFrame,
    y: np.ndarray,
    param_iter,
    fit_one_fold: Callable[[Any, np.ndarray, np.ndarray], np.ndarray],
    record_extra_columns: Callable[[Dict[str, Any]], Dict[str, Any]] = lambda p: {},
    selection_metric: str = "accuracy",
    n_splits: int = 5,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Generic CV-search loop with OOF threshold tuning. Used by both searches."""
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    rows = []
    for params in param_iter:
        oof_prob = np.zeros(len(y), dtype=float)
        fold_acc, fold_f1, fold_auc = [], [], []
        for tr_idx, val_idx in cv.split(X, y):
            prob = fit_one_fold(params, tr_idx, val_idx)
            pred = (prob >= 0.5).astype(int)
            oof_prob[val_idx] = prob
            fold_acc.append(accuracy_score(y[val_idx], pred))
            fold_f1.append(f1_score(y[val_idx], pred))
            fold_auc.append(roc_auc_score(y[val_idx], prob))
        tuned_threshold, tuned_score = _best_threshold(y, oof_prob, metric_name=selection_metric)
        tuned_pred = (oof_prob >= tuned_threshold).astype(int)
        rows.append({
            **record_extra_columns(params),
            **params,
            "mean_cv_accuracy_at_0.5": float(np.mean(fold_acc)),
            "mean_cv_f1_at_0.5": float(np.mean(fold_f1)),
            "mean_cv_auc": float(np.mean(fold_auc)),
            "oof_best_threshold": float(tuned_threshold),
            f"oof_{selection_metric}": float(tuned_score),
            "oof_f1_tuned": float(f1_score(y, tuned_pred)),
        })
    results = pd.DataFrame(rows).sort_values(
        [f"oof_{selection_metric}", "mean_cv_auc", "oof_f1_tuned"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    return results, results.iloc[0].to_dict()
def _cv_search_logreg(X: pd.DataFrame, y: np.ndarray, selection_metric: str = "accuracy"):
    grid = list(ParameterGrid({
        "C": [0.05, 0.10, 0.25, 1.0, 4.0, 8.0, 16.0],
        "l1_ratio": [0.0, 0.05, 0.15, 0.35, 0.50],
        "class_weight": [None, "balanced"],
    }))
    def fit_one_fold(params, tr_idx, val_idx):
        model = _make_logreg_candidate(
            C=params["C"], l1_ratio=params["l1_ratio"], class_weight=params["class_weight"],
        )
        model.fit(X.iloc[tr_idx], y[tr_idx])
        return _predict_positive_proba(model, X.iloc[val_idx])
    return _run_cv_search(
        X, y,
        param_iter=grid,
        fit_one_fold=fit_one_fold,
        selection_metric=selection_metric,
        n_splits=LOGREG_CV_FOLDS,
    )
def _cv_search_boosted(
    X: pd.DataFrame,
    y: np.ndarray,
    selection_metric: str = "accuracy",
    n_iter: int = BOOSTED_RANDOM_SEARCH_N_ITER,
):
    space = {
        "n_estimators": [150, 250, 400, 600],
        "max_depth": [2, 3, 4, 6],
        "learning_rate": [0.02, 0.03, 0.05, 0.10],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
        "min_child_weight": [1, 2, 4],
        "reg_lambda": [1.0, 3.0, 5.0],
        "min_samples_leaf": [5, 10, 20],
        "l2_regularization": [0.0, 0.01, 0.1],
    }
    sampled = list(ParameterSampler(space, n_iter=n_iter, random_state=SEED))
    last_model_name = {"value": "XGBoost"}
    def fit_one_fold(params, tr_idx, val_idx):
        X_tr_proc, fold_pre = _fit_boost_preprocessor(X.iloc[tr_idx].copy())
        X_val_proc = _transform_boost_features(X.iloc[val_idx].copy(), fold_pre)
        model, model_name = _make_boosted_candidate(params, y[tr_idx])
        last_model_name["value"] = model_name
        model = _fit_with_optional_weights(model, X_tr_proc, y[tr_idx])
        return _predict_positive_proba(model, X_val_proc)
    def record_extra_columns(_params):
        return {"model_family": last_model_name["value"]}
    return _run_cv_search(
        X, y,
        param_iter=sampled,
        fit_one_fold=fit_one_fold,
        record_extra_columns=record_extra_columns,
        selection_metric=selection_metric,
        n_splits=BOOSTED_CV_FOLDS,
    )
use_cached_task_a = (
    CONFIG.get("task_a_load_from_artifacts", False)
    and task_a_cache_is_compatible()
    and not CONFIG.get("force_retrain", False)
)
if use_cached_task_a:
    print("Loading compatible cached Task a artifacts.")
    task_a_metrics = load_json(task_a_metrics_path)
    task_a_preds = load_joblib(task_a_preds_path)
    task_a_models = load_joblib(task_a_models_path) if artifact_exists(task_a_models_path) else {}
    task_a_thresholds = load_json(task_a_thresholds_path) if artifact_exists(task_a_thresholds_path) else {}
    task_a_logreg_cv_results = (
        pd.read_csv(task_a_logreg_cv_path) if artifact_exists(task_a_logreg_cv_path) else pd.DataFrame()
    )
    task_a_boosted_cv_results = (
        pd.read_csv(task_a_boosted_cv_path) if artifact_exists(task_a_boosted_cv_path) else pd.DataFrame()
    )
else:
    task_a_logreg_cv_results, best_logreg = _cv_search_logreg(
        X_train_a, y_train_a, selection_metric=SELECTION_METRIC,
    )
    save_df(task_a_logreg_cv_results, task_a_logreg_cv_path)
    task_a_logreg = _make_logreg_candidate(
        C=best_logreg["C"], l1_ratio=best_logreg["l1_ratio"],
        class_weight=best_logreg["class_weight"],
    )
    task_a_logreg.fit(X_train_a, y_train_a)
    prob_logreg = _predict_positive_proba(task_a_logreg, X_test_a)
    thr_logreg = float(best_logreg["oof_best_threshold"])
    pred_logreg = (prob_logreg >= thr_logreg).astype(int)
    task_a_boosted_cv_results, best_boosted = _cv_search_boosted(
        X_train_a, y_train_a, selection_metric=SELECTION_METRIC,
    )
    save_df(task_a_boosted_cv_results, task_a_boosted_cv_path)
    X_train_a_boosted, task_a_boosted_preprocessor = _fit_boost_preprocessor(X_train_a)
    X_test_a_boosted = _transform_boost_features(X_test_a, task_a_boosted_preprocessor)
    task_a_boosted, task_a_boosted_name = _make_boosted_candidate(best_boosted, y_train_a)
    task_a_boosted = _fit_with_optional_weights(task_a_boosted, X_train_a_boosted, y_train_a)
    prob_boosted = _predict_positive_proba(task_a_boosted, X_test_a_boosted)
    thr_boosted = float(best_boosted["oof_best_threshold"])
    pred_boosted = (prob_boosted >= thr_boosted).astype(int)
    task_a_metrics = {
        "Elastic-net logistic regression": compute_binary_metrics(y_test_a, pred_logreg, prob_logreg),
        task_a_boosted_name: compute_binary_metrics(y_test_a, pred_boosted, prob_boosted),
    }
    task_a_preds = {
        "Elastic-net logistic regression": {"pred": pred_logreg, "prob": prob_logreg},
        task_a_boosted_name: {"pred": pred_boosted, "prob": prob_boosted},
    }
    task_a_models = {
        "Elastic-net logistic regression": task_a_logreg,
        task_a_boosted_name: {"model": task_a_boosted, "preprocessor": task_a_boosted_preprocessor},
    }
    task_a_thresholds = {
        "selection_metric": SELECTION_METRIC,
        "Elastic-net logistic regression": {"threshold": thr_logreg, "best_cv_row": best_logreg},
        task_a_boosted_name: {"threshold": thr_boosted, "best_cv_row": best_boosted},
    }
    save_json(task_a_metrics, task_a_metrics_path)
    save_joblib(task_a_preds, task_a_preds_path)
    save_joblib(task_a_models, task_a_models_path)
    save_json(task_a_thresholds, task_a_thresholds_path)
    save_json(current_task_a_meta, task_a_meta_path)
task_a_metrics_df = pd.DataFrame(task_a_metrics).T.round(4)
task_a_metrics_display_df = task_a_metrics_df.reset_index().rename(columns={"index": "model"})
display_report_table(
    task_a_metrics_display_df,
    row_color_col="model",
    row_colors=REPORT_MODEL_COLORS,
    text_cols=["model"],
    gradient_cols=["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"],
    gradient_kwargs={"cmap": "YlGn"},
    highlight_min_cols=["brier"],
)
if "task_a_logreg_cv_results" in globals() and len(task_a_logreg_cv_results) > 0:
    display(Markdown("### Task a logistic CV results (top 10)"))
    display_report_table(
        task_a_logreg_cv_results.head(10).round(4),
        gradient_cols=["mean_accuracy", "mean_balanced_accuracy", "mean_f1", "oof_accuracy", "oof_balanced_accuracy", "oof_f1"],
        gradient_kwargs={"cmap": "YlGn"},
        highlight_max_cols=["mean_accuracy", "oof_accuracy", "mean_f1", "oof_f1"],
    )
if "task_a_boosted_cv_results" in globals() and len(task_a_boosted_cv_results) > 0:
    display(Markdown("### Task a boosted-model CV results (top 10)"))
    display_report_table(
        task_a_boosted_cv_results.head(10).round(4),
        gradient_cols=["mean_accuracy", "mean_balanced_accuracy", "mean_f1", "oof_accuracy", "oof_balanced_accuracy", "oof_f1"],
        gradient_kwargs={"cmap": "YlGn"},
        highlight_max_cols=["mean_accuracy", "oof_accuracy", "mean_f1", "oof_f1"],
    )
if "task_a_thresholds" in globals():
    display(Markdown("### Task a tuned thresholds"))
    task_a_thresholds_display_df = pd.DataFrame({
        name: {"threshold": info["threshold"]}
        for name, info in task_a_thresholds.items()
        if isinstance(info, dict) and "threshold" in info
    }).T.round(4).reset_index().rename(columns={"index": "model"})
    display_report_table(
        task_a_thresholds_display_df,
        row_color_col="model",
        row_colors=REPORT_MODEL_COLORS,
        gradient_cols=["threshold"],
        gradient_kwargs={"cmap": "YlGnBu"},
    )


In [ ]:
for _task_a_fig_idx, (model_name, model_pred) in enumerate(task_a_preds.items()):
    pred = model_pred["pred"]
    prob = model_pred["prob"]
    display(Markdown(f"### Figure {33 + 2 * _task_a_fig_idx}. Task A normalized confusion matrix: {model_name}"))
    plot_confusion(
        y_test_a,
        pred,
        labels=list(mi_label_encoder.classes_),
        title=f"Task a – {model_name} (normalized confusion matrix)",
        save_path=FIG_DIR / f"task_a_cm_{model_name.replace(' ', '_').replace('/', '_')}.png",
    )
    fpr, tpr, _ = roc_curve(y_test_a, prob)
    precision, recall, _ = precision_recall_curve(y_test_a, prob)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(fpr, tpr)
    axes[0].plot([0, 1], [0, 1], linestyle="--")
    axes[0].set_title(f"Task a ROC – {model_name}")
    axes[0].set_xlabel("FPR")
    axes[0].set_ylabel("TPR")
    axes[1].plot(recall, precision)
    axes[1].set_title(f"Task a PR curve – {model_name}")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"task_a_curves_{model_name.replace(' ', '_').replace('/', '_')}.png", dpi=300, bbox_inches="tight")
    display(Markdown(f"### Figure {34 + 2 * _task_a_fig_idx}. Task A ROC and precision-recall curves: {model_name}"))
    plt.show()


In [ ]:
def _bootstrap_binary_metric_cis(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: np.ndarray,
    n_boot: int = 5000,
    seed: int = 42,
) -> Dict[str, Tuple[float, float, float]]:
    """Returns {metric: (ci_low, median, ci_high)} for accuracy, F1, ROC AUC, AP."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    acc, f1m, auc, ap = [], [], [], []
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        acc.append(accuracy_score(y_true[idx], y_pred[idx]))
        f1m.append(f1_score(y_true[idx], y_pred[idx], zero_division=0))
        try:
            auc.append(roc_auc_score(y_true[idx], y_prob[idx]))
            ap.append(average_precision_score(y_true[idx], y_prob[idx]))
        except Exception:
            pass
    def _ci(values):
        if len(values) == 0:
            return (float("nan"), float("nan"), float("nan"))
        lo, med, hi = np.percentile(values, [2.5, 50, 97.5])
        return (float(lo), float(med), float(hi))
    return {
        "accuracy": _ci(acc),
        "f1": _ci(f1m),
        "roc_auc": _ci(auc),
        "average_precision": _ci(ap),
    }
task_a_ci_rows = []
for model_name, model_pred in task_a_preds.items():
    cis = _bootstrap_binary_metric_cis(
        y_true=np.asarray(y_test_a),
        y_pred=np.asarray(model_pred["pred"]),
        y_prob=np.asarray(model_pred["prob"]),
        n_boot=5000,
        seed=SEED,
    )
    point = task_a_metrics[model_name]
    row = {"model": model_name}
    for metric in ["accuracy", "f1", "roc_auc", "average_precision"]:
        ci_low, ci_med, ci_high = cis[metric]
        row[f"{metric}_point"] = float(point.get(metric, float("nan")))
        row[f"{metric}_ci95_low"] = ci_low
        row[f"{metric}_ci95_high"] = ci_high
    task_a_ci_rows.append(row)
task_a_ci_df = pd.DataFrame(task_a_ci_rows).round(4)
display(Markdown(
    "### Appendix A: test metrics with non-parametric bootstrap 95% CIs\n\n"
    f"Bootstrap is over the {len(y_test_a)} test transcripts "
    f"(5,000 resamples, seed={SEED}). One transcript misclassification "
    f"corresponds to roughly {100/len(y_test_a):.1f} pp of accuracy, so the "
    "CIs on this small test set are necessarily wide; they should be read as "
    "the realistic uncertainty around each point estimate."
))
display_report_table(
    task_a_ci_df,
    row_color_col="model",
    row_colors=REPORT_MODEL_COLORS,
    text_cols=["model"],
    gradient_cols=["accuracy_point", "f1_point", "roc_auc_point", "average_precision_point"],
    gradient_kwargs={"cmap": "YlGn"},
)
save_df(task_a_ci_df, task_a_dir / "task_a_test_metric_cis.csv")


In [ ]:
task_c_dir = ARTIFACT_DIR / "task_c"
task_c_dir.mkdir(parents=True, exist_ok=True)
task_c_history_max = max(int(CONFIG.get("task_c_history_k", 5)), 10)
def _safe_text(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).strip()
def _last_or_default(seq, default="none"):
    return seq[-1] if len(seq) > 0 else default
def _second_last_or_default(seq, default="none"):
    return seq[-2] if len(seq) > 1 else default
def _run_length_from_end(seq, value) -> int:
    c = 0
    for item in reversed(seq):
        if item == value:
            c += 1
        else:
            break
    return c
def _window_count(seq, label, k: int) -> int:
    if len(seq) == 0:
        return 0
    return int(seq[-k:].count(label))
def build_task_c_examples(df: pd.DataFrame, history_k: int = 10) -> pd.DataFrame:
    rows = []
    sort_cols = ["transcript_id", "utterance_id"]
    for transcript_id, g in df.sort_values(sort_cols).groupby("transcript_id", sort=False):
        g = g.reset_index(drop=True).copy()
        topic = g.loc[0, "topic_norm"] if "topic_norm" in g.columns else "unknown"
        n_turns = len(g)
        past_tagged_turns: List[str] = []
        past_behaviours: List[str] = []
        past_client_talks: List[str] = []
        past_ther_texts: List[str] = []
        past_client_texts: List[str] = []
        has_turn_ratio = "turn_ratio" in g.columns
        iter_cols = ["utterance_id", "utterance_text", "interlocutor", "main_therapist_behaviour", "client_talk_type"]
        if has_turn_ratio:
            iter_cols.append("turn_ratio")
        for local_idx, row in enumerate(g[iter_cols].itertuples(index=False)):
            utterance_id = row.utterance_id
            interlocutor = row.interlocutor
            therapist_behaviour = row.main_therapist_behaviour
            client_talk_type = row.client_talk_type
            current_text = _safe_text(row.utterance_text)
            role = _safe_text(interlocutor).upper()
            tagged_turn = f"[{role}] {current_text}".strip()
            if interlocutor == "therapist" and therapist_behaviour != "n/a":
                history_turns = past_tagged_turns[-history_k:].copy()
                hist_client = past_client_talks[-history_k:].copy()
                hist_ther = past_behaviours[-history_k:].copy()
                last_client_text = _last_or_default(past_client_texts, default="")
                prev_ther_text = _last_or_default(past_ther_texts, default="")
                last_client_talk = _last_or_default(hist_client, default="none")
                prev_client_talk_2 = _second_last_or_default(hist_client, default="none")
                last_ther_behaviour = _last_or_default(hist_ther, default="none")
                prev_ther_behaviour_2 = _second_last_or_default(hist_ther, default="none")
                last_exchange_text = " ".join(
                    [
                        f"[PREV_THERAPIST] {prev_ther_text}".strip() if prev_ther_text else "",
                        f"[LAST_CLIENT] {last_client_text}".strip() if last_client_text else "",
                    ]
                ).strip()
                if not last_exchange_text:
                    last_exchange_text = " ".join(history_turns[-2:]).strip()
                row_dict = {
                    "transcript_id": transcript_id,
                    "utterance_id": int(utterance_id),
                    "topic_norm": topic,
                    "turn_index": int(local_idx),
                    "n_turns_transcript": int(n_turns),
                    "turn_ratio": float(row.turn_ratio) if has_turn_ratio else float(local_idx / max(n_turns - 1, 1)),
                    "history_turns_list": history_turns,
                    "history_only_text": " ".join(history_turns).strip(),
                    "last_client_text": last_client_text,
                    "last_exchange_text": last_exchange_text,
                    "history_len_turns": int(len(history_turns)),
                    "history_char_len": int(sum(len(t) for t in history_turns)),
                    "history_token_len": int(sum(len(str(t).split()) for t in history_turns)),
                    "last_client_text_len": int(len(last_client_text.split())) if last_client_text else 0,
                    "prev_ther_text_len": int(len(prev_ther_text.split())) if prev_ther_text else 0,
                    "last_client_has_qmark": int("?" in last_client_text),
                    "last_client_has_exclaim": int("!" in last_client_text),
                    "last_client_talk": last_client_talk,
                    "prev_client_talk_2": prev_client_talk_2,
                    "last_ther_behaviour": last_ther_behaviour,
                    "prev_ther_behaviour_2": prev_ther_behaviour_2,
                    "streak_last_client_talk": int(_run_length_from_end(hist_client, last_client_talk)) if last_client_talk != "none" else 0,
                    "streak_last_ther_behaviour": int(_run_length_from_end(hist_ther, last_ther_behaviour)) if last_ther_behaviour != "none" else 0,
                    "target_next_therapist_behaviour": therapist_behaviour,
                }
                for label in ["change", "neutral", "sustain"]:
                    row_dict[f"hist_client_{label}"] = int(hist_client.count(label))
                for label, out_name in [
                    ("reflection", "reflection"),
                    ("question", "question"),
                    ("therapist_input", "input"),
                    ("other", "other"),
                ]:
                    row_dict[f"hist_ther_{out_name}"] = int(hist_ther.count(label))
                for k in [3, 5]:
                    for label in ["change", "neutral", "sustain"]:
                        row_dict[f"hist{k}_client_{label}"] = _window_count(hist_client, label, k)
                    for label, out_name in [
                        ("reflection", "reflection"),
                        ("question", "question"),
                        ("therapist_input", "input"),
                        ("other", "other"),
                    ]:
                        row_dict[f"hist{k}_ther_{out_name}"] = _window_count(hist_ther, label, k)
                rows.append(row_dict)
            past_tagged_turns.append(tagged_turn)
            if interlocutor == "client":
                past_client_texts.append(current_text)
                if client_talk_type != "n/a":
                    past_client_talks.append(client_talk_type)
            elif interlocutor == "therapist":
                past_ther_texts.append(current_text)
                if therapist_behaviour != "n/a":
                    past_behaviours.append(therapist_behaviour)
    return pd.DataFrame(rows)
task_c_df = build_task_c_examples(utterance_df, history_k=task_c_history_max).copy()
task_c_df = task_c_df[task_c_df["target_next_therapist_behaviour"] != "n/a"].reset_index(drop=True)
task_c_label_encoder = LabelEncoder()
task_c_df["y_code"] = task_c_label_encoder.fit_transform(task_c_df["target_next_therapist_behaviour"])
task_c_train = task_c_df[task_c_df["transcript_id"].isin(TRAIN_TRANSCRIPTS)].copy().reset_index(drop=True)
task_c_test = task_c_df[task_c_df["transcript_id"].isin(TEST_TRANSCRIPTS)].copy().reset_index(drop=True)
task_c_history_text_col = "history_only_text"
task_c_last_client_text_col = "last_client_text"
task_c_exchange_text_col = "last_exchange_text"
task_c_numeric = [
    "turn_ratio",
    "turn_index",
    "n_turns_transcript",
    "history_len_turns",
    "history_char_len",
    "history_token_len",
    "last_client_text_len",
    "prev_ther_text_len",
    "last_client_has_qmark",
    "last_client_has_exclaim",
    "streak_last_client_talk",
    "streak_last_ther_behaviour",
    "hist_client_change",
    "hist_client_neutral",
    "hist_client_sustain",
    "hist_ther_reflection",
    "hist_ther_question",
    "hist_ther_input",
    "hist_ther_other",
    "hist3_client_change",
    "hist3_client_neutral",
    "hist3_client_sustain",
    "hist3_ther_reflection",
    "hist3_ther_question",
    "hist3_ther_input",
    "hist3_ther_other",
    "hist5_client_change",
    "hist5_client_neutral",
    "hist5_client_sustain",
    "hist5_ther_reflection",
    "hist5_ther_question",
    "hist5_ther_input",
    "hist5_ther_other",
]
task_c_categorical = [
    "topic_norm",
    "last_client_talk",
    "prev_client_talk_2",
    "last_ther_behaviour",
    "prev_ther_behaviour_2",
]
task_c_required_feature_cols = (
    [task_c_history_text_col, task_c_last_client_text_col, task_c_exchange_text_col]
    + task_c_numeric
    + task_c_categorical
)
def shorten_for_table(value: Any, max_chars: int = 120) -> str:
    text = _safe_text(value)
    return text if len(text) <= max_chars else text[: max_chars - 3].rstrip() + "..."
task_c_target_distribution = (
    task_c_df["target_next_therapist_behaviour"]
    .value_counts()
    .rename_axis("target_next_therapist_behaviour")
    .reset_index(name="n_examples")
)
task_c_target_distribution["share_of_dataset"] = (
    task_c_target_distribution["n_examples"] / len(task_c_df)
)
task_c_target_distribution = task_c_target_distribution.merge(
    task_c_train["target_next_therapist_behaviour"]
    .value_counts()
    .rename_axis("target_next_therapist_behaviour")
    .reset_index(name="train_examples"),
    on="target_next_therapist_behaviour",
    how="left",
).merge(
    task_c_test["target_next_therapist_behaviour"]
    .value_counts()
    .rename_axis("target_next_therapist_behaviour")
    .reset_index(name="test_examples"),
    on="target_next_therapist_behaviour",
    how="left",
)
task_c_target_distribution[["train_examples", "test_examples"]] = (
    task_c_target_distribution[["train_examples", "test_examples"]]
    .fillna(0)
    .astype(int)
)
task_c_target_distribution["class_role"] = np.where(
    task_c_target_distribution["n_examples"].eq(task_c_target_distribution["n_examples"].max()),
    "majority forecasting target",
    "minority forecasting target",
)
task_c_preview_cols = [
    "transcript_id",
    "utterance_id",
    "topic_norm",
    "turn_index",
    "turn_ratio",
    "history_len_turns",
    "last_client_talk",
    "last_ther_behaviour",
    "target_next_therapist_behaviour",
    "last_client_text",
    "last_exchange_text",
]
task_c_preview_df = task_c_df.loc[:, task_c_preview_cols].head(8).copy()
task_c_preview_df["last_client_text"] = task_c_preview_df["last_client_text"].replace("", "[no prior client turn]")
task_c_preview_df["last_exchange_text"] = task_c_preview_df["last_exchange_text"].replace("", "[start of transcript - no prior exchange]")
for text_col in ["topic_norm", "last_client_text", "last_exchange_text"]:
    task_c_preview_df[text_col] = task_c_preview_df[text_col].map(shorten_for_table)
_task_c_class_colors = {
    "reflection": REPORT_BEHAVIOUR_COLORS["reflection"],
    "question": REPORT_BEHAVIOUR_COLORS["question"],
    "therapist_input": REPORT_BEHAVIOUR_COLORS["therapist_input"],
    "other": REPORT_BEHAVIOUR_COLORS["other"],
}
display(Markdown("### Table B.1A. Task C forecasting dataset preview"))
display_report_table(
    task_c_preview_df,
    formats={"turn_ratio": "{:.3f}", "history_len_turns": "{:,}"},
    row_color_col="target_next_therapist_behaviour",
    row_colors=_task_c_class_colors,
    text_cols=[
        "topic_norm",
        "last_client_talk",
        "last_ther_behaviour",
        "target_next_therapist_behaviour",
        "last_client_text",
        "last_exchange_text",
    ],
    gradient_cols=["turn_ratio", "history_len_turns"],
    gradient_kwargs={"cmap": "YlGnBu"},
)
display(Markdown("### Table B.1B. Task C target-class distribution"))
display_report_table(
    task_c_target_distribution,
    formats={
        "n_examples": "{:,}",
        "share_of_dataset": "{:.1%}",
        "train_examples": "{:,}",
        "test_examples": "{:,}",
    },
    row_color_col="target_next_therapist_behaviour",
    row_colors=_task_c_class_colors,
    text_cols=["target_next_therapist_behaviour", "class_role"],
    gradient_cols=["n_examples", "share_of_dataset", "train_examples", "test_examples"],
    gradient_kwargs={"cmap": "YlGnBu"},
    highlight_max_cols=["n_examples", "share_of_dataset"],
)
save_df(task_c_preview_df, task_c_dir / "task_c_forecasting_dataset_preview.csv")
save_df(task_c_target_distribution, task_c_dir / "task_c_target_class_distribution.csv")


In [ ]:
assert "task_c_train" in globals(), "task_c_train must be in memory."
assert "task_c_label_encoder" in globals(), "task_c_label_encoder must be in memory."
_marg = task_c_train["target_next_therapist_behaviour"].value_counts(normalize=True)
task_c_majority_label = str(_marg.idxmax())
task_c_majority_acc_train = float(_marg.max())
_m1 = task_c_train[
    ["last_ther_behaviour", "target_next_therapist_behaviour"]
].copy()
_m1_table = pd.crosstab(
    _m1["last_ther_behaviour"],
    _m1["target_next_therapist_behaviour"],
    normalize="index",
)
_m1_pred_map = _m1_table.idxmax(axis=1).to_dict()
_m1_pred = _m1["last_ther_behaviour"].map(_m1_pred_map).fillna(task_c_majority_label)
task_c_markov1_acc_train = float((_m1_pred == _m1["target_next_therapist_behaviour"]).mean())
task_c_markov1_f1_train = float(f1_score(
    _m1["target_next_therapist_behaviour"],
    _m1_pred,
    average="macro",
    zero_division=0,
))
_m2 = task_c_train[
    ["last_ther_behaviour", "last_client_talk", "target_next_therapist_behaviour"]
].copy()
_m2["state"] = list(zip(_m2["last_ther_behaviour"], _m2["last_client_talk"]))
_m2_table = (
    _m2.groupby(["state", "target_next_therapist_behaviour"]).size()
       .unstack(fill_value=0)
)
_m2_table_norm = _m2_table.div(_m2_table.sum(axis=1), axis=0)
_m2_pred_map = _m2_table_norm.idxmax(axis=1).to_dict()
_m2_pred = _m2["state"].map(_m2_pred_map).fillna(task_c_majority_label)
task_c_markov2_acc_train = float((_m2_pred == _m2["target_next_therapist_behaviour"]).mean())
task_c_markov2_f1_train = float(f1_score(
    _m2["target_next_therapist_behaviour"],
    _m2_pred,
    average="macro",
    zero_division=0,
))
_p = _marg.to_numpy()
task_c_label_entropy_bits = float(-np.sum(_p * np.log2(_p + 1e-12)))
task_c_ceilings_df = pd.DataFrame([
    {
        "predictor": "Marginal majority class",
        "description": f"Always predict '{task_c_majority_label}'",
        "accuracy_train": task_c_majority_acc_train,
        "macro_f1_train": float(f1_score(
            task_c_train["target_next_therapist_behaviour"],
            np.full(len(task_c_train), task_c_majority_label, dtype=object),
            average="macro",
            zero_division=0,
        )),
    },
    {
        "predictor": "Markov-1 oracle",
        "description": "argmax P(next | last_ther_behaviour) — ground-truth prev label",
        "accuracy_train": task_c_markov1_acc_train,
        "macro_f1_train": task_c_markov1_f1_train,
    },
    {
        "predictor": "Markov-2 oracle",
        "description": "argmax P(next | last_ther_behaviour, last_client_talk)",
        "accuracy_train": task_c_markov2_acc_train,
        "macro_f1_train": task_c_markov2_f1_train,
    },
])
display(Markdown(
    "### Task c theoretical ceilings on the training set\n\n"
    f"Marginal label entropy = **{task_c_label_entropy_bits:.3f} bits** "
    "(uniform 4-class entropy is 2.0). The label sequence is moderately "
    "informative but has weak autocorrelation: even given oracle previous "
    "labels, the conditional distribution P(next | prev) does not collapse "
    "to a single mode. Both oracle Markov ceilings sit barely above 41% "
    "accuracy, which means a model with no better information than 'what "
    "did the therapist last do' cannot exceed this without learning "
    "richer signals from the utterance text itself.\n\n"
    "Wu et al. (2022, Interspeech) report a best-published macro-F1 of "
    "**0.40** for this exact task on this exact dataset using a fine-tuned "
    "RoBERTa-base — i.e. essentially at this ceiling. The Task c results "
    "below should be read against this band, not against arbitrary "
    "absolute thresholds."
))
task_c_ceilings_display_df = (
    task_c_ceilings_df.round(4)
    .assign(
        description=lambda d: d["description"].replace(
            {
                "argmax P(next | last_ther_behaviour) — ground-truth prev label": "Previous therapist label",
                "argmax P(next | last_ther_behaviour, last_client_talk)": "Previous therapist label + last client talk",
            }
        )
    )
    .rename(
        columns={
            "predictor": "Predictor",
            "description": "Oracle Inputs",
            "accuracy_train": "Train Accuracy",
            "macro_f1_train": "Train Macro-F1",
        }
    )
)
display_report_table(
    task_c_ceilings_display_df,
    formats={"Train Accuracy": "{:.4f}", "Train Macro-F1": "{:.4f}"},
    row_color_col="Predictor",
    row_colors={
        "Marginal majority class": "#f8fafc",
        "Markov-1 oracle": "#fff7ed",
        "Markov-2 oracle": "#e7f0ff",
    },
    text_cols=["Predictor", "Oracle Inputs"],
    highlight_max_cols=["Train Accuracy", "Train Macro-F1"],
)
save_df(task_c_ceilings_df, task_c_dir / "task_c_theoretical_ceilings.csv")
save_json(
    {
        "majority_label": task_c_majority_label,
        "majority_accuracy_train": task_c_majority_acc_train,
        "markov1_accuracy_train": task_c_markov1_acc_train,
        "markov1_macro_f1_train": task_c_markov1_f1_train,
        "markov2_accuracy_train": task_c_markov2_acc_train,
        "markov2_macro_f1_train": task_c_markov2_f1_train,
        "label_entropy_bits": task_c_label_entropy_bits,
        "published_reference": (
            "Wu et al. (2022), 'Towards Automated Counselling Decision-Making: "
            "Remarks on Therapist Action Forecasting on the AnnoMI Dataset', "
            "Interspeech 2022, pp. 1906–1910. Best macro-F1 reported = 0.40 "
            "with roberta-base, with the authors recommending top-K "
            "reformulation in the conclusion."
        ),
    },
    task_c_dir / "task_c_theoretical_ceilings.json",
)


In [ ]:
task_c_baseline_cv_path = task_c_dir / "task_c_baseline_cv_results.csv"
task_c_metrics_path = task_c_dir / "task_c_baseline_metrics.json"
task_c_preds_path = task_c_dir / "task_c_baseline_preds.joblib"
task_c_model_path = task_c_dir / "task_c_baseline_model.joblib"
task_c_name_path = task_c_dir / "task_c_baseline_name.json"
task_c_transition_tables_path = task_c_dir / "task_c_transition_tables.joblib"
task_c_baseline_meta_path = task_c_dir / "task_c_baseline_meta.json"
TASK_C_BASELINE_VERSION = "transition_first_v4_catboost_xgboost_compete"
def _task_c_safe_json_save(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=str)
def _task_c_safe_json_load(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)
def _task_c_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)
def _task_c_cache_is_compatible() -> bool:
    needed = [
        task_c_baseline_cv_path,
        task_c_metrics_path,
        task_c_preds_path,
        task_c_model_path,
        task_c_name_path,
        task_c_baseline_meta_path,
    ]
    if not all(Path(p).exists() for p in needed):
        return False
    try:
        meta = _task_c_safe_json_load(task_c_baseline_meta_path)
        if meta.get("version") != TASK_C_BASELINE_VERSION:
            return False
        if meta.get("n_test") != int(len(task_c_test)):
            return False
        if meta.get("test_transcript_ids") != sorted(task_c_test["transcript_id"].astype(str).tolist()):
            return False
        cached_preds = load_joblib(task_c_preds_path)
        if len(cached_preds["pred"]) != len(task_c_test):
            return False
        return True
    except Exception:
        return False
def _task_c_macro_metrics(y_true, pred, prob=None):
    out = {
        "accuracy": float(accuracy_score(y_true, pred)),
        "f1_macro": float(f1_score(y_true, pred, average="macro")),
    }
    if prob is not None and "compute_multiclass_metrics" in globals():
        try:
            more = compute_multiclass_metrics(y_true, pred, prob)
            out.update({k: float(v) for k, v in more.items()})
        except Exception:
            pass
    return out
def _task_c_backend_available(name: str) -> bool:
    if name == "catboost":
        return bool(globals().get("HAVE_CATBOOST", False) and globals().get("CatBoostClassifier") is not None)
    if name == "xgboost":
        return bool(globals().get("HAVE_XGBOOST", False) and globals().get("XGBClassifier") is not None)
    return False
def fit_task_c_transition_tables(train_df: pd.DataFrame, alpha: float = 1.0) -> Dict[str, Any]:
    classes = list(task_c_label_encoder.classes_)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    n_classes = len(classes)
    global_counts = np.zeros(n_classes, dtype=float)
    markov1_counts: Dict[str, np.ndarray] = {}
    markov2_counts: Dict[Tuple[str, str], np.ndarray] = {}
    transition_cols = ["target_next_therapist_behaviour", "last_ther_behaviour", "last_client_talk"]
    for row in train_df[transition_cols].itertuples(index=False):
        target = row.target_next_therapist_behaviour
        y_idx = class_to_idx[target]
        global_counts[y_idx] += 1.0
        key1 = str(row.last_ther_behaviour)
        if key1 not in markov1_counts:
            markov1_counts[key1] = np.zeros(n_classes, dtype=float)
        markov1_counts[key1][y_idx] += 1.0
        key2 = (str(row.last_client_talk), str(row.last_ther_behaviour))
        if key2 not in markov2_counts:
            markov2_counts[key2] = np.zeros(n_classes, dtype=float)
        markov2_counts[key2][y_idx] += 1.0
    def smooth(counts: np.ndarray) -> np.ndarray:
        counts = counts.astype(float)
        return (counts + alpha) / (counts.sum() + alpha * n_classes)
    global_prob = smooth(global_counts)
    markov1_prob = {k: smooth(v) for k, v in markov1_counts.items()}
    markov2_prob = {k: smooth(v) for k, v in markov2_counts.items()}
    return {
        "classes": classes,
        "class_to_idx": class_to_idx,
        "alpha": float(alpha),
        "global_prob": global_prob,
        "markov1_prob": markov1_prob,
        "markov2_prob": markov2_prob,
    }
def predict_task_c_markov_probs(df: pd.DataFrame, tables: Dict[str, Any], order: int = 2) -> np.ndarray:
    n = len(df)
    n_classes = len(tables["classes"])
    probs = np.zeros((n, n_classes), dtype=float)
    transition_cols = ["last_client_talk", "last_ther_behaviour"]
    for i, row in enumerate(df[transition_cols].itertuples(index=False)):
        key2 = (str(row.last_client_talk), str(row.last_ther_behaviour))
        key1 = str(row.last_ther_behaviour)
        if order >= 2 and key2 in tables["markov2_prob"]:
            p = tables["markov2_prob"][key2]
        elif key1 in tables["markov1_prob"]:
            p = tables["markov1_prob"][key1]
        else:
            p = tables["global_prob"]
        probs[i] = p
    return probs
def augment_task_c_with_transition_priors(reference_train_df: pd.DataFrame, target_df: pd.DataFrame) -> pd.DataFrame:
    tables = fit_task_c_transition_tables(reference_train_df, alpha=1.0)
    probs = predict_task_c_markov_probs(target_df, tables, order=2)
    out = target_df.copy()
    for j, cls_name in enumerate(task_c_label_encoder.classes_):
        safe_name = str(cls_name).replace("therapist_", "").replace(" ", "_")
        out[f"prior_prob_{safe_name}"] = probs[:, j]
        out[f"prior_logprob_{safe_name}"] = np.log(np.clip(probs[:, j], 1e-8, 1.0))
    return out
task_c_prior_prob_cols = [f"prior_prob_{str(c).replace('therapist_', '').replace(' ', '_')}" for c in task_c_label_encoder.classes_]
task_c_prior_logprob_cols = [f"prior_logprob_{str(c).replace('therapist_', '').replace(' ', '_')}" for c in task_c_label_encoder.classes_]
task_c_prior_cols = task_c_prior_prob_cols + task_c_prior_logprob_cols
def _task_c_make_logreg(C: float, l1_ratio: float, class_weight=None):
    kwargs = {
        "penalty": "elasticnet",
        "solver": "saga",
        "C": float(C),
        "l1_ratio": float(l1_ratio),
        "class_weight": class_weight,
        "max_iter": 5000,
        "random_state": SEED,
    }
    if "multi_class" in inspect.signature(LogisticRegression).parameters:
        kwargs["multi_class"] = "auto"
    return LogisticRegression(**kwargs)
def build_task_c_hybrid_logreg_pipeline(
    min_df: int = 2,
    C: float = 1.0,
    l1_ratio: float = 0.15,
    class_weight: Optional[str] = None,
) -> Pipeline:
    numeric_cols = task_c_numeric + task_c_prior_cols
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
            ("scaler", MaxAbsScaler()),
        ]
    )
    pre = ColumnTransformer(
        transformers=[
            (
                "last_client_word",
                TfidfVectorizer(
                    ngram_range=(1, 2),
                    min_df=min_df,
                    sublinear_tf=True,
                    max_features=4000,
                ),
                task_c_last_client_text_col,
            ),
            (
                "last_client_char",
                TfidfVectorizer(
                    analyzer="char_wb",
                    ngram_range=(3, 5),
                    min_df=min_df,
                    sublinear_tf=True,
                    max_features=2500,
                ),
                task_c_last_client_text_col,
            ),
            (
                "last_exchange_word",
                TfidfVectorizer(
                    ngram_range=(1, 2),
                    min_df=min_df,
                    sublinear_tf=True,
                    max_features=4000,
                ),
                task_c_exchange_text_col,
            ),
            ("num", numeric_transformer, numeric_cols),
            ("cat", _task_c_onehot_encoder(), task_c_categorical),
        ],
        sparse_threshold=0.3,
    )
    clf = _task_c_make_logreg(
        C=float(C),
        l1_ratio=float(l1_ratio),
        class_weight=class_weight,
    )
    return Pipeline([("pre", pre), ("clf", clf)])
def _prepare_task_c_structured_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df[task_c_numeric + task_c_prior_cols + task_c_categorical].copy()
    for c in task_c_categorical:
        out[c] = out[c].astype(str).fillna("none")
    return out
def build_task_c_catboost(params: Dict[str, Any]):
    model = CatBoostClassifier(
        loss_function="MultiClass",
        eval_metric="TotalF1:average=Macro",
        iterations=int(params.get("n_estimators", 400)),
        depth=int(params.get("max_depth", 6)),
        learning_rate=float(params.get("learning_rate", 0.05)),
        l2_leaf_reg=float(params.get("l2_leaf_reg", 3.0)),
        random_seed=SEED,
        verbose=False,
        auto_class_weights="Balanced",
        allow_writing_files=False,
    )
    return {
        "family_name": "CatBoost structured state model",
        "backend": "catboost",
        "model": model,
    }
def build_task_c_xgboost(params: Dict[str, Any]):
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler", StandardScaler()),
            ]), task_c_numeric + task_c_prior_cols),
            ("cat", _task_c_onehot_encoder(), task_c_categorical),
        ],
        sparse_threshold=0.3,
    )
    clf = XGBClassifier(
        n_estimators=int(params.get("n_estimators", 400)),
        max_depth=int(params.get("max_depth", 6)),
        learning_rate=float(params.get("learning_rate", 0.05)),
        subsample=float(params.get("subsample", 0.9)),
        colsample_bytree=float(params.get("colsample_bytree", 0.9)),
        reg_lambda=float(params.get("reg_lambda", 3.0)),
        objective="multi:softprob",
        num_class=len(task_c_label_encoder.classes_),
        eval_metric="mlogloss",
        random_state=SEED,
        tree_method="hist",
    )
    return {
        "family_name": "XGBoost structured state model",
        "backend": "xgboost",
        "model": Pipeline([("pre", pre), ("clf", clf)]),
    }
def build_task_c_histgb(params: Dict[str, Any]):
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
                ("scaler", StandardScaler()),
            ]), task_c_numeric + task_c_prior_cols),
            ("cat", _task_c_onehot_encoder(), task_c_categorical),
        ],
        sparse_threshold=0.0,
    )
    clf = HistGradientBoostingClassifier(
        learning_rate=float(params.get("learning_rate", 0.05)),
        max_depth=int(params.get("max_depth", 6)),
        max_iter=int(params.get("n_estimators", 400)),
        l2_regularization=float(params.get("l2_leaf_reg", 0.01)),
        random_state=SEED,
    )
    return {
        "family_name": "HistGradientBoosting structured state model",
        "backend": "histgb",
        "model": Pipeline([("pre", pre), ("clf", clf)]),
    }
def build_task_c_structured_model(params: Dict[str, Any]):
    family = params["family"]
    if family == "structured_catboost":
        return build_task_c_catboost(params)
    if family == "structured_xgboost":
        return build_task_c_xgboost(params)
    if family == "structured_histgb":
        return build_task_c_histgb(params)
    raise ValueError(f"Unknown structured Task c family: {family}")
def _fit_task_c_structured_model(model_bundle, X_train_df, y_train):
    model = model_bundle["model"]
    if model_bundle["backend"] == "catboost":
        X_fit = X_train_df.copy()
        for c in task_c_categorical:
            X_fit[c] = X_fit[c].astype(str)
        model.fit(X_fit, y_train, cat_features=task_c_categorical)
        return model
    if model_bundle["backend"] == "histgb":
        classes = np.unique(y_train)
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
        mapping = {cls: w for cls, w in zip(classes, weights)}
        sample_weight = np.array([mapping[v] for v in y_train], dtype=float)
        model.fit(X_train_df, y_train, clf__sample_weight=sample_weight)
        return model
    model.fit(X_train_df, y_train)
    return model
def _predict_task_c_structured_model(model_bundle, X_df):
    model = model_bundle["model"]
    if model_bundle["backend"] == "catboost":
        X_in = X_df.copy()
        for c in task_c_categorical:
            X_in[c] = X_in[c].astype(str)
        prob = model.predict_proba(X_in)
        pred = np.asarray(prob).argmax(axis=1)
        return pred, np.asarray(prob)
    pred = model.predict(X_df)
    if hasattr(model, "predict_proba"):
        prob = model.predict_proba(X_df)
    else:
        prob = None
    return pred, prob
def search_task_c_baselines(train_df: pd.DataFrame) -> pd.DataFrame:
    candidate_rows = []
    logreg_space = list(
        ParameterGrid(
            {
                "family": ["hybrid_logreg"],
                "min_df": [2, 5],
                "C": [0.5, 1.0, 4.0],
                "l1_ratio": [0.0, 0.15],
                "class_weight": [None, "balanced"],
            }
        )
    )
    markov_space = [
        {"family": "majority_dummy"},
        {"family": "markov1"},
        {"family": "markov2"},
    ]
    structured_spaces = []
    if _task_c_backend_available("catboost"):
        structured_spaces += list(
            ParameterSampler(
                {
                    "family": ["structured_catboost"],
                    "n_estimators": [250, 400, 600],
                    "max_depth": [4, 6, 8],
                    "learning_rate": [0.03, 0.05, 0.08],
                    "l2_leaf_reg": [3.0, 8.0, 12.0],
                },
                n_iter=8,
                random_state=SEED,
            )
        )
    if _task_c_backend_available("xgboost"):
        structured_spaces += list(
            ParameterSampler(
                {
                    "family": ["structured_xgboost"],
                    "n_estimators": [250, 400, 600],
                    "max_depth": [4, 6, 8],
                    "learning_rate": [0.03, 0.05, 0.08],
                    "subsample": [0.8, 0.9, 1.0],
                    "colsample_bytree": [0.8, 0.9, 1.0],
                    "reg_lambda": [1.0, 3.0, 5.0],
                },
                n_iter=8,
                random_state=SEED,
            )
        )
    if (not _task_c_backend_available("catboost")) and (not _task_c_backend_available("xgboost")):
        structured_spaces += list(
            ParameterSampler(
                {
                    "family": ["structured_histgb"],
                    "n_estimators": [250, 400, 600],
                    "max_depth": [4, 6, 8],
                    "learning_rate": [0.03, 0.05, 0.08],
                    "l2_leaf_reg": [0.0, 0.01, 0.1],
                },
                n_iter=6,
                random_state=SEED,
            )
        )
    search_space = markov_space + logreg_space + structured_spaces
    splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)
    y = train_df["y_code"].to_numpy()
    groups = train_df["transcript_id"].to_numpy()
    n_classes = len(task_c_label_encoder.classes_)
    for cfg in tqdm(search_space, desc="Task c baseline search"):
        fold_f1 = []
        fold_acc = []
        for tr_idx, val_idx in splitter.split(train_df, y, groups):
            tr_df = train_df.iloc[tr_idx].copy().reset_index(drop=True)
            val_df = train_df.iloc[val_idx].copy().reset_index(drop=True)
            tr_aug = augment_task_c_with_transition_priors(tr_df, tr_df)
            val_aug = augment_task_c_with_transition_priors(tr_df, val_df)
            y_val = val_aug["y_code"].to_numpy()
            if cfg["family"] == "majority_dummy":
                majority_class = int(tr_aug["y_code"].value_counts().idxmax())
                pred = np.full(len(val_aug), majority_class, dtype=int)
                prob = np.zeros((len(val_aug), n_classes), dtype=float)
                prob[:, majority_class] = 1.0
            elif cfg["family"] == "markov1":
                tables = fit_task_c_transition_tables(tr_df, alpha=1.0)
                prob = predict_task_c_markov_probs(val_aug, tables, order=1)
                pred = prob.argmax(axis=1)
            elif cfg["family"] == "markov2":
                tables = fit_task_c_transition_tables(tr_df, alpha=1.0)
                prob = predict_task_c_markov_probs(val_aug, tables, order=2)
                pred = prob.argmax(axis=1)
            elif cfg["family"] == "hybrid_logreg":
                model = build_task_c_hybrid_logreg_pipeline(
                    min_df=int(cfg["min_df"]),
                    C=float(cfg["C"]),
                    l1_ratio=float(cfg["l1_ratio"]),
                    class_weight=cfg["class_weight"],
                )
                X_tr = tr_aug[
                    [task_c_last_client_text_col, task_c_exchange_text_col]
                    + task_c_numeric + task_c_prior_cols + task_c_categorical
                ]
                X_val = val_aug[
                    [task_c_last_client_text_col, task_c_exchange_text_col]
                    + task_c_numeric + task_c_prior_cols + task_c_categorical
                ]
                model.fit(X_tr, tr_aug["y_code"].to_numpy())
                pred = model.predict(X_val)
                prob = model.predict_proba(X_val) if hasattr(model, "predict_proba") else None
            elif cfg["family"] in {"structured_catboost", "structured_xgboost", "structured_histgb"}:
                model_bundle = build_task_c_structured_model(cfg)
                X_tr = _prepare_task_c_structured_df(tr_aug)
                X_val = _prepare_task_c_structured_df(val_aug)
                _fit_task_c_structured_model(model_bundle, X_tr, tr_aug["y_code"].to_numpy())
                pred, prob = _predict_task_c_structured_model(model_bundle, X_val)
            else:
                raise ValueError(f"Unknown Task c baseline family: {cfg['family']}")
            fold_f1.append(f1_score(y_val, pred, average="macro"))
            fold_acc.append(accuracy_score(y_val, pred))
        row = dict(cfg)
        row["cv_f1_macro_mean"] = float(np.mean(fold_f1))
        row["cv_f1_macro_std"] = float(np.std(fold_f1))
        row["cv_accuracy_mean"] = float(np.mean(fold_acc))
        row["cv_accuracy_std"] = float(np.std(fold_acc))
        candidate_rows.append(row)
    return (
        pd.DataFrame(candidate_rows)
        .sort_values(
            ["cv_f1_macro_mean", "cv_accuracy_mean", "cv_f1_macro_std"],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )
if _task_c_cache_is_compatible() and not CONFIG.get("force_retrain", False):
    task_c_baseline_cv_results = pd.read_csv(task_c_baseline_cv_path)
    task_c_baseline_metrics = load_json(task_c_metrics_path)
    task_c_baseline_preds = load_joblib(task_c_preds_path)
    task_c_baseline_model = load_joblib(task_c_model_path)
    task_c_baseline_name = _task_c_safe_json_load(task_c_name_path)["task_c_baseline_name"]
    task_c_transition_tables = load_joblib(task_c_transition_tables_path) if Path(task_c_transition_tables_path).exists() else None
else:
    task_c_baseline_cv_results = search_task_c_baselines(task_c_train)
    save_df(task_c_baseline_cv_results, task_c_baseline_cv_path)
    best_row = task_c_baseline_cv_results.iloc[0].to_dict()
    task_c_transition_tables = fit_task_c_transition_tables(task_c_train, alpha=1.0)
    save_joblib(task_c_transition_tables, task_c_transition_tables_path)
    task_c_train_aug = augment_task_c_with_transition_priors(task_c_train, task_c_train)
    task_c_test_aug = augment_task_c_with_transition_priors(task_c_train, task_c_test)
    y_train_c = task_c_train_aug["y_code"].to_numpy()
    y_test_c = task_c_test_aug["y_code"].to_numpy()
    n_classes = len(task_c_label_encoder.classes_)
    if best_row["family"] == "majority_dummy":
        majority_class = int(task_c_train_aug["y_code"].value_counts().idxmax())
        pred = np.full(len(task_c_test_aug), majority_class, dtype=int)
        prob = np.zeros((len(task_c_test_aug), n_classes), dtype=float)
        prob[:, majority_class] = 1.0
        task_c_baseline_model = {"family": "majority_dummy", "majority_class": majority_class}
        task_c_baseline_name = "Majority dummy baseline"
    elif best_row["family"] == "markov1":
        prob = predict_task_c_markov_probs(task_c_test_aug, task_c_transition_tables, order=1)
        pred = prob.argmax(axis=1)
        task_c_baseline_model = {"family": "markov1", "transition_tables": task_c_transition_tables}
        task_c_baseline_name = "Smoothed Markov-1 transition baseline"
    elif best_row["family"] == "markov2":
        prob = predict_task_c_markov_probs(task_c_test_aug, task_c_transition_tables, order=2)
        pred = prob.argmax(axis=1)
        task_c_baseline_model = {"family": "markov2", "transition_tables": task_c_transition_tables}
        task_c_baseline_name = "Smoothed Markov-2 transition baseline"
    elif best_row["family"] == "hybrid_logreg":
        task_c_baseline_model = build_task_c_hybrid_logreg_pipeline(
            min_df=int(best_row["min_df"]),
            C=float(best_row["C"]),
            l1_ratio=float(best_row["l1_ratio"]),
            class_weight=best_row["class_weight"] if not pd.isna(best_row["class_weight"]) else None,
        )
        X_train_c = task_c_train_aug[
            [task_c_last_client_text_col, task_c_exchange_text_col]
            + task_c_numeric + task_c_prior_cols + task_c_categorical
        ]
        X_test_c = task_c_test_aug[
            [task_c_last_client_text_col, task_c_exchange_text_col]
            + task_c_numeric + task_c_prior_cols + task_c_categorical
        ]
        task_c_baseline_model.fit(X_train_c, y_train_c)
        pred = task_c_baseline_model.predict(X_test_c)
        prob = task_c_baseline_model.predict_proba(X_test_c)
        task_c_baseline_name = "Hybrid elastic-net logistic baseline"
    elif best_row["family"] in {"structured_catboost", "structured_xgboost", "structured_histgb"}:
        model_bundle = build_task_c_structured_model(best_row)
        X_train_c = _prepare_task_c_structured_df(task_c_train_aug)
        X_test_c = _prepare_task_c_structured_df(task_c_test_aug)
        _fit_task_c_structured_model(model_bundle, X_train_c, y_train_c)
        pred, prob = _predict_task_c_structured_model(model_bundle, X_test_c)
        task_c_baseline_model = model_bundle
        task_c_baseline_name = model_bundle["family_name"]
    else:
        raise ValueError(f"Unknown selected Task c family: {best_row['family']}")
    if prob is not None and "compute_multiclass_metrics" in globals():
        try:
            task_c_baseline_metrics = compute_multiclass_metrics(y_test_c, pred, prob)
        except Exception:
            task_c_baseline_metrics = _task_c_macro_metrics(y_test_c, pred, prob)
    else:
        task_c_baseline_metrics = _task_c_macro_metrics(y_test_c, pred, prob)
    task_c_baseline_preds = {"pred": pred, "prob": prob, "y_true": y_test_c}
    save_json(task_c_baseline_metrics, task_c_metrics_path)
    save_joblib(task_c_baseline_preds, task_c_preds_path)
    save_joblib(task_c_baseline_model, task_c_model_path)
    save_json({"task_c_baseline_name": task_c_baseline_name}, task_c_name_path)
    save_json(
        {
            "version": TASK_C_BASELINE_VERSION,
            "n_test": int(len(task_c_test)),
            "test_transcript_ids": sorted(task_c_test["transcript_id"].astype(str).tolist()),
        },
        task_c_baseline_meta_path,
    )
display(Markdown("### Task c backend availability"))
task_c_backend_df = pd.DataFrame([{
    "catboost_available": _task_c_backend_available("catboost"),
    "xgboost_available": _task_c_backend_available("xgboost"),
}])
display_report_table(
    task_c_backend_df,
    bool_cols=["catboost_available", "xgboost_available"],
)
display(Markdown(f"### Task c selected baseline: **{task_c_baseline_name}**"))
display_report_table(
    task_c_baseline_cv_results.head(30).round(4),
    row_color_col="family",
    row_colors=REPORT_FAMILY_COLORS,
    text_cols=["family", "class_weight"],
    gradient_cols=["cv_f1_macro_mean", "cv_accuracy_mean"],
    gradient_kwargs={"cmap": "YlGn"},
    highlight_min_cols=["cv_f1_macro_std", "cv_accuracy_std"],
    highlight_max_cols=["cv_f1_macro_mean", "cv_accuracy_mean"],
)
display(Markdown("### Table B.2A. Task C selected-baseline held-out metrics"))
display_report_table(
    pd.DataFrame([task_c_baseline_metrics]).round(4),
    gradient_cols=["accuracy", "precision_macro", "recall_macro", "f1_macro", "precision_weighted", "recall_weighted", "f1_weighted"],
    gradient_kwargs={"cmap": "YlGn"},
    highlight_min_cols=["brier_multiclass"],
)


In [ ]:
task_c_seq_metrics_path = task_c_dir / "task_c_seq_metrics.json"
task_c_seq_preds_path = task_c_dir / "task_c_seq_preds.joblib"
task_c_seq_model_path = task_c_dir / "task_c_seq_models.joblib"
task_c_seq_cv_path = task_c_dir / "task_c_seq_cv_results.csv"
task_c_seq_best_cfg_path = task_c_dir / "task_c_seq_best_cfg.json"
task_c_seq_meta_path = task_c_dir / "task_c_seq_meta.json"
TASK_C_SEQ_VERSION = "hybrid_gru_transition_prior_v2"
def _task_c_seq_cache_is_compatible() -> bool:
    needed = [
        task_c_seq_metrics_path,
        task_c_seq_preds_path,
        task_c_seq_cv_path,
        task_c_seq_best_cfg_path,
        task_c_seq_meta_path,
    ]
    if not all(Path(p).exists() for p in needed):
        return False
    try:
        meta = load_json(task_c_seq_meta_path)
        if meta.get("version") != TASK_C_SEQ_VERSION:
            return False
        if meta.get("n_test") != int(len(task_c_test)):
            return False
        if meta.get("test_transcript_ids") != sorted(task_c_test["transcript_id"].astype(str).tolist()):
            return False
        preds = load_joblib(task_c_seq_preds_path)
        if len(preds["pred"]) != len(task_c_test):
            return False
        return True
    except Exception:
        return False
def _load_task_c_tokenizer(model_name_or_path: str, max_length: int = 96):
    if "load_tokenizer_safe" in globals():
        return load_tokenizer_safe(model_name_or_path, max_length=max_length)
    try:
        tok = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True, use_fast=True)
    except Exception:
        tok = AutoTokenizer.from_pretrained(model_name_or_path, trust_remote_code=True, use_fast=False)
    tok.model_max_length = max_length
    return tok
def encode_unique_texts_for_task_c(
    texts: List[str],
    model_name: str,
    cache_path: Path,
    max_length: int = 96,
    batch_size: int = 32,
) -> Dict[str, np.ndarray]:
    if cache_path.exists() and not CONFIG.get("force_recompute", False):
        return load_joblib(cache_path)
    texts = ["" if pd.isna(t) else str(t) for t in texts]
    unique_texts = sorted(set(texts))
    if len(unique_texts) == 0:
        save_joblib({}, cache_path)
        return {}
    tokenizer = _load_task_c_tokenizer(model_name, max_length=max_length)
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(DEVICE)
    model.eval()
    text_to_emb: Dict[str, np.ndarray] = {}
    for i in tqdm(range(0, len(unique_texts), batch_size), desc=f"Encoding Task c texts with {model_name}"):
        batch = unique_texts[i:i + batch_size]
        tok = tokenizer(
            batch,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(DEVICE)
        with torch.no_grad():
            out = model(**tok)
            emb = out.last_hidden_state[:, 0, :].detach().cpu().numpy()
        for txt, vec in zip(batch, emb):
            text_to_emb[txt] = vec.astype(np.float32)
    save_joblib(text_to_emb, cache_path)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return text_to_emb
def fit_task_c_struct_preprocessor(train_df: pd.DataFrame):
    num_cols = task_c_numeric + task_c_prior_cols
    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
                ("scaler", StandardScaler()),
            ]), num_cols),
            ("cat", _task_c_onehot_encoder(), task_c_categorical),
        ],
        sparse_threshold=0.0,
    )
    X = pre.fit_transform(train_df[num_cols + task_c_categorical])
    if sp_sparse.issparse(X):
        X = X.toarray()
    return pre, np.asarray(X, dtype=np.float32)
def transform_task_c_struct_preprocessor(df: pd.DataFrame, pre):
    num_cols = task_c_numeric + task_c_prior_cols
    X = pre.transform(df[num_cols + task_c_categorical])
    if sp_sparse.issparse(X):
        X = X.toarray()
    return np.asarray(X, dtype=np.float32)
def build_task_c_hybrid_arrays(
    df: pd.DataFrame,
    history_text_to_emb: Dict[str, np.ndarray],
    client_text_to_emb: Dict[str, np.ndarray],
    seq_len: int,
    struct_preprocessor=None,
    precomputed_struct: Optional[np.ndarray] = None,
):
    if len(history_text_to_emb) == 0:
        raise RuntimeError("Task c history embedding cache is empty.")
    emb_dim = next(iter(history_text_to_emb.values())).shape[0]
    client_dim = next(iter(client_text_to_emb.values())).shape[0] if len(client_text_to_emb) > 0 else emb_dim
    seq_list = []
    client_vecs = []
    prior_vecs = []
    y_list = []
    g_list = []
    if precomputed_struct is None:
        if struct_preprocessor is None:
            raise ValueError("Either struct_preprocessor or precomputed_struct must be supplied.")
        struct_arr = transform_task_c_struct_preprocessor(df, struct_preprocessor)
    else:
        struct_arr = np.asarray(precomputed_struct, dtype=np.float32)
    history_values = df["history_turns_list"].tolist()
    client_values = df[task_c_last_client_text_col].tolist()
    prior_arr = df[task_c_prior_prob_cols].to_numpy(dtype=np.float32)
    y_values = df["y_code"].to_numpy()
    group_values = df["transcript_id"].to_numpy()
    for hist_turns, client_text_raw, prior_vec, y_code, transcript_id in zip(
        history_values, client_values, prior_arr, y_values, group_values
    ):
        if isinstance(hist_turns, str):
            try:
                hist_turns = ast.literal_eval(hist_turns)
            except Exception:
                hist_turns = []
        hist_turns = hist_turns[-seq_len:] if len(hist_turns) > 0 else []
        seq_embs = [history_text_to_emb[t] for t in hist_turns if t in history_text_to_emb]
        seq_padded = np.zeros((seq_len, emb_dim), dtype=np.float32)
        if len(seq_embs) > 0:
            start = seq_len - len(seq_embs)
            seq_padded[start:] = np.stack(seq_embs, axis=0)
        seq_list.append(seq_padded)
        client_text = "" if pd.isna(client_text_raw) else str(client_text_raw)
        client_vecs.append(client_text_to_emb.get(client_text, np.zeros(client_dim, dtype=np.float32)))
        prior_vecs.append(prior_vec)
        y_list.append(int(y_code))
        g_list.append(transcript_id)
    return (
        np.stack(seq_list).astype(np.float32),
        np.stack(client_vecs).astype(np.float32),
        struct_arr.astype(np.float32),
        np.stack(prior_vecs).astype(np.float32),
        np.array(y_list, dtype=np.int64),
        np.array(g_list),
    )
class TaskCHybridDataset(Dataset):
    def __init__(self, seq_x, client_x, struct_x, prior_x, y):
        self.seq_x = torch.tensor(seq_x, dtype=torch.float32)
        self.client_x = torch.tensor(client_x, dtype=torch.float32)
        self.struct_x = torch.tensor(struct_x, dtype=torch.float32)
        self.prior_x = torch.tensor(prior_x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return (
            self.seq_x[idx],
            self.client_x[idx],
            self.struct_x[idx],
            self.prior_x[idx],
            self.y[idx],
        )
class TaskCHybridGRU(nn.Module):
    def __init__(
        self,
        seq_input_dim: int,
        client_input_dim: int,
        struct_input_dim: int,
        n_classes: int,
        hidden_size: int = 160,
        struct_hidden: int = 96,
        fusion_hidden: int = 192,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.gru = nn.GRU(
            input_size=seq_input_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=False,
        )
        self.hist_proj = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Dropout(dropout),
        )
        self.client_proj = nn.Sequential(
            nn.Linear(client_input_dim, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.struct_proj = nn.Sequential(
            nn.Linear(struct_input_dim, struct_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.fusion = nn.Sequential(
            nn.Linear(hidden_size + hidden_size + struct_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, n_classes),
        )
        self.prior_proj = nn.Linear(n_classes, n_classes, bias=False)
    def forward(self, seq_x, client_x, struct_x, prior_x):
        _, h = self.gru(seq_x)
        hist_repr = self.hist_proj(h[-1])
        client_repr = self.client_proj(client_x)
        struct_repr = self.struct_proj(struct_x)
        fusion_in = torch.cat([hist_repr, client_repr, struct_repr], dim=1)
        fusion_logits = self.fusion(fusion_in)
        prior_logits = self.prior_proj(prior_x)
        return fusion_logits + prior_logits
def train_task_c_hybrid_once(
    seq_train,
    client_train,
    struct_train,
    prior_train,
    y_train,
    seq_val,
    client_val,
    struct_val,
    prior_val,
    y_val,
    cfg: Dict[str, Any],
    n_classes: int,
    seed: int = 42,
):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    train_ds = TaskCHybridDataset(seq_train, client_train, struct_train, prior_train, y_train)
    val_ds = TaskCHybridDataset(seq_val, client_val, struct_val, prior_val, y_val)
    train_loader = DataLoader(train_ds, batch_size=int(cfg["batch_size"]), shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=int(cfg["batch_size"]), shuffle=False)
    model = TaskCHybridGRU(
        seq_input_dim=seq_train.shape[-1],
        client_input_dim=client_train.shape[-1],
        struct_input_dim=struct_train.shape[-1],
        n_classes=n_classes,
        hidden_size=int(cfg["hidden_size"]),
        struct_hidden=int(cfg["struct_hidden"]),
        fusion_hidden=int(cfg["fusion_hidden"]),
        dropout=float(cfg["dropout"]),
    ).to(DEVICE)
    present_classes = np.unique(y_train)
    present_weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_train,
    )
    full_weights = np.ones(n_classes, dtype=np.float32)
    for cls_id, w in zip(present_classes, present_weights):
        full_weights[int(cls_id)] = float(w)
    class_weights = torch.tensor(full_weights, dtype=torch.float32, device=DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(cfg["learning_rate"]), weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_state = None
    best_f1 = -1.0
    best_epoch = 1
    patience = int(cfg.get("patience", 3))
    wait = 0
    history_rows = []
    for epoch in range(1, int(cfg["epochs"]) + 1):
        model.train()
        running_loss = 0.0
        for seq_x, client_x, struct_x, prior_x, yb in train_loader:
            seq_x = seq_x.to(DEVICE)
            client_x = client_x.to(DEVICE)
            struct_x = struct_x.to(DEVICE)
            prior_x = prior_x.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(seq_x, client_x, struct_x, prior_x)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            running_loss += float(loss.item()) * len(yb)
        model.eval()
        all_pred = []
        all_true = []
        with torch.no_grad():
            for seq_x, client_x, struct_x, prior_x, yb in val_loader:
                seq_x = seq_x.to(DEVICE)
                client_x = client_x.to(DEVICE)
                struct_x = struct_x.to(DEVICE)
                prior_x = prior_x.to(DEVICE)
                logits = model(seq_x, client_x, struct_x, prior_x)
                pred = logits.argmax(dim=1).cpu().numpy()
                all_pred.extend(pred.tolist())
                all_true.extend(yb.numpy().tolist())
        val_f1 = f1_score(all_true, all_pred, average="macro")
        train_loss = running_loss / max(len(train_ds), 1)
        history_rows.append({"epoch": epoch, "train_loss": train_loss, "val_f1_macro": float(val_f1)})
        if val_f1 > best_f1:
            best_f1 = float(val_f1)
            best_epoch = int(epoch)
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state is None:
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, best_f1, best_epoch, pd.DataFrame(history_rows)
def fit_task_c_hybrid_fixed_epochs(
    seq_train,
    client_train,
    struct_train,
    prior_train,
    y_train,
    cfg: Dict[str, Any],
    n_classes: int,
    seed: int = 42,
):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    train_ds = TaskCHybridDataset(seq_train, client_train, struct_train, prior_train, y_train)
    train_loader = DataLoader(train_ds, batch_size=int(cfg["batch_size"]), shuffle=True)
    model = TaskCHybridGRU(
        seq_input_dim=seq_train.shape[-1],
        client_input_dim=client_train.shape[-1],
        struct_input_dim=struct_train.shape[-1],
        n_classes=n_classes,
        hidden_size=int(cfg["hidden_size"]),
        struct_hidden=int(cfg["struct_hidden"]),
        fusion_hidden=int(cfg["fusion_hidden"]),
        dropout=float(cfg["dropout"]),
    ).to(DEVICE)
    present_classes = np.unique(y_train)
    present_weights = compute_class_weight(
        class_weight="balanced",
        classes=present_classes,
        y=y_train,
    )
    full_weights = np.ones(n_classes, dtype=np.float32)
    for cls_id, w in zip(present_classes, present_weights):
        full_weights[int(cls_id)] = float(w)
    class_weights = torch.tensor(full_weights, dtype=torch.float32, device=DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(cfg["learning_rate"]), weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    for _ in range(int(cfg["epochs_fixed"])):
        model.train()
        for seq_x, client_x, struct_x, prior_x, yb in train_loader:
            seq_x = seq_x.to(DEVICE)
            client_x = client_x.to(DEVICE)
            struct_x = struct_x.to(DEVICE)
            prior_x = prior_x.to(DEVICE)
            yb = yb.to(DEVICE)
            optimizer.zero_grad()
            logits = model(seq_x, client_x, struct_x, prior_x)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
    return model
def predict_task_c_hybrid(model, seq_x, client_x, struct_x, prior_x, batch_size=256):
    ds = TaskCHybridDataset(seq_x, client_x, struct_x, prior_x, np.zeros(len(seq_x), dtype=np.int64))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    model.eval()
    all_logits = []
    with torch.no_grad():
        for seq_b, client_b, struct_b, prior_b, _ in loader:
            seq_b = seq_b.to(DEVICE)
            client_b = client_b.to(DEVICE)
            struct_b = struct_b.to(DEVICE)
            prior_b = prior_b.to(DEVICE)
            logits = model(seq_b, client_b, struct_b, prior_b).detach().cpu().numpy()
            all_logits.append(logits)
    logits = np.vstack(all_logits)
    prob = softmax(logits, axis=1)
    pred = prob.argmax(axis=1)
    return pred, prob
def search_task_c_hybrid_gru(train_df: pd.DataFrame):
    history_turn_lists = train_df["history_turns_list"].tolist()
    train_history_texts = [t for seq in history_turn_lists for t in (seq if isinstance(seq, list) else [])]
    train_client_texts = train_df[task_c_last_client_text_col].fillna("").astype(str).tolist()
    hist_emb_cache = task_c_dir / f"task_c_history_turn_embeddings_trainonly_{CONFIG['task_c_turn_encoder'].replace('/', '__')}.joblib"
    client_emb_cache = task_c_dir / f"task_c_last_client_embeddings_trainonly_{CONFIG['task_c_turn_encoder'].replace('/', '__')}.joblib"
    history_text_to_emb = encode_unique_texts_for_task_c(
        texts=train_history_texts,
        model_name=CONFIG["task_c_turn_encoder"],
        cache_path=hist_emb_cache,
        max_length=96,
        batch_size=32,
    )
    client_text_to_emb = encode_unique_texts_for_task_c(
        texts=train_client_texts,
        model_name=CONFIG["task_c_turn_encoder"],
        cache_path=client_emb_cache,
        max_length=96,
        batch_size=32,
    )
    search_space = list(
        ParameterSampler(
            {
                "seq_len": [3, 5, 10],
                "hidden_size": [128, 160, 192],
                "struct_hidden": [64, 96, 128],
                "fusion_hidden": [128, 192, 256],
                "dropout": [0.1, 0.2, 0.3],
                "learning_rate": [1e-3, 5e-4, 3e-4],
                "batch_size": [32, 64],
                "epochs": [12],
                "patience": [3],
            },
            n_iter=8,
            random_state=SEED,
        )
    )
    splitter = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
    y = train_df["y_code"].to_numpy()
    groups = train_df["transcript_id"].to_numpy()
    rows = []
    best_cfg = None
    best_score = -1.0
    for cfg in tqdm(search_space, desc="Task c hybrid GRU search"):
        fold_f1 = []
        fold_acc = []
        for tr_idx, val_idx in splitter.split(train_df, y, groups):
            tr_df = train_df.iloc[tr_idx].copy().reset_index(drop=True)
            val_df = train_df.iloc[val_idx].copy().reset_index(drop=True)
            tr_aug = augment_task_c_with_transition_priors(tr_df, tr_df)
            val_aug = augment_task_c_with_transition_priors(tr_df, val_df)
            struct_pre, struct_tr = fit_task_c_struct_preprocessor(tr_aug)
            struct_val = transform_task_c_struct_preprocessor(val_aug, struct_pre)
            seq_tr, client_tr, _, prior_tr, y_tr, _ = build_task_c_hybrid_arrays(
                tr_aug,
                history_text_to_emb=history_text_to_emb,
                client_text_to_emb=client_text_to_emb,
                seq_len=int(cfg["seq_len"]),
                precomputed_struct=struct_tr,
            )
            seq_val, client_val, _, prior_val, y_val, _ = build_task_c_hybrid_arrays(
                val_aug,
                history_text_to_emb=history_text_to_emb,
                client_text_to_emb=client_text_to_emb,
                seq_len=int(cfg["seq_len"]),
                precomputed_struct=struct_val,
            )
            model, val_f1, _, _ = train_task_c_hybrid_once(
                seq_train=seq_tr,
                client_train=client_tr,
                struct_train=struct_tr,
                prior_train=prior_tr,
                y_train=y_tr,
                seq_val=seq_val,
                client_val=client_val,
                struct_val=struct_val,
                prior_val=prior_val,
                y_val=y_val,
                cfg=cfg,
                n_classes=len(task_c_label_encoder.classes_),
                seed=SEED,
            )
            pred_val, _ = predict_task_c_hybrid(model, seq_val, client_val, struct_val, prior_val, batch_size=int(cfg["batch_size"]))
            fold_f1.append(f1_score(y_val, pred_val, average="macro"))
            fold_acc.append(accuracy_score(y_val, pred_val))
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
        row = dict(cfg)
        row["cv_f1_macro_mean"] = float(np.mean(fold_f1))
        row["cv_f1_macro_std"] = float(np.std(fold_f1))
        row["cv_accuracy_mean"] = float(np.mean(fold_acc))
        rows.append(row)
        if row["cv_f1_macro_mean"] > best_score:
            best_score = row["cv_f1_macro_mean"]
            best_cfg = dict(cfg)
    cv_df = (
        pd.DataFrame(rows)
        .sort_values(["cv_f1_macro_mean", "cv_accuracy_mean", "cv_f1_macro_std"], ascending=[False, False, True])
        .reset_index(drop=True)
    )
    return cv_df, best_cfg
if _task_c_seq_cache_is_compatible() and not CONFIG.get("force_retrain", False):
    task_c_seq_metrics = load_json(task_c_seq_metrics_path)
    task_c_seq_preds = load_joblib(task_c_seq_preds_path)
    task_c_seq_cv_results = pd.read_csv(task_c_seq_cv_path)
    task_c_seq_best_cfg = load_json(task_c_seq_best_cfg_path)
else:
    task_c_seq_cv_results, task_c_seq_best_cfg = search_task_c_hybrid_gru(task_c_train)
    save_df(task_c_seq_cv_results, task_c_seq_cv_path)
    task_c_seed_list = CONFIG.get("task_c_multiseed_seeds", [17, 42, 101])
    task_c_seq_best_cfg = dict(task_c_seq_best_cfg)
    task_c_seq_best_cfg["seed_list"] = [int(s) for s in task_c_seed_list]
    final_history_texts = [t for seq in task_c_train["history_turns_list"].tolist() for t in (seq if isinstance(seq, list) else [])]
    final_history_texts += [t for seq in task_c_test["history_turns_list"].tolist() for t in (seq if isinstance(seq, list) else [])]
    final_client_texts = task_c_train[task_c_last_client_text_col].fillna("").astype(str).tolist()
    final_client_texts += task_c_test[task_c_last_client_text_col].fillna("").astype(str).tolist()
    final_hist_emb_cache = task_c_dir / f"task_c_history_turn_embeddings_final_{CONFIG['task_c_turn_encoder'].replace('/', '__')}.joblib"
    final_client_emb_cache = task_c_dir / f"task_c_last_client_embeddings_final_{CONFIG['task_c_turn_encoder'].replace('/', '__')}.joblib"
    final_history_text_to_emb = encode_unique_texts_for_task_c(
        texts=final_history_texts,
        model_name=CONFIG["task_c_turn_encoder"],
        cache_path=final_hist_emb_cache,
        max_length=96,
        batch_size=32,
    )
    final_client_text_to_emb = encode_unique_texts_for_task_c(
        texts=final_client_texts,
        model_name=CONFIG["task_c_turn_encoder"],
        cache_path=final_client_emb_cache,
        max_length=96,
        batch_size=32,
    )
    seed_prob_list = []
    seed_metric_rows = []
    saved_seed_states = []
    for seed in task_c_seed_list:
        splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=int(seed))
        y_all = task_c_train["y_code"].to_numpy()
        g_all = task_c_train["transcript_id"].to_numpy()
        train_idx, val_idx = next(splitter.split(task_c_train, y_all, g_all))
        early_train_df = task_c_train.iloc[train_idx].copy().reset_index(drop=True)
        early_val_df = task_c_train.iloc[val_idx].copy().reset_index(drop=True)
        early_train_aug = augment_task_c_with_transition_priors(early_train_df, early_train_df)
        early_val_aug = augment_task_c_with_transition_priors(early_train_df, early_val_df)
        early_struct_pre, early_struct_train = fit_task_c_struct_preprocessor(early_train_aug)
        early_struct_val = transform_task_c_struct_preprocessor(early_val_aug, early_struct_pre)
        early_seq_train, early_client_train, _, early_prior_train, early_y_train, _ = build_task_c_hybrid_arrays(
            early_train_aug,
            history_text_to_emb=final_history_text_to_emb,
            client_text_to_emb=final_client_text_to_emb,
            seq_len=int(task_c_seq_best_cfg["seq_len"]),
            precomputed_struct=early_struct_train,
        )
        early_seq_val, early_client_val, _, early_prior_val, early_y_val, _ = build_task_c_hybrid_arrays(
            early_val_aug,
            history_text_to_emb=final_history_text_to_emb,
            client_text_to_emb=final_client_text_to_emb,
            seq_len=int(task_c_seq_best_cfg["seq_len"]),
            precomputed_struct=early_struct_val,
        )
        _, _, best_epoch, train_hist_df = train_task_c_hybrid_once(
            seq_train=early_seq_train,
            client_train=early_client_train,
            struct_train=early_struct_train,
            prior_train=early_prior_train,
            y_train=early_y_train,
            seq_val=early_seq_val,
            client_val=early_client_val,
            struct_val=early_struct_val,
            prior_val=early_prior_val,
            y_val=early_y_val,
            cfg=task_c_seq_best_cfg,
            n_classes=len(task_c_label_encoder.classes_),
            seed=int(seed),
        )
        full_train_aug = augment_task_c_with_transition_priors(task_c_train, task_c_train)
        full_test_aug = augment_task_c_with_transition_priors(task_c_train, task_c_test)
        full_struct_pre, full_struct_train = fit_task_c_struct_preprocessor(full_train_aug)
        full_struct_test = transform_task_c_struct_preprocessor(full_test_aug, full_struct_pre)
        full_seq_train, full_client_train, _, full_prior_train, full_y_train, _ = build_task_c_hybrid_arrays(
            full_train_aug,
            history_text_to_emb=final_history_text_to_emb,
            client_text_to_emb=final_client_text_to_emb,
            seq_len=int(task_c_seq_best_cfg["seq_len"]),
            precomputed_struct=full_struct_train,
        )
        full_seq_test, full_client_test, _, full_prior_test, full_y_test, _ = build_task_c_hybrid_arrays(
            full_test_aug,
            history_text_to_emb=final_history_text_to_emb,
            client_text_to_emb=final_client_text_to_emb,
            seq_len=int(task_c_seq_best_cfg["seq_len"]),
            precomputed_struct=full_struct_test,
        )
        seed_cfg = dict(task_c_seq_best_cfg)
        seed_cfg["epochs_fixed"] = int(best_epoch)
        final_model = fit_task_c_hybrid_fixed_epochs(
            seq_train=full_seq_train,
            client_train=full_client_train,
            struct_train=full_struct_train,
            prior_train=full_prior_train,
            y_train=full_y_train,
            cfg=seed_cfg,
            n_classes=len(task_c_label_encoder.classes_),
            seed=int(seed),
        )
        seed_pred, seed_prob = predict_task_c_hybrid(
            final_model,
            full_seq_test,
            full_client_test,
            full_struct_test,
            full_prior_test,
            batch_size=int(task_c_seq_best_cfg["batch_size"]),
        )
        seed_prob_list.append(seed_prob)
        seed_metric_rows.append({
            "seed": int(seed),
            "best_epoch": int(best_epoch),
            "accuracy": float(accuracy_score(full_y_test, seed_pred)),
            "f1_macro": float(f1_score(full_y_test, seed_pred, average="macro")),
        })
        saved_seed_states.append({
            "seed": int(seed),
            "best_epoch": int(best_epoch),
            "state_dict": {k: v.detach().cpu() for k, v in final_model.state_dict().items()},
            "config": dict(seed_cfg),
        })
        del final_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()
    mean_prob = np.mean(np.stack(seed_prob_list, axis=0), axis=0)
    final_pred = mean_prob.argmax(axis=1)
    final_y_true = task_c_test["y_code"].to_numpy()
    task_c_seq_metrics = compute_multiclass_metrics(final_y_true, final_pred, mean_prob)
    task_c_seq_preds = {
        "pred": final_pred,
        "prob": mean_prob,
        "y_true": final_y_true,
        "per_seed_rows": seed_metric_rows,
    }
    save_json(task_c_seq_best_cfg, task_c_seq_best_cfg_path)
    save_df(task_c_seq_cv_results, task_c_seq_cv_path)
    save_json(task_c_seq_metrics, task_c_seq_metrics_path)
    save_joblib(task_c_seq_preds, task_c_seq_preds_path)
    save_joblib(
        {
            "version": TASK_C_SEQ_VERSION,
            "seed_states": saved_seed_states,
            "best_cfg": task_c_seq_best_cfg,
        },
        task_c_seq_model_path,
    )
    save_json(
        {
            "version": TASK_C_SEQ_VERSION,
            "n_test": int(len(task_c_test)),
            "test_transcript_ids": sorted(task_c_test["transcript_id"].astype(str).tolist()),
        },
        task_c_seq_meta_path,
    )
display(Markdown("### Task c hybrid-GRU search results"))
display_report_table(
    task_c_seq_cv_results.head(20).round(4),
    gradient_cols=["cv_f1_macro_mean", "cv_accuracy_mean"],
    gradient_kwargs={"cmap": "YlGn"},
    highlight_min_cols=["cv_f1_macro_std", "cv_accuracy_std"],
    highlight_max_cols=["cv_f1_macro_mean", "cv_accuracy_mean"],
)
display(Markdown("### Task c selected hybrid-GRU configuration"))
task_c_seq_best_cfg_display = pd.DataFrame([task_c_seq_best_cfg]).T.reset_index()
task_c_seq_best_cfg_display.columns = ["setting", "value"]
display_report_table(task_c_seq_best_cfg_display, text_cols=["setting", "value"])
display_report_table(
    pd.DataFrame([task_c_seq_metrics]).round(4),
    gradient_cols=["accuracy", "precision_macro", "recall_macro", "f1_macro", "precision_weighted", "recall_weighted", "f1_weighted"],
    gradient_kwargs={"cmap": "YlGn"},
    highlight_min_cols=["brier_multiclass"],
)
if isinstance(task_c_seq_preds, dict) and "per_seed_rows" in task_c_seq_preds:
    display(Markdown("### Task c per-seed hybrid-GRU test results"))
    display_report_table(
        pd.DataFrame(task_c_seq_preds["per_seed_rows"]).round(4),
        gradient_cols=["accuracy", "f1_macro"],
        gradient_kwargs={"cmap": "YlGn"},
        highlight_max_cols=["accuracy", "f1_macro"],
    )


In [ ]:
_cm = confusion_matrix
display(Markdown("### Figure 37. Task C selected-baseline confusion matrix"))
plot_confusion(
    task_c_baseline_preds["y_true"],
    task_c_baseline_preds["pred"],
    labels=list(task_c_label_encoder.classes_),
    title=f"Task c – {task_c_baseline_name}",
    save_path=FIG_DIR / "task_c_cm_baseline.png",
)
display(Markdown("### Figure 38. Task C hybrid-GRU confusion matrix"))
plot_confusion(
    task_c_seq_preds["y_true"],
    task_c_seq_preds["pred"],
    labels=list(task_c_label_encoder.classes_),
    title="Task c – hybrid GRU with local-state fusion",
    save_path=FIG_DIR / "task_c_cm_gru.png",
)
task_c_baseline_report = pd.DataFrame(
    classification_report(
        task_c_baseline_preds["y_true"],
        task_c_baseline_preds["pred"],
        target_names=list(task_c_label_encoder.classes_),
        output_dict=True,
        zero_division=0,
    )
).T
task_c_seq_report = pd.DataFrame(
    classification_report(
        task_c_seq_preds["y_true"],
        task_c_seq_preds["pred"],
        target_names=list(task_c_label_encoder.classes_),
        output_dict=True,
        zero_division=0,
    )
).T
display(Markdown("### Task c per-class report — selected baseline"))
task_c_baseline_report_display = task_c_baseline_report.round(4).reset_index().rename(columns={"index": "class"})
display_report_table(
    task_c_baseline_report_display,
    row_color_col="class",
    row_colors=REPORT_BEHAVIOUR_COLORS,
    text_cols=["class"],
    gradient_cols=["precision", "recall", "f1-score"],
    gradient_kwargs={"cmap": "YlGn"},
)
display(Markdown("### Task c per-class report — hybrid GRU"))
task_c_seq_report_display = task_c_seq_report.round(4).reset_index().rename(columns={"index": "class"})
display_report_table(
    task_c_seq_report_display,
    row_color_col="class",
    row_colors=REPORT_BEHAVIOUR_COLORS,
    text_cols=["class"],
    gradient_cols=["precision", "recall", "f1-score"],
    gradient_kwargs={"cmap": "YlGn"},
)
label_rows = list(task_c_label_encoder.classes_)
compare_f1_df = pd.DataFrame({
    "label": label_rows,
    task_c_baseline_name: [task_c_baseline_report.loc[l, "f1-score"] for l in label_rows],
    "Hybrid GRU": [task_c_seq_report.loc[l, "f1-score"] for l in label_rows],
})
fig, ax = plt.subplots(figsize=(9, 4.5))
compare_f1_df.set_index("label").plot(kind="bar", ax=ax, width=0.75)
ax.set_title("Task c per-class F1: baseline vs hybrid GRU")
ax.set_ylabel("F1-score")
ax.set_xlabel("")
ax.legend(title="")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(FIG_DIR / "task_c_per_class_f1_comparison.png", dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 39. Task C per-class F1 comparison"))
display(IPyImage(filename=str(FIG_DIR / "task_c_per_class_f1_comparison.png")))
labels_idx = list(range(len(task_c_label_encoder.classes_)))
def _normalised_cm(y_true, y_pred, labels):
    cm = _cm(y_true, y_pred, labels=labels).astype(float)
    row_sums = cm.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    return cm / row_sums
cm_baseline_norm = _normalised_cm(
    task_c_baseline_preds["y_true"], task_c_baseline_preds["pred"], labels_idx,
)
cm_gru_norm = _normalised_cm(
    task_c_seq_preds["y_true"], task_c_seq_preds["pred"], labels_idx,
)
cm_delta = cm_gru_norm - cm_baseline_norm
fig, ax = plt.subplots(figsize=(7.5, 6.5))
_sns_local = sns
_sns_local.heatmap(
    cm_delta,
    annot=True, fmt=".2f", cmap="RdBu_r", center=0.0, vmin=-0.20, vmax=0.20,
    xticklabels=list(task_c_label_encoder.classes_),
    yticklabels=list(task_c_label_encoder.classes_),
    cbar_kws={"label": "GRU(row-normalised CM) − baseline(row-normalised CM)"},
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(
    "Task c — CM delta (Hybrid GRU − baseline)\n"
    "Positive on the diagonal = GRU is more often right for that true class\n"
    "Off-diagonal sign = where the GRU re-routes probability mass"
)
plt.tight_layout()
cm_delta_path = FIG_DIR / "task_c_cm_delta_gru_minus_baseline.png"
plt.savefig(cm_delta_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 40. Task C confusion-matrix delta for hybrid GRU vs baseline"))
display(IPyImage(filename=str(cm_delta_path)))
cm_delta_df = pd.DataFrame(
    cm_delta,
    index=list(task_c_label_encoder.classes_),
    columns=list(task_c_label_encoder.classes_),
)
save_df(cm_delta_df.reset_index().rename(columns={"index": "true_class"}),
        task_c_dir / "task_c_cm_delta_gru_minus_baseline.csv")
y_true_test_c = np.asarray(task_c_baseline_preds["y_true"])
n_classes_c_local = len(task_c_label_encoder.classes_)
maj_class_c_local = int(pd.Series(task_c_train["y_code"]).value_counts().idxmax())
y_pred_majority_c_local = np.full_like(y_true_test_c, maj_class_c_local)
y_prob_majority_c_local = np.zeros((len(y_true_test_c), n_classes_c_local), dtype=float)
y_prob_majority_c_local[:, maj_class_c_local] = 1.0
majority_metrics_test_c = compute_multiclass_metrics(
    y_true_test_c, y_pred_majority_c_local, y_prob_majority_c_local
)
majority_label_str = task_c_label_encoder.classes_[maj_class_c_local]
test_rows = [
    {
        "model": f"Majority-class floor (test, always predicts '{majority_label_str}')",
        **majority_metrics_test_c,
    },
    {"model": task_c_baseline_name, **task_c_baseline_metrics},
    {"model": "Hybrid GRU with local-state fusion", **task_c_seq_metrics},
]
display_metric_cols = ["accuracy", "f1_macro", "f1_weighted", "brier_multiclass"]
test_rows_df = (
    pd.DataFrame(test_rows)
    .reindex(columns=["model"] + display_metric_cols)
    .round(4)
)
display(Markdown(
    "### Task c headline comparison — test set\n\n"
    "Reading order is `floor → baseline → neural`, i.e. ascending capability. "
    "Both trained models reach 44.6% accuracy and ~0.43 macro-F1, which is "
    "above the published macro-F1 = 0.40 band reported by Wu et al. (2022, "
    "Interspeech) on this exact task and dataset."
))
display_report_table(
    test_rows_df,
    row_color_col="model",
    row_colors=REPORT_MODEL_COLORS,
    text_cols=["model"],
    gradient_cols=["accuracy", "f1_macro", "f1_weighted"],
    gradient_kwargs={"cmap": "YlGn"},
    highlight_min_cols=["brier_multiclass"],
)
ceiling_rows = []
if "task_c_markov1_acc_train" in globals():
    ceiling_rows.append({
        "predictor": "Markov-1 oracle",
        "predictor_inputs": "ground-truth previous therapist label",
        "accuracy_train": float(task_c_markov1_acc_train),
        "macro_f1_train": float(task_c_markov1_f1_train),
    })
if "task_c_markov2_acc_train" in globals():
    ceiling_rows.append({
        "predictor": "Markov-2 oracle",
        "predictor_inputs": "ground-truth previous therapist label + last client talk",
        "accuracy_train": float(task_c_markov2_acc_train),
        "macro_f1_train": float(task_c_markov2_f1_train),
    })
if ceiling_rows:
    ceiling_rows_df = pd.DataFrame(ceiling_rows).round(4)
    display(Markdown(
        "### Task c training-set ceiling reference\n\n"
        "These rows use ground-truth previous labels at training-set scale; "
        "they are *not* achievable by a real-time forecasting model and are "
        "shown only as reference ceilings. Both ceilings sit at roughly "
        "41–42% accuracy, which means simply knowing the previous oracle "
        "labels gets you a few points above the marginal floor — anything "
        "more requires learning from the utterance text itself."
    ))
    ceiling_display_df = (
        ceiling_rows_df
        .assign(
            predictor_inputs=lambda d: d["predictor_inputs"].replace(
                {
                    "ground-truth previous therapist label": "Previous therapist label",
                    "ground-truth previous therapist label + last client talk": "Previous therapist label + last client talk",
                }
            )
        )
        .rename(
            columns={
                "predictor": "Predictor",
                "predictor_inputs": "Oracle Inputs",
                "accuracy_train": "Train Accuracy",
                "macro_f1_train": "Train Macro-F1",
            }
        )
    )
    display_report_table(
        ceiling_display_df,
        formats={"Train Accuracy": "{:.4f}", "Train Macro-F1": "{:.4f}"},
        row_color_col="Predictor",
        row_colors={"Markov-1 oracle": "#fff7ed", "Markov-2 oracle": "#e7f0ff"},
        text_cols=["Predictor", "Oracle Inputs"],
        highlight_max_cols=["Train Accuracy", "Train Macro-F1"],
    )
combined_headline_df = pd.concat(
    [
        test_rows_df.assign(scope="test_set"),
        (
            pd.DataFrame(ceiling_rows).rename(
                columns={"accuracy_train": "accuracy", "macro_f1_train": "f1_macro"}
            ).assign(scope="train_set_oracle_ceiling")
            if ceiling_rows
            else pd.DataFrame()
        ),
    ],
    ignore_index=True,
)
save_df(combined_headline_df, task_c_dir / "task_c_headline_comparison.csv")


In [ ]:
assert "task_c_baseline_preds" in globals() and "prob" in task_c_baseline_preds, (
    "task_c_baseline_preds with a 'prob' field not in memory. "
    "Run the Task c baseline cell first."
)
assert "task_c_seq_preds" in globals() and "prob" in task_c_seq_preds, (
    "task_c_seq_preds with a 'prob' field not in memory. "
    "Run the Task c hybrid GRU cell first."
)
n_classes_c = len(task_c_label_encoder.classes_)
labels_c = list(range(n_classes_c))
y_true_c = np.asarray(task_c_baseline_preds["y_true"])
maj_class_c = int(pd.Series(task_c_train["y_code"]).value_counts().idxmax())
y_pred_majority_c = np.full_like(y_true_c, maj_class_c)
y_prob_majority_c = np.zeros((len(y_true_c), n_classes_c), dtype=float)
y_prob_majority_c[:, maj_class_c] = 1.0
def _topk_metrics(y_true, y_prob, y_pred):
    return {
        "top_1_accuracy": float((y_pred == y_true).mean()),
        "top_2_accuracy": float(
            top_k_accuracy_score(y_true, y_prob, k=2, labels=labels_c)
        ),
        "top_3_accuracy": float(
            top_k_accuracy_score(y_true, y_prob, k=3, labels=labels_c)
        ),
    }
task_c_topk_rows = [
    {
        "model": (
            "Majority-class baseline "
            f"(always predicts '{task_c_label_encoder.classes_[maj_class_c]}')"
        ),
        **_topk_metrics(y_true_c, y_prob_majority_c, y_pred_majority_c),
    },
    {
        "model": task_c_baseline_name,
        **_topk_metrics(
            np.asarray(task_c_baseline_preds["y_true"]),
            np.asarray(task_c_baseline_preds["prob"]),
            np.asarray(task_c_baseline_preds["pred"]),
        ),
    },
    {
        "model": "Hybrid GRU with local-state fusion",
        **_topk_metrics(
            np.asarray(task_c_seq_preds["y_true"]),
            np.asarray(task_c_seq_preds["prob"]),
            np.asarray(task_c_seq_preds["pred"]),
        ),
    },
]
task_c_topk_df = pd.DataFrame(task_c_topk_rows).round(4)
display(Markdown(
    "### Appendix B: top-K accuracy on next-action forecasting\n\n"
    "Top-1 accuracy reflects only one of potentially multiple legitimate "
    "next therapist actions. Top-2 and top-3 give a more honest measure "
    "of how informative the model's predicted distribution is over the "
    "four candidate actions. The majority-class baseline is the floor."
))
task_c_topk_display_df = task_c_topk_df.rename(
    columns={
        "model": "Model",
        "top_1_accuracy": "Top-1 Accuracy",
        "top_2_accuracy": "Top-2 Accuracy",
        "top_3_accuracy": "Top-3 Accuracy",
    }
)
display_report_table(
    task_c_topk_display_df,
    formats={"Top-1 Accuracy": "{:.4f}", "Top-2 Accuracy": "{:.4f}", "Top-3 Accuracy": "{:.4f}"},
    row_color_col="Model",
    row_colors={
        "Majority-class floor": "#f8fafc",
        "CatBoost structured state model": "#f5f3ff",
        "Hybrid GRU with local-state fusion": "#ecfdf5",
    },
    text_cols=["Model"],
    highlight_max_cols=["Top-1 Accuracy", "Top-2 Accuracy", "Top-3 Accuracy"],
)
save_df(task_c_topk_df, task_c_dir / "task_c_topk_accuracy.csv")
plot_df = task_c_topk_df.set_index("model")[
    ["top_1_accuracy", "top_2_accuracy", "top_3_accuracy"]
].rename(columns={
    "top_1_accuracy": "top-1",
    "top_2_accuracy": "top-2",
    "top_3_accuracy": "top-3",
})
fig, ax = plt.subplots(figsize=(11, 5.5))
plot_df.plot(kind="bar", ax=ax, width=0.75)
ax.set_ylim(0.0, 1.0)
ax.set_ylabel("Accuracy on the held-out test set")
ax.set_xlabel("")
ax.set_title(
    "Task c: top-1 / top-2 / top-3 accuracy on next-action forecasting"
)
ax.grid(True, alpha=0.3, axis="y")
ax.axhline(0.25, color="grey", linestyle=":", linewidth=1)
ax.text(
    -0.45, 0.252, "uniform-random floor (0.25)",
    fontsize=9, color="grey",
)
ax.axhline(0.40, color="#b71c1c", linestyle="--", linewidth=1)
ax.text(
    -0.45, 0.402, "Wu et al. 2022 published top-1 macro-F1 band (0.40)",
    fontsize=9, color="#b71c1c",
)
ax.set_xticklabels(plot_df.index, rotation=15, ha="right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=2, fontsize=8)
ax.legend(title="metric", loc="upper left")
plt.tight_layout()
topk_path = FIG_DIR / "task_c_topk_accuracy_bar.png"
plt.savefig(topk_path, dpi=300, bbox_inches="tight")
plt.close(fig)
display(Markdown("### Figure 41. Task C top-K accuracy on next-action forecasting"))
display(IPyImage(filename=str(topk_path)))
save_json(
    {
        "task": "next-turn therapist action forecasting on AnnoMI",
        "test_n": int(len(y_true_c)),
        "n_classes": int(n_classes_c),
        "majority_class": task_c_label_encoder.classes_[maj_class_c],
        "majority_top_1_accuracy": float((y_pred_majority_c == y_true_c).mean()),
        "baseline_top_1_accuracy": float(
            (np.asarray(task_c_baseline_preds["pred"]) == y_true_c).mean()
        ),
        "gru_top_1_accuracy": float(
            (np.asarray(task_c_seq_preds["pred"])
             == np.asarray(task_c_seq_preds["y_true"])).mean()
        ),
        "gru_top_2_accuracy": float(task_c_topk_df.iloc[2]["top_2_accuracy"]),
        "gru_top_3_accuracy": float(task_c_topk_df.iloc[2]["top_3_accuracy"]),
        "uniform_random_floor": 0.25,
        "wu_2022_published_top1_macro_f1": 0.40,
        "published_reference": (
            "Wu et al. (2022), 'Towards Automated Counselling Decision-Making: "
            "Remarks on Therapist Action Forecasting on the AnnoMI Dataset', "
            "Interspeech 2022, pp. 1906-1910. Reported best macro-F1 = 0.40 "
            "with roberta-base; concluded the task is intrinsically multi-"
            "modal in its response space and recommended top-K reformulation."
        ),
    },
    task_c_dir / "task_c_published_context.json",
)


In [ ]:
RESEARCH_ARTIFACT_DIR = ARTIFACT_DIR / "research_track"
RESEARCH_FIG_DIR = FIG_DIR / "research_track"
RESEARCH_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESEARCH_FIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG.update({
    "run_research_track": True,
    "research_track_use_cached": True,
    "research_task_b_context_ablation": True,
    "research_task_b_topic_shift": True,
    "research_task_b_topic_shift_min_transcripts": 5,
    "research_task_a_nested_cv": True,
    "research_task_a_nested_cv_seeds": [17, 42, 101],
    "research_task_a_nested_cv_outer_splits": 5,
    "research_task_a_nested_cv_selection_metric": "balanced_accuracy",
    "research_task_a_boosted_n_iter": 18,
})
save_json(CONFIG, CONFIG_JSON)
required_research_names = [
    "artifact_exists",
    "save_df",
    "save_json",
    "load_json",
    "load_joblib",
    "label_encoder",
    "build_logreg_pipeline",
    "prepare_logreg_input",
    "compute_multiclass_metrics",
]
missing_research_names = [n for n in required_research_names if n not in globals()]
if missing_research_names:
    raise NameError(
        "The research-track section is missing required upstream objects. "
        f"Run the main modelling cells first: {missing_research_names}"
    )
TASK_B_ALL_LABELS = np.arange(len(label_encoder.classes_))
def compute_multiclass_metrics_fixed_labels(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: Optional[np.ndarray] = None,
    labels: Optional[np.ndarray] = None,
) -> Dict[str, float]:
    labels = TASK_B_ALL_LABELS if labels is None else np.asarray(labels)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        average="macro",
        zero_division=0,
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=labels,
        average="weighted",
        zero_division=0,
    )
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_macro),
        "recall_macro": float(recall_macro),
        "f1_macro": float(f1_macro),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
    }
    if y_prob is not None:
        y_onehot = np.eye(len(labels))[y_true]
        out["brier_multiclass"] = float(np.mean(np.sum((y_prob - y_onehot) ** 2, axis=1)))
    return out
def _research_get_best_task_b_logreg_cfg() -> Dict[str, Any]:
    if "best_logreg_cfg" in globals() and isinstance(best_logreg_cfg, dict) and len(best_logreg_cfg) > 0:
        cfg = dict(best_logreg_cfg)
    else:
        task_b_cv_path = ARTIFACT_DIR / "task_b" / "task_b_logreg_cv_results.csv"
        if not artifact_exists(task_b_cv_path):
            raise FileNotFoundError(
                f"Could not find Task b logistic CV results at {task_b_cv_path}. "
                "Run the Task b sparse baseline cell first."
            )
        cv_df = pd.read_csv(task_b_cv_path)
        sort_cols = [c for c in ["cv_f1_macro_mean", "cv_f1_macro_std"] if c in cv_df.columns]
        if sort_cols:
            ascending = [False, True][:len(sort_cols)]
            cv_df = cv_df.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)
        cfg = cv_df.iloc[0].to_dict()
    if isinstance(cfg.get("char_ngram"), str):
        cfg["char_ngram"] = tuple(ast.literal_eval(cfg["char_ngram"]))
    if pd.isna(cfg.get("class_weight", None)):
        cfg["class_weight"] = None
    if "context_k" in cfg and not pd.isna(cfg["context_k"]):
        cfg["context_k"] = int(cfg["context_k"])
    return cfg
research_track_plan = pd.DataFrame([
    {
        "scope": "Task B",
        "analysis": "Context-size ablation",
        "enabled": bool(CONFIG["research_task_b_context_ablation"]),
        "cache_mode": "reuse cached outputs" if CONFIG["research_track_use_cached"] else "recompute outputs",
        "artifact_output": "artifacts/research_track/task_b_context_ablation.csv",
        "figure_output": "figures/research_track/task_b_context_ablation_macro_f1.png",
        "purpose": "Checks whether causal context improves the sparse baseline",
    },
    {
        "scope": "Task B",
        "analysis": "Topic-shift robustness",
        "enabled": bool(CONFIG["research_task_b_topic_shift"]),
        "cache_mode": "reuse cached outputs" if CONFIG["research_track_use_cached"] else "recompute outputs",
        "artifact_output": "artifacts/research_track/task_b_topic_shift.csv",
        "figure_output": "figures/research_track/task_b_topic_shift_macro_f1.png",
        "purpose": "Holds out topic families to test robustness under topic shift",
    },
    {
        "scope": "Task A",
        "analysis": "Repeated nested CV",
        "enabled": bool(CONFIG["research_task_a_nested_cv"]),
        "cache_mode": "reuse cached outputs" if CONFIG["research_track_use_cached"] else "recompute outputs",
        "artifact_output": "artifacts/research_track/task_a_nested_cv/",
        "figure_output": "figures/research_track/task_a_nested_cv_boxplot.png",
        "purpose": "Adds repeated outer-fold stability estimates without changing the official holdout",
    },
])
_research_plan_colors = {
    "Task A": "#f3e8ff",
    "Task B": "#e7f0ff",
}
display(Markdown("### Table C.0. Research-track activation plan and output locations"))
display_report_table(
    research_track_plan,
    row_color_col="scope",
    row_colors=_research_plan_colors,
    text_cols=["scope", "analysis", "cache_mode", "artifact_output", "figure_output", "purpose"],
    bool_cols=["enabled"],
)
save_df(research_track_plan, RESEARCH_ARTIFACT_DIR / "research_track_activation_plan.csv")
print("Research-track outputs will be written to:", RESEARCH_ARTIFACT_DIR)

In [ ]:
if CONFIG.get("run_research_track", False) and CONFIG.get("research_task_b_context_ablation", True):
    best_logreg_cfg = _research_get_best_task_b_logreg_cfg()
    context_ablation_csv = RESEARCH_ARTIFACT_DIR / "task_b_context_ablation.csv"
    context_ablation_plot = RESEARCH_FIG_DIR / "task_b_context_ablation_macro_f1.png"
    if context_ablation_csv.exists() and CONFIG.get("research_track_use_cached", True) and not CONFIG.get("force_recompute", False):
        context_ablation_df = pd.read_csv(context_ablation_csv)
    else:
        train_df = prepare_logreg_input(therapist_train.copy())
        test_df = prepare_logreg_input(therapist_test.copy())
        rows = []
        for k in CONFIG["context_grid"]:
            context_col = f"context_k{int(k)}_lc"
            if context_col not in train_df.columns or context_col not in test_df.columns:
                continue
            model = build_logreg_pipeline(
                context_col=context_col,
                min_df=int(best_logreg_cfg["min_df"]),
                char_ngram=tuple(best_logreg_cfg["char_ngram"]),
                C=float(best_logreg_cfg["C"]),
                l1_ratio=float(best_logreg_cfg["l1_ratio"]),
                class_weight=best_logreg_cfg.get("class_weight", None),
                random_state=SEED,
            )
            model.fit(train_df, train_df["y_code"].to_numpy())
            pred = model.predict(test_df)
            prob = model.predict_proba(test_df)
            metrics = compute_multiclass_metrics_fixed_labels(
                test_df["y_code"].to_numpy(),
                pred,
                prob,
                labels=TASK_B_ALL_LABELS,
            )
            rows.append(
                {
                    "context_k": int(k),
                    **metrics,
                }
            )
        context_ablation_df = (
            pd.DataFrame(rows)
            .sort_values("context_k")
            .reset_index(drop=True)
        )
        if not context_ablation_df.empty:
            baseline_k0 = context_ablation_df.loc[context_ablation_df["context_k"] == 0, "f1_macro"]
            if len(baseline_k0) == 1:
                context_ablation_df["delta_f1_vs_k0"] = context_ablation_df["f1_macro"] - float(baseline_k0.iloc[0])
            else:
                context_ablation_df["delta_f1_vs_k0"] = np.nan
        save_df(context_ablation_df, context_ablation_csv)
    display(Markdown("### Table C.1. Task B sparse-baseline context-size ablation"))
    display_report_table(
        context_ablation_df.round(4),
        row_color_col="context_k",
        row_colors=REPORT_CONTEXT_COLORS,
        gradient_cols=["accuracy", "precision_macro", "recall_macro", "f1_macro", "precision_weighted", "recall_weighted", "f1_weighted"],
        gradient_kwargs={"cmap": "YlGn"},
        signed_cols=["delta_f1_vs_k0"],
        highlight_min_cols=["brier_multiclass"],
        highlight_max_cols=["f1_macro"],
    )
    if not context_ablation_df.empty:
        best_context_k = int(best_logreg_cfg.get("context_k", context_ablation_df.loc[context_ablation_df["f1_macro"].idxmax(), "context_k"]))
        fig, ax = plt.subplots(figsize=(7.5, 4.5))
        ax.plot(context_ablation_df["context_k"], context_ablation_df["f1_macro"], marker="o", label="macro-F1")
        ax.axvline(best_context_k, linestyle="--", label=f"selected k={best_context_k}")
        ax.set_title("Research track: Task b sparse baseline context ablation")
        ax.set_xlabel("Number of prior turns included")
        ax.set_ylabel("Held-out test macro-F1")
        ax.legend()
        plt.tight_layout()
        plt.savefig(context_ablation_plot, dpi=300, bbox_inches="tight")
        display(Markdown("### Figure 42. Research track Task B context-size ablation"))
        plt.show()


In [ ]:
if CONFIG.get("run_research_track", False) and CONFIG.get("research_task_b_topic_shift", True):
    best_logreg_cfg = _research_get_best_task_b_logreg_cfg()
    topic_shift_csv = RESEARCH_ARTIFACT_DIR / "task_b_topic_shift.csv"
    topic_shift_plot = RESEARCH_FIG_DIR / "task_b_topic_shift_macro_f1.png"
    if topic_shift_csv.exists() and CONFIG.get("research_track_use_cached", True) and not CONFIG.get("force_recompute", False):
        topic_shift_df = pd.read_csv(topic_shift_csv)
    else:
        full_df = prepare_logreg_input(therapist_df.copy())
        transcript_topics = full_df.groupby("transcript_id")["topic_norm"].first()
        topic_counts = transcript_topics.value_counts()
        min_transcripts = int(CONFIG.get("research_task_b_topic_shift_min_transcripts", 5))
        eligible_topics = topic_counts[topic_counts >= min_transcripts].index.tolist(), 
        rows = []
        for topic in eligible_topics:
            test_ids = set(transcript_topics[transcript_topics == topic].index.tolist())
            train_ids = set(transcript_topics[transcript_topics != topic].index.tolist())
            train_sub = full_df[full_df["transcript_id"].isin(train_ids)].copy()
            test_sub = full_df[full_df["transcript_id"].isin(test_ids)].copy()
            if len(test_sub) < 20 or train_sub["y_code"].nunique() < 2:
                continue
            model = build_logreg_pipeline(
                context_col=f"context_k{int(best_logreg_cfg['context_k'])}_lc",
                min_df=int(best_logreg_cfg["min_df"]),
                char_ngram=tuple(best_logreg_cfg["char_ngram"]),
                C=float(best_logreg_cfg["C"]),
                l1_ratio=float(best_logreg_cfg["l1_ratio"]),
                class_weight=best_logreg_cfg.get("class_weight", None),
                random_state=SEED,
            )
            model.fit(train_sub, train_sub["y_code"].to_numpy())
            pred = model.predict(test_sub)
            prob = model.predict_proba(test_sub)
            metrics = compute_multiclass_metrics_fixed_labels(
                test_sub["y_code"].to_numpy(),
                pred,
                prob,
                labels=TASK_B_ALL_LABELS,
            )
            rows.append(
                {
                    "held_out_topic": topic,
                    "n_test_transcripts": int(len(test_ids)),
                    "n_test_rows": int(len(test_sub)),
                    **metrics,
                }
            )
        topic_shift_df = (
            pd.DataFrame(rows)
            .sort_values(["f1_macro", "n_test_rows"], ascending=[False, False])
            .reset_index(drop=True)
        )
        save_df(topic_shift_df, topic_shift_csv)
    display(Markdown("### Table C.2. Task B topic-shift robustness by held-out topic"))
    display_report_table(
        topic_shift_df.round(4),
        text_cols=["held_out_topic"],
        gradient_cols=["accuracy", "precision_macro", "recall_macro", "f1_macro", "precision_weighted", "recall_weighted", "f1_weighted"],
        gradient_kwargs={"cmap": "YlGn"},
        highlight_min_cols=["brier_multiclass"],
        highlight_max_cols=["f1_macro"],
    )
    if not topic_shift_df.empty:
        weighted_macro_f1 = np.average(
            topic_shift_df["f1_macro"],
            weights=topic_shift_df["n_test_rows"].clip(lower=1),
        )
        print(f"Weighted mean macro-F1 across held-out topics: {weighted_macro_f1:.4f}")
        plot_df = topic_shift_df.sort_values("f1_macro", ascending=True)
        fig, ax = plt.subplots(figsize=(9, max(4, 0.45 * len(plot_df))))
        ax.barh(plot_df["held_out_topic"], plot_df["f1_macro"])
        ax.set_title("Research track: Task b topic-shift robustness")
        ax.set_xlabel("Macro-F1 on held-out topic")
        ax.set_ylabel("Held-out topic")
        plt.tight_layout()
        plt.savefig(topic_shift_plot, dpi=300, bbox_inches="tight")
        display(Markdown("### Figure 43. Research track Task B topic-shift robustness"))
        plt.show()


In [ ]:
if CONFIG.get("run_research_track", False) and CONFIG.get("research_task_a_nested_cv", True):
    _nested_plotting_available = globals().get("plt") is not None and globals().get("sns") is not None
    _nested_plotting_error = "" if _nested_plotting_available else "matplotlib/seaborn unavailable from the main imports cell"
    task_a_research_dir = RESEARCH_ARTIFACT_DIR / "task_a_nested_cv"
    task_a_research_dir.mkdir(parents=True, exist_ok=True)
    nested_outer_csv = task_a_research_dir / "task_a_nested_outer_results.csv"
    nested_preds_csv = task_a_research_dir / "task_a_nested_prediction_records.csv"
    nested_summary_csv = task_a_research_dir / "task_a_nested_summary.csv"
    nested_plot_png = RESEARCH_FIG_DIR / "task_a_nested_cv_boxplot.png"
    _nested_cache_ready = (
        nested_outer_csv.exists()
        and nested_preds_csv.exists()
        and nested_summary_csv.exists()
        and CONFIG.get("research_track_use_cached", True)
        and not CONFIG.get("force_recompute", False)
    )
    if "_make_task_a_logreg_candidate" not in globals() and "_make_logreg_candidate" in globals():
        _make_task_a_logreg_candidate = _make_logreg_candidate
    if "_make_task_a_boosted_candidate" not in globals() and "_make_boosted_candidate" in globals():
        _make_task_a_boosted_candidate = _make_boosted_candidate
    required_task_a_names = [
        "transcript_df",
        "transcript_feature_cols",
        "_cv_search_logreg",
        "_cv_search_boosted",
        "_make_task_a_logreg_candidate",
        "_make_task_a_boosted_candidate",
        "_fit_boost_preprocessor",
        "_transform_boost_features",
        "_fit_with_optional_weights",
        "_predict_positive_proba",
        "compute_binary_metrics",
    ]
    missing_task_a_names = [] if _nested_cache_ready else [n for n in required_task_a_names if n not in globals()]
    if missing_task_a_names:
        raise NameError(
            "Appendix A research CV is missing required upstream objects. "
            f"Run the Appendix A preparation/training cells first, or keep cached outputs available: {missing_task_a_names}"
        )
    @contextmanager
    def _temporary_task_a_seed(seed: int):
        previous_seed = globals().get("SEED", 42)
        try:
            globals()["SEED"] = int(seed)
            np.random.seed(int(seed))
            random.seed(int(seed))
            yield
        finally:
            globals()["SEED"] = int(previous_seed)
            np.random.seed(int(previous_seed))
            random.seed(int(previous_seed))
    if (
        _nested_cache_ready
    ):
        task_a_nested_outer_df = pd.read_csv(nested_outer_csv)
        task_a_nested_pred_df = pd.read_csv(nested_preds_csv)
        task_a_nested_summary_df = pd.read_csv(nested_summary_csv)
    else:
        X_full = transcript_df[transcript_feature_cols].copy().reset_index(drop=True)
        y_full = transcript_df["mi_y"].to_numpy()
        transcript_ids_full = transcript_df["transcript_id"].astype(str).to_numpy()
        outer_rows = []
        pred_rows = []
        seeds = [int(s) for s in CONFIG.get("research_task_a_nested_cv_seeds", [17, 42, 101])]
        outer_splits = int(CONFIG.get("research_task_a_nested_cv_outer_splits", 5))
        selection_metric = str(CONFIG.get("research_task_a_nested_cv_selection_metric", globals().get("SELECTION_METRIC", "accuracy")))
        boost_n_iter = int(CONFIG.get("research_task_a_boosted_n_iter", 18))
        for seed in seeds:
            outer_cv = StratifiedKFold(
                n_splits=outer_splits,
                shuffle=True,
                random_state=int(seed),
            )
            for outer_fold, (tr_idx, te_idx) in enumerate(outer_cv.split(X_full, y_full), start=1):
                X_tr = X_full.iloc[tr_idx].reset_index(drop=True)
                X_te = X_full.iloc[te_idx].reset_index(drop=True)
                y_tr = y_full[tr_idx]
                y_te = y_full[te_idx]
                te_ids = transcript_ids_full[te_idx]
                with _temporary_task_a_seed(seed):
                    log_cv_df, best_log = _cv_search_logreg(
                        X_tr,
                        y_tr,
                        selection_metric=selection_metric,
                    )
                log_model = _make_task_a_logreg_candidate(
                    C=best_log["C"],
                    l1_ratio=best_log["l1_ratio"],
                    class_weight=best_log["class_weight"],
                )
                log_model.fit(X_tr, y_tr)
                log_prob = _predict_positive_proba(log_model, X_te)
                log_thr = float(best_log["oof_best_threshold"])
                log_pred = (log_prob >= log_thr).astype(int)
                log_metrics = compute_binary_metrics(y_te, log_pred, log_prob)
                log_metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_te, log_pred))
                outer_rows.append(
                    {
                        "model": "Elastic-net logistic regression",
                        "seed": int(seed),
                        "outer_fold": int(outer_fold),
                        "selection_metric": selection_metric,
                        "threshold": float(log_thr),
                        "best_params": json.dumps(
                            {
                                "C": float(best_log["C"]),
                                "l1_ratio": float(best_log["l1_ratio"]),
                                "class_weight": best_log["class_weight"],
                            }
                        ),
                        **{k: float(v) for k, v in log_metrics.items()},
                    }
                )
                pred_rows.extend(
                    [
                        {
                            "model": "Elastic-net logistic regression",
                            "seed": int(seed),
                            "outer_fold": int(outer_fold),
                            "transcript_id": str(tid),
                            "y_true": int(yt),
                            "pred": int(yp),
                            "prob": float(pb),
                        }
                        for tid, yt, yp, pb in zip(te_ids, y_te, log_pred, log_prob)
                    ]
                )
                with _temporary_task_a_seed(seed):
                    boost_cv_df, best_boost = _cv_search_boosted(
                        X_tr,
                        y_tr,
                        selection_metric=selection_metric,
                        n_iter=boost_n_iter,
                    )
                X_tr_boost, boost_pre = _fit_boost_preprocessor(X_tr)
                X_te_boost = _transform_boost_features(X_te, boost_pre)
                boost_model, boost_model_name = _make_task_a_boosted_candidate(best_boost, y_tr)
                boost_model = _fit_with_optional_weights(boost_model, X_tr_boost, y_tr)
                boost_prob = _predict_positive_proba(boost_model, X_te_boost)
                boost_thr = float(best_boost["oof_best_threshold"])
                boost_pred = (boost_prob >= boost_thr).astype(int)
                boost_metrics = compute_binary_metrics(y_te, boost_pred, boost_prob)
                boost_metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_te, boost_pred))
                outer_rows.append(
                    {
                        "model": str(boost_model_name),
                        "seed": int(seed),
                        "outer_fold": int(outer_fold),
                        "selection_metric": selection_metric,
                        "threshold": float(boost_thr),
                        "best_params": json.dumps(best_boost, default=str),
                        **{k: float(v) for k, v in boost_metrics.items()},
                    }
                )
                pred_rows.extend(
                    [
                        {
                            "model": str(boost_model_name),
                            "seed": int(seed),
                            "outer_fold": int(outer_fold),
                            "transcript_id": str(tid),
                            "y_true": int(yt),
                            "pred": int(yp),
                            "prob": float(pb),
                        }
                        for tid, yt, yp, pb in zip(te_ids, y_te, boost_pred, boost_prob)
                    ]
                )
        task_a_nested_outer_df = pd.DataFrame(outer_rows)
        task_a_nested_pred_df = pd.DataFrame(pred_rows)
        task_a_nested_summary_df = (
            task_a_nested_outer_df
            .groupby("model", as_index=False)
            .agg(
                outer_runs=("accuracy", "size"),
                accuracy_mean=("accuracy", "mean"),
                accuracy_std=("accuracy", "std"),
                balanced_accuracy_mean=("balanced_accuracy", "mean"),
                balanced_accuracy_std=("balanced_accuracy", "std"),
                f1_mean=("f1", "mean"),
                f1_std=("f1", "std"),
                roc_auc_mean=("roc_auc", "mean"),
                roc_auc_std=("roc_auc", "std"),
                average_precision_mean=("average_precision", "mean"),
                average_precision_std=("average_precision", "std"),
                brier_mean=("brier", "mean"),
                brier_std=("brier", "std"),
            )
            .sort_values(["balanced_accuracy_mean", "f1_mean"], ascending=[False, False])
            .reset_index(drop=True)
        )
        save_df(task_a_nested_outer_df, nested_outer_csv)
        save_df(task_a_nested_pred_df, nested_preds_csv)
        save_df(task_a_nested_summary_df, nested_summary_csv)
    display(Markdown("### Table C.3A. Appendix A repeated nested-CV stability summary"))
    display_report_table(
        task_a_nested_summary_df.round(4),
        row_color_col="model",
        row_colors=REPORT_MODEL_COLORS,
        text_cols=["model"],
        gradient_cols=["accuracy_mean", "balanced_accuracy_mean", "f1_mean", "roc_auc_mean", "average_precision_mean"],
        gradient_kwargs={"cmap": "YlGn"},
        highlight_min_cols=["accuracy_std", "balanced_accuracy_std", "f1_std", "roc_auc_std", "average_precision_std", "brier_mean", "brier_std"],
        highlight_max_cols=["balanced_accuracy_mean", "f1_mean"],
    )
    def _compact_nested_model_name(model_name: Any) -> str:
        normalized = _report_normalize_key(model_name)
        if "elastic" in normalized and "logistic" in normalized:
            return "Elastic-net LR"
        if "xgboost" in normalized:
            return "XGBoost"
        if "hist" in normalized and "boost" in normalized:
            return "HistGB"
        return str(model_name)
    def _compact_nested_params(params_text: Any) -> str:
        if pd.isna(params_text) or str(params_text).strip() == "":
            return "cached metrics only"
        try:
            params = json.loads(str(params_text))
        except Exception:
            return shorten_for_table(params_text, max_chars=95) if "shorten_for_table" in globals() else str(params_text)[:95]
        family = str(params.get("model_family", "")).lower()
        if "xgboost" in family or "max_depth" in params or "n_estimators" in params:
            pieces = []
            for key, label in [
                ("n_estimators", "n"),
                ("max_depth", "depth"),
                ("learning_rate", "lr"),
                ("subsample", "sub"),
                ("reg_lambda", "lambda"),
                ("min_child_weight", "child"),
            ]:
                if key in params:
                    value = params[key]
                    if isinstance(value, float):
                        value = f"{value:g}"
                    pieces.append(f"{label}={value}")
            return "; ".join(pieces[:6])
        pieces = []
        if "C" in params:
            pieces.append(f"C={float(params['C']):g}")
        if "l1_ratio" in params:
            pieces.append(f"l1={float(params['l1_ratio']):g}")
        if "class_weight" in params:
            pieces.append(f"cw={params['class_weight'] or 'none'}")
        return "; ".join(pieces) if pieces else str(params)[:95]
    task_a_nested_outer_display_df = task_a_nested_outer_df.copy()
    task_a_nested_outer_display_df["outer_run"] = task_a_nested_outer_display_df.apply(
        lambda row: f"s{int(row['seed'])}-f{int(row['outer_fold'])}",
        axis=1,
    )
    task_a_nested_outer_display_df["model"] = task_a_nested_outer_display_df["model"].map(_compact_nested_model_name)
    if "best_params" in task_a_nested_outer_display_df.columns:
        task_a_nested_outer_display_df["best_config"] = task_a_nested_outer_display_df["best_params"].map(_compact_nested_params)
    else:
        task_a_nested_outer_display_df["best_config"] = "cached metrics only"
    _nested_detail_cols = [
        "outer_run",
        "model",
        "threshold",
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "average_precision",
        "brier",
        "best_config",
    ]
    _nested_detail_cols = [c for c in _nested_detail_cols if c in task_a_nested_outer_display_df.columns]
    task_a_nested_outer_display_df = (
        task_a_nested_outer_display_df
        .sort_values(["seed", "outer_fold", "model"])
        .loc[:, _nested_detail_cols]
        .head(20)
        .reset_index(drop=True)
        .rename(columns={
            "outer_run": "run",
            "threshold": "thr",
            "accuracy": "acc",
            "balanced_accuracy": "bal_acc",
            "precision": "prec",
            "recall": "rec",
            "average_precision": "avg_prec",
        })
    )
    save_df(task_a_nested_outer_display_df, task_a_research_dir / "task_a_nested_outer_fold_details_display.csv")
    display(Markdown("### Table C.3B. Appendix A repeated nested-CV outer-fold details"))
    display_report_table(
        task_a_nested_outer_display_df.round(4),
        formats={
            "thr": "{:.2f}",
            "acc": "{:.3f}",
            "bal_acc": "{:.3f}",
            "prec": "{:.3f}",
            "rec": "{:.3f}",
            "f1": "{:.3f}",
            "roc_auc": "{:.3f}",
            "avg_prec": "{:.3f}",
            "brier": "{:.3f}",
        },
        row_color_col="model",
        row_colors={
            "Elastic-net LR": "#e7f0ff",
            "XGBoost": "#f3e8ff",
            "HistGB": "#fff3d9",
        },
        text_cols=["run", "model", "best_config"],
        gradient_cols=["acc", "bal_acc", "prec", "rec", "f1", "roc_auc", "avg_prec"],
        gradient_kwargs={"cmap": "YlGn"},
        highlight_min_cols=["brier"],
    )
    if not task_a_nested_outer_df.empty and _nested_plotting_available:
        plot_df = task_a_nested_outer_df.copy()
        plot_df["model"] = plot_df["model"].astype(str)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        sns.boxplot(data=plot_df, x="model", y="balanced_accuracy", ax=axes[0])
        sns.stripplot(data=plot_df, x="model", y="balanced_accuracy", ax=axes[0], alpha=0.5)
        axes[0].set_title("Task a repeated nested CV – balanced accuracy")
        axes[0].tick_params(axis="x", rotation=15)
        sns.boxplot(data=plot_df, x="model", y="f1", ax=axes[1])
        sns.stripplot(data=plot_df, x="model", y="f1", ax=axes[1], alpha=0.5)
        axes[1].set_title("Task a repeated nested CV – F1")
        axes[1].tick_params(axis="x", rotation=15)
        plt.tight_layout()
        plt.savefig(nested_plot_png, dpi=300, bbox_inches="tight")
        display(Markdown("### Figure 44. Research track Appendix A repeated nested-CV stability"))
        plt.show()
    elif not task_a_nested_outer_df.empty:
        print(f"Skipping nested-CV plot because plotting libraries are unavailable: {_nested_plotting_error}")


In [ ]:
if CONFIG.get("run_research_track", False):
    research_outputs = [
        {
            "analysis": "Task b context-size ablation",
            "artifact": RESEARCH_ARTIFACT_DIR / "task_b_context_ablation.csv",
            "figure": RESEARCH_FIG_DIR / "task_b_context_ablation_macro_f1.png",
        },
        {
            "analysis": "Task b topic-shift robustness",
            "artifact": RESEARCH_ARTIFACT_DIR / "task_b_topic_shift.csv",
            "figure": RESEARCH_FIG_DIR / "task_b_topic_shift_macro_f1.png",
        },
        {
            "analysis": "Task a repeated nested CV",
            "artifact": RESEARCH_ARTIFACT_DIR / "task_a_nested_cv" / "task_a_nested_summary.csv",
            "figure": RESEARCH_FIG_DIR / "task_a_nested_cv_boxplot.png",
        },
    ]
    research_manifest_df = pd.DataFrame(
        [
            {
                "analysis": row["analysis"],
                "artifact_exists": Path(row["artifact"]).exists(),
                "figure_exists": Path(row["figure"]).exists(),
                "artifact_path": str(Path(row["artifact"]).relative_to(PROJECT_DIR)) if Path(row["artifact"]).exists() else str(row["artifact"]),
                "figure_path": str(Path(row["figure"]).relative_to(PROJECT_DIR)) if Path(row["figure"]).exists() else str(row["figure"]),
            }
            for row in research_outputs
        ]
    )
    display(Markdown("### Table C.4. Research-track manifest and output status"))
    display_report_table(
        research_manifest_df,
        text_cols=["analysis", "artifact_path", "figure_path"],
        bool_cols=["artifact_exists", "figure_exists"],
    )
    save_df(research_manifest_df, RESEARCH_ARTIFACT_DIR / "research_track_manifest.csv")


In [ ]:
produced_files = []
for root, _, files in os.walk(ARTIFACT_DIR):
    for fn in files:
        path = Path(root) / fn
        produced_files.append({
            "path": str(path.relative_to(PROJECT_DIR)) if path.exists() else str(path),
            "type": path.suffix,
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
        })
for root, _, files in os.walk(FIG_DIR):
    for fn in files:
        path = Path(root) / fn
        produced_files.append({
            "path": str(path.relative_to(PROJECT_DIR)) if path.exists() else str(path),
            "type": path.suffix,
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
        })
produced_files_df = pd.DataFrame(produced_files, columns=["path", "type", "size_kb"]).sort_values("path")
produced_files_df["size_kb"] = pd.to_numeric(produced_files_df["size_kb"], errors="coerce")
produced_files_df["folder"] = produced_files_df["path"].map(lambda p: str(p).replace("\\", "/").split("/")[0])
produced_files_df["type"] = produced_files_df["type"].replace("", "[none]")
artifact_type_summary = (
    produced_files_df.groupby("type", dropna=False)
    .agg(files=("path", "count"), total_mb=("size_kb", lambda s: s.sum() / 1024))
    .reset_index()
    .sort_values(["total_mb", "files"], ascending=[False, False])
    .rename(columns={"type": "File Type", "files": "Files", "total_mb": "Total Size (MB)"})
)
artifact_folder_summary = (
    produced_files_df.groupby("folder", dropna=False)
    .agg(files=("path", "count"), total_mb=("size_kb", lambda s: s.sum() / 1024))
    .reset_index()
    .sort_values(["total_mb", "files"], ascending=[False, False])
    .rename(columns={"folder": "Folder", "files": "Files", "total_mb": "Total Size (MB)"})
)
largest_artifacts_report = (
    produced_files_df.nlargest(12, "size_kb")
    .loc[:, ["path", "type", "size_kb"]]
    .rename(columns={"path": "Path", "type": "File Type", "size_kb": "Size (KB)"})
    .reset_index(drop=True)
)
if "_display_report_table" not in globals():
    def _display_report_table(df, formats=None, text_cols=None, gradient_cols=None, gradient_kwargs=None) -> None:
        formats = formats or {}
        text_cols = text_cols or []
        try:
            styler = df.style.hide(axis="index")
            if formats:
                styler = styler.format(formats)
            if text_cols:
                styler = styler.set_properties(subset=text_cols, **{"text-align": "left", "max-width": "620px"})
            display(styler)
        except Exception:
            fallback = df.copy()
            for col, fmt in formats.items():
                if col in fallback.columns:
                    fallback[col] = fallback[col].map(lambda v: "" if pd.isna(v) else fmt.format(v))
            display(fallback.reset_index(drop=True))
display(Markdown("### Final artifact inventory summary"))
_display_report_table(artifact_type_summary, formats={"Total Size (MB)": "{:.2f}"})
_display_report_table(artifact_folder_summary, formats={"Total Size (MB)": "{:.2f}"})
display(Markdown("### Largest generated files"))
_display_report_table(largest_artifacts_report, formats={"Size (KB)": "{:,.1f}"}, text_cols=["Path"])
manifest = {
    "config_path": str(CONFIG_JSON),
    "n_figures": int((produced_files_df["type"] == ".png").sum()),
    "n_artifacts": int(len(produced_files_df)),
    "tasks_included": ["Task 1", "Task 2.1", "Task 2.2", "Task 2.3", "Task 2.4 main task", "Task 2.5", "Appendix A", "Appendix B"],
}
save_json(manifest, MANIFEST_JSON)
def _project_relative_or_str(path_value: Any) -> str:
    path = Path(path_value)
    try:
        return str(path.relative_to(PROJECT_DIR))
    except ValueError:
        return str(path)
manifest_report = pd.DataFrame([
    {
        "Config": _project_relative_or_str(manifest["config_path"]),
        "Figures": manifest["n_figures"],
        "Artifacts": manifest["n_artifacts"],
        "Included Sections": ", ".join(manifest["tasks_included"]),
    }
])
display(Markdown("### release manifest"))
_display_report_table(manifest_report, text_cols=["Included Sections"])
